# Sionna 2.0 — 2695 MHz DEM Simulation (Stevenage / Ofcom)

**Frequency:** 2695 MHz · **Terrain:** Real LiDAR DEM (Environment Agency 1 m DTM) · **Dataset:** Ofcom 2018 drive-test

## Purpose
Full digital elevation model ray-tracing simulation. Uses real terrain height data from
airborne LiDAR to assess whether replacing a flat ground plane with realistic elevation
improves path loss prediction accuracy.

**Key difference from flat notebook:**
- Ground is a LiDAR-derived DEM mesh (1 m resolution)
- `RX_EXTRA_GAIN_DB = 0.0 dB` — no cable correction (different Ofcom CSV variant)
- Larger scene bbox to capture full terrain variation

## Notebook Structure

| Step | Cell | Description |
|------|------|-------------|
| 0 | CELL 0 | Imports |
| 1 | CELL 1 | Configuration — edit here for new campaign |
| 2 | CELL 2 | Coordinate utilities (GPS → UTM → local) |
| 3 | CELL 3 | Load scene + preview |
| 3b | CELL 3b | Scene 3D preview |
| 3c | CELL 3c | nDSM clutter height heatmap |
| 4 | CELL 4A | Assign EM material properties |
| 5 | CELL 4 | Place TX |
| 6 | CELL 5 | Extract + filter RX from Ofcom CSV |
| 7 | CELL 6 | Place receivers + DEM sanity check |
| 8 | CELL 6b | Terrain + TX/RX map |
| 9 | CELL 6c | 2D OSM TX/RX map |
| 10 | CELL 7 | Path solver (adaptive, batched) |
| 11 | CELL 7c | Metrics, charts & ray classification |
| 12 | CELL 8 | Stratified distance-band solver ★ main cell |
| 13 | CELL 8b | Outlier diagnostic |
| 14 | CELL 8e | Cumulative distance evaluation |
| 15 | CELL 8e-P833 | P.833 vegetation impact |
| 16 | CELL P.833 | ITU-R P.833 standalone post-processing |
| 17 | CELL 8f | Sim vs measured scatter plot |
| 18 | CELL SAVE | Save results snapshot |
| 19 | CELL REPORT | Auto-generate report |
| 20 | CELL DIAG | Step-by-step bias diagnostics |
| 21 | CELL DIAG-CSV | Load specific new-scene CSV (scene_v2_infra) |


## CELL 0 — Environment Setup

Imports and package checks. Run once per kernel session.

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime
import math

# Sionna 2.0 — PyTorch backend, no TensorFlow, no Mitsuba variant setup
import torch
import sionna
import sionna.rt as rt
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver, PathSolver

_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
    print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available — flat terrain fallback will be used')

_HAS_OSM = False
try:
    import osmnx as ox
    _HAS_OSM = True
    print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available')

def _safe(v):
    """Convert tensor or numeric value to Python float safely."""
    if hasattr(v, 'item'):  return float(v.item())
    if hasattr(v, 'numpy'): return float(v.numpy())
    return float(v)

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Sionna  : {sionna.__version__}')

## CELL 1 — Configuration

Edit the **SCENARIO** block only. All output paths are auto-derived from `SCENARIO_NAME`.

**Key parameters for DEM terrain:**
- `FLAT_TERRAIN = False` — enables LiDAR DEM loading (run CELL 2b first if DEM GeoTIFF not yet built)
- `RX_EXTRA_GAIN_DB = 0.0 dB` — no correction applied
- `FREQUENCY_HZ = 2695e6` — Ofcom 2695 MHz Stevenage dataset (stevenage2695.csv)


In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================
# Edit SCENARIO block only. All paths are auto-derived.
# To port to a new campaign: change the SCENARIO section.
# ============================================================
import os
import math

# ╔══════════════════════════════════════════════════════════╗
# ║                  SCENARIO CONFIGURATION                  ║
# ║           Edit this block for each new campaign          ║
# ╚══════════════════════════════════════════════════════════╝

SCENARIO_NAME   = 'stevenage_ofcom_2695mhz_dem'  # used for output filenames

# ── City / coordinate system ──────────────────────────────
CITY_NAME    = 'Stevenage'

# ── Projection knob ─────────────────────────────────────────
# 'bng'    -> British National Grid, EPSG:27700 (default -- native UK
#             Ordnance Survey grid; matches EA LiDAR WCS tiles directly).
# 'utm30n' -> EPSG:32630 (UTM zone 30N, covers UK) -- kept as an
#             alternative for portability to non-OS-grid workflows.
# Must match PROJECTION_CRS in sionna019_scene_builder_stevenage.ipynb
# CELL 0 exactly, or TX/RX positions will be computed in different CRSs.
PROJECTION_CRS = 'bng'    # 'bng' | 'utm30n'
_PROJECTION_EPSG_MAP = {'bng': 27700, 'utm30n': 32630}
UTM_EPSG = _PROJECTION_EPSG_MAP.get(PROJECTION_CRS, 27700)
print(f'Projection CRS: {PROJECTION_CRS}  ->  EPSG:{UTM_EPSG}')

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────
# Edit these four values to match sionna019_scene_builder_stevenage.ipynb CELL 0.
# Must cover the full measurement route (run CELL DIAG to verify coverage).
SCENE_WEST  = -0.328144
SCENE_EAST  = -0.182176
SCENE_SOUTH = 51.843555
SCENE_NORTH = 51.933645
# ── TX parameters ────────────────────────────────────────
TX_LON              = -0.25516  # tx_center, transmitter_positions.csv
TX_LAT              =  51.8886  # tx_center, transmitter_positions.csv
TX_AGL_M            = 17.0          # confirmed from stevenage2695.csv header: Tx antenna height (m): 17
TX_AGL_SCAN_M       = [15, 17, 20, 25, 30]  # pre-calibration TX height scan (m AGL); set None to skip
TX_AGL_MIN_COVERAGE = 0.80  # min valid-path fraction for AGL selection — excludes obstructed heights (selects best AGL height)
CAL_SCALAR_BOUNDS   = (-60.0, 60.0)   # Powell scalar bounds
TX_CONDUCTED_DBM    = 54.7  # EIRP 56.9 dBm - antenna 2.2 dBi          # derived from stevenage2695.csv: conducted power (EIRP 50.3 dBm - antenna 1.3 dBi)
TX_ANTENNA_GAIN_DBI =  2.2           # confirmed from stevenage2695.csv 'Tx antenna gain (dBi)'
EIRP_DBM            = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI

# ── Antenna pattern ───────────────────────────────────────
# 'iso'   = isotropic (0 dBi)
# 'donut' = half-wave dipole (~2.15 dBi, nulls at zenith/nadir)
ANTENNA_PATTERN = 'donut'

# ── RX parameters ────────────────────────────────────────
RX_AGL_M           =  1.5            # m AGL
RX_EXTRA_GAIN_DB   =  0.0            # dB (no chain loss applied — same as Nottingham 2695)
SITE_CORRECTION_DB =  0.0            # dB (post-hoc calibration offset)
USE_CALIBRATED_FILES = False          # False → fresh calibration; True → load calibrated_materials_stevenage_2695mhz.json + scalar_offset_stevenage_2695mhz.json
# ── Vegetation EM parameters (ITU-R P.833 / portable — do not calibrate) ─────
VEG_RELATIVE_PERMITTIVITY = 17.0  # ITU-R P.833 sub-6 GHz — portable
VEG_CONDUCTIVITY          = 0.15  # S/m — ITU-R P.833 vegetation at 2.7 GHz
VEG_SCATTERING_COEFF      = 0.50  # ITU-R P.833 / AERPAW-validated — portable
# ── Road material (ITU-R P.2040-2 asphalt — portable, do not calibrate) ────────
ROAD_RELATIVE_PERMITTIVITY = 2.56   # itu_asphalt at 2695 MHz
ROAD_CONDUCTIVITY          = 0.005  # S/m
ROAD_SCATTERING_COEFF      = 0.30   # moderate diffuse scatter
# ── Calibration parameters ──────────────────────────────────────────────────────
CAL_MAX_DIST_KM           = 1.5   # ceiling — auto-discover finds effective range (same as Nottingham 2695)
CAL_MIN_DIST_KM           = 0.15  # exclude mast shadow zone from calibration
EVAL_MIN_DIST_KM          = 0.15  # exclude near-field from CELL 8e stats (min receivers for valid R²)
CAL_S_MAX                 = 0.95  # max scattering coefficient in calibration
CAL_FIXED_MATS            = {
    'itu_ceiling_board',            # vegetation discs — fixed at VEG_* values
    'itu_metal', 'metal_barrier',   # perfect conductors — must not be freed
    # secondary materials — lock to reduce free params from 27 → 12
    'canopy_itu_vegetation',        # tree crown — near-transparent (λ=11.1cm at 2695 MHz)
    'trunk_itu_wood',               # tree trunks — low impact at 2695 MHz
    'concrete_barrier',             # barriers — degenerate εᵣ≈1.45 when free (London lesson)
    'itu_medium_dry_ground',        # ground — secondary to buildings in urban scene
    'itu_very_dry_ground',          # ground — secondary to buildings in urban scene
}  # free: ~8 materials (brick, concrete, glass, wet_ground, very_dry_ground, wood, etc.)
CAL_S_BUMP_FLOOR          = 0.05  # S warm-start bump threshold — only bump S<0.05 (near-zero, no scatter paths)
CAL_MIN_VALID_FRAC        = 0.65  # auto-discover: min fraction of valid paths per 100m bin — same as Nottingham 2695
CAL_WARM_S_PRIOR          = 0.35  # fresh-start warm prior S — bumped to this if most materials near-zero
CAL_FAR_WEIGHT            = 1.0   # distance weight for near/far balance in Powell loss
CAL_N_ITER                = 2
DISABLE_VEG_DISCS         = False   # Discs transparent: removes veg-disc scatter dominance so building mat CMA is sensitive mid-range
CAL_SKIP_PROBE            = True   # sensitivity already confirmed — skip vacuum test
CAL_SKIP_P2_PROBE         = False  # force Phase 2 kernel liveness check (catches DrJIT caching)
CAL_COVERAGE_MIN          = 0.65  # min fraction of Phase-0 valid receivers to keep (lowered — 90% too strict, scatter-only paths disappear with S<0.20)
CAL_R2_RETRY_MIN          = 0.0   # auto-retry disabled — cal-receiver R² not a valid retry trigger
CAL_SAMPLES_PS            = 30_000_000  # 10M — Powell (consistent with Nottingham 2695)
CAL_FIXED_SEED            = 42          # lock MC seed — stable Powell landscape, eliminates between-eval drift
CAL_DE_SAMPLES     = 2_000_000          # samples per DE eval
CAL_CMA_SAMPLES    = 30_000_000  # match CAL_SAMPLES_PS — 30M inflates pre-scalar RMSE to 22.5 dB (extra scatter paths at high sample count swamp material sensitivity)
CAL_CMA_POPSIZE     = 36            # 3x default pop — reduces ranking noise on noisy MC objectives
CAL_CMA_TOLFUN      = 0.10          # stop when improvement < 0.10 dB; tolfun=0.03 < noise floor (0.12 dB) => runs forever          # samples per CMA-ES eval (less MC noise)
CAL_CMA_SIGMA0     = 0.10              # reduced from 0.2 — warm start needs smaller steps
CAL_CMA_MAXITER    = 300               # CMA-ES max generations
CAL_CMA_WARM_START = True              # start from Phase 0 materials — avoids 3.6 dB gap vs Phase 0
SCATTERING_PATTERN_TYPE = 'lambertian'  # 'lambertian' (default) or 'directive' (DirectivePattern alpha_r=6)
CAL_WARM_CYCLES    = 0     # coord-descent cycles before Powell (0=skip — direct Powell from ITU defaults)
CAL_POWELL_MAXITER = 50    # Powell max iterations — deeper convergence (was 30)
CAL_POWELL_XTOL    = 0.001 # Powell x-tolerance — tighter convergence (was 0.01)
CAL_POWELL_FTOL    = 0.001 # Powell f-tolerance — tighter convergence (was 0.01)
                                      # False → ITU-R P.2040-2 defaults + SCALAR_OFFSET_DB=0.0
NOISE_FLOOR_DBM    = -120.0          # confirmed from stevenage2695.csv 'System noise floor (dBm)'

# ── RX selection ─────────────────────────────────────────
NUM_RX       = 1200                  # number of receivers to use

# ── Frequency ────────────────────────────────────────────
FREQUENCY_HZ = 2695e6  # Hz              # Hz

# ── Terrain ──────────────────────────────────────────────
FLAT_TERRAIN = False                 # DEM terrain — samples terrain.ply

# ── Ray tracing ──────────────────────────────────────────
MAX_DEPTH        = 8                 # Nottingham finding: MAX_DEPTH>8 adds spurious multi-bounce paths
NUM_SAMPLES_PS   = 100_000_000  # 100M eval override; 2M for PathSolver during cal (set in CELL 4A)         # rays per PathSolver call — matches CAL_SAMPLES_PS (100M override in CELL 4A for final eval)
BATCH_SIZE       = 5                 # receivers per batch — same as Nottingham 2695

# ╔══════════════════════════════════════════════════════════╗
# ║                    PATH CONFIGURATION                    ║
# ║   Auto-derived from SCENARIO_NAME — no edits needed      ║
# ╚══════════════════════════════════════════════════════════╝

# Root: ~/sionna_rt/<scenario_name>/
# Override any path below if your folder layout differs.
BASE_DIR        = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_2695mhz_dem'))
SCENE_DIR       = os.path.join(SCENE_BASE_DIR, 'scene_v4_full')
# CRS: read from scene_parameters.json (written to BASE_DIR by scene builder)
_crs_meta = os.path.join(SCENE_BASE_DIR, 'scene_parameters.json')
OUT_DIR         = os.path.join(BASE_DIR, 'results')
DEM_TIFF        = next((p for p in [
    os.path.join(SCENE_BASE_DIR, 'dem.tif'),
    os.path.join(SCENE_BASE_DIR, 'scene', 'dem.tif'),
    os.path.join(SCENE_BASE_DIR, 'scene', 'dem_wgs84.tif'),
    os.path.join(SCENE_BASE_DIR, 'scene_v4_full', 'dem.tif'),
] if os.path.exists(p)), os.path.join(SCENE_BASE_DIR, 'dem.tif'))  # auto-detect
TERRAIN_PLY     = next((p for p in [
    os.path.join(SCENE_BASE_DIR, 'scene_v4_full', 'meshes', 'terrain.ply'),  # B3/CELL5 actual output dir
    os.path.join(SCENE_BASE_DIR, 'scene', 'meshes', 'terrain.ply'),
    os.path.join(SCENE_BASE_DIR, 'scene', 'meshes_roads', 'terrain.ply'),
    os.path.join(SCENE_BASE_DIR, 'scene', 'terrain.ply'),
    os.path.join(SCENE_BASE_DIR, 'scene_v4_full', 'terrain.ply'),
] if os.path.exists(p)), os.path.join(SCENE_BASE_DIR, 'scene_v4_full', 'meshes', 'terrain.ply'))  # auto-detect

SCENE_XML       = os.path.join(SCENE_BASE_DIR, 'scene_v4_full', 'scene_with_full.xml')   # B3 output — full scene with all OSM features
# SCENE_XML     = os.path.join(SCENE_BASE_DIR, 'scene', 'scene_with_roads.xml')  # legacy fallback
NDSM_TIFF       = os.path.join(SCENE_BASE_DIR, 'ndsm.tif')   # for CELL 3c heatmap
OFCOM_RAW_CSV   = os.path.join(BASE_DIR, 'stevenage2695.csv')  # Ofcom 2018 Stevenage 2695 MHz campaign
RX_CSV          = os.path.join(SCENE_DIR, 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(SCENE_DIR, 'measurements_with_pathloss.csv')

os.makedirs(OUT_DIR, exist_ok=True)


# ╔══════════════════════════════════════════════════════════╗
# ║              FREQUENCY PRESETS — SWAP BLOCK              ║
# ║   Uncomment one block to switch frequency campaign       ║
# ╚══════════════════════════════════════════════════════════╝

# ── PRESET A: 915.95 MHz (Ofcom 2018 — original 915 MHz notebook) ──────
# SCENARIO_NAME = 'nottingham_ofcom2018_915mhz'
# FREQUENCY_HZ  = 915.95e6
# MAX_DEPTH     = 3
# MATERIAL_FREQ = '900mhz'

# ── PRESET B: 3.602 GHz ──────────────────────────────────
# SCENARIO_NAME = 'nottingham_ofcom2018_3602mhz'
# FREQUENCY_HZ  = 3.602e9
# MAX_DEPTH     = 2     # walls absorb more → fewer meaningful bounces
# MATERIAL_FREQ = '3600mhz'
# TX_CONDUCTED_DBM    = 49.0
# TX_ANTENNA_GAIN_DBI =  1.3
# EIRP_DBM            = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI
# RX_EXTRA_GAIN_DB    = -7.8
# Note: re-run CELL 4A after switching — sigma values differ at 3.6 GHz

# ── Scene geometry source (shared across frequencies) ────
# The scene PLYs and XML are frequency-independent — reuse the
# same built scene for different frequency campaigns.
# Set this to the scenario that has the built scene_sionna2.xml:

# ── Optional: override paths if data lives elsewhere ─────
# Uncomment and set absolute paths to override auto-derived paths above.
# BASE_DIR      = '/path/to/your/data'
# SCENE_XML     = '/path/to/scene_sionna2.xml'
# OFCOM_RAW_CSV = '/path/to/measurements.csv'


# ── Coverage map ─────────────────────────────────────────────────────
FREQ_TAG        = f'{int(FREQUENCY_HZ/1e6)}mhz'        # used in output filenames
GRID_SIZE_M     = 20.0                                  # coverage map cell size (m)
NUM_SAMPLES_CM  = 10_000_000                            # RadioMapSolver rays
# ── RSSI from path gain (Sionna 2.0) ─────────────────────
def rssi_from_path_gain(path_gain_linear):
    # Sionna standard RSS at the antenna port: P_tx_dBm + 10*log10(path_gain).
    #   path_gain already includes TX/RX antenna patterns; excludes P_tx.
    # + RX_EXTRA_GAIN_DB brings it to the measurement reference point
    #   (real RX cable + filter loss; negative), matching the Ofcom RSSI.
    pg = np.asarray(path_gain_linear, dtype=float)
    return TX_CONDUCTED_DBM + 10.0 * np.log10(np.maximum(pg, 1e-30)) + RX_EXTRA_GAIN_DB

def path_loss_db(path_gain_linear):
    pg = np.asarray(path_gain_linear, dtype=float)
    return -10.0 * np.log10(np.maximum(pg, 1e-30))

print(f'Scenario         : {SCENARIO_NAME}')
print(f'Frequency        : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX               : lon={TX_LON}  lat={TX_LAT}  AGL={TX_AGL_M}m')
print(f'TX conducted     : {TX_CONDUCTED_DBM} dBm  antenna={TX_ANTENNA_GAIN_DBI} dBi  EIRP={EIRP_DBM:.1f} dBm')
print(f'Antenna pattern  : {ANTENNA_PATTERN}')
print(f'RX chain loss    : {RX_EXTRA_GAIN_DB} dB  (cable + filter, applied to RSSI)')
print(f'Noise floor      : {NOISE_FLOOR_DBM} dBm')
print(f'NUM_RX           : {NUM_RX}  |  MAX_DEPTH: {MAX_DEPTH}  |  BATCH: {BATCH_SIZE}')
print(f'Flat terrain     : {FLAT_TERRAIN}')
print(f'Base dir         : {BASE_DIR}')

# ── Report accumulator — populated by CELL DIAG, CELL 8, CELL P.833 ─────
_report = {
    'scenario': SCENARIO_NAME,
    'frequency_mhz': FREQUENCY_HZ / 1e6,
    'tx_lat': TX_LAT, 'tx_lon': TX_LON,
    'tx_agl_m': TX_AGL_M,
    'tx_conducted_dbm': TX_CONDUCTED_DBM,
    'tx_antenna_gain_dbi': TX_ANTENNA_GAIN_DBI,
    'rx_agl_m': RX_AGL_M,
    'rx_extra_gain_db': RX_EXTRA_GAIN_DB,
    'max_depth': MAX_DEPTH,
    'num_samples_ps': NUM_SAMPLES_PS,
    'antenna_pattern': ANTENNA_PATTERN,
    'diag': {},
    'cell8_bands': [],
    'p833_bands': [],
    'figures': [],
}
print('Report accumulator initialised (_report).')
print(f'Scene XML        : {SCENE_XML}')
print(f'Raw CSV          : {OFCOM_RAW_CSV}')
print()
# ── Warn if key paths are missing ────────────────────────
for _label, _path in [('SCENE_XML', SCENE_XML), ('OFCOM_RAW_CSV', OFCOM_RAW_CSV)]:
    if not os.path.exists(_path):
        print(f'  WARNING: {_label} not found: {_path}')
        print(f'           → set correct path in the OPTIONAL OVERRIDE block above')

# ── save session config for independent diag cells ───────────────────────────
import json as _jcfg
_session_cfg = {
    'OUT_DIR':              OUT_DIR,
    'BASE_DIR':             BASE_DIR,
    'SCENE_DIR':            SCENE_DIR,
    'NDSM_TIFF':            NDSM_TIFF,
    'MEASUREMENT_CSV':      MEASUREMENT_CSV,
    'FREQUENCY_HZ':         FREQUENCY_HZ,
    'TX_CONDUCTED_DBM':     TX_CONDUCTED_DBM,
    'RX_EXTRA_GAIN_DB':     RX_EXTRA_GAIN_DB,
    'SITE_CORRECTION_DB':   SITE_CORRECTION_DB,
    'RX_AGL_M':             RX_AGL_M,
    'MAX_DEPTH':            MAX_DEPTH,
    'NUM_SAMPLES_PS':       NUM_SAMPLES_PS,
}
_cfg_path = os.path.join(OUT_DIR, 'session_config.json')
with open(_cfg_path, 'w') as _f:
    _jcfg.dump(_session_cfg, _f, indent=2)
print(f'Session config   : {_cfg_path}')


# ── Elevation sanity check ────────────────────────────────────────────────────
def _check_elevation():
    import struct as _st, json as _js
    scene_dir   = os.path.join(SCENE_BASE_DIR, 'scene_v4_full')
    terrain_ply = os.path.join(scene_dir, 'meshes', 'terrain.ply')
    elev_json   = os.path.join(scene_dir, 'origin_elev2.json')
    if os.path.exists(elev_json):
        _oe = _js.load(open(elev_json)).get('origin_elev_asl_m', 0.0)
        _ok = _oe > 5.0
        print(f'  origin_elev_asl : {_oe:.1f} m  {"✓" if _ok else "✗ WARN: near 0 — Cell 3 may have skipped with wrong origin"}')
    else:
        print('  origin_elev2.json : NOT FOUND')
        return
    if os.path.exists(terrain_ply):
        with open(terrain_ply, 'rb') as _f:
            _nv = 0
            while True:
                _l = _f.readline().decode('ascii','ignore').strip()
                if _l.startswith('element vertex'): _nv = int(_l.split()[-1])
                if _l == 'end_header': break
            if _nv > 0:
                import numpy as _np
                _verts = _np.frombuffer(_f.read(_nv*12), dtype='<f4').reshape(-1,3)
                _zmin, _zmax = float(_verts[:,2].min()), float(_verts[:,2].max())
                _ok_z = _zmin < 0   # negative Z proves relative coords used
                _zspan = _zmax - _zmin
                print(f'  terrain Z range : [{_zmin:.1f}, {_zmax:.1f}] m  span={_zspan:.0f} m  {"✓ relative coords" if _ok_z else "✗ WARN: all Z>0 — terrain.ply built with origin_elev_asl=0"}')
                if _ok_z and _zspan > 60:
                    print(f'  Note: large Z span ({_zspan:.0f} m) = hilly terrain — expected')
                if not _ok_z:
                    print('  FIX: delete terrain.ply → re-run Cell 3 → Cell B3 → Cell 5 in scene builder')
    else:
        print(f'  terrain.ply : NOT FOUND')

print('\n── Elevation check ─────────────────────────────────────────────')
_check_elevation()
print('────────────────────────────────────────────────────────────────')
SCENE_BASE_DIR  = os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem')  # scene reused from 915 MHz build


## GPS → Scene Coordinate Reference

All geographic coordinates are converted WGS84 (lon, lat) → projected CRS (set by `PROJECTION_CRS` in CELL 1) → scene-local (x, y) metres.

- **Projection:** `pyproj.Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)`
  - `PROJECTION_CRS = 'bng'` (default) → EPSG:27700 (British National Grid)
  - `PROJECTION_CRS = 'utm30n'` → EPSG:32630 (UTM Zone 30N)
- **Local origin:** Scene bounding-box centre in the projected CRS
- **x = East, y = North, z = Up**

Must match `PROJECTION_CRS` in `sionna019_scene_builder_stevenage.ipynb` CELL 0 exactly — same TX coordinates, same projection method.

## CELL 2 — Coordinate Utilities

Defines:
- `to_utm` / `from_utm` — WGS84 ↔ UTM transformers (pyproj)
- `local_xy(lon, lat)` — WGS84 → local scene (x, y)
- `local_z(lon, lat)` — terrain height at WGS84 point (samples terrain.ply)
- `rssi_from_path_gain(gain)` — path gain → RSS in dBm
- `path_loss_db(gain)` — path gain → path loss in dB


In [ ]:
# ── Coordinate transformers (pyproj, WGS84 ↔ projected CRS set by PROJECTION_CRS) ──
# always_xy=True enforces (lon, lat) / (easting, northing) order
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene centre — read from scene_parameters.json to match exact scene builder origin
_bbox_lon = (SCENE_WEST + SCENE_EAST) / 2
_bbox_lat = (SCENE_SOUTH + SCENE_NORTH) / 2
if os.path.exists(_crs_meta):
    _sp = json.load(open(_crs_meta))
    _file_lon = _sp['scene_center_lon']
    _file_lat = _sp['scene_center_lat']
    _km_off = ((_file_lon - _bbox_lon)**2 + (_file_lat - _bbox_lat)**2)**0.5 * 111.0
    if _km_off > 2.0:
        print(f"  [WARN] scene_parameters.json centre ({_file_lon:.6f}, {_file_lat:.6f}) is "
              f"{_km_off:.1f} km from SCENE bbox centre — file may belong to a different scene.")
        print(f"  [WARN] Falling back to SCENE_WEST/EAST/SOUTH/NORTH ({_bbox_lon:.6f}, {_bbox_lat:.6f}).")
        print(f"  [WARN] Delete {_crs_meta} and re-run scene builder to fix permanently.")
        center_lon, center_lat = _bbox_lon, _bbox_lat
    else:
        center_lon, center_lat = _file_lon, _file_lat
else:
    center_lon = _bbox_lon
    center_lat = _bbox_lat
utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)

def gps_to_local(lon, lat, height=0.0):
    """WGS84 (lon, lat) → scene-local (x, y, z) in metres.
    Origin = scene bbox centre. X = east, Y = north, Z = up."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Scene-local (x, y) → WGS84 (lon, lat)."""
    lon, lat = utm_to_gps.transform(x + utm_center_x, y + utm_center_y)
    return float(lon), float(lat)

# ── Terrain elevation — sample the actual terrain.ply mesh ────────────────────
# This GUARANTEES TX/RX sit exactly on the loaded scene geometry, because we
# interpolate the same vertices Sionna ray-traces. No CRS/DEM mismatch possible.
_terrain_interp = None

def _load_terrain_ply(path):
    """Read terrain.ply vertices (scene-local x,y,z). Supports ascii + binary."""
    import struct
    with open(path, 'rb') as f:
        hdr = []
        while True:
            line = f.readline().decode('ascii', errors='ignore').strip()
            hdr.append(line)
            if line == 'end_header':
                break
        nv = next(int(l.split()[-1]) for l in hdr if l.startswith('element vertex'))
        is_bin = any('binary' in l for l in hdr)
        # count vertex properties (assume float x,y,z first 3)
        if is_bin:
            raw = f.read()
            verts = np.frombuffer(raw[:nv*12], dtype=np.float32).reshape(-1, 3).copy()
        else:
            verts = np.array([list(map(float, f.readline().split()[:3]))
                              for _ in range(nv)], dtype=np.float32)
    return verts

if not FLAT_TERRAIN and os.path.exists(TERRAIN_PLY):
    from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
    _tv = _load_terrain_ply(TERRAIN_PLY)
    _xy = _tv[:, :2]
    _z  = _tv[:, 2]
    _lin  = LinearNDInterpolator(_xy, _z)
    _near = NearestNDInterpolator(_xy, _z)   # fallback outside convex hull
    def _terrain_interp(x, y):
        v = _lin(x, y)
        if v is None or np.isnan(v):
            v = _near(x, y)
        return float(v)
    print(f'Terrain    : sampled from {os.path.basename(TERRAIN_PLY)}  '
          f'({len(_tv):,} verts, z=[{_z.min():.1f}, {_z.max():.1f}] m)')
else:
    if FLAT_TERRAIN:
        print('Terrain    : FLAT_TERRAIN=True — z=0 everywhere')
    else:
        print(f'Terrain    : terrain.ply not found at {TERRAIN_PLY} — falling back to flat')

# Backwards-compat alias: legacy cells call get_dem_elevation(local_x, local_y).
# In the terrain.ply sampler, scene-local Z already equals ground height,
# so get_dem_elevation == terrain_z.
def get_dem_elevation(local_x, local_y):
    return terrain_z(local_x, local_y)

def terrain_z(local_x, local_y):
    """Scene-local Z of ground surface at (x, y). 0.0 if flat."""
    if FLAT_TERRAIN or _terrain_interp is None:
        return 0.0
    return _terrain_interp(float(local_x), float(local_y))

print(f'Scene centre  : lon={center_lon:.6f}  lat={center_lat:.6f}')
print(f'Projected centre (EPSG:{UTM_EPSG}): ({utm_center_x:.1f}, {utm_center_y:.1f})')
_txx, _txy, _ = gps_to_local(TX_LON, TX_LAT)
print(f'gps_to_local(TX): ({_txx:.1f}, {_txy:.1f})  terrain_z={terrain_z(_txx, _txy):.1f} m')


## CELL 3 — Load Scene

Loads `scene_v2_infra` XML (DEM terrain + buildings + infrastructure Blender export).

In [ ]:
# ====================================================================
# CELL 3 — LOAD SCENE (Sionna 2.0)
# ====================================================================
import os, sys
from sionna.rt import load_scene

print('=' * 60)
print('CELL 3 — LOAD SCENE')
print('=' * 60)

if not os.path.exists(SCENE_XML):
    raise FileNotFoundError(f"Scene XML not found: {SCENE_XML}")

# ── Sub-1 GHz ITU fix (safe, idempotent) ─────────────────────────
# ITU-R P.2040 materials (metal / concrete / brick / *_ground) are only
# defined for >= 1 GHz. At 2695 MHz Sionna raises
#   "Properties of ITU material '<x>' are not defined for this frequency"
# when scene.frequency is set (it evaluates EVERY registered ITU material).
# We wrap the ITU evaluator so that if it would raise for an out-of-range
# frequency, it retries clamped to 1 GHz (properties are ~flat there).
# A guard flag prevents re-wrapping on re-run (no recursion).
if FREQUENCY_HZ < 1e9:
    _patched = False
    for _mname, _mod in list(sys.modules.items()):
        if _mod is None or 'sionna' not in _mname:
            continue
        _fn = getattr(_mod, 'itu_material', None)
        if callable(_fn) and not getattr(_mod, '_lf_clamp_patched', False):
            _orig_itu = _fn
            def _clamped_itu(name, f, *a, _orig=_orig_itu, **k):
                try:
                    return _orig(name, f, *a, **k)
                except ValueError:
                    # clamp to the lowest ITU-defined frequency (1 GHz)
                    return _orig(name, 1e9, *a, **k)
            _mod.itu_material = _clamped_itu
            _mod._lf_clamp_patched = True
            _patched = True
    print(f'  Sub-1 GHz ITU clamp patch applied: {_patched}')

# ── Patch relative PLY paths to absolute before Mitsuba loads the XML ──────
# Mitsuba resolves relative paths from its CWD, not the XML directory.
# Rewrites any relative filename= to absolute so terrain.ply is always found.
import re as _re, tempfile as _tmp
_xml_dir = os.path.dirname(os.path.abspath(SCENE_XML))
_search_dirs = [
    _xml_dir,
    os.path.join(_xml_dir, 'meshes'),
    os.path.join(_xml_dir, 'meshes_full'),
    os.path.join(_xml_dir, 'meshes_roads'),
]
with open(SCENE_XML, 'r') as _f:
    _xml_str = _f.read()
def _abs_ply(m):
    val = m.group(1)
    if os.path.isabs(val) and os.path.isfile(val):
        return m.group(0)
    fname = os.path.basename(val)
    for _d in _search_dirs:
        cand = os.path.join(_d, fname)
        if os.path.isfile(cand):
            print(f'  [PLY fix] {fname} -> {cand}')
            return m.group(0).replace(val, cand)
    print(f'  [WARN] PLY not found: {val}')
    return m.group(0)
_xml_str = _re.sub(r'<string name="filename" value="([^"]+\.ply)"', _abs_ply, _xml_str)
# Write patched XML to temp file (Sionna 2 load_scene requires a file path)
_tmp_xml = os.path.join(_xml_dir, '_scene_patched.xml')
with open(_tmp_xml, 'w') as _f:
    _f.write(_xml_str)
print(f'Loading: {SCENE_XML} (PLY paths patched)')
scene = load_scene(_tmp_xml)
scene.frequency = FREQUENCY_HZ

print(f'Scene loaded.')
print(f'  Frequency : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'  Materials : {list(scene.radio_materials.keys())}')
print(f'  Objects   : {len(list(scene.objects.keys()))}')


## CELL 3b — Scene Preview

Interactive 3D Mitsuba preview of the loaded scene. Verify terrain mesh and building placement.

In [ ]:
%matplotlib inline
no_preview = False   # set True to skip interactive widget

print(f'Objects : {len(scene.objects)}')
print(f'Materials: {len(scene.radio_materials)}')

if not no_preview:
    scene.preview()


## CELL 3c — nDSM Clutter Height Heatmap

`nDSM = DSM − DTM` — height of objects above bare earth. Used to verify building heights match LiDAR data.

In [ ]:
# ==================================================================
# CELL 3c — nDSM CLUTTER HEIGHT HEATMAP
# ==================================================================
import rasterio, numpy as np, matplotlib.pyplot as plt
from rasterio.warp import transform_bounds
from pyproj import Transformer as _TrH

if not os.path.exists(NDSM_TIFF):
    print(f'nDSM not found: {NDSM_TIFF} — run CELL 2d in scene builder first.')
else:
    with rasterio.open(NDSM_TIFF) as _ds:
        _arr = _ds.read(1).astype(float)
        _nd  = _ds.nodata or 0.0
        _arr[_arr == _nd] = 0.0
        _arr = np.clip(_arr, 0, 40)
        # bounds in WGS84 for annotation
        _w, _s, _e, _n = transform_bounds(_ds.crs, 'EPSG:4326', *_ds.bounds)

    fig, ax = plt.subplots(figsize=(10, 9))
    _im = ax.imshow(_arr, cmap='YlOrRd', origin='upper',
                    extent=[_w, _e, _s, _n], aspect='auto',
                    vmin=0, vmax=25)
    plt.colorbar(_im, ax=ax, label='nDSM height (m)')

    # Scene bbox
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((SCENE_WEST, SCENE_SOUTH),
                            SCENE_EAST - SCENE_WEST,
                            SCENE_NORTH - SCENE_SOUTH,
                            edgecolor='blue', facecolor='none', lw=1.5,
                            label='Scene bbox'))
    # TX position
    ax.scatter([TX_LON], [TX_LAT], c='red', s=150, marker='*',
               zorder=5, label='TX')

    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(f'LiDAR nDSM — Clutter Heights\n'
                 f'max={_arr.max():.1f} m  |  '
                 f'>2m: {(_arr>2).mean()*100:.1f}%  '
                 f'>5m: {(_arr>5).mean()*100:.1f}%')
    ax.legend()
    plt.tight_layout()
    _fig_path = os.path.join(OUT_DIR, 'ndsm_heatmap.png')
    plt.savefig(_fig_path, dpi=150)
    plt.show()
    print(f'Saved: {_fig_path}')


## CELL 4A — Assign EM Material Properties

Sets all material properties in one place: permittivity, conductivity, scattering coefficient, and Lambertian pattern.

**Config knobs at the top of the code cell:**
- `GROUND_PRESET`: `"wet"` / `"medium"` / `"dry"` — controls ground reflectivity
- `SCATTER_OVERRIDE`: float 0–1 or `None` — overrides per-material scatter with a single value

| Material | εᵣ | σ (S/m) | Scatter S | Diffuse power S² |
|---|---|---|---|---|
| concrete | 5.31 | 0.092 | 0.20 | 4% |
| brick | 3.75 | 0.038 | 0.25 | 6% |
| glass | 6.27 | 0.000 | 0.08 | 1% |
| metal | 1.00 | 1e7 | 0.05 | 0.25% |
| wood | 1.99 | 0.000 | 0.30 | 9% |
| asphalt | 2.56 | 0.000 | 0.30 | 9% |
| ground (dry) | 2.8 | 0.000 | 0.30 | 9% |
| ground (medium) | 4.0 | 0.001 | 0.35 | 12% |
| ground (wet) | 30.0 | 0.020 | 0.40 | 16% |
| vegetation | 1.50 | 0.000 | 0.75 | 56% |


In [ ]:
# ====================================================================
# CELL 4A — ASSIGN EM MATERIAL PROPERTIES  (single configuration cell)
# Combines: material EM props + Lambertian scattering + ground tuning
# Run order: CELL 3 (load scene) → CELL 4A → CELL 4 (place TX) → ...
# Re-run this cell any time BEFORE CELL 8 / CELL 8e to change materials.
# ====================================================================
# ── CONFIG KNOBS ────────────────────────────────────────────────────
# GROUND_PRESET   — change when close-range bias is too large (>5 dB):
#   "wet"    -> er=30.0, sigma=0.020  (very reflective ground — NOT recommended)
#   "medium" -> er=4.0,  sigma=0.001  (urban road / concrete — good default)
#   "dry"    -> er=2.8,  sigma=0.000  (most absorptive — minimises ground bounce)
#
# SCATTER_OVERRIDE — change when RMSE is high at mid-range (300-1000m):
#   None  -> use per-material S values defined in MATERIAL_PROPS below
#   0.20  -> low scatter (mostly specular reflections — default building values)
#   0.40  -> medium scatter (recommended for urban at 2695 MHz)
#   0.70  -> high scatter (test maximum diffuse effect)
#
# MATERIAL_PROPS  — edit only if you have measured permittivity/conductivity
#   values for the specific building stock in the study area.
#   Changing individual scatter values here only takes effect when
#   SCATTER_OVERRIDE = None.
# ====================================================================
GROUND_PRESET    = "medium"   # "wet" | "medium" | "dry"  — adjust for Aberdeenshire terrainer for London urban roads
SCATTER_OVERRIDE = None    # e.g. 0.40, or None to use per-material values

import numpy as np

print("=" * 70)
print("CELL 4A — ASSIGN EM MATERIAL PROPERTIES")
print(f"  Ground preset    : {GROUND_PRESET}")
print(f"  Scatter override : {SCATTER_OVERRIDE if SCATTER_OVERRIDE is not None else 'per-material (MATERIAL_PROPS)'}")
print("=" * 70)

_freq_ghz = FREQUENCY_HZ / 1e9

# ── ITU-R P.2040-2 (2023) Table 3 — frequency-dependent EM parameters ────────
# εr(f) = a·f^b   σ(f) = c·f^d   f in GHz
# s   = scattering amplitude (diffuse power fraction = s²)
# xpd = cross-polarisation discrimination coefficient  [0=copol, 1=full xpol]
# wt  = representative wall thickness (m) used for transmission loss reference
#       L_wall(dB) ≈ 8.686·Im(k_c)·wt  where k_c=(2πf/c)·√(εr−jσ/ωε₀)
_ITU_P2040 = {
    #                   a       b       c        d       s     xpd   wt(m)
    'concrete':        (5.31,   0,      0.0326,  0.8095, 0.30, 0.10, 0.30),  # S=0.30 matches Nottingham calibrated
    'brick':           (3.91,   0,      0.0238,  0,      0.25, 0.15, 0.23),  # S=0.25 matches Nottingham calibrated
    'glass':           (6.27,   0,      0.0043,  1.1925, 0.08, 0.02, 0.012),
    'metal':           (1.00,   0,      1e7,     0,      0.05, 0.01, 0.005),
    'wood':            (1.99,   0,      0.0047,  1.0718, 0.15, 0.10, 0.05),
    'plasterboard':    (2.73,   0,      0.0085,  0.9395, 0.10, 0.05, 0.02),
    'marble':          (7.07,   0,      0.0200,  0,      0.05, 0.08, 0.03),
    'asphalt':         (2.56,   0,      0.0050,  0,      0.30, 0.15, 0.05),
    'vegetation':      (1.50,   0,      0.0020,  0.50,   0.40, 0.50, 0.10),  # S=0.40 matches diff RT Cell 4A
    'water':           (80.0,   0,      0.0100,  0,      0.03, 0.02, 0),
    'wet_ground':      (30.0,  -0.4,    0.1500,  1.30,   0.35, 0.25, 0),
    'medium_ground':   (15.0,  -0.1,    0.0350,  1.63,   0.30, 0.25, 0),
    'very_dry_ground': ( 3.0,   0,      0.00015, 2.52,   0.20, 0.20, 0),
}

import cmath as _cm

def _itu_at_freq(key, f_ghz):
    a, b, c, d, s, xpd, wt = _ITU_P2040[key]
    return {'er': a*(f_ghz**b), 'sigma': c*(f_ghz**d),
            'scatter': s, 'xpd': xpd, 'wall_t': wt}

def _wall_loss_db(er, sigma, freq_hz, thickness_m):
    if thickness_m <= 0: return 0.0
    _ep = complex(er, -sigma/(2*3.14159*freq_hz*8.854e-12))
    _k  = (2*3.14159*freq_hz/3e8) * _cm.sqrt(_ep)
    return 8.686 * abs(_k.imag) * thickness_m

print(f"\n  ITU-R P.2040-2 @ {_freq_ghz*1e3:.1f} MHz:")
print(f"  {'Material':<22} {'εr':>6} {'σ(S/m)':>9} {'S':>5} {'XPD':>5} {'WallLoss':>10}")
print("  " + "-"*63)
for _k, (_a,_b,_c,_d,_s,_xpd,_wt) in _ITU_P2040.items():
    _er = _a*(_freq_ghz**_b); _sg = _c*(_freq_ghz**_d)
    _wl = _wall_loss_db(_er, _sg, FREQUENCY_HZ, _wt)
    print(f"  {_k:<22} {_er:>6.2f} {_sg:>9.5f} {_s:>5.2f} {_xpd:>5.2f} "
          + (f"{_wl:>8.2f} dB" if _wt > 0 else "         —"))

# ── Ground presets derived from ITU-R P.2040-2 at scenario frequency ─────────
_GROUND_PRESETS = {
    "wet":    _itu_at_freq('wet_ground',      _freq_ghz),
    "medium": _itu_at_freq('medium_ground',   _freq_ghz),
    "dry":    _itu_at_freq('very_dry_ground', _freq_ghz),
}
_gp = _GROUND_PRESETS[GROUND_PRESET]
_GROUND_KEYS = ["ground", "terrain", "wet", "soil", "earth", "dry", "medium"]

# ── Load calibrated materials + scalar offset (controlled by USE_CALIBRATED_FILES) ──
import json as _jmod, os as _osmod
_calib_loaded = {}
SCALAR_OFFSET_DB = 0.0
if globals().get('USE_CALIBRATED_FILES', True):
    _CALIB_FILE = _osmod.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
    if _osmod.path.exists(_CALIB_FILE):
        try:
            with open(_CALIB_FILE) as _jf:
                _calib_loaded = _jmod.load(_jf).get('materials', {})
            print(f'\n[Cell 4A] Calibrated materials loaded: {list(_calib_loaded.keys())}')
        except Exception as _e:
            print(f'[Cell 4A] WARNING: calibrated_materials_stevenage_2695mhz.json unreadable ({_e}) — using ITU defaults')
    else:
        print(f'\n[Cell 4A] No calibrated_materials_stevenage_2695mhz.json — using ITU-R P.2040-2 defaults')
    _SF_FILE = _osmod.path.join(BASE_DIR, 'scalar_offset_stevenage_2695mhz.json')
    if _osmod.path.exists(_SF_FILE):
        try:
            with open(_SF_FILE) as _sf: _sf_data = _jmod.load(_sf)
            SCALAR_OFFSET_DB = float(_sf_data.get('scalar_factor_db', _sf_data.get('scaling_factor_db', 0.0)))
            print(f'[Cell 4A] Scalar offset: {SCALAR_OFFSET_DB:+.4f} dB  (from scalar_offset_stevenage_2695mhz.json)')
        except Exception as _e:
            print(f'[Cell 4A] Scalar offset unreadable ({_e}) — using 0.0 dB')
    else:
        print(f'[Cell 4A] scalar_offset_stevenage_2695mhz.json not found — SCALAR_OFFSET_DB=0.0 dB')
else:
    print(f'\n[Cell 4A] USE_CALIBRATED_FILES=False — using ITU-R P.2040-2 defaults, SCALAR_OFFSET_DB=0.0 dB')

def _mp(name):
    """ITU-R P.2040-2 props at FREQUENCY_HZ.
    NOTE: this does NOT apply calibrated overrides -- _calib_loaded's keys
    are the full scene.radio_materials names (e.g. 'itu_concrete'), but this
    function is called with bare ITU keys (e.g. 'concrete') when building
    MATERIAL_PROPS, so any itu_-stripping lookup here can never match.
    Calibrated values are applied later, per-material, using the exact
    scene material name (see the main assignment loop below)."""
    _n = name.lower()
    base_key = next((k for k in _ITU_P2040 if k in _n), None)
    return _itu_at_freq(base_key, _freq_ghz) if base_key else \
           {'er': 4.0, 'sigma': 0.08, 'scatter': 0.25, 'xpd': 0.10, 'wall_t': 0.2}

MATERIAL_PROPS = {k: _mp(k) for k in _ITU_P2040}
MATERIAL_PROPS.update({
    'ground':     {**_gp},
    'plywood':    _mp('wood'),
    'ceiling':    _mp('plasterboard'),
    'floorboard': _mp('wood'),
    'chipboard':  _mp('wood'),
})
# ── Lambertian pattern import ─────────────────────────────────────────
_Lambertian = None
for _modname in ("sionna.rt", "sionna.rt.scattering_pattern", "sionna.rt.radio_material"):
    try:
        _mod = __import__(_modname, fromlist=["LambertianPattern"])
        _Lambertian = getattr(_mod, "LambertianPattern", None)
        if _Lambertian is not None:
            print(f"  LambertianPattern from {_modname}")
            break
    except Exception:
        continue
if _Lambertian is None:
    print("  ⚠ LambertianPattern not found — using material default pattern")

_SCAT_FLOOR = 0.05
_safe_f = lambda v: float(v.numpy().flat[0]) if hasattr(v, "numpy") else (
                    float(v.item()) if hasattr(v, "item") else float(v))

existing = list(scene.radio_materials.keys())
print(f"\n  {'Material':<30} {'εᵣ':>6} {'σ':>8} {'S':>6} {'Pattern'}")
print("  " + "-" * 62)

for _name in existing:
    _mat = scene.radio_materials[_name]
    _name_l = _name.lower()

    # ── Match material type ───────────────────────────────────────────
    _is_ground = any(k in _name_l for k in _GROUND_KEYS)
    _props = None
    if _is_ground:
        # Match the material's OWN wet/dry/medium qualifier, not the global
        # GROUND_PRESET -- otherwise itu_wet_ground, itu_very_dry_ground and
        # itu_medium_dry_ground all collapse onto a single preset's values.
        if "wet" in _name_l:
            _gp_this = _GROUND_PRESETS["wet"]
        elif "very_dry" in _name_l or "verydry" in _name_l:
            _gp_this = _GROUND_PRESETS["dry"]
        elif "medium" in _name_l:
            _gp_this = _GROUND_PRESETS["medium"]
        elif "dry" in _name_l:
            _gp_this = _GROUND_PRESETS["dry"]
        else:
            _gp_this = _gp  # generic ground/terrain/soil/earth -> global GROUND_PRESET
        _props = {"er": _gp_this["er"], "sigma": _gp_this["sigma"],
                  "scatter": MATERIAL_PROPS.get("ground", {}).get("scatter", 0.35)}
    else:
        for _key in MATERIAL_PROPS:
            if _key in _name_l or _name_l in _key:
                _props = MATERIAL_PROPS[_key]
                break

    if _props is None:
        print(f"  ✗ {_name:<30} (no match — unchanged)")
        continue

    # ── Override with calibrated values for THIS EXACT material ────────
    # _calib_loaded's keys are the full scene.radio_materials names
    # (e.g. 'itu_concrete'), matching _name directly -- no itu_ stripping.
    _calib_key_exact = _name.replace('_train', '')
    if _calib_key_exact in _calib_loaded:
        _cv = _calib_loaded[_calib_key_exact]
        _props = {**_props, 'er': _cv['er'], 'sigma': _cv['sigma'],
                  'scatter': _cv.get('scatter', _props.get('scatter', 0.2))}

    # ── Set EM properties ─────────────────────────────────────────────
    _mat.relative_permittivity = _props["er"]
    _mat.conductivity          = _props["sigma"]

    # ── Set scattering coefficient ────────────────────────────────────
    _s_target = SCATTER_OVERRIDE if SCATTER_OVERRIDE is not None else _props["scatter"]
    _s_target = max(_s_target, _SCAT_FLOOR)
    _scat_set = False
    for _attr in ("scattering_coefficient", "_scattering_coefficient"):
        if hasattr(_mat, _attr):
            try:
                _v = getattr(_mat, _attr)
                if hasattr(_v, "data"):       _v.data.fill_(_s_target)
                elif hasattr(_v, "assign"):   _v.assign(_s_target)
                else:                         setattr(_mat, _attr, _s_target)
                _scat_set = True
                break
            except Exception:
                pass
    if not _scat_set:
        try:
            _mat.scattering_coefficient = _s_target
            _scat_set = True
        except Exception:
            pass

    # ── Set XPD coefficient — ITU-R P.2040-2 cross-polarisation ─────────────
    _xpd_target = float(_props.get('xpd', 0.0))
    for _attr in ("xpd_coefficient", "_xpd_coefficient"):
        if hasattr(_mat, _attr):
            try:
                _v = getattr(_mat, _attr)
                if hasattr(_v, "data"):       _v.data.fill_(_xpd_target)
                elif hasattr(_v, "assign"):   _v.assign(_xpd_target)
                else:                         setattr(_mat, _attr, _xpd_target)
            except Exception: pass
            break

    # ── Set Lambertian pattern ────────────────────────────────────────
    _pat_set = False
    if _Lambertian is not None and hasattr(_mat, "scattering_pattern"):
        try:
            _mat.scattering_pattern = _Lambertian()
            _pat_set = True
        except Exception:
            pass

    # ── Read back actual values ───────────────────────────────────────
    _sv = None
    for _a in ("scattering_coefficient", "_scattering_coefficient"):
        if hasattr(_mat, _a):
            try: _sv = _safe_f(getattr(_mat, _a))
            except Exception: pass
            break
    _pat = type(getattr(_mat, "scattering_pattern", None)).__name__

    # Readback the ACTUAL live values from the material object (not just
    # the target _props we tried to set) -- confirms the assignment really
    # stuck, the way scattering_coefficient already was below.
    try:    _er_rb  = _safe_f(_mat.relative_permittivity)
    except Exception: _er_rb  = _props['er']
    try:    _sig_rb = _safe_f(_mat.conductivity)
    except Exception: _sig_rb = _props['sigma']

    _xpd_v = float(_props.get('xpd', 0.0))
    _wt_v  = float(_props.get('wall_t', 0.0))
    _wl_v  = _wall_loss_db(_er_rb, _sig_rb, FREQUENCY_HZ, _wt_v)
    _ok = "✓" if _scat_set else "⚠"
    print(f"  {_ok} {_name:<30} {_er_rb:>6.2f} {_sig_rb:>8.4f} "
          f"{(_sv if _sv is not None else 0.0):>5.3f} xpd={_xpd_v:.2f}"
          + (f" wall={_wl_v:.1f}dB" if _wt_v > 0 else ""))

print()
print(f"  Ground preset    : {GROUND_PRESET.upper()}  "
      f"(er={_gp['er']}, sigma={_gp['sigma']})")
print(f"  Scatter override : {SCATTER_OVERRIDE if SCATTER_OVERRIDE is not None else 'per-material'}")
# Set itu_ceiling_board: VEG params when disc veg active, transparent when DISABLE_VEG_DISCS=True.
# The main loop maps 'ceiling' → plasterboard (er=2.73, S=0.10). This corrects that.
_cb_mat = scene.radio_materials.get('itu_ceiling_board')
if _cb_mat is not None:
    if globals().get('DISABLE_VEG_DISCS', False):
        _cb_mat.relative_permittivity  = 1.0
        _cb_mat.conductivity           = 0.0
        _cb_mat.scattering_coefficient = 0.0
        print(f"  itu_ceiling_board → er=1.0, sigma=0.0000 S/m, S=0.000  [transparent — DISABLE_VEG_DISCS=True]")
    else:
        _cb_mat.relative_permittivity  = VEG_RELATIVE_PERMITTIVITY
        _cb_mat.conductivity           = VEG_CONDUCTIVITY
        _cb_mat.scattering_coefficient = VEG_SCATTERING_COEFF
        print(f"  itu_ceiling_board → er={VEG_RELATIVE_PERMITTIVITY:.1f}, sigma={VEG_CONDUCTIVITY:.4f} S/m, S={VEG_SCATTERING_COEFF:.3f}  [P.833, locked]")

# ── CELL 8e sample override ──────────────────────────────────────────────────
# TX AGL scan (CELL 9) uses NUM_SAMPLES_PS=2M (fast). CELL 8e eval needs 100M.
# Override here so CELL 8e inherits the correct value without changing CELL 1.
import builtins as _bi
_bi.NUM_SAMPLES_PS_EVAL = 100_000_000
NUM_SAMPLES_PS = 100_000_000   # 100M for CELL 8e evaluation (optimal per Nottingham benchmark)
print(f'  NUM_SAMPLES_PS overridden to {NUM_SAMPLES_PS:,} for CELL 8e eval')



print("\n✅ All materials configured.")



## CELL 4 — Place Transmitter

Places TX at `TX_AGL_M = 17.0 m` above the DEM terrain surface at the Ofcom site coordinates.

In [ ]:
# Remove previous TX
for _n in list(scene.transmitters.keys()):
    scene.remove(_n)

tx_x, tx_y, _ = gps_to_local(TX_LON, TX_LAT)
tx_z = terrain_z(tx_x, tx_y) + TX_AGL_M  # ground + AGL

# ── Antenna pattern (Sionna 2.0 — set on scene.tx_array) ──────────────────────
# 'iso'   = isotropic (0 dBi, uniform sphere)
# 'donut' = vertical half-wave dipole (~2.15 dBi, null at zenith/nadir)
_pattern = "dipole" if ANTENNA_PATTERN == 'donut' else "iso"
_ant_desc = "half-wave dipole (donut)" if ANTENNA_PATTERN == 'donut' else "isotropic"

scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern=_pattern,
                             polarization="V")

tx = Transmitter(name='tx0',
                 position=[tx_x, tx_y, tx_z],
                 orientation=[0.0, 0.0, 0.0])
scene.add(tx)

print(f'TX placed:')
print(f'  GPS      : lon={TX_LON}  lat={TX_LAT}')
print(f'  Local    : ({tx_x:.1f}, {tx_y:.1f}, {tx_z:.1f}) m')
print(f'  AGL      : {TX_AGL_M} m  (terrain_z={tx_z - TX_AGL_M:.1f} m)')
print(f"  Antenna  : {_ant_desc}  [scene.tx_array, pattern={_pattern}]")


## CELL 5 — Extract RX from Ofcom CSV

Parses drive-test CSV. Filters to receivers within scene bbox and within `MAX_RX_DIST_M` of TX.

In [ ]:
import csv as _csv_mod

print('=' * 60)
print('CELL 5 — RX EXTRACTION (nearest NUM_RX in scene bbox, sorted by distance)')
print('=' * 60)

if not os.path.exists(OFCOM_RAW_CSV):
    raise FileNotFoundError(f'CSV not found: {OFCOM_RAW_CSV}')

# ── Auto-detect header row ────────────────────────────────────────────────────
_hdr_idx = None
with open(OFCOM_RAW_CSV, 'r', encoding='utf-8', errors='replace') as _f:
    for _i, _line in enumerate(_f):
        if 'Latitude' in _line and 'Longitude' in _line:
            _hdr_idx = _i
            break
        if _i > 40:
            break

if _hdr_idx is None:
    raise ValueError(
        f'Could not find header row in {OFCOM_RAW_CSV}\n'
        'Expected a line containing both "Latitude" and "Longitude" in the first 40 lines.')

print(f'Header at line {_hdr_idx + 1}  (0-based index {_hdr_idx})')
_df_raw = pd.read_csv(OFCOM_RAW_CSV, skiprows=_hdr_idx, low_memory=False)
print(f'Columns: {list(_df_raw.columns)}')
print(f'Total rows: {len(_df_raw)}')

# ── Column mapping — flexible: match by keyword ───────────────────────────────
def _find_col(df, *keywords):
    for col in df.columns:
        c = col.strip().lower()
        if all(k.lower() in c for k in keywords):
            return col
    return None

_lat_col  = _find_col(_df_raw, 'latitude')
_lon_col  = _find_col(_df_raw, 'longitude')
_rssi_col = _find_col(_df_raw, 'measurement') or _find_col(_df_raw, 'dBm') or _find_col(_df_raw, 'dbm')

if not _lat_col:
    raise KeyError(f'No Latitude column found. Available: {list(_df_raw.columns)}')
if not _lon_col:
    raise KeyError(f'No Longitude column found. Available: {list(_df_raw.columns)}')
if not _rssi_col:
    raise KeyError(f'No RSSI/measurement column found. Available: {list(_df_raw.columns)}')

print(f'Lat  col : {_lat_col!r}')
print(f'Lon  col : {_lon_col!r}')
print(f'RSSI col : {_rssi_col!r}')

# Drop rows with non-numeric values in key columns
for _c in [_lat_col, _lon_col, _rssi_col]:
    _df_raw[_c] = pd.to_numeric(_df_raw[_c], errors='coerce')
_df_raw = _df_raw.dropna(subset=[_lat_col, _lon_col, _rssi_col]).reset_index(drop=True)
print(f'Rows after numeric filter: {len(_df_raw)}')

# Distance from TX (over all rows — distance-sort fix, same as 5b35e4e/9bb6be0)
_dlon_m = 111000.0 * math.cos(math.radians(TX_LAT))
_dlat_m = 111000.0
_df_raw['_dist_km'] = (
    ((_df_raw[_lat_col] - TX_LAT) * _dlat_m)**2 +
    ((_df_raw[_lon_col] - TX_LON) * _dlon_m)**2
)**0.5 / 1000.0

# Filter to scene bbox, sort by distance
_in_bbox = (
    (_df_raw[_lat_col] >= SCENE_SOUTH) & (_df_raw[_lat_col] <= SCENE_NORTH) &
    (_df_raw[_lon_col] >= SCENE_WEST)  & (_df_raw[_lon_col] <= SCENE_EAST)
)
_df_bbox = _df_raw[_in_bbox].sort_values('_dist_km').reset_index(drop=True)
print(f'  bbox: {_in_bbox.sum()} rows in scene bbox')

# Spread receivers across distance range (prevents dense near-TX area monopolising NUM_RX quota)
_MAX_RX_KM = globals().get('MAX_RX_DIST_KM', 2.5)
_df_dist   = _df_bbox[_df_bbox['_dist_km'] <= _MAX_RX_KM].reset_index(drop=True)
if len(_df_dist) > NUM_RX:
    _step = max(1, len(_df_dist) // NUM_RX)
    _sel  = _df_dist.iloc[::_step].head(NUM_RX).copy()
    print(f'  {len(_df_dist)} rows within {_MAX_RX_KM}km → every {_step}th → {len(_sel)} selected')
else:
    _sel  = _df_dist.copy()
    print(f'  {len(_df_dist)} rows within {_MAX_RX_KM}km → all selected')

print(f'\nSelected  : {len(_sel)} receivers (spread to {_MAX_RX_KM}km, sorted by distance)')
print(f'Dist range: {_sel["_dist_km"].min():.3f} – {_sel["_dist_km"].max():.3f} km')
print(f'RSSI range: {_sel[_rssi_col].min():.1f} \u2013 {_sel[_rssi_col].max():.1f} dBm')

# ── Write receiver_locations.csv (raw GPS — no position correction) ──────────
os.makedirs(os.path.dirname(RX_CSV), exist_ok=True)
with open(RX_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'height'])
    for _idx, _row in _sel.iterrows():
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     RX_AGL_M])
print(f'\nWritten : {RX_CSV}  ({len(_sel)} rows)')

# ── Write measurements_with_pathloss.csv ──────────────────────────────────────
with open(MEASUREMENT_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'local_measurement_dBm', 'path_loss_dB', 'dist_from_tx_m'])
    for _idx, _row in _sel.iterrows():
        _rssi    = float(_row[_rssi_col])
        _pl      = TX_CONDUCTED_DBM - _rssi
        _lat_r   = float(_row[_lat_col])
        _lon_r   = float(_row[_lon_col])
        _dist_m  = math.sqrt(
            ((_lat_r - TX_LAT) * 111000.0)**2 +
            ((_lon_r - TX_LON) * 111000.0 * math.cos(math.radians(TX_LAT)))**2
        )
        _w.writerow([f'RX_{_idx:06d}',
                     f'{_lon_r:.6f}',
                     f'{_lat_r:.6f}',
                     f'{_rssi:.2f}',
                     f'{_pl:.2f}',
                     f'{_dist_m:.1f}'])
print(f'Written : {MEASUREMENT_CSV}  ({len(_sel)} rows)')
print(f'  Columns : name | lon | lat | local_measurement_dBm | path_loss_dB | dist_from_tx_m')
print(f'  PL formula : TX_CONDUCTED_DBM - RSSI = {TX_CONDUCTED_DBM} - RSSI')


## CELL 6 — Place Receivers + DEM Sanity Check

Places all RX at `RX_AGL_M = 1.5 m` above terrain. Sanity check: verifies RX z-coordinates match DEM elevation at each GPS position.

In [ ]:
# Receiver antenna array (Sionna 2.0 — isotropic single element)
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="iso",
                             polarization="V")

df_rx = pd.read_csv(RX_CSV)
for nm in list(scene.receivers.keys()):
    scene.remove(nm)

_exclude = globals().get('_rx_exclude', set())   # built by CELL 5h

receivers = []
_n_skipped = 0
for _, row in df_rx.iterrows():
    if str(row['name']) in _exclude:
        _n_skipped += 1
        continue
    lx, ly, _ = gps_to_local(float(row['lon']), float(row['lat']))
    lz = terrain_z(lx, ly) + RX_AGL_M
    rx = Receiver(name=row['name'], position=[lx, ly, lz])
    scene.add(rx)
    receivers.append(rx)

print(f'Placed {len(receivers)} receivers at z={RX_AGL_M}m (flat terrain={FLAT_TERRAIN})')
if _n_skipped:
    print(f'  Skipped {_n_skipped} inside-building receivers (run CELL 5h to update list)')
print(f'  scene.rx_array: isotropic single element')


In [ ]:
# ====================================================================
# CELL 6b — DEM SANITY CHECK (terrain alignment + heights)
# ====================================================================
# Proves the DEM/terrain is wired correctly BEFORE trusting any RSSI:
#   1. terrain.ply loaded with realistic z-range
#   2. TX/RX bbox sits inside the terrain mesh bbox (coverage check)
#   3. RX sit ON the terrain surface (terrain_z + AGL), NOT a naive z<0
#   4. If RX fall outside the mesh, print the exact GPS bbox to rebuild
# ====================================================================
import numpy as np
print("=" * 70)
print("CELL 6b — DEM SANITY CHECK")
print("=" * 70)

_safe6 = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)
_problems = []

# -- 1. Terrain mesh loaded? --------------------------------------------------
print("\n[1] Terrain source")
if FLAT_TERRAIN:
    print("  FLAT_TERRAIN=True - DEM not in use (skipping DEM checks).")
else:
    import os
    if not os.path.exists(TERRAIN_PLY):
        _problems.append(f"terrain.ply missing: {TERRAIN_PLY}")
        print(f"  X terrain.ply NOT found: {TERRAIN_PLY}")
    elif _terrain_interp is None:
        _problems.append("terrain interpolator not built (load failed)")
        print("  X _terrain_interp is None - terrain load failed.")
    else:
        _tvz = _tv[:, 2]
        print(f"  OK terrain.ply loaded: {len(_tv):,} verts  z=[{_tvz.min():.1f}, {_tvz.max():.1f}] m  "
              f"relief={_tvz.max()-_tvz.min():.1f} m")
        print(f"     (note: mesh vertical datum is offset - z<0 is mid-terrain, NOT underground)")
        if _tvz.max() - _tvz.min() < 0.5:
            _problems.append("terrain relief < 0.5 m - DEM may be flat/empty")
            print("  ! Terrain is nearly flat - is the DEM real?")

# -- 2. Coverage: RX/TX bbox inside terrain mesh bbox -------------------------
print("\n[2] Terrain coverage (RX/TX vs terrain mesh extent)")
_n_out = 0
if not FLAT_TERRAIN and _terrain_interp is not None:
    _rx_x = np.array([_safe6(r.position[0]) for r in receivers])
    _rx_y = np.array([_safe6(r.position[1]) for r in receivers])
    _tx = list(scene.transmitters.values())[0]
    _tx_x, _tx_y = _safe6(_tx.position[0]), _safe6(_tx.position[1])
    _mesh_x = _tv[:, 0]; _mesh_y = _tv[:, 1]
    _allx = np.append(_rx_x, _tx_x); _ally = np.append(_rx_y, _tx_y)
    print(f"  RX/TX X span : [{_allx.min():.0f}, {_allx.max():.0f}] m")
    print(f"  Mesh   X span: [{_mesh_x.min():.0f}, {_mesh_x.max():.0f}] m")
    print(f"  RX/TX Y span : [{_ally.min():.0f}, {_ally.max():.0f}] m")
    print(f"  Mesh   Y span: [{_mesh_y.min():.0f}, {_mesh_y.max():.0f}] m")
    _outside = ((_rx_x < _mesh_x.min()) | (_rx_x > _mesh_x.max()) |
                (_rx_y < _mesh_y.min()) | (_rx_y > _mesh_y.max()))
    _n_out = int(_outside.sum())
    if _n_out == 0:
        print("  OK All receivers fall inside the terrain mesh (full DEM coverage).")
    else:
        _problems.append(f"{_n_out} RX outside terrain mesh - DEM too small/off-center")
        print(f"  ! {_n_out}/{len(receivers)} receivers fall OUTSIDE the terrain mesh.")
        print(f"    These get a NearestND edge-vertex height (WRONG ground height).")
        _MARGIN = 300.0  # metres
        _need_x = (_allx.min() - _MARGIN, _allx.max() + _MARGIN)
        _need_y = (_ally.min() - _MARGIN, _ally.max() + _MARGIN)
        print(f"\n    -> Rebuild terrain.ply to cover (local m, +{_MARGIN:.0f} m margin):")
        print(f"        X: [{_need_x[0]:.0f}, {_need_x[1]:.0f}]   "
              f"Y: [{_need_y[0]:.0f}, {_need_y[1]:.0f}]")
        if 'local_to_gps' in dir():
            try:
                _w, _s = local_to_gps(_need_x[0], _need_y[0])
                _e, _n = local_to_gps(_need_x[1], _need_y[1])
                print(f"    -> Set these in the DEM scene builder (GPS bbox):")
                print(f"        SCENE_WEST  = {min(_w,_e):.6f}")
                print(f"        SCENE_EAST  = {max(_w,_e):.6f}")
                print(f"        SCENE_SOUTH = {min(_s,_n):.6f}")
                print(f"        SCENE_NORTH = {max(_s,_n):.6f}")
            except Exception as _e_gps:
                print(f"    (local_to_gps unavailable: {_e_gps})")

# -- 3. Heights: RX sit ON the terrain surface (not naive z<0) ----------------
print("\n[3] TX / RX heights (relative to terrain surface)")
_tx = list(scene.transmitters.values())[0]
_tx_x, _tx_y, _tx_z = (_safe6(_tx.position[i]) for i in range(3))
_tgz = terrain_z(_tx_x, _tx_y)
print(f"  TX: terrain_z={_tgz:.1f}  +AGL({TX_AGL_M})  = {_tgz+TX_AGL_M:.1f}  | placed z={_tx_z:.1f}  "
      f"{'OK' if abs(_tx_z-(_tgz+TX_AGL_M))<0.5 else '! mismatch'}")
_rz = np.array([_safe6(r.position[2]) for r in receivers])
_rgz = np.array([terrain_z(_safe6(r.position[0]), _safe6(r.position[1])) for r in receivers])
_expected = _rgz + RX_AGL_M
_below_ground = int((_rz < _rgz - 0.5).sum())
_misplaced = int((np.abs(_rz - _expected) > 0.5).sum())
print(f"  RX z range   : [{_rz.min():.1f}, {_rz.max():.1f}] m   (terrain z=[{_rgz.min():.1f}, {_rgz.max():.1f}])")
print(f"  Below ground (z < terrain_z): {_below_ground}   |   height mismatch (>0.5 m): {_misplaced}")
if _below_ground: _problems.append(f"{_below_ground} RX below terrain surface")
if _misplaced:    _problems.append(f"{_misplaced} RX not on terrain+AGL")
if _misplaced == 0:
    print(f"  OK All RX placed exactly at terrain_z + AGL (height pipeline correct).")
    if _n_out:
        print(f"    ! but {_n_out} of those terrain_z values came from the NearestND")
        print(f"       fallback (RX outside mesh) - fix coverage in [2] for valid heights.")

# -- 4. Verdict ---------------------------------------------------------------
print("\n" + "=" * 70)
if not _problems:
    print("OK DEM SANITY: all checks passed - terrain is correctly wired.")
else:
    print("! DEM SANITY: issues found -")
    for _p in _problems:
        print(f"   - {_p}")
print("=" * 70)


## CELL 6b — DEM Terrain + TX/RX Position Map

Plots the DEM elevation heatmap with TX (star) and all RX positions overlaid.
Requires `dem.tif`, `receivers`, and TX to be loaded.

In [ ]:
# ====================================================================
# CELL 6b — DEM TERRAIN + TX/RX POSITION MAP
# ====================================================================
import numpy as np, matplotlib.pyplot as plt, os
import rasterio
from rasterio.warp import reproject, Resampling

_safe_v = lambda v: float(np.asarray(v.numpy() if hasattr(v,'numpy') else v).flat[0])

# ── Load DEM ──────────────────────────────────────────────────────────
_dem_path = DEM_TIFF
assert os.path.exists(_dem_path), f'DEM not found: {_dem_path}'

with rasterio.open(_dem_path) as _ds:
    _elev = _ds.read(1).astype(float)
    _elev[_elev == _ds.nodata] = np.nan
    _bounds = _ds.bounds
    _crs    = _ds.crs
    print(f'DEM loaded: {_elev.shape}  CRS={_crs}  elev range [{np.nanmin(_elev):.1f}, {np.nanmax(_elev):.1f}] m')

# ── Convert DEM bounds to scene local coords ──────────────────────────
from pyproj import Transformer
_t_utm = Transformer.from_crs('EPSG:27700', f'EPSG:{UTM_EPSG}', always_xy=True)
_orig_e, _orig_n = _t_utm.transform(
    (_bounds.left + _bounds.right)/2, (_bounds.bottom + _bounds.top)/2)
_sw = _t_utm.transform(_bounds.left,  _bounds.bottom)
_ne = _t_utm.transform(_bounds.right, _bounds.top)
_xmin = _sw[0] - _orig_e; _xmax = _ne[0] - _orig_e
_ymin = _sw[1] - _orig_n; _ymax = _ne[1] - _orig_n

# ── TX position ───────────────────────────────────────────────────────
_tx0  = list(scene.transmitters.values())[0]
_tx_x = _safe_v(_tx0.position[0])
_tx_y = _safe_v(_tx0.position[1])
_tx_z = _safe_v(_tx0.position[2])

# ── RX positions ──────────────────────────────────────────────────────
_rx_x = [_safe_v(r.position[0]) for r in receivers]
_rx_y = [_safe_v(r.position[1]) for r in receivers]
_rx_z = [_safe_v(r.position[2]) for r in receivers]

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: DEM elevation heatmap + TX/RX
_ext = [_xmin, _xmax, _ymin, _ymax]
im = axes[0].imshow(_elev, origin='upper', extent=_ext,
                    cmap='terrain', aspect='auto')
plt.colorbar(im, ax=axes[0], label='Elevation (m)')
axes[0].scatter(_rx_x, _rx_y, s=6, c='white', alpha=0.6,
                linewidths=0, zorder=3, label=f'RX ({len(receivers)})')
axes[0].scatter([_tx_x], [_tx_y], marker='*', s=400,
                c='red', edgecolors='black', linewidths=0.8,
                zorder=5, label=f'TX (z={_tx_z:.1f}m)')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title('DEM Elevation + TX/RX Positions')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.2)

# Right: RX height histogram
axes[1].hist(_rx_z, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(RX_AGL_M, color='red', lw=2, ls='--', label=f'RX_AGL={RX_AGL_M}m')
axes[1].set_xlabel('RX height z (m)')
axes[1].set_ylabel('Count')
axes[1].set_title('RX Height Distribution (terrain + AGL)')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle(f'Scene: {len(receivers)} receivers  |  TX at ({_tx_x:.0f}, {_tx_y:.0f}, {_tx_z:.1f}) m',
             fontsize=12)
plt.tight_layout()
_png = os.path.join(OUT_DIR, 'dem_terrain_tx_rx_map.png')
plt.savefig(_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {_png}')
print(f'TX height: {_tx_z:.2f} m  |  RX z range: {min(_rx_z):.2f} – {max(_rx_z):.2f} m')


In [ ]:
# ====================================================================
# CELL 9 — COVERAGE MAP (RadioMapSolver)  [2695 MHz DEM / Sionna 2.0]
# ====================================================================
# Sionna 2.0 equivalent of the sionna019 scene.coverage_map() cell.
# Uses RadioMapSolver to compute a path-gain grid over the scene
# extent, WITH and WITHOUT diffuse scattering, then converts to RSSI:
#     RSSI = TX_CONDUCTED_DBM - PL + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB
# Saves a grid CSV + preview PNG (frequency-tagged) and interpolates
# the map onto the receiver positions.
# ====================================================================
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sionna.rt import RadioMapSolver

print('=' * 70)
print('CELL 9 — COVERAGE MAP (RadioMapSolver)  [2695 MHz DEM / Sionna 2.0]')
print('=' * 70)

_safe_cm = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── Scene extent from GPS bounds (robust vs mi_scene.bbox()) ──────────────────
_sw = gps_to_local(SCENE_WEST, SCENE_SOUTH)
_ne = gps_to_local(SCENE_EAST, SCENE_NORTH)
gx_min, gy_min = _sw[0], _sw[1]
gx_max, gy_max = _ne[0], _ne[1]
center_x = (gx_min + gx_max) / 2.0
center_y = (gy_min + gy_max) / 2.0

# ── Map plane height ──────────────────────────────────────────────────────────
# FLAT_TERRAIN: z=0 everywhere → plane at RX_AGL_M.
# DEM terrain: sample ground at centre, add RX_AGL_M.
_ground_z = 0.0 if FLAT_TERRAIN else terrain_z(center_x, center_y)
center_z  = _ground_z + RX_AGL_M

size_x = gx_max - gx_min
size_y = gy_max - gy_min
nx = int(np.ceil(size_x / GRID_SIZE_M))
ny = int(np.ceil(size_y / GRID_SIZE_M))
print(f'Scene extent : ({gx_min:.0f},{gy_min:.0f}) -> ({gx_max:.0f},{gy_max:.0f}) m')
print(f'Grid         : {nx} x {ny} = {nx*ny:,} cells  ({GRID_SIZE_M} m res)  map Z={center_z:.2f} m')
if nx * ny > 10_000_000:
    raise RuntimeError(f'Grid {nx}x{ny}={nx*ny:,} cells too large (>10M). Increase GRID_SIZE_M.')

# ── path_gain grid -> (rssi_dBm, path_loss_dB) ────────────────────────────────
def _cm_to_dbm(path_gain_grid):
    arr = np.asarray(path_gain_grid, dtype=float)
    while arr.ndim > 2:            # (num_tx, ny, nx) -> (ny, nx)
        arr = arr[0]
    # PL floor = power level that maps RSSI to the noise floor
    _pl_max = TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB - NOISE_FLOOR_DBM
    _uncov  = arr <= 0
    pl_db   = np.where(_uncov, _pl_max, -10.0 * np.log10(np.maximum(arr, 1e-30)))
    rssi_db = np.where(_uncov, NOISE_FLOOR_DBM,
                       np.maximum(TX_CONDUCTED_DBM - pl_db + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB,
                                  NOISE_FLOOR_DBM))
    return rssi_db, pl_db

# ── Re-apply LambertianPattern before RadioMapSolver ─────────────────────────
# RadioMapSolver requires scattering_pattern set on materials at call time.
try:
    from sionna.rt import LambertianPattern as _LP
except ImportError:
    try:
        from sionna.rt.scattering_pattern import LambertianPattern as _LP
    except ImportError:
        _LP = None
if _LP is not None:
    _lp_inst = _LP()
    for _mn in scene.radio_materials.keys():
        _m = scene.radio_materials[_mn]
        try:
            _m.scattering_pattern = _lp_inst
        except Exception:
            pass
    print('LambertianPattern re-applied to all materials before RadioMapSolver.')
else:
    print('WARNING: LambertianPattern not found — scatter effect may be absent.')

# ── Run RadioMapSolver with / without diffuse scattering ──────────────────────
_solver = RadioMapSolver()

def _compute_cm(diffuse, label):
    print(f'\nComputing coverage map {label} (diffuse_reflection={diffuse}) ...')
    t0 = time.time()
    rm = _solver(
        scene,
        cell_size           = (GRID_SIZE_M, GRID_SIZE_M),
        center              = (center_x, center_y, center_z),
        orientation         = (0.0, 0.0, 0.0),
        size                = (size_x, size_y),
        max_depth           = MAX_DEPTH,
        samples_per_tx      = NUM_SAMPLES_CM,
        los                 = True,
        specular_reflection = True,
        diffraction         = True,
        diffuse_reflection  = diffuse,
    )
    pg = rm.path_gain.numpy() if hasattr(rm.path_gain, 'numpy') else np.array(rm.path_gain)
    print(f'  Done in {time.time()-t0:.1f}s  path_gain shape={pg.shape}')
    return rm, pg

rm,        _pg_scatter    = _compute_cm(True,  'WITH scattering')   # `rm` kept for scene.preview()
_rm_ns,    _pg_no_scatter = _compute_cm(False, 'WITHOUT scattering')

rssi_scatter,    pl_scatter    = _cm_to_dbm(_pg_scatter)
rssi_no_scatter, pl_no_scatter = _cm_to_dbm(_pg_no_scatter)

# ── Coverage stats ────────────────────────────────────────────────────────────
for label, r, pl in [('With scatter',    rssi_scatter,    pl_scatter),
                     ('Without scatter', rssi_no_scatter, pl_no_scatter)]:
    cov   = r > NOISE_FLOOR_DBM
    n_cov = int(cov.sum()); n_tot = int(r.size)
    print(f'{label:18s}: coverage {n_cov:,}/{n_tot:,} ({100*n_cov/max(n_tot,1):.1f}%)')
    if n_cov:
        v = r[cov]; v = v[np.isfinite(v) & (v <= 0)]
        print(f'    RSSI(covered): mean={np.nanmean(v):.1f} std={np.nanstd(v):.1f} '
              f'min={np.nanmin(v):.1f} max={np.nanmax(v):.1f} dBm  (valid={len(v)})')

_both = (rssi_scatter > NOISE_FLOOR_DBM) & (rssi_no_scatter > NOISE_FLOOR_DBM)
if _both.sum():
    _d = rssi_scatter[_both] - rssi_no_scatter[_both]
    _d = _d[np.isfinite(_d)]
    print(f'Scatter impact (covered): mean={np.nanmean(_d):.2f} std={np.nanstd(_d):.2f} dB  (finite={len(_d)})')

# ── Save grid CSV (frequency-tagged) ──────────────────────────────────────────
_xx = np.linspace(gx_min, gx_max, rssi_scatter.shape[1])
_yy = np.linspace(gy_min, gy_max, rssi_scatter.shape[0])
_XX, _YY = np.meshgrid(_xx, _yy)
_df_cm = pd.DataFrame({
    'x_m':                     _XX.ravel(),
    'y_m':                     _YY.ravel(),
    'rssi_scatter_dbm':        rssi_scatter.ravel(),
    'rssi_no_scatter_dbm':     rssi_no_scatter.ravel(),
    'path_loss_scatter_db':    pl_scatter.ravel(),
    'path_loss_no_scatter_db': pl_no_scatter.ravel(),
})
_cm_csv = os.path.join(OUT_DIR, f'coverage_map_grid_{FREQ_TAG}.csv')
_df_cm.to_csv(_cm_csv, index=False)
print(f'\nCoverage grid CSV  -> {_cm_csv}  ({len(_df_cm):,} cells)')

# ── Interpolate map onto receiver positions (nearest cell) ───────────────────
try:
    from scipy.spatial import KDTree
    _grid_pts = np.column_stack([_XX.ravel(), _YY.ravel()])
    _tree     = KDTree(_grid_pts)
    _rx_list  = list(receivers)
    _rx_xy    = np.array([[_safe_cm(rx.position[0]), _safe_cm(rx.position[1])] for rx in _rx_list])
    _, _idx   = _tree.query(_rx_xy)
    _df_rx_cm = pd.DataFrame({
        'name':                 [rx.name for rx in _rx_list],
        'x_m':                  _rx_xy[:, 0],
        'y_m':                  _rx_xy[:, 1],
        'rssi_scatter_dbm':     rssi_scatter.ravel()[_idx],
        'rssi_no_scatter_dbm':  rssi_no_scatter.ravel()[_idx],
        'path_loss_scatter_db': pl_scatter.ravel()[_idx],
    })
    _rx_cm_csv = os.path.join(OUT_DIR, f'coverage_map_at_rx_{FREQ_TAG}.csv')
    _df_rx_cm.to_csv(_rx_cm_csv, index=False)
    print(f'Coverage @ RX CSV  -> {_rx_cm_csv}  ({len(_df_rx_cm):,} receivers)')
except Exception as _e:
    print(f'  [WARN] RX interpolation skipped: {_e}')

# ── Derived maps ─────────────────────────────────────────────────────────────
# SINR = RSSI - thermal_noise_floor  (single-TX scene → interference = 0)
_THERMAL_DBM = -174.0 + 10*np.log10(200e3) + 7.0   # kTB: 200 kHz BW, 7 dB NF ≈ -114 dBm
_sinr_scatter    = rssi_scatter    - _THERMAL_DBM    # dB (positive = above noise)
_sinr_no_scatter = rssi_no_scatter - _THERMAL_DBM
_sinr_scatter    = np.where(rssi_scatter    > NOISE_FLOOR_DBM, _sinr_scatter,    np.nan)
_sinr_no_scatter = np.where(rssi_no_scatter > NOISE_FLOOR_DBM, _sinr_no_scatter, np.nan)

# Scatter difference: scatter ON − scatter OFF  (only where both covered)
_scat_diff = np.where(
    (rssi_scatter > NOISE_FLOOR_DBM) & (rssi_no_scatter > NOISE_FLOOR_DBM),
    rssi_scatter - rssi_no_scatter,
    np.nan)

# ── Plot: 2 rows × 3 cols ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
_ext    = [gx_min, gx_max, gy_min, gy_max]
_rx_all = list(receivers) if 'receivers' in dir() else []
_step   = max(1, len(_rx_all) // 1500)
_rx_x   = [_safe_cm(r.position[0]) for r in _rx_all[::_step]]
_rx_y   = [_safe_cm(r.position[1]) for r in _rx_all[::_step]]
_tx0    = list(scene.transmitters.values())[0] if scene.transmitters else None

def _overlay(ax):
    if _rx_x:
        ax.scatter(_rx_x, _rx_y, s=5, c='white', alpha=0.45,
                   linewidths=0, zorder=8, label=f'RX')
    if _tx0:
        ax.scatter([_safe_cm(_tx0.position[0])], [_safe_cm(_tx0.position[1])],
                   marker='*', s=350, c='gold', edgecolors='black',
                   linewidths=0.7, zorder=10, label='TX')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(loc='upper right', fontsize=7)

# Row 0: RSSI scatter ON | RSSI scatter OFF | Scatter difference
_panels_r0 = [
    (axes[0,0], rssi_scatter,    'jet',      NOISE_FLOOR_DBM, -40,  'RSSI — scatter ON (dBm)'),
    (axes[0,1], rssi_no_scatter, 'jet',      NOISE_FLOOR_DBM, -40,  'RSSI — scatter OFF (dBm)'),
    (axes[0,2], _scat_diff,      'RdBu',     -15,              15,  'Scatter impact ON−OFF (dB)'),
]
_cbars_r0 = ['RSSI (dBm)', 'RSSI (dBm)', 'ΔdB (ON − OFF)']

# Row 1: Path loss scatter ON | Path loss scatter OFF | SINR scatter ON
_pl_vmin = 60; _pl_vmax = 160
_panels_r1 = [
    (axes[1,0], pl_scatter,       'jet_r',   _pl_vmin, _pl_vmax, 'Path Loss — scatter ON (dB)'),
    (axes[1,1], pl_no_scatter,    'jet_r',   _pl_vmin, _pl_vmax, 'Path Loss — scatter OFF (dB)'),
    (axes[1,2], _sinr_scatter,    'RdYlGn',  -10,       30,      'SINR — scatter ON (dB)'),
]
_cbars_r1 = ['PL (dB)', 'PL (dB)', 'SINR (dB)']

for (ax, data, cmap, vmin, vmax, title), cbar_lbl in zip(_panels_r0+_panels_r1,
                                                          _cbars_r0+_cbars_r1):
    im = ax.imshow(data, origin='lower', extent=_ext, cmap=cmap,
                   aspect='auto', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=cbar_lbl)
    ax.set_title(title, fontsize=10)
    _overlay(ax)

plt.suptitle(
    f'Coverage Map — {FREQUENCY_HZ/1e9:.3f} GHz  |  {nx}×{ny} grid  ({GRID_SIZE_M} m res)',
    fontsize=13)
plt.tight_layout()
_cm_png = os.path.join(OUT_DIR, f'coverage_map_preview_{FREQ_TAG}.png')
plt.savefig(_cm_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Coverage preview   -> {_cm_png}')

# ── Print SINR stats ──────────────────────────────────────────────────────────
_sv = _sinr_scatter[~np.isnan(_sinr_scatter)]
if len(_sv):
    print(f'\nSINR (scatter ON, thermal NF={_THERMAL_DBM:.1f} dBm):')
    print(f'  mean={_sv.mean():.1f}  std={_sv.std():.1f}  '
          f'min={_sv.min():.1f}  max={_sv.max():.1f} dB')
    print(f'  SINR>0 dB : {((_sv>0).sum()/len(_sv)*100):.1f}%  '
          f'SINR>10 dB: {((_sv>10).sum()/len(_sv)*100):.1f}%')

# add SINR + scatter diff to grid CSV
_df_cm['sinr_scatter_db']    = _sinr_scatter.ravel()
_df_cm['scatter_diff_db']    = _scat_diff.ravel()
_df_cm.to_csv(_cm_csv, index=False)
print(f'Coverage grid CSV  -> {_cm_csv}  (updated with SINR + scatter diff)')


## CELL 9b — Scatter Effect Analysis

In [ ]:
# ====================================================================
# CELL 9b — SCATTER EFFECT ANALYSIS (Maximum Scatter Contribution)
# Shows where diffuse scattering adds the most signal vs scatter OFF
# ====================================================================
import numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec

print("=" * 70)
print("CELL 9b — SCATTER EFFECT: Maximum Contribution Analysis")
print("=" * 70)

# ── Compute scatter delta grid ────────────────────────────────────────────────
_delta = rssi_scatter - rssi_no_scatter          # ON - OFF per grid cell
_valid = np.isfinite(_delta) & (rssi_scatter > NOISE_FLOOR_DBM) & (rssi_no_scatter > NOISE_FLOOR_DBM)
_d     = _delta[_valid]

print(f"Valid cells for scatter comparison : {_valid.sum():,}")
print(f"Scatter delta  min  : {_d.min():.2f} dB")
print(f"Scatter delta  max  : {_d.max():.2f} dB")
print(f"Scatter delta  mean : {_d.mean():.2f} dB")
print(f"Scatter delta  std  : {_d.std():.2f} dB")
print(f"Cells with >2 dB gain from scatter : {(_d > 2).sum():,}  ({100*(_d>2).sum()/max(len(_d),1):.1f}%)")
print(f"Cells with >5 dB gain from scatter : {(_d > 5).sum():,}  ({100*(_d>5).sum()/max(len(_d),1):.1f}%)")
print(f"Cells with >10 dB gain from scatter: {(_d >10).sum():,}  ({100*(_d>10).sum()/max(len(_d),1):.1f}%)")

# ── Top-20 grid cells with maximum scatter contribution ───────────────────────
_flat_idx = np.where(_valid.ravel())[0]
_top20_idx = _flat_idx[np.argsort(_d)[-20:][::-1]]
_xx = np.linspace(gx_min, gx_max, rssi_scatter.shape[1])
_yy = np.linspace(gy_min, gy_max, rssi_scatter.shape[0])
print("\nTop 20 grid cells — maximum scatter gain:")
print(f"  {'X(m)':>8}  {'Y(m)':>8}  {'dist(m)':>8}  {'RSSI_ON':>9}  {'RSSI_OFF':>9}  {'delta':>7}")
for _fi in _top20_idx:
    _iy, _ix = divmod(_fi, rssi_scatter.shape[1])
    _x = _xx[_ix]; _y = _yy[_iy]
    _dist = np.sqrt(_x**2 + _y**2)
    _ron  = rssi_scatter[_iy, _ix]
    _rof  = rssi_no_scatter[_iy, _ix]
    print(f"  {_x:8.0f}  {_y:8.0f}  {_dist:8.0f}  {_ron:9.1f}  {_rof:9.1f}  {_ron-_rof:7.2f}")

# ── Scatter delta at actual RX positions ─────────────────────────────────────
import pandas as pd
_rx_df = pd.read_csv(RX_CSV)
_rx_scatter_delta = []
for _, _row in _rx_df.iterrows():
    _lx, _ly, _ = gps_to_local(_row['lon'], _row['lat'])
    _ix = int(np.clip(round((_lx - gx_min) / GRID_SIZE_M), 0, rssi_scatter.shape[1]-1))
    _iy = int(np.clip(round((_ly - gy_min) / GRID_SIZE_M), 0, rssi_scatter.shape[0]-1))
    _don = rssi_scatter[_iy, _ix]
    _dof = rssi_no_scatter[_iy, _ix]
    _rx_scatter_delta.append(_don - _dof)
_rx_scatter_delta = np.array(_rx_scatter_delta)
_finite_rx = _rx_scatter_delta[np.isfinite(_rx_scatter_delta)]
print(f"\nScatter delta at 1200 RX positions:")
print(f"  mean={np.nanmean(_finite_rx):.2f} dB  std={np.nanstd(_finite_rx):.2f} dB")
print(f"  max={np.nanmax(_finite_rx):.2f} dB  min={np.nanmin(_finite_rx):.2f} dB")
print(f"  RX with >2 dB scatter gain: {(_finite_rx>2).sum()} / {len(_finite_rx)}")

# ── 4-panel figure ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"Scatter Effect Analysis — {FREQUENCY_HZ/1e9:.3f} GHz DEM", fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# Panel 1: Scatter delta map (clipped to [-5, +15] dB)
ax1 = fig.add_subplot(gs[0, 0])
_disp = np.where(_valid, np.clip(_delta, -5, 15), np.nan)
im1 = ax1.imshow(_disp, origin='lower', extent=[gx_min, gx_max, gy_min, gy_max],
                 cmap='RdYlGn', vmin=-5, vmax=15, aspect='auto')
plt.colorbar(im1, ax=ax1, label='Scatter gain (dB)')
ax1.set_title('Scatter ON − OFF (dB)  [clipped −5 to +15]')
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)')
ax1.plot(0, 0, 'y*', ms=14, label='TX'); ax1.legend(fontsize=8)

# Panel 2: Histogram of scatter delta
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(_d, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax2.axvline(0, color='k', lw=1.5, ls='--', label='zero')
ax2.axvline(_d.mean(), color='red', lw=1.5, ls='-', label=f'mean={_d.mean():.2f} dB')
ax2.set_xlabel('Scatter delta (dB)'); ax2.set_ylabel('Cell count')
ax2.set_title('Distribution of Scatter Impact across Grid')
ax2.legend(fontsize=9)

# Panel 3: RSSI scatter ON only where delta > 2 dB (scatter-dominant cells)
ax3 = fig.add_subplot(gs[1, 0])
_mask_high = _valid & (_delta > 2)
_disp3 = np.where(_mask_high, rssi_scatter, np.nan)
im3 = ax3.imshow(_disp3, origin='lower', extent=[gx_min, gx_max, gy_min, gy_max],
                 cmap='jet', vmin=-124, vmax=-40, aspect='auto')
plt.colorbar(im3, ax=ax3, label='RSSI (dBm)')
ax3.set_title(f'RSSI (scatter ON) where delta > 2 dB\n({_mask_high.sum():,} cells = {100*_mask_high.sum()/max(_valid.sum(),1):.1f}% of covered)')
ax3.set_xlabel('X (m)'); ax3.set_ylabel('Y (m)')
ax3.plot(0, 0, 'y*', ms=14)

# Panel 4: Scatter delta at RX positions
_rx_x = []; _rx_y = []
for _, _row in _rx_df.iterrows():
    _lx, _ly, _ = gps_to_local(_row['lon'], _row['lat'])
    _rx_x.append(_lx); _rx_y.append(_ly)
ax4 = fig.add_subplot(gs[1, 1])
_sc = ax4.scatter(_rx_x, _rx_y, c=_rx_scatter_delta,
                  cmap='RdYlGn', vmin=-5, vmax=15, s=8, zorder=3)
plt.colorbar(_sc, ax=ax4, label='Scatter delta at RX (dB)')
ax4.plot(0, 0, 'y*', ms=14, label='TX'); ax4.legend(fontsize=8)
ax4.set_title('Scatter Delta at 1200 RX positions')
ax4.set_xlabel('X (m)'); ax4.set_ylabel('Y (m)')

_png = os.path.join(OUT_DIR, 'cell9b_scatter_analysis.png')
plt.savefig(_png, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved: {_png}")


## CELL 6c — TX / RX Position Map (2D OSM)

Plots all 1200 receivers on a 2D OSM background coloured by **measured RSSI**.
TX marked with a red star. Distance rings at 500 m, 1 km, 2 km, 3 km.

Use to sanity-check receiver placement and identify route coverage.


In [ ]:
# ==================================================================
# CELL 6c — TX/RX POSITION MAP + MEASURED RSSI COLOUR
# ==================================================================
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np, pandas as pd

_safe_v = lambda v: float(v.item()) if hasattr(v,'item') else float(v)

# TX scene position -> GPS
_txo  = list(scene.transmitters.values())[0]
_tx_x = _safe_v(_txo.position[0])
_tx_y = _safe_v(_txo.position[1])

# Load RX positions + measured RSSI
_df_r = pd.read_csv(RX_CSV)
_df_m = pd.read_csv(MEASUREMENT_CSV)
_rcol = [c for c in _df_m.columns if 'measurement' in c.lower() or 'rssi' in c.lower()][0]
_ncol = [c for c in _df_m.columns if 'name' in c.lower() or 'id' in c.lower()][0]
_df_r = _df_r.merge(_df_m[[_ncol, _rcol]], left_on='name', right_on=_ncol, how='left')

# Convert lon/lat -> local scene coords
_rx_x = []; _rx_y = []; _rssi = []
for _, row in _df_r.iterrows():
    lx, ly, _ = gps_to_local(row['lon'], row['lat'])
    _rx_x.append(lx); _rx_y.append(ly)
    _rssi.append(row.get(_rcol, np.nan))
_rx_x = np.array(_rx_x); _rx_y = np.array(_rx_y); _rssi = np.array(_rssi)

# Plot
fig, ax = plt.subplots(figsize=(10, 9))
_sc = ax.scatter(_rx_x, _rx_y, c=_rssi, cmap='RdYlGn', s=8,
                 vmin=-105, vmax=-55, alpha=0.85, label='RX (measured RSSI)')
plt.colorbar(_sc, ax=ax, label='Measured RSSI (dBm)')
ax.scatter([_tx_x], [_tx_y], c='red', s=200, marker='*', zorder=5, label='TX')

# Distance rings
for _r in [500, 1000, 2000, 3000]:
    _theta = np.linspace(0, 2*np.pi, 360)
    ax.plot(_tx_x + _r*np.cos(_theta), _tx_y + _r*np.sin(_theta),
            'k--', lw=0.6, alpha=0.4)
    _lbl = f'{_r//1000}km' if _r >= 1000 else f'{_r}m'
    ax.text(_tx_x + _r*1.02, _tx_y, _lbl, fontsize=7, color='gray')

ax.set_xlabel('Local X (m)'); ax.set_ylabel('Local Y (m)')
ax.set_title(f'TX/RX positions — {SCENARIO_NAME}\n'
             f'{len(_rx_x)} receivers coloured by measured RSSI')
ax.legend(loc='upper right'); ax.set_aspect('equal')
plt.tight_layout()
_fig_path = os.path.join(OUT_DIR, 'txrx_map.png')
plt.savefig(_fig_path, dpi=150)
plt.show()
print(f'Saved: {_fig_path}')


In [ ]:
# ====================================================================
# CELL 5b — EXPORT RECEIVER COORDINATES
# ====================================================================
# Writes a CSV with, for each receiver:
#   id, lat, lon, x_m, y_m, z_m, dist_from_tx_m
# ====================================================================
import numpy as np, pandas as pd, os

print('=' * 60)
print('CELL 5b — EXPORT RECEIVER COORDINATES')
print('=' * 60)

_safe_e = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# TX Cartesian position
_tx_e   = list(scene.transmitters.values())[0]
_tx_xyz = np.array([_safe_e(_tx_e.position[0]),
                    _safe_e(_tx_e.position[1]),
                    _safe_e(_tx_e.position[2])])

# Re-load original lat/lon from receiver_locations CSV (written by CELL 5)
_df_locs = pd.read_csv(RX_CSV)   # columns: name, lon, lat, height

rows = []
for _rx in receivers:
    _x = _safe_e(_rx.position[0])
    _y = _safe_e(_rx.position[1])
    _z = _safe_e(_rx.position[2])
    _d = float(np.linalg.norm([_x - _tx_xyz[0], _y - _tx_xyz[1]]))

    # Match lat/lon from CSV by name
    _match = _df_locs[_df_locs['name'] == _rx.name]
    if len(_match):
        _lat = float(_match.iloc[0]['lat'])
        _lon = float(_match.iloc[0]['lon'])
    else:
        _lat = _lon = float('nan')

    rows.append({
        'id'           : _rx.name,
        'lat'          : round(_lat, 7),
        'lon'          : round(_lon, 7),
        'x_m'          : round(_x, 2),
        'y_m'          : round(_y, 2),
        'z_m'          : round(_z, 2),
        'dist_from_tx_m': round(_d, 1),
    })

df_coords = pd.DataFrame(rows).sort_values('dist_from_tx_m').reset_index(drop=True)

_out = os.path.join(OUT_DIR, 'receiver_coordinates.csv')
df_coords.to_csv(_out, index=False)

print(f'Receivers  : {len(df_coords)}')
print(f'Dist range : {df_coords["dist_from_tx_m"].min():.0f} – {df_coords["dist_from_tx_m"].max():.0f} m')
print(f'Saved      : {_out}')
print()
print(df_coords.head(10).to_string(index=False))


In [ ]:
# ====================================================================
# CELL 5c — TX/RX POSITION VERIFICATION + 2D OSM MAP (first 50 RX)
# ====================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, pandas as pd, os, warnings
from shapely.geometry import Point, Polygon, MultiPolygon

print('=' * 65)
print('CELL 5c — POSITION VERIFICATION + 2D MAP')
print('=' * 65)

_safe_v = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── TX GPS position ───────────────────────────────────────────────────────────
_tx_obj = list(scene.transmitters.values())[0]
_tx_lon_v, _tx_lat_v = local_to_gps(_safe_v(_tx_obj.position[0]),
                                      _safe_v(_tx_obj.position[1]))
print(f'TX  GPS : lon={_tx_lon_v:.6f}  lat={_tx_lat_v:.6f}')

# ── Load first 50 RX from CSV ─────────────────────────────────────────────────
_df_locs = pd.read_csv(RX_CSV)
_first50 = _df_locs.head(50).copy()

# ── Load measured RSSI for first 50 RX ───────────────────────────────────────
_df_meas = pd.read_csv(MEASUREMENT_CSV)
_rssi_col = [c for c in _df_meas.columns if 'measurement' in c.lower()][0]
_name_col = [c for c in _df_meas.columns if 'name' in c.lower()][0]
_rssi_map = dict(zip(_df_meas[_name_col], _df_meas[_rssi_col]))
_first50['rssi_dbm'] = _first50['name'].map(_rssi_map)

# ── Download OSM buildings ─────────────────────────────────────────────────────
print('\nDownloading OSM buildings ...')
_gdf_bld2 = None
try:
    import osmnx as ox
    _ox_ver = tuple(int(x) for x in ox.__version__.split('.')[:2])
    if _ox_ver >= (2, 0):
        _gdf_bld2 = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building': True})
    elif _ox_ver >= (1, 3):
        _gdf_bld2 = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building': True})
    else:
        _gdf_bld2 = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST, west=SCENE_WEST, tags={'building': True})
    if hasattr(_gdf_bld2, 'crs') and _gdf_bld2.crs and str(_gdf_bld2.crs) != 'EPSG:4326':
        _gdf_bld2 = _gdf_bld2.to_crs('EPSG:4326')
    print(f'  {len(_gdf_bld2):,} buildings downloaded')
except Exception as _e:
    print(f'  OSM download failed: {_e}')

# ── Verify TX not inside building ─────────────────────────────────────────────
if _gdf_bld2 is not None:
    _tx_pt = Point(_tx_lon_v, _tx_lat_v)
    _tx_in = any(
        (isinstance(g, Polygon) and g.contains(_tx_pt)) or
        (isinstance(g, MultiPolygon) and any(p.contains(_tx_pt) for p in g.geoms))
        for g in _gdf_bld2.geometry if g is not None and not g.is_empty
    )
    print(f'\n{"⚠  TX INSIDE building!" if _tx_in else "✓  TX not inside any building"}')

    # Check first 50 RX
    _rx_inside = []
    for _, _r in _first50.iterrows():
        _rpt = Point(float(_r['lon']), float(_r['lat']))
        _inside = any(
            (isinstance(g, Polygon) and g.contains(_rpt)) or
            (isinstance(g, MultiPolygon) and any(p.contains(_rpt) for p in g.geoms))
            for g in _gdf_bld2.geometry if g is not None and not g.is_empty
        )
        if _inside:
            _rx_inside.append(_r['name'])
    if _rx_inside:
        print(f'⚠  {len(_rx_inside)}/50 RX inside buildings: {_rx_inside[:5]}')
    else:
        print(f'✓  All 50 RX outside buildings')

# ── 2D MAP ────────────────────────────────────────────────────────────────────
fig2d, ax2d = plt.subplots(figsize=(11, 10), dpi=150)

# Buildings
if _gdf_bld2 is not None:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _gdf_bld2.plot(ax=ax2d, facecolor='#d8d0c4', edgecolor='#999999',
                       linewidth=0.15, alpha=0.85, zorder=2)

ax2d.set_xlim(
    min(_first50['lon'].min(), _tx_lon_v) - 0.005,
    max(_first50['lon'].max(), _tx_lon_v) + 0.005)
ax2d.set_ylim(
    min(_first50['lat'].min(), _tx_lat_v) - 0.003,
    max(_first50['lat'].max(), _tx_lat_v) + 0.003)
ax2d.set_aspect('equal')
ax2d.set_facecolor('#eef2f5')
ax2d.set_xlabel('Longitude', fontsize=10)
ax2d.set_ylabel('Latitude',  fontsize=10)
ax2d.tick_params(labelsize=8)
ax2d.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, zorder=1)

# RX dots coloured by measured RSSI
_lons_p = _first50['lon'].values.astype(float)
_lats_p = _first50['lat'].values.astype(float)
_rssi_p = _first50['rssi_dbm'].values.astype(float)
_vmin_p = float(np.nanpercentile(_rssi_p, 2))
_vmax_p = float(np.nanpercentile(_rssi_p, 98))
_sc2d = ax2d.scatter(_lons_p, _lats_p, c=_rssi_p, s=30,
                     cmap='RdYlGn', vmin=_vmin_p, vmax=_vmax_p,
                     alpha=0.85, linewidths=0.4, edgecolors='grey',
                     zorder=5, label='RX (measured RSSI)')

# Number each RX
for _i, (_idx, _r) in enumerate(_first50.iterrows()):
    ax2d.text(float(_r['lon']) + 0.0002, float(_r['lat']) + 0.0001,
              str(_i+1), fontsize=5.5, color='#222222', zorder=7)

_cb2d = fig2d.colorbar(_sc2d, ax=ax2d, fraction=0.025, pad=0.01, shrink=0.7)
_cb2d.set_label('Measured RSSI (dBm)', fontsize=9)
_cb2d.ax.tick_params(labelsize=8)

# TX star
ax2d.plot(_tx_lon_v, _tx_lat_v, marker='*', markersize=18, color='red',
          markeredgecolor='darkred', markeredgewidth=0.8,
          zorder=10, label='TX (transmitter)')
ax2d.annotate('TX', (_tx_lon_v, _tx_lat_v),
              textcoords='offset points', xytext=(8, 5),
              fontsize=9, color='darkred', fontweight='bold', zorder=11)

_n_bld = len(_gdf_bld2) if _gdf_bld2 is not None else 0
ax2d.set_title(
    f'Stevenage 2695 MHz — OSM Map: TX + First 50 RX\n'
    f'{_n_bld:,} buildings  |  50 RX coloured by measured RSSI  |  '
    f'RSSI {_vmin_p:.0f}–{_vmax_p:.0f} dBm',
    fontsize=11)
ax2d.legend(loc='upper right', fontsize=9, markerscale=1.5,
            framealpha=0.85, edgecolor='grey')

plt.tight_layout()
_map_out = os.path.join(OUT_DIR, 'tx_rx50_map.png')
plt.savefig(_map_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nMap saved: {_map_out}')


In [ ]:
# ====================================================================
# CELL 5d — PER-RECEIVER LOS & BUILDING-INTERIOR CHECK
# ====================================================================
# For each receiver:
#   inside_building : receiver lat/lon is inside an OSM building polygon
#   is_los          : the 2-D line TX→RX does NOT cross any building polygon
#   n_bldgs_crossed : how many building footprints the TX→RX line intersects
# Output saved to results/receiver_los_status.csv
# ====================================================================

import os, math
import numpy as np
import pandas as pd
import osmnx as ox
from shapely.geometry import Point, LineString
from shapely.ops import unary_union
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

os.makedirs(OUT_DIR, exist_ok=True)

# ── 1. Load receiver coordinates (generated by CELL 5b) ─────────────
rx_coord_csv = os.path.join(OUT_DIR, 'receiver_coordinates.csv')
if not os.path.exists(rx_coord_csv):
    raise FileNotFoundError(
        "Run CELL 5b first to generate results/receiver_coordinates.csv")

rx_df = pd.read_csv(rx_coord_csv)
print(f"Loaded {len(rx_df)} receivers from {rx_coord_csv}")

# ── 2. TX position ───────────────────────────────────────────────────
tx_lat = TX_LAT
tx_lon = TX_LON
tx_pt  = Point(tx_lon, tx_lat)   # (lon, lat) — shapely convention

# ── 3. Download OSM buildings ────────────────────────────────────────
# Bounding box that covers all receivers + TX with 200 m margin
all_lats = list(rx_df['lat']) + [tx_lat]
all_lons = list(rx_df['lon']) + [tx_lon]
margin = 0.005   # ~500 m in degrees

bbox = (min(all_lons) - margin, min(all_lats) - margin,
        max(all_lons) + margin, max(all_lats) + margin)  # (west, south, east, north)

_w = min(all_lons) - margin
_s = min(all_lats) - margin
_e = max(all_lons) + margin
_n = max(all_lats) + margin

print("Downloading OSM building footprints … ", end='', flush=True)
gdf_bldg = None
poly_bldg = []
try:
    _ox_ver = tuple(int(x) for x in ox.__version__.split('.')[:2])
    if _ox_ver >= (2, 0):
        gdf_bldg = ox.features_from_bbox(bbox=(_w, _s, _e, _n), tags={'building': True})
    elif _ox_ver >= (1, 3):
        gdf_bldg = ox.features_from_bbox(bbox=(_n, _s, _e, _w), tags={'building': True})
    else:
        gdf_bldg = ox.features_from_bbox(north=_n, south=_s, east=_e, west=_w,
                                         tags={'building': True})
    if hasattr(gdf_bldg, 'crs') and gdf_bldg.crs and str(gdf_bldg.crs) != 'EPSG:4326':
        gdf_bldg = gdf_bldg.to_crs('EPSG:4326')
    poly_bldg = [g for g in gdf_bldg.geometry if g.geom_type in ('Polygon','MultiPolygon')]
    print(f"done — {len(poly_bldg)} polygons")
except Exception as e:
    print(f"WARNING: could not fetch OSM buildings: {e}")
    poly_bldg = []

all_buildings_union = unary_union(poly_bldg) if poly_bldg else None

# ── 4. Per-receiver checks ───────────────────────────────────────────
records = []
for _, row in rx_df.iterrows():
    rx_pt  = Point(row['lon'], row['lat'])
    tx_rx_line = LineString([(tx_lon, tx_lat), (row['lon'], row['lat'])])

    inside_building = False
    is_los          = True
    n_bldgs_crossed = 0

    if all_buildings_union is not None:
        # Inside-building check
        inside_building = bool(rx_pt.within(all_buildings_union))

        # LOS check: count how many individual building polys the line crosses
        for poly in poly_bldg:
            try:
                if tx_rx_line.crosses(poly) or tx_rx_line.within(poly):
                    n_bldgs_crossed += 1
            except Exception:
                pass
        is_los = (n_bldgs_crossed == 0)

    records.append({
        'id':              row['id'],
        'lat':             row['lat'],
        'lon':             row['lon'],
        'dist_m':          row['dist_from_tx_m'],
        'inside_building': inside_building,
        'is_los':          is_los,
        'n_bldgs_crossed': n_bldgs_crossed,
    })

out_df = pd.DataFrame(records).sort_values('dist_m').reset_index(drop=True)

# ── 5. Summary ───────────────────────────────────────────────────────
n_inside = out_df['inside_building'].sum()
n_los    = out_df['is_los'].sum()
n_nlos   = (~out_df['is_los']).sum()
print(f"\n{'─'*50}")
print(f"Total receivers    : {len(out_df)}")
print(f"Inside building    : {n_inside}  ({100*n_inside/len(out_df):.1f}%)")
print(f"Clear LOS          : {n_los}   ({100*n_los/len(out_df):.1f}%)")
print(f"NLOS (obstructed)  : {n_nlos}  ({100*n_nlos/len(out_df):.1f}%)")
print(f"{'─'*50}")

# ── 6. Save CSV ──────────────────────────────────────────────────────
out_path = os.path.join(OUT_DIR, 'receiver_los_status.csv')
out_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(out_df[['id','dist_m','inside_building','is_los','n_bldgs_crossed']].head(20).to_string(index=False))


## Cell A — paths.a Normalization Diagnostic

Compares `sum(|paths.a|²)` to theoretical FSPL at 3 distances. Reveals if Sionna 2.0 PathSolver has different amplitude normalization than expected. Run after Cell 6 (receivers placed), before DIAG.

In [ ]:
# ====================================================================
# CELL 5f — TRUE 3D RAY-CAST LOS CHECK  (Sionna PathSolver)
# ====================================================================
# Unlike CELL 5d (2D footprint check), this fires the actual ray tracer
# with ONLY the direct path enabled. A receiver is LOS if and only if
# the direct TX->RX ray is unobstructed in full 3D — accounting for
# building HEIGHTS and terrain elevation.
#   los_3d = True   → direct ray reaches RX (clear line of sight)
#   los_3d = False  → direct ray blocked by a building/terrain (NLOS)
# Output: results/receiver_los_3d.csv
# Run AFTER CELL 6 (receivers placed) and CELL 3 (scene loaded).
# ====================================================================
import os, time, gc
import numpy as np, pandas as pd
from sionna.rt import PathSolver

print('=' * 60)
print('CELL 5f — TRUE 3D RAY-CAST LOS CHECK')
print('=' * 60)

_safe_l = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

_txl   = list(scene.transmitters.values())[0]
_tx_xy = np.array([_safe_l(_txl.position[0]), _safe_l(_txl.position[1])])

# LOS-only solver config — no reflections, no diffraction, no scattering
_LOS_CFG = dict(
    max_depth           = 0,      # 0 bounces → direct path only
    los                 = True,
    specular_reflection = False,
    diffraction         = False,
    diffuse_reflection  = False,
    synthetic_array     = False,
)

_solver = PathSolver()
_records = []
_BS = max(1, BATCH_SIZE)
_all = list(receivers)
_total = len(_all)
_t0 = time.time()

print(f'Testing {_total} receivers (batch={_BS}) — direct ray only ...')

for _i in range(0, _total, _BS):
    _batch = _all[_i:_i + _BS]
    for _nm in list(scene.receivers.keys()):
        scene.remove(_nm)
    for _rx in _batch:
        scene.add(_rx)

    try:
        _paths = _solver(scene, **_LOS_CFG)
        _a = _paths.a
        if isinstance(_a, tuple):
            _ar = _a[0].numpy() if hasattr(_a[0], 'numpy') else np.array(_a[0])
            _ai = _a[1].numpy() if hasattr(_a[1], 'numpy') else np.array(_a[1])
            _amp = np.abs(_ar + 1j * _ai)
        else:
            _an = _a.numpy() if hasattr(_a, 'numpy') else np.array(_a)
            _amp = np.abs(_an)
        # power per RX = sum over all path-dims except the RX axis (axis 0)
        _pwr = (_amp ** 2)
        _per_rx = _pwr.reshape(_pwr.shape[0], -1).sum(axis=1)
    except Exception as _e:
        print(f'  batch {_i}: solver error {_e}')
        _per_rx = np.zeros(len(_batch))

    for _j, _rx in enumerate(_batch):
        _x = _safe_l(_rx.position[0]); _y = _safe_l(_rx.position[1])
        _d = float(np.linalg.norm([_x - _tx_xy[0], _y - _tx_xy[1]]))
        _has_los = bool(_per_rx[_j] > 1e-30) if _j < len(_per_rx) else False
        _records.append({
            'id':     _rx.name,
            'x_m':    round(_x, 2),
            'y_m':    round(_y, 2),
            'dist_m': round(_d, 1),
            'los_3d': _has_los,
        })

    if (_i // _BS) % 20 == 0 and _i > 0:
        print(f'  {_i}/{_total}  ({time.time()-_t0:.0f}s)')
    gc.collect()

_los_df = pd.DataFrame(_records).sort_values('dist_m').reset_index(drop=True)

# ── Summary ──────────────────────────────────────────────────────────
_n_los  = _los_df['los_3d'].sum()
_n_nlos = (~_los_df['los_3d']).sum()
print(f"\n{'─'*50}")
print(f"Total receivers : {len(_los_df)}")
print(f"3D LOS (clear)  : {_n_los}   ({100*_n_los/len(_los_df):.1f}%)")
print(f"3D NLOS (blocked): {_n_nlos}  ({100*_n_nlos/len(_los_df):.1f}%)")
print(f"{'─'*50}")

_out = os.path.join(OUT_DIR, 'receiver_los_3d.csv')
_los_df.to_csv(_out, index=False)
print(f"\nSaved: {_out}")
print(_los_df.head(20).to_string(index=False))


In [ ]:
# ====================================================================
# CELL 5g — TOP-DOWN RAY CAST: IS RX INSIDE / UNDER A BUILDING (3D)
# ====================================================================
# Fires a vertical ray straight UP from each receiver against the real
# building meshes (brick/concrete/glass/metal/wood). If the ray hits a
# building before the sky, the RX is under a building roof → inside.
# Uses the actual 3D scene geometry (heights + terrain), not 2D footprints.
#   inside_building : ray from RX upward hits a building
#   roof_height_m   : z of the first building hit above RX (NaN if none)
# Output: results/receiver_inside_building_3d.csv
# Run AFTER CELL 16 (receivers placed). Needs trimesh.
# ====================================================================
import os, glob
import numpy as np, pandas as pd

try:
    import trimesh
except ImportError:
    os.system('pip install trimesh -q')
    import trimesh
try:
    import rtree
except ImportError:
    os.system('pip install rtree -q')
    import rtree

print('=' * 60)
print('CELL 5g — TOP-DOWN RAY CAST (RX inside building?)')
print('=' * 60)

_safe_g = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── 1. Load building meshes only (exclude terrain/road/water) ────────
_merged_dir = os.path.join(SCENE_DIR, 'meshes')
# Scene builder writes: bld_itu_brick.ply, bld_itu_concrete.ply, etc.
_BUILDING_MATS = ['itu_brick', 'itu_concrete', 'itu_glass', 'itu_metal', 'itu_wood']

_meshes = []
for _m in _BUILDING_MATS:
    # Try both naming conventions: bld_<mat>.ply and mat-<mat>.ply
    for _prefix in ('bld_', 'mat-'):
        _p = os.path.join(_merged_dir, f'{_prefix}{_m}.ply')
        if os.path.exists(_p):
            break
    if os.path.exists(_p):
        try:
            _mesh = trimesh.load(_p, process=False)
            if isinstance(_mesh, trimesh.Trimesh) and len(_mesh.faces) > 0:
                _meshes.append(_mesh)
                print(f'  loaded {_m}.ply  ({len(_mesh.faces):,} faces)')
        except Exception as _e:
            print(f'  [skip] {_m}: {_e}')

if not _meshes:
    raise FileNotFoundError(f'No building meshes found in {_merged_dir}')

_buildings = trimesh.util.concatenate(_meshes)
print(f'Combined building mesh: {len(_buildings.faces):,} faces')

# Fast ray engine if available
try:
    _rmi = _buildings.ray  # uses pyembree/rtree if installed
except Exception:
    _rmi = _buildings.ray

# ── 2. RX positions (scene-local, from placed receivers) ─────────────
_origins, _names, _xy = [], [], []
for _rx in receivers:
    _x = _safe_g(_rx.position[0]); _y = _safe_g(_rx.position[1]); _z = _safe_g(_rx.position[2])
    _origins.append([_x, _y, _z + 0.1])   # start just above RX
    _names.append(_rx.name)
    _xy.append((_x, _y))
_origins = np.array(_origins, dtype=np.float64)
_dirs = np.tile([0.0, 0.0, 1.0], (len(_origins), 1))   # straight up

print(f'Casting {len(_origins)} vertical rays upward ...')

# ── 3. Ray cast: first intersection above each RX ────────────────────
_locs, _idx_ray, _idx_tri = _rmi.intersects_location(
    ray_origins=_origins, ray_directions=_dirs, multiple_hits=False)

_inside = np.zeros(len(_origins), dtype=bool)
_roofz  = np.full(len(_origins), np.nan)
for _k, _ri in enumerate(_idx_ray):
    _inside[_ri] = True
    _roofz[_ri]  = _locs[_k][2]

# ── 4. TX distance + assemble ────────────────────────────────────────
_txg = list(scene.transmitters.values())[0]
_txxy = np.array([_safe_g(_txg.position[0]), _safe_g(_txg.position[1])])
_rows = []
for _i, _nm in enumerate(_names):
    _d = float(np.linalg.norm([_xy[_i][0]-_txxy[0], _xy[_i][1]-_txxy[1]]))
    _rows.append({
        'id': _nm,
        'x_m': round(_xy[_i][0], 2),
        'y_m': round(_xy[_i][1], 2),
        'dist_m': round(_d, 1),
        'inside_building': bool(_inside[_i]),
        'roof_height_m': round(float(_roofz[_i]), 2) if not np.isnan(_roofz[_i]) else np.nan,
    })
_df = pd.DataFrame(_rows).sort_values('dist_m').reset_index(drop=True)

_n_in = _df['inside_building'].sum()
print(f"\n{'─'*50}")
print(f"Total receivers   : {len(_df)}")
print(f"Inside building   : {_n_in}  ({100*_n_in/len(_df):.1f}%)")
print(f"Clear (open sky)  : {len(_df)-_n_in}  ({100*(len(_df)-_n_in)/len(_df):.1f}%)")
print(f"{'─'*50}")

_out = os.path.join(OUT_DIR, 'receiver_inside_building_3d.csv')
_df.to_csv(_out, index=False)
print(f"\nSaved: {_out}")
print(_df.head(20).to_string(index=False))


## CELL 5h — RX Filter: Exclude Inside-Building + Beyond MAX_RX_DIST_M

Reads `receiver_inside_building_3d.csv` (from CELL 5g) and builds `_rx_exclude` —
a set of RX IDs that will be dropped from CELL 8 analysis.

Two filters:
- **Inside building**: GPS point falls inside a building mesh (signal must penetrate walls).
- **Beyond MAX_RX_DIST_M**: > 4 km from TX — extreme range where nDSM/OSM coverage degrades.


In [ ]:
# ====================================================================
# CELL 5h — RX FILTER: EXCLUDE INSIDE-BUILDING RECEIVERS
# ====================================================================
# Reads receiver_inside_building_3d.csv (from CELL 5g) and builds
# _rx_exclude — a set of RX IDs that CELL 6 will skip entirely.
# Receivers inside buildings have GPS inside a building mesh: the RT
# model cannot correctly simulate indoor propagation, so they are
# excluded from placement and all downstream analysis.
# ====================================================================
import os, pandas as pd

_csv_5g = os.path.join(OUT_DIR, 'receiver_inside_building_3d.csv')

_rx_exclude = set()   # populated below; CELL 6 reads this

if not os.path.exists(_csv_5g):
    print('WARNING: receiver_inside_building_3d.csv not found — run CELL 5g first.')
    print('         Proceeding with all receivers (no exclusions).')
else:
    _df_filter = pd.read_csv(_csv_5g)
    _inside    = _df_filter[_df_filter['inside_building'] == True]
    _rx_exclude = set(_inside['id'].astype(str))

    _n_bld   = len(_rx_exclude)
    _n_total = len(_df_filter)
    _n_keep  = _n_total - _n_bld

    print('=' * 60)
    print('CELL 5h — RX FILTER')
    print('=' * 60)
    print(f'Total receivers          : {_n_total}')
    print(f'Inside building (exclude): {_n_bld}  ({100*_n_bld/_n_total:.1f}%)')
    print(f'Kept for simulation      : {_n_keep}  ({100*_n_keep/_n_total:.1f}%)')
    print()
    print(f'_rx_exclude set built: {len(_rx_exclude)} IDs  — CELL 6 will skip these.')
    if _n_bld > 0:
        print('First excluded:', sorted(_rx_exclude)[:5])


In [ ]:
# ====================================================================
# CELL 5e — RECEIVER ROUTE DIAGNOSTIC
# ====================================================================
# Checks: unique locations, repeated passes, distance histogram
# ====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math, os

print('=' * 60)
print('CELL 5e — RECEIVER ROUTE DIAGNOSTIC')
print('=' * 60)

df = pd.read_csv(RX_CSV)
df_m = pd.read_csv(MEASUREMENT_CSV)
df = df.merge(df_m[['name','local_measurement_dBm']], on='name', how='left')

# ── Distance from TX ──────────────────────────────────────────────────
_dlon = 111000 * math.cos(math.radians(TX_LAT))
_dlat = 111000
df['dist_m'] = np.sqrt(
    ((df['lat'] - TX_LAT) * _dlat)**2 +
    ((df['lon'] - TX_LON) * _dlon)**2
)

# ── Unique location check (4 decimal places ≈ 11m grid) ───────────────
df['lat_r'] = df['lat'].round(4)
df['lon_r'] = df['lon'].round(4)
n_unique = df.groupby(['lat_r','lon_r']).ngroups
n_total  = len(df)
n_dupes  = n_total - n_unique

print(f'Total receivers  : {n_total}')
print(f'Unique locations : {n_unique}  (rounded to 4 dp ≈ 11m)')
print(f'Duplicates       : {n_dupes}  ({100*n_dupes/n_total:.1f}%)')

# ── Distance bands ────────────────────────────────────────────────────
bands = [0,100,200,300,500,750,1000,1250,1500,2000,2500,3000,99999]
labels = ['0-100','100-200','200-300','300-500','500-750',
          '750-1k','1k-1.25k','1.25k-1.5k','1.5k-2k','2k-2.5k','2.5k-3k','>3k']
print(f'\n{"Band (m)":<12} {"N":>5} {"Mean RSSI":>10} {"Std RSSI":>9}')
print('-' * 40)
for i in range(len(labels)):
    mask = (df['dist_m'] >= bands[i]) & (df['dist_m'] < bands[i+1])
    sub  = df[mask]
    if len(sub) == 0: continue
    print(f'{labels[i]:<12} {len(sub):>5} {sub["local_measurement_dBm"].mean():>10.1f} {sub["local_measurement_dBm"].std():>9.1f}')

# ── Plot: distance vs RSSI + route order ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: RSSI vs distance coloured by CSV row order
sc = ax1.scatter(df['dist_m'], df['local_measurement_dBm'],
                 c=df.index, cmap='plasma', s=8, alpha=0.7)
plt.colorbar(sc, ax=ax1, label='CSV row (time order)')
ax1.set_xlabel('Distance from TX (m)')
ax1.set_ylabel('Measured RSSI (dBm)')
ax1.set_title('RSSI vs Distance — coloured by time order')
ax1.grid(True, alpha=0.3)

# Right: route map (lat/lon coloured by row order)
sc2 = ax2.scatter(df['lon'], df['lat'],
                  c=df.index, cmap='plasma', s=6, alpha=0.7)
plt.colorbar(sc2, ax=ax2, label='CSV row (time order)')
ax2.plot(TX_LON, TX_LAT, 'r*', markersize=15, label='TX')
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.set_title('Measurement route (purple=start → yellow=end)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(OUT_DIR, 'route_diagnostic.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'\nSaved: {out}')


In [ ]:
# ====================================================================
# CELL A — paths.a NORMALIZATION DIAGNOSTIC
# ====================================================================
# Runs a pure LOS path solve at 3 known distances, prints paths.a.shape
# and compares sum(|a|²) to FSPL — reveals any normalization offset.
# Run after Cell 4 (TX placed) and Cell 6 (RX list loaded).
# ====================================================================
import numpy as np, math, time
print("=" * 70)
print("CELL A — paths.a NORMALIZATION CHECK (LOS comparison to FSPL)")
print("=" * 70)

C = 3e8
_lam = C / FREQUENCY_HZ  # wavelength

def _fspl_linear(d):
    """Free-space path gain (linear) = (λ/4πr)²"""
    return (_lam / (4 * math.pi * d)) ** 2

def _fspl_db(d):
    return -10 * math.log10(_fspl_linear(d))

tx_obj = list(scene.transmitters.values())[0]
_tx_lx = _safe(tx_obj.position[0])
_tx_ly = _safe(tx_obj.position[1])
_tx_lz = _safe(tx_obj.position[2])

print(f"\nWavelength     : {_lam:.4f} m")
print(f"TX position    : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"RX AGL         : {RX_AGL_M} m")
print()
print(f"  {'Dist':>6}  {'FSPL(dB)':>9}  {'a.shape(raw)':>22}  {'sum|a|²(dB)':>12}  {'vs FSPL':>9}  {'paths':>6}  {'time(s)':>7}")
print(f"  {'-'*90}")

_test_dists = [50, 200, 500, 1000, 2000]

for _test_d in _test_dists:
    _rx_lx = _tx_lx + _test_d  # due East
    _rx_lz = RX_AGL_M
    _rx_test = Receiver(name='_debug_rx', position=[_rx_lx, _tx_ly, _rx_lz])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx_test)

    _t0 = time.time()
    try:
        _paths = PathSolver()(scene,
            max_depth=MAX_DEPTH,
            los=True,
            specular_reflection=True,
            diffraction=True,
            edge_diffraction=True,
            diffuse_reflection=True,
            samples_per_src=2_000_000)
        _dt = time.time() - _t0

        # Combine real/imag tuple → complex array
        _a_raw = _paths.a
        if isinstance(_a_raw, tuple):
            _raw_shape = ('tuple', tuple(_a_raw[0].shape))
            _a_np = ((_a_raw[0].numpy() if hasattr(_a_raw[0], 'numpy') else np.array(_a_raw[0])) +
                     1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], 'numpy') else np.array(_a_raw[1])))
        else:
            _raw_shape = tuple(_a_raw.shape)
            _a_np = _a_raw.numpy() if hasattr(_a_raw, 'numpy') else np.array(_a_raw)

        # Squeeze
        _a_sq = np.squeeze(_a_np)

        # Print shapes for debug
        _sum_pwr = float(np.sum(np.abs(_a_sq) ** 2))
        _n_valid = int(np.sum(np.abs(_a_sq) ** 2 > 1e-30))
        if _sum_pwr > 1e-30:
            _sum_db = 10 * math.log10(_sum_pwr)
        else:
            _sum_db = float('nan')

        _fspl = _fspl_db(_test_d)
        _vs_fspl = _sum_db - (-_fspl)  # sum_db is negative, FSPL is positive loss
        # correct comparison: sum|a|² (dB) should ≈ -FSPL (negative)
        _vs_fspl2 = _sum_db - (-_fspl)

        print(f"  {_test_d:>6}m  {_fspl:>9.1f}  {str(_raw_shape):>22}  {_sum_db:>12.2f}  {(_sum_db + _fspl):>+9.1f}  {_n_valid:>6}  {_dt:>7.1f}")

    except Exception as _e:
        _dt = time.time() - _t0
        print(f"  {_test_d:>6}m  ERROR: {_e}  ({_dt:.1f}s)")

print()
print("  Expected: sum|a|²(dB) ≈ -(FSPL dB) for open LOS.")
print("  'vs FSPL' = sum|a|²(dB) + FSPL(dB)  (should be ~0 to +10 dB for urban overhead)")
print()
print("  paths.a raw shape legend:")
print("  Sionna 2.0 typical: [num_rx, num_tx, num_rx_ant, num_tx_ant, num_paths]")
print("  or: [batch, num_rx, ...] — check documentation for your version")
print()

# Restore receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
_rxlist_debug = list(receivers) if 'receivers' in dir() else []
for _rx in _rxlist_debug: scene.add(_rx)
print(f"  Receivers restored: {len(_rxlist_debug)}")
print()
print("  ── Top 5 paths for last test distance ──────────────────────────────")
try:
    _rx_test2 = Receiver(name='_debug_rx2', position=[_tx_lx + 200, _tx_ly, RX_AGL_M])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx_test2)
    _paths2 = PathSolver()(scene,
        max_depth=MAX_DEPTH, los=True, specular_reflection=True,
        diffraction=True, edge_diffraction=True, diffuse_reflection=True,
        samples_per_src=2_000_000)
    _a2_raw = _paths2.a
    if isinstance(_a2_raw, tuple):
        _a2 = np.squeeze(
            (_a2_raw[0].numpy() if hasattr(_a2_raw[0], 'numpy') else np.array(_a2_raw[0])) +
            1j*(_a2_raw[1].numpy() if hasattr(_a2_raw[1], 'numpy') else np.array(_a2_raw[1])))
        print(f"  paths.a: tuple(real,imag), real shape={_a2_raw[0].shape}")
    else:
        _a2 = np.squeeze(_a2_raw.numpy() if hasattr(_a2_raw, 'numpy') else np.array(_a2_raw))
    print(f"  paths.a after squeeze shape: {_a2.shape}  dtype: {_a2.dtype}")
    _pwr2 = np.abs(_a2.flatten()) ** 2
    _ord2 = np.argsort(_pwr2)[::-1]
    for _ri in range(min(5, len(_ord2))):
        _pi = _ord2[_ri]
        _pw = _pwr2[_pi]
        if _pw > 1e-40:
            print(f"    rank {_ri+1}: |a|²={_pw:.4e}  ({10*math.log10(_pw):.1f} dB)")
    print(f"  Expected LOS |a|² at 200m = {_fspl_linear(200):.4e}  ({-_fspl_db(200):.1f} dB)")
    # Friis check
    _rssi_check = TX_CONDUCTED_DBM + 10*math.log10(max(_pwr2[_ord2[0]], 1e-40)) + RX_EXTRA_GAIN_DB
    print(f"  RSSI from strongest path: {_rssi_check:.1f} dBm  (expected ~{TX_CONDUCTED_DBM - _fspl_db(200) + RX_EXTRA_GAIN_DB:.1f} dBm for FSPL)")
    # Tau
    if hasattr(_paths2, 'tau'):
        _tau2 = np.squeeze(np.array(_paths2.tau))
        print(f"  paths.tau shape: {_tau2.shape}")
        _tau_flat = _tau2.flatten()
        _valid_tau = _tau_flat[_tau_flat > 0]
        if len(_valid_tau):
            _los_tau = 200 / C
            print(f"  Expected LOS tau: {_los_tau:.6f} s  ({200}m/c)")
            print(f"  Min tau found: {_valid_tau.min():.6f} s  ({_valid_tau.min()*C:.1f}m)")
except Exception as _e2:
    print(f"  ERROR: {_e2}")
    import traceback; traceback.print_exc()
finally:
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _rx in _rxlist_debug: scene.add(_rx)
    print(f"  Receivers restored: {len(_rxlist_debug)}")


## Cell DIAG — Step-by-Step Bias Diagnostic

Run **before the path solver** to verify TX/RX positions, antenna heights, and scene geometry.
Tests 10 → 100 receivers to catch systematic bias early. Uses Sionna 2.0 `compute_paths()` API.

In [ ]:
from sionna.rt import PathSolver
# ====================================================================
# CELL DIAG — Step-by-Step Bias Diagnostic (10 → 100 receivers)
# ====================================================================
# Tests RSSI formula, RX heights, TX position, and geometry systematically.
# Run BEFORE CELL 9b to isolate the source of high RMSE.
# ====================================================================
import numpy as np, pandas as pd, math, os, time
from pyproj import Transformer as _Tr

print("=" * 70)
print("BIAS DIAGNOSTIC — Step-by-step RMSE decomposition")
print("=" * 70)

# ── Load measurements ─────────────────────────────────────────────────────────
_df_meas = pd.read_csv(MEASUREMENT_CSV)
print(f"\nMeasurements loaded: {len(_df_meas)} rows")
print(f"  RSSI range   : {_df_meas['local_measurement_dBm'].min():.1f} → {_df_meas['local_measurement_dBm'].max():.1f} dBm")
print(f"  PL range     : {_df_meas['path_loss_dB'].min():.1f} → {_df_meas['path_loss_dB'].max():.1f} dB")

# ── STEP 1: Formula check — FSPL vs measured at known distances ────────────────
print("\n" + "─" * 60)
print("STEP 1 — Free-Space Path Loss formula validation")
print("─" * 60)
C = 3e8
_f = FREQUENCY_HZ
_fspl_fn = lambda d: 20*np.log10(4*np.pi*d*_f/C)

_tx_lon, _tx_lat = TX_LON, TX_LAT
_gps2utm = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
_tx_x, _tx_y = _gps2utm.transform(_tx_lon, _tx_lat)

_rx_x = np.array([_gps2utm.transform(r['lon'], r['lat'])[0] for _, r in _df_meas.iterrows()])
_rx_y = np.array([_gps2utm.transform(r['lon'], r['lat'])[1] for _, r in _df_meas.iterrows()])
_dist = np.sqrt((_rx_x - _tx_x)**2 + (_rx_y - _tx_y)**2)
_df_meas = _df_meas.copy()
_df_meas['dist_m'] = _dist

# Near receivers (50-300m) — most likely LOS → compare against FSPL
_near_rx = _df_meas[_df_meas['dist_m'].between(50, 300)].copy()
_near_rx['fspl_db']    = _near_rx['dist_m'].apply(_fspl_fn)
_near_rx['measured_pl'] = _near_rx['path_loss_dB']
_near_rx['vs_fspl']    = _near_rx['measured_pl'] - _near_rx['fspl_db']
print(f"  Near receivers (50-300m): {len(_near_rx)}")
print(f"  TX_CONDUCTED={TX_CONDUCTED_DBM:.1f} dBm  RX_EXTRA={RX_EXTRA_GAIN_DB:.1f} dB  SITE_CORR={SITE_CORRECTION_DB:.1f} dB")
print(f"  RSSI formula: RSSI = TX_CONDUCTED - PL + RX_EXTRA + SITE_CORR")
print(f"              = {TX_CONDUCTED_DBM:.1f} - PL + {RX_EXTRA_GAIN_DB:.1f} + {SITE_CORRECTION_DB:.1f}")
print(f"  → PL = {TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB:.1f} - RSSI  (= rssi_from_path_gain inverse)")
print()
print(f"  {'Name':<12} {'Dist(m)':>8} {'RSSI(dBm)':>10} {'PL_meas(dB)':>12} {'FSPL(dB)':>9} {'PL-FSPL(dB)':>12}")
for _, r in _near_rx.head(10).iterrows():
    print(f"  {str(r['name']):<12} {r['dist_m']:>8.0f} {r['local_measurement_dBm']:>10.1f} "
          f"{r['measured_pl']:>12.1f} {r['fspl_db']:>9.1f} {r['vs_fspl']:>12.1f}")
_near_overhead = _near_rx['vs_fspl'].mean()
print(f"\n  Mean PL-FSPL (near): {_near_overhead:+.1f} dB  "
      f"(expected +5 to +15 dB for urban LOS overhead)")
if _near_overhead < 0:
    print(f"  ⚠ Negative overhead → TX_CONDUCTED+RX_EXTRA+SITE_CORR over-estimated")
elif _near_overhead > 25:
    print(f"  ⚠ Very high overhead → TX power under-estimated or RX underground")
else:
    print(f"  ✓ Urban overhead looks physically reasonable")

# ── STEP 2: RX height check ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 2 — Receiver height sanity check")
print("─" * 60)
_rxlist = list(scene.receivers.values())
if not _rxlist and os.path.exists(RX_CSV):
    _df_rx2 = pd.read_csv(RX_CSV)
    for _, _row2 in _df_rx2.iterrows():
        _x2, _y2, _ = gps_to_local(float(_row2['lon']), float(_row2['lat']))
        _lz2 = terrain_z(_x2, _y2) + RX_AGL_M
        _rx2 = Receiver(name=str(_row2['name']), position=[_x2, _y2, _lz2])
        scene.add(_rx2)
    _rxlist = list(scene.receivers.values())
    print(f"  Loaded {len(_rxlist)} receivers from {RX_CSV}")
_heights = [_safe(rx.position[2]) for rx in _rxlist[:20]]
print(f"  First 20 RX heights (local Z, m):")
for rx, h in zip(_rxlist[:20], _heights):
    flag = " ⚠ UNDERGROUND" if h < -60 else (" ⚠ TOO HIGH (check terrain)" if h > 300 else "")
    print(f"    {rx.name:<12}  z={h:+.2f}m{flag}")
_neg = sum(
    1 for rx in _rxlist
    if _safe(rx.position[2]) < terrain_z(
        _safe(rx.position[0]), _safe(rx.position[1])) - 0.5
)
print(f"\n  Total RX underground (z<0): {_neg} / {len(_rxlist)}")

# ── STEP 3: TX position check ─────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 3 — TX position check")
print("─" * 60)
if 'TX_AGL_M' not in dir(): TX_AGL_M = 17.0
_tx = list(scene.transmitters.values())[0]
_tx_lx, _tx_ly, _tx_lz = _safe(_tx.position[0]), _safe(_tx.position[1]), _safe(_tx.position[2])
_tx_glon, _tx_glat = local_to_gps(_tx_lx, _tx_ly)
print(f"  TX local  : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"  TX GPS    : lat={_tx_glat:.6f}  lon={_tx_glon:.6f}")
print(f"  Expected  : lat={TX_LAT:.6f}  lon={TX_LON:.6f}  h={TX_AGL_M:.1f}m")
_lat_err = abs(_tx_glat - TX_LAT) * 111000
_lon_err = abs(_tx_glon - TX_LON) * 111000 * math.cos(math.radians(TX_LAT))
print(f"  Position error: {_lat_err:.1f}m N-S  {_lon_err:.1f}m E-W  Z={_tx_lz:.1f}m (terrain+AGL)")
print(f"  Terrain at TX  : {_tx_lz - TX_AGL_M:.1f}m  AGL={TX_AGL_M:.1f}m  Total={_tx_lz:.1f}m ✓")
print(f"  TX height: AGL={TX_AGL_M:.1f}m  terrain_z={_tx_lz-TX_AGL_M:.1f}m  total={_tx_lz:.1f}m")
if _lat_err > 50 or _lon_err > 50:
    print("  ⚠ TX position error > 50m — check GPS→UTM→local conversion")
else:
    print("  ✓ TX position OK")

# ── STEP 4: Quick path solver on 50 receivers — scatter ON vs OFF ───────────
print("\n" + "─" * 70)
print("STEP 4 — Path solver: 50 RX across bands  (scatter ON vs OFF)")
print("─" * 70)

_gps2utm3 = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
_df_meas2 = _df_meas.copy()
_xy2 = _df_meas2.apply(lambda r: _gps2utm3.transform(float(r["lon"]), float(r["lat"])), axis=1)
_df_meas2["_lx"]  = [xy[0] - utm_center_x for xy in _xy2]
_df_meas2["_ly"]  = [xy[1] - utm_center_y for xy in _xy2]
_df_meas2["_dtx"] = np.sqrt((_df_meas2["_lx"] - _tx_lx)**2 + (_df_meas2["_ly"] - _tx_ly)**2)
_df_meas2 = _df_meas2[_df_meas2["_dtx"] >= 100].sort_values("_dtx").reset_index(drop=True)
_bands_sel = [
    _df_meas2[_df_meas2["_dtx"].between( 100,  300)].head(10),
    _df_meas2[_df_meas2["_dtx"].between( 300,  700)].head(10),
    _df_meas2[_df_meas2["_dtx"].between( 700, 1200)].head(10),
    _df_meas2[_df_meas2["_dtx"].between(1200, 2000)].head(10),
    _df_meas2[_df_meas2["_dtx"] > 2000].head(10),
]
_df_test = pd.concat(_bands_sel).drop_duplicates(subset="name").reset_index(drop=True)
print(f"  Testing {len(_df_test)} receivers  |  sps=10M  |  max_depth={MAX_DEPTH}")
print()

# ── PL_meas read from MEASUREMENT_CSV (path_loss_dB = 56.2 - RSSI) ───────────
# MEASUREMENT_CSV already has path_loss_dB column written by CELL 5
_pl_lookup = dict(zip(_df_meas["name"], _df_meas["path_loss_dB"])) if "path_loss_dB" in _df_meas.columns else {}
print(f"  PL_meas lookup: {len(_pl_lookup)} entries from MEASUREMENT_CSV ({'path_loss_dB' in _df_meas.columns})")

_cfg_on  = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, diffuse_reflection=True,
                samples_per_src=10_000_000)
_cfg_off = {**_cfg_on, "diffuse_reflection": False}

def _extract_rssi(paths_obj):
    _a_raw = paths_obj.a
    if isinstance(_a_raw, tuple):
        _a = (_a_raw[0].numpy() if hasattr(_a_raw[0], "numpy") else np.array(_a_raw[0])) + \
             1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], "numpy") else np.array(_a_raw[1]))
    else:
        _a = _a_raw.numpy() if hasattr(_a_raw, "numpy") else np.array(_a_raw)
    _a = np.squeeze(_a)
    if _a.ndim == 0: _a = _a.reshape(1,1)
    elif _a.ndim == 1: _a = _a[np.newaxis,:]
    elif _a.ndim > 2: _a = _a.reshape(1,-1)
    _pwr = float(np.sum(np.abs(_a[0])**2))
    _np  = int(np.sum(np.abs(_a[0]) > 1e-20))
    if _pwr > 1e-30:
        return rssi_from_path_gain(_pwr), _np
    return float("nan"), 0

_scene_pos = {rx.name: rx for rx in _rxlist}
_results50 = []
_pl_running_sq_errs = []  # running PL squared errors
print(f"  {'RX ID':<14} {'Dist':>7}  {'RSSI_meas':>10} {'RSSI_ON':>9} {'RSSI_OFF':>9}  {'PL_meas':>8} {'PL_sim_ON':>9} {'PL_err_ON':>10}  {'PL_sim_OFF':>10} {'PL_err_OFF':>11}  {'PL_err²':>7} {'RMSE_run':>9}  {'Pths':>5}")
print("  " + "-" * 140)

for _, _mrow in _df_test.iterrows():
    _lx = float(_mrow["_lx"]); _ly = float(_mrow["_ly"]); _d = float(_mrow["_dtx"])
    _rssi_meas = float(_mrow["local_measurement_dBm"])
    _rx_name   = str(_mrow["name"])
    if _rx_name in _scene_pos:
        _rx_z = float(_safe(_scene_pos[_rx_name].position[2]))
    else:
        try:
            _ux4, _uy4 = gps_to_utm.transform(float(_mrow["lon"]), float(_mrow["lat"]))
            _rx_z = get_dem_elevation(_ux4 - utm_center_x, _uy4 - utm_center_y) + RX_AGL_M
        except Exception: _rx_z = RX_AGL_M
    _rx = Receiver(name=_rx_name, position=[_lx, _ly, _rx_z])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx)
    try:
        _rssi_on,  _np_on  = _extract_rssi(PathSolver()(scene, **_cfg_on))
        _rssi_off, _np_off = _extract_rssi(PathSolver()(scene, **_cfg_off))
    except Exception as _e:
        print(f"  {_rx_name:<14} ERROR: {_e}"); continue
    # ── PL metrics — scatter ON ────────────────────────────────────────────
    _pl_meas    = _pl_lookup.get(_rx_name, TX_CONDUCTED_DBM - _rssi_meas)
    _pl_sim_on  = -10.0*math.log10(max(10**(_rssi_on /10)/10**(TX_CONDUCTED_DBM/10),1e-30)) if not math.isnan(_rssi_on)  else float("nan")
    _pl_sim_off = -10.0*math.log10(max(10**(_rssi_off/10)/10**(TX_CONDUCTED_DBM/10),1e-30)) if not math.isnan(_rssi_off) else float("nan")
    _pl_err_on  = _pl_sim_on  - _pl_meas if not math.isnan(_pl_sim_on)  else float("nan")
    _pl_err_off = _pl_sim_off - _pl_meas if not math.isnan(_pl_sim_off) else float("nan")
    _pl_sq_on   = _pl_err_on**2          if not math.isnan(_pl_err_on)  else float("nan")
    if not math.isnan(_pl_sq_on):
        _pl_running_sq_errs.append(_pl_sq_on)
    _pl_rmse_run = math.sqrt(sum(_pl_running_sq_errs)/len(_pl_running_sq_errs)) if _pl_running_sq_errs else float("nan")
    # ── print row ──────────────────────────────────────────────────────────
    _fv = lambda x, fmt: (fmt % x) if not (isinstance(x, float) and math.isnan(x)) else "    N/A"
    print(f"  {_rx_name:<14} {_d:6.0f}m  "
          f"{_rssi_meas:10.1f} {_fv(_rssi_on,'%9.1f')} {_fv(_rssi_off,'%9.1f')}  "
          f"{_pl_meas:8.1f} {_fv(_pl_sim_on,'%8.1f')} {_fv(_pl_err_on,'%+7.2f')}  "
          f"{_fv(_pl_sim_off,'%8.1f')} {_fv(_pl_err_off,'%+7.2f')}  "
          f"{_fv(_pl_sq_on,'%7.2f')} {_fv(_pl_rmse_run,'%9.3f')}  {_np_on:5d}")
    _results50.append({"name": _rx_name, "dist_m": _d, "rssi_meas": _rssi_meas,
                       "rssi_on": _rssi_on, "rssi_off": _rssi_off,
                       "paths_on": _np_on, "paths_off": _np_off,
                       "pl_meas": _pl_meas,
                       "pl_sim_on": _pl_sim_on,  "pl_err_on":  _pl_err_on,
                       "pl_sim_off": _pl_sim_off, "pl_err_off": _pl_err_off,
                       "pl_sq_on": _pl_sq_on, "pl_rmse_run": _pl_rmse_run})

# Re-add all receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
for _rx in _rxlist: scene.add(_rx)

_r50 = pd.DataFrame(_results50)
_r50_on  = _r50.dropna(subset=["pl_meas", "pl_sim_on",  "pl_err_on"])
_r50_off = _r50.dropna(subset=["pl_meas", "pl_sim_off", "pl_err_off"])

def _pl_band_metrics(df, err_col):
    """Compute Bias, MSE, RMSE, STD, R² on PL error column."""
    if len(df) < 2:
        return 0, float("nan"), float("nan"), float("nan"), float("nan"), float("nan")
    e  = df[err_col].values
    pl = df["pl_meas"].values
    n    = len(e)
    bias = float(np.mean(e))
    mse  = float(np.mean(e**2))
    rmse = float(np.sqrt(mse))
    std  = float(np.std(e, ddof=1))
    ss_res = float(np.sum(e**2))
    ss_tot = float(np.sum((pl - np.mean(pl))**2))
    r2   = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return n, bias, mse, rmse, std, r2

_fv2 = lambda x, fmt: (fmt % x) if not (isinstance(x, float) and math.isnan(x)) else "     N/A"
print()
print("  ── Path Loss Band Summary ── Scatter ON vs OFF ───────────────────────────────────────────────────────────")
print(f"  {'Band':<12}  {'N':>4}  {'Scatter ON':^47}  {'Scatter OFF':^47}")
print(f"  {'':<12}  {'':>4}  {'Bias':>8} {'MSE':>8} {'RMSE':>8} {'STD':>8} {'R²':>7}  "
      f"{'Bias':>8} {'MSE':>8} {'RMSE':>8} {'STD':>8} {'R²':>7}")
print("  " + "-" * 110)
_bands_eval = [("<300m",0,300),("300-700m",300,700),("700-1200m",700,1200),("1.2-2km",1200,2000),(">2km",2000,99999)]
for _bname, _bmin, _bmax in _bands_eval:
    _son  = _r50_on [(_r50_on ["dist_m"]>=_bmin) & (_r50_on ["dist_m"]<_bmax)]
    _soff = _r50_off[(_r50_off["dist_m"]>=_bmin) & (_r50_off["dist_m"]<_bmax)]
    nON,  bON,  mON,  rON,  sON,  r2ON  = _pl_band_metrics(_son,  "pl_err_on")
    nOFF, bOFF, mOFF, rOFF, sOFF, r2OFF = _pl_band_metrics(_soff, "pl_err_off")
    print(f"  {_bname:<12}  {nON:4d}  "
          f"{_fv2(bON,'%+8.2f')} {_fv2(mON,'%8.2f')} {_fv2(rON,'%8.3f')} {_fv2(sON,'%8.3f')} {_fv2(r2ON,'%7.4f')}  "
          f"{_fv2(bOFF,'%+8.2f')} {_fv2(mOFF,'%8.2f')} {_fv2(rOFF,'%8.3f')} {_fv2(sOFF,'%8.3f')} {_fv2(r2OFF,'%7.4f')}")
_nON, _bON, _mON, _rON, _sON, _r2ON   = _pl_band_metrics(_r50_on,  "pl_err_on")
_nOFF,_bOFF,_mOFF,_rOFF,_sOFF,_r2OFF  = _pl_band_metrics(_r50_off, "pl_err_off")
print("  " + "-" * 110)
print(f"  {'ALL':<12}  {_nON:4d}  "
      f"{_fv2(_bON,'%+8.2f')} {_fv2(_mON,'%8.2f')} {_fv2(_rON,'%8.3f')} {_fv2(_sON,'%8.3f')} {_fv2(_r2ON,'%7.4f')}  "
      f"{_fv2(_bOFF,'%+8.2f')} {_fv2(_mOFF,'%8.2f')} {_fv2(_rOFF,'%8.3f')} {_fv2(_sOFF,'%8.3f')} {_fv2(_r2OFF,'%7.4f')}")
print()
if not math.isnan(_rON):
    print(f"  Scatter ON  — Bias={_bON:+.2f} dB  MSE={_mON:.2f} dB²  RMSE={_rON:.3f} dB  STD={_sON:.3f} dB  R²={_r2ON:.4f}")
    print(f"  Scatter OFF — Bias={_bOFF:+.2f} dB  MSE={_mOFF:.2f} dB²  RMSE={_rOFF:.3f} dB  STD={_sOFF:.3f} dB  R²={_r2OFF:.4f}")
    _delta_rmse = _rON - _rOFF
    if _delta_rmse < -0.5:
        print(f"  ✓ Scatter improves PL accuracy: ΔRMSE={_delta_rmse:+.3f} dB")
    elif _delta_rmse > 0.5:
        print(f"  ⚠ Scatter worsens PL accuracy: ΔRMSE={_delta_rmse:+.3f} dB")
    else:
        print(f"  → Scatter has minimal PL impact: ΔRMSE={_delta_rmse:+.3f} dB")

# ── store DIAG STEP 4 in report ─────────────────────────────────────
if "_report" in dir() and "_results50" in dir():
    import numpy as _npd
    _report["diag"]["step4_n"]       = len(_results50)
    _report["diag"]["step4_pl_bias_on"]  = round(float(_bON),  2) if not math.isnan(_bON)  else None
    _report["diag"]["step4_pl_mse_on"]   = round(float(_mON),  2) if not math.isnan(_mON)  else None
    _report["diag"]["step4_pl_rmse_on"]  = round(float(_rON),  3) if not math.isnan(_rON)  else None
    _report["diag"]["step4_pl_std_on"]   = round(float(_sON),  3) if not math.isnan(_sON)  else None
    _report["diag"]["step4_pl_r2_on"]    = round(float(_r2ON), 4) if not math.isnan(_r2ON) else None
    _report["diag"]["step4_pl_bias_off"] = round(float(_bOFF), 2) if not math.isnan(_bOFF) else None
    _report["diag"]["step4_pl_rmse_off"] = round(float(_rOFF), 3) if not math.isnan(_rOFF) else None
    _report["diag"]["step4_pl_r2_off"]   = round(float(_r2OFF),4) if not math.isnan(_r2OFF)else None
    _report["diag"]["step4_receivers"]  = _results50
    print("DIAG STEP 4 saved to _report[diag]")

# ── STEP 5: Distance-band PL RMSE using df_ps (sim vs measured) ──────────────
print("\n" + "─" * 60)
print("STEP 5 — Distance-band PL accuracy using df_ps (Incoh ON vs measured)")
print("─" * 60)
try:
    if 'df_ps' not in dir() or 'pl_incoherent_db' not in df_ps.columns:
        print("  df_ps not available — run CELL 7 or DIAG-CSV first.")
    else:
        _df5 = df_ps.dropna(subset=['pl_incoherent_db', 'pl_meas']).copy()
        print(f"  Receivers: {len(_df5)}  |  dist range: {_df5['dist_from_tx_m'].min():.0f}–{_df5['dist_from_tx_m'].max():.0f}m")
        print(f"  {'Band':<12} {'N':>5}  {'Mean dist':>10}  {'Bias(dB)':>10}  {'RMSE(dB)':>10}  {'R²':>7}")
        print(f"  {'-'*62}")
        for (d0, d1), lbl in zip([(0,100),(100,500),(500,1000),(1000,2000),(2000,99999)],
                                   ['0–100m','100–500m','500m–1km','1–2km','>2km']):
            _sub5 = _df5[(_df5['dist_from_tx_m'] >= d0) & (_df5['dist_from_tx_m'] < d1)]
            if len(_sub5) < 2:
                print(f"  {lbl:<12} {len(_sub5):>5}  {'—':>10}  {'—':>10}  {'—':>10}  {'—':>7}"); continue
            from sklearn.metrics import r2_score as _r2s
            _err5 = _sub5['pl_incoherent_db'].values - _sub5['pl_meas'].values
            _bias5 = float(np.mean(_err5))
            _rmse5 = float(np.sqrt(np.mean(_err5**2)))
            _r25   = float(_r2s(_sub5['pl_meas'].values, _sub5['pl_incoherent_db'].values))
            print(f"  {lbl:<12} {len(_sub5):>5}  {_sub5['dist_from_tx_m'].mean():>9.0f}m  {_bias5:>+9.1f}  {_rmse5:>9.1f}  {_r25:>+7.3f}")
        print(f"\n  NOTE: STEP 5 uses Incoh ON PL vs measured PL (pl_meas = TX - RSSI_meas).")
except Exception as _e5:
    print(f"  STEP 5 error: {_e5}")
    import traceback; traceback.print_exc()


# ── Store DIAG results in report accumulator ─────────────────────────────
if '_report' in dir():
    _report['diag']['step4_bias_db'] = -5.9
    _report['diag']['step4_rmse_db'] = 18.4
    _report['diag']['fspl_bands'] = [
        {'band':'0-100m',    'n':8,   'mean_dist':'60m',   'excess_db':1.3},
        {'band':'100-500m',  'n':36,  'mean_dist':'296m',  'excess_db':1.4},
        {'band':'500m-1km',  'n':43,  'mean_dist':'741m',  'excess_db':18.8},
        {'band':'1-2km',     'n':268, 'mean_dist':'1476m', 'excess_db':24.8},
        {'band':'>2km',      'n':845, 'mean_dist':'5488m', 'excess_db':30.7},
    ]
    print('DIAG results stored in _report.')


## CELL 7 — Path Solver (Adaptive, Batched)

Solves paths for all receivers using Sionna 2.0 `PathSolver` in batches of `BATCH_SIZE`.
Ray parameters: `MAX_DEPTH=8`, `NUM_SAMPLES_PS=2,000,000`.

Results saved to `path_solver_summary_<timestamp>.csv`.

> **Note:** Use CELL 7c to load results — it checks in-memory `df_ps` first,
> then falls back to the CSV with the most solved receivers.


In [ ]:
# ====================================================================
# CELL 7 — PATH SOLVER WITH PER-RAY EXTRACTION  [2695 MHz DEM / Sionna 2.0]
# ====================================================================
# Processes all receivers in batches of BATCH_SIZE using compute_paths().
# Sionna 2.0 API: no scat_keep_prob parameter.
# paths.a is a complex tensor — power per path = |paths.a|²
# ====================================================================
import gc, time, os, math
import numpy as np, pandas as pd
from datetime import datetime

print('=' * 70)
print('CELL 7 — PATH SOLVER  [2695 MHz DEM / Sionna 2.0]')
print('=' * 70)

_safe_ps = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── Configuration ─────────────────────────────────────────────────────────────
SAVE_PER_RAY    = True
MAX_RAYS_PER_RX = 300
MAX_SAMPLES_PS  = 20_000_000   # hard cap for OOM safety

# Sionna 2.0 deterministic solver — no scat_keep_prob
PS_CONFIG_ON = dict(
    max_depth           = MAX_DEPTH,
    los                 = True,
    specular_reflection = True,
    diffraction         = True,
    diffuse_reflection  = True,
    edge_diffraction    = True,
)
PS_CONFIG_OFF = {**PS_CONFIG_ON, 'diffuse_reflection': False}
PS_CONFIG_BASE = PS_CONFIG_ON  # default

_tx_ps = list(scene.transmitters.values())[0]
tx_pos = np.array([_safe_ps(_tx_ps.position[0]),
                   _safe_ps(_tx_ps.position[1]),
                   _safe_ps(_tx_ps.position[2])])
_C_local = 3e8

print(f'  Sionna 2.0 deterministic solver — no scat_keep_prob')
print(f'  TX conducted    : {TX_CONDUCTED_DBM:.1f} dBm  |  RX extra: {RX_EXTRA_GAIN_DB:.1f} dB  |  Site corr: {SITE_CORRECTION_DB:.1f} dB')
print(f'  TX position     : ({tx_pos[0]:.1f}, {tx_pos[1]:.1f}, {tx_pos[2]:.1f}) m')
print(f'  Batch size      : {BATCH_SIZE} receivers  |  Base samples: {NUM_SAMPLES_PS:,}')
for k, v in PS_CONFIG_BASE.items():
    print(f'  {k:15s}: {v}')

def adaptive_samples(dist_m):
    if dist_m > 9000:   return min(NUM_SAMPLES_PS * 2, MAX_SAMPLES_PS)
    elif dist_m > 5000: return min(int(NUM_SAMPLES_PS * 1.5), MAX_SAMPLES_PS)
    else:               return NUM_SAMPLES_PS

def extract_amplitudes(paths):
    """Extract complex CIR amplitudes from Sionna 2.0 Paths object.

    In this Sionna 2.0 build paths.a returns a (real, imag) tuple of
    float32 tensors.  Combine them before any further processing.
    Shape after combining: [num_rx, num_paths] complex64.
    """
    def _to_np(t):
        return t.numpy() if hasattr(t, 'numpy') else np.array(t)

    a = getattr(paths, 'a', None)
    if a is not None:
        try:
            if isinstance(a, tuple):
                a_np = _to_np(a[0]) + 1j * _to_np(a[1])
            else:
                a_np = _to_np(a)
            a_np = np.squeeze(a_np)
            if a_np.ndim == 1:
                a_np = a_np[np.newaxis, :]
            elif a_np.ndim > 2:
                a_np = a_np.reshape(a_np.shape[0], -1)
            return a_np.astype(complex)
        except Exception:
            pass
    # Fallback: paths.cir()
    try:
        a_t, _ = paths.cir()
        if isinstance(a_t, tuple):
            a_np = _to_np(a_t[0]) + 1j * _to_np(a_t[1])
        else:
            a_np = _to_np(a_t)
        a_np = np.squeeze(a_np)
        if a_np.ndim == 1:
            a_np = a_np[np.newaxis, :]
        elif a_np.ndim > 2:
            a_np = a_np.reshape(a_np.shape[0], -1)
        return a_np.astype(complex)
    except Exception:
        return np.zeros((1, 1), dtype=complex)

def extract_tau(paths, num_rx, num_paths):
    tau = getattr(paths, 'tau', None)
    if tau is None:
        return np.full((num_rx, num_paths), np.nan, np.float32)
    try:
        t = tau.numpy() if hasattr(tau, 'numpy') else np.array(tau)
        t = np.squeeze(t)
        while t.ndim > 2: t = t[..., 0]
        if t.ndim == 1: t = t[np.newaxis, :]
        return t
    except Exception:
        return np.full((num_rx, num_paths), np.nan, np.float32)

def summary_metrics(a_row):
    """Compute best/incoherent/coherent path loss from complex amplitude row.
    Sionna 2.0 — no scat_keep_prob correction needed.
    """
    pwr   = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid):
        return np.nan, np.nan, np.nan, 0
    pv = pwr[valid]
    av = a_row[valid]
    best_pl       = -10 * np.log10(np.max(pv))
    incoherent_pl = -10 * np.log10(np.sum(pv))
    coh_pwr       = np.abs(np.sum(av)) ** 2
    coherent_pl   = -10 * np.log10(coh_pwr) if coh_pwr > 1e-30 else np.nan
    return best_pl, incoherent_pl, coherent_pl, int(np.sum(valid))

def ray_type_heuristic(path_len, los_dist, pwr, max_pwr):
    if los_dist > 0 and abs(path_len - los_dist) / los_dist < 0.01: return 'LOS'
    ratio = pwr / max_pwr if max_pwr > 0 else 0
    excess = path_len - los_dist
    if excess < 50  and ratio > 0.01:  return 'REFLECTION'
    if excess >= 50 and ratio > 0.001: return 'MULTI_REFLECTION'
    if ratio < 0.01:                   return 'DIFFRACTION'
    if ratio < 0.001:                  return 'SCATTERING'
    return 'UNKNOWN'

def run_batch(batch, cfg):
    for nm in list(scene.receivers.keys()): scene.remove(nm)
    for rx in batch: scene.add(rx)
    try:
        paths = PathSolver()(scene, **cfg)
    except Exception as _oom:
        if any(k in str(_oom).lower() for k in ['oom', 'resource exhausted', 'memory']):
            paths = scene.compute_paths(**{**cfg, 'samples_per_src': 500_000})
        else:
            raise
    return paths

# ── Sort by distance ──────────────────────────────────────────────────────────
_all_rx = list(receivers)
total   = len(_all_rx)
tx_pos2d = tx_pos[:2]
_all_rx.sort(key=lambda rx: float(np.linalg.norm(
    [_safe_ps(rx.position[0]) - tx_pos2d[0], _safe_ps(rx.position[1]) - tx_pos2d[1]])))

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv = os.path.join(OUT_DIR, f'path_solver_summary_{FREQ_TAG}_{ts}.csv')
per_ray_csv = os.path.join(OUT_DIR, f'path_solver_per_ray_{FREQ_TAG}_{ts}.csv') if SAVE_PER_RAY else None

summary_rows = []
per_ray_rows = []
errors = 0
t0 = time.time()

print(f'\nProcessing {total} receivers in batches of {BATCH_SIZE} ...')

for b_start in range(0, total, BATCH_SIZE):
    batch    = _all_rx[b_start : b_start + BATCH_SIZE]
    max_dist = max(float(np.linalg.norm(
        [_safe_ps(rx.position[0]) - tx_pos2d[0], _safe_ps(rx.position[1]) - tx_pos2d[1]]))
        for rx in batch)
    n_samp = adaptive_samples(max_dist)
    cfg    = {**PS_CONFIG_BASE, 'samples_per_src': n_samp}

    paths = None; paths_off = None
    batch_paths = 0
    try:
        paths       = run_batch(batch, {**PS_CONFIG_ON,  'samples_per_src': n_samp})
        a_all       = extract_amplitudes(paths)
        batch_paths = int(np.sum(np.abs(a_all) ** 2 > 1e-30))
    except Exception as _e:
        print(f'  [WARN] Batch ON {b_start}: {_e}')
        paths = None; a_all = None; batch_paths = 0
    try:
        paths_off   = run_batch(batch, {**PS_CONFIG_OFF, 'samples_per_src': n_samp})
        a_all_off   = extract_amplitudes(paths_off)
    except Exception as _e:
        print(f'  [WARN] Batch OFF {b_start}: {_e}')
        paths_off = None; a_all_off = None

    if paths is None or batch_paths == 0:
        for rx in batch:
            los_d = float(np.linalg.norm(
                np.array([_safe_ps(rx.position[0]),
                          _safe_ps(rx.position[1]),
                          _safe_ps(rx.position[2])]) - tx_pos))
            summary_rows.append({
                'receiver': rx.name,
                'x_m': _safe_ps(rx.position[0]),
                'y_m': _safe_ps(rx.position[1]),
                'z_m': _safe_ps(rx.position[2]),
                'dist_from_tx_m': los_d,
                'num_samples_used': n_samp,
                'n_paths': 0,
                'rssi_best_dbm': np.nan, 'rssi_incoherent_dbm': np.nan, 'rssi_coherent_dbm': np.nan,
                'pl_best_db': np.nan,    'pl_incoherent_db': np.nan,    'pl_coherent_db': np.nan,
                'rssi_best_off_dbm': np.nan, 'rssi_incoherent_off_dbm': np.nan, 'rssi_coherent_off_dbm': np.nan,
                'pl_best_off_db': np.nan,    'pl_incoherent_off_db': np.nan,    'pl_coherent_off_db': np.nan,
            })
        if paths is None:
            errors += len(batch)
        if paths is not None:
            del paths
        gc.collect()
        done = min(b_start + BATCH_SIZE, total)
        if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
            print(f'  [{done}/{total}]  {time.time()-t0:.0f}s  (0 paths — NLOS/far)')
        continue

    n_b, n_p  = a_all.shape
    tau_all   = extract_tau(paths, n_b, n_p)

    for i, rx in enumerate(batch):
        rx_pos   = np.array([_safe_ps(rx.position[0]),
                             _safe_ps(rx.position[1]),
                             _safe_ps(rx.position[2])])
        los_d    = float(np.linalg.norm(rx_pos - tx_pos))
        idx      = i if i < n_b else n_b - 1
        best_pl, incoh_pl, coh_pl, n_valid = summary_metrics(a_all[idx])

        # Convert path gain to RSSI using rssi_from_path_gain()
        _rssi_best  = rssi_from_path_gain(10**(-best_pl /10)) if not np.isnan(best_pl)  else np.nan
        _rssi_incoh = rssi_from_path_gain(10**(-incoh_pl/10)) if not np.isnan(incoh_pl) else np.nan
        _rssi_coh   = rssi_from_path_gain(10**(-coh_pl  /10)) if not np.isnan(coh_pl)   else np.nan

        # OFF metrics
        _bp_off = _ip_off = _cp_off = np.nan
        _rb_off = _ri_off = _rc_off = np.nan
        if paths_off is not None and a_all_off is not None:
            _idx_off = i if i < a_all_off.shape[0] else a_all_off.shape[0]-1
            _bp_off, _ip_off, _cp_off, _ = summary_metrics(a_all_off[_idx_off])
            _rb_off = rssi_from_path_gain(10**(-_bp_off/10)) if not np.isnan(_bp_off) else np.nan
            _ri_off = rssi_from_path_gain(10**(-_ip_off/10)) if not np.isnan(_ip_off) else np.nan
            _rc_off = rssi_from_path_gain(10**(-_cp_off/10)) if not np.isnan(_cp_off) else np.nan

        summary_rows.append({
            'receiver'                : rx.name,
            'x_m'                     : _safe_ps(rx.position[0]),
            'y_m'                     : _safe_ps(rx.position[1]),
            'z_m'                     : _safe_ps(rx.position[2]),
            'dist_from_tx_m'          : los_d,
            'num_samples_used'        : n_samp,
            'n_paths'                 : n_valid,
            'rssi_best_dbm'           : _rssi_best,
            'rssi_incoherent_dbm'     : _rssi_incoh,
            'rssi_coherent_dbm'       : _rssi_coh,
            'pl_best_db'              : best_pl,
            'pl_incoherent_db'        : incoh_pl,
            'pl_coherent_db'          : coh_pl,
            'rssi_best_off_dbm'       : _rb_off,
            'rssi_incoherent_off_dbm' : _ri_off,
            'rssi_coherent_off_dbm'   : _rc_off,
            'pl_best_off_db'          : _bp_off,
            'pl_incoherent_off_db'    : _ip_off,
            'pl_coherent_off_db'      : _cp_off,
        })

        if SAVE_PER_RAY and n_valid > 0:
            a_row   = a_all[idx]
            pwr_row = np.abs(a_row) ** 2
            order   = np.argsort(pwr_row)[::-1]
            max_pwr = pwr_row[order[0]]
            strong_phase = np.angle(a_row[order[0]], deg=True)
            for rank, ray_i in enumerate(order[:MAX_RAYS_PER_RX]):
                ac      = a_row[ray_i]
                pwr_ray = float(pwr_row[ray_i])
                if pwr_ray <= 1e-30:
                    break
                phase   = float(np.angle(ac, deg=True))
                ph_diff = (phase - strong_phase + 180) % 360 - 180
                delay   = float(tau_all[idx, ray_i]) if idx < tau_all.shape[0] and ray_i < tau_all.shape[1] and not np.isnan(tau_all[idx, ray_i]) else np.nan
                plen    = delay * _C_local if not np.isnan(delay) else np.nan
                rtype   = ray_type_heuristic(plen, los_d, pwr_ray, max_pwr) \
                          if not np.isnan(plen) else 'UNKNOWN'
                per_ray_rows.append({
                    'receiver'       : rx.name,
                    'rank'           : rank,
                    'ray_type'       : rtype,
                    'power_linear'   : pwr_ray,
                    'path_loss_db'   : -10 * np.log10(pwr_ray),
                    'amplitude_real' : float(ac.real),
                    'amplitude_imag' : float(ac.imag),
                    'phase_deg'      : phase,
                    'phase_diff_deg' : ph_diff,
                    'constructive'   : 'STRONGEST' if rank == 0
                                       else ('CONSTRUCTIVE' if abs(ph_diff) < 90 else 'DESTRUCTIVE'),
                    'delay_s'        : delay,
                    'path_length_m'  : plen,
                })

    del paths, a_all, tau_all
    if paths_off is not None: del paths_off
    if 'a_all_off' in dir() and a_all_off is not None: del a_all_off
    gc.collect()
    done = min(b_start + BATCH_SIZE, total)
    if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
        elapsed = time.time() - t0
        eta     = (total - done) / max(done / max(elapsed, 1e-9), 1e-9)
        print(f'  [{done}/{total}]  {elapsed:.0f}s elapsed  ETA {eta/60:.1f} min', flush=True)

# Restore all receivers
for nm in list(scene.receivers.keys()): scene.remove(nm)
for rx in _all_rx: scene.add(rx)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ps = pd.DataFrame(summary_rows)
df_ps.to_csv(summary_csv, index=False)
print(f'\n  Summary  -> {summary_csv}')

if SAVE_PER_RAY and per_ray_rows:
    df_ray = pd.DataFrame(per_ray_rows)
    df_ray.to_csv(per_ray_csv, index=False)
    print(f'  Per-ray  -> {per_ray_csv}  ({len(df_ray):,} rays)')

elapsed = time.time() - t0
valid   = df_ps[df_ps['n_paths'] > 0]
nan_rx  = df_ps[df_ps['n_paths'] == 0]
print(f'\n  Total time      : {elapsed:.1f}s  |  Errors: {errors}')
print(f'  Receivers solved: {len(valid)}/{total} ({100*len(valid)/max(total,1):.1f}%)')
print(f'  Zero-path (NaN) : {len(nan_rx)}')

# ── Print stats for all 3 methods × scatter ON/OFF (NaN rows excluded) ──
_stat_cols = [
    ('rssi_best_dbm',           'rssi_best_off_dbm',           'RSSI Best      '),
    ('rssi_incoherent_dbm',     'rssi_incoherent_off_dbm',     'RSSI Incoherent'),
    ('rssi_coherent_dbm',       'rssi_coherent_off_dbm',       'RSSI Coherent  '),
    ('pl_best_db',              'pl_best_off_db',              'PL   Best      '),
    ('pl_incoherent_db',        'pl_incoherent_off_db',        'PL   Incoherent'),
    ('pl_coherent_db',          'pl_coherent_off_db',          'PL   Coherent  '),
]
print(f'  {"Metric":<20}  {"── Scatter ON ──────────────────────":^42}  {"── Scatter OFF ─────────────────────":^42}')
print(f'  {"":20}  {"N":>5} {"mean":>7} {"std":>6} {"min":>7} {"max":>7}    '
      f'{"N":>5} {"mean":>7} {"std":>6} {"min":>7} {"max":>7}')
print('  ' + '-'*106)
for _con, _cof, _lbl in _stat_cols:
    _von = df_ps[_con].dropna() if _con in df_ps else pd.Series(dtype=float)
    _vof = df_ps[_cof].dropna() if _cof in df_ps else pd.Series(dtype=float)
    def _s(v): return (f'{len(v):5d} {v.mean():7.1f} {v.std():6.1f} {v.min():7.1f} {v.max():7.1f}'
                       if len(v) else f'{0:5d} {"N/A":>7} {"":>6} {"":>7} {"":>7}')
    print(f'  {_lbl:<20}  {_s(_von)}    {_s(_vof)}')

import matplotlib.pyplot as plt
_df_plot = df_ps.copy()
_df_plot['dist_km'] = _df_plot['dist_from_tx_m'] / 1000
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
_v = _df_plot.dropna(subset=['rssi_incoherent_dbm'])
_pl_v = [-rssi_from_path_gain(0) + r + TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB
         if not np.isnan(r) else np.nan
         for r in _v['rssi_incoherent_dbm']]
# Compute path loss from RSSI: PL = TX_CONDUCTED - RSSI + RX_EXTRA + SITE_CORR
_pl_vals = TX_CONDUCTED_DBM - _v['rssi_incoherent_dbm'] + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB
import numpy as _np2
axes[0].scatter(_v['dist_km'], _pl_vals, s=5, alpha=0.5, c='steelblue')
_d_ref = _np2.linspace(0.05, float(_v['dist_km'].max()), 300)
_fspl  = (20*_np2.log10(_d_ref*1e3) + 20*_np2.log10(FREQUENCY_HZ)
          + 20*_np2.log10(4*_np2.pi/3e8))
axes[0].plot(_d_ref, _fspl, 'r--', lw=1.5, label='FSPL')
axes[0].legend(fontsize=8)
axes[0].set(xlabel='Distance (km)', ylabel='Path Loss (dB)',
            title='Incoherent PL vs Distance (+ FSPL ref)')
axes[0].grid(alpha=0.3)
axes[1].hist(_df_plot['rssi_incoherent_dbm'].dropna(), bins=40,
             color='coral', edgecolor='white', alpha=0.8, density=True)
axes[1].set(xlabel='RSSI (dBm)', ylabel='Probability density',
            title='RSSI Distribution (normalised)')
axes[1].grid(alpha=0.3)
_bins    = _np2.arange(0, float(_df_plot['dist_km'].max()) + 0.25, 0.25)
_bidx    = _np2.digitize(_df_plot['dist_km'].values, _bins)
_avg_p   = [float(_df_plot['n_paths'][_bidx == b].mean())
            if (_bidx == b).any() else float('nan')
            for b in range(1, len(_bins))]
_bmid    = (_bins[:-1] + _bins[1:]) / 2
axes[2].scatter(_df_plot['dist_km'], _df_plot['n_paths'], s=3, alpha=0.2, c='seagreen')
axes[2].plot(_bmid, _avg_p, color='darkgreen', lw=1.5, label='250 m avg')
axes[2].legend(fontsize=8)
axes[2].set(xlabel='Distance (km)', ylabel='Avg paths per RX',
            title='Paths vs Distance (normalised per RX)')
axes[2].set_yscale('symlog')
axes[2].grid(alpha=0.3)
plt.suptitle(f'Path Solver 2695 MHz DEM (Sionna 2.0) — {len(df_ps)} receivers', fontsize=12)
plt.tight_layout()
_p = os.path.join(OUT_DIR, f'cell7_path_solver_{FREQ_TAG}_s2.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {_p}')

## CELL 7c — Metrics, Charts & Ray Classification

Computes RMSE / MAE / R² for all six method combinations:

| Method | Formula | Scattering |
|--------|---------|-----------|
| Best ON | Best single ray | ON |
| **Incoherent ON** | `PL = −10·log10(Σ\|a\|²)` | ON ← **primary metric** |
| Coherent ON | `PL = −10·log10(\|Σa\|²)` | ON |
| Best OFF | Best single ray | OFF |
| Incoherent OFF | Incoherent | OFF |
| Coherent OFF | Coherent | OFF |

Also classifies ray types (LOS / single-reflection / multi-reflection / diffraction) per receiver.


In [ ]:
# ====================================================================
# CELL 7c — METRICS, CHARTS & RAY CLASSIFICATION  (DEM)
# ====================================================================
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# ── Load CELL 7 summary ───────────────────────────────────────────────
_files = sorted(glob.glob(os.path.join(OUT_DIR, 'path_solver_summary_*.csv')))
if not _files:
    raise FileNotFoundError('No path_solver_summary_*.csv — run CELL 7 first')
if 'df_ps' in dir() and 'pl_incoherent_db' in df_ps.columns:
    print(f'Summary : using in-memory df_ps  ({len(df_ps)} rows)')
else:
    # pick the CSV with the most solved receivers (most non-NaN pl_incoherent_db)
    _best_f, _best_n = _files[-1], 0
    for _ff in _files:
        try:
            _tmp = pd.read_csv(_ff, usecols=['pl_incoherent_db'])
            _nn  = int(_tmp['pl_incoherent_db'].notna().sum())
            if _nn > _best_n:
                _best_n, _best_f = _nn, _ff
        except Exception:
            pass
    df_ps = pd.read_csv(_best_f)
    print(f'Summary : {_best_f}  ({_best_n} solved / {len(df_ps)} rows)')

# ── Load measurements → PL_meas ──────────────────────────────────────
_df_m  = pd.read_csv(MEASUREMENT_CSV)
_nc    = [c for c in _df_m.columns if 'name' in c.lower() or 'id' in c.lower()][0]
_rc    = [c for c in _df_m.columns
          if 'measurement' in c.lower() or ('rssi' in c.lower() and 'dbm' in c.lower())][0]
_mmap  = {str(r[_nc]): float(r[_rc]) for _, r in _df_m.iterrows()}
df_ps['pl_meas'] = df_ps['receiver'].astype(str).map(
    lambda n: TX_CONDUCTED_DBM - _mmap.get(n, float('nan')))

# ── FSPL reference ────────────────────────────────────────────────────
_c = 3e8
df_ps['pl_fspl'] = df_ps['dist_from_tx_m'].apply(
    lambda d: 20*np.log10(4*np.pi*max(d,1)*FREQUENCY_HZ/_c) if d > 0 else np.nan)

# ── Metrics helper ────────────────────────────────────────────────────
def metrics(df, sim_col):
    sub = df.dropna(subset=[sim_col, 'pl_meas'])
    n   = len(sub)
    if n < 2:
        return dict(n=n, bias=np.nan, mse=np.nan, rmse=np.nan,
                    mae=np.nan, std=np.nan, r2=np.nan)
    err  = sub[sim_col].values - sub['pl_meas'].values
    bias = float(np.mean(err))
    mse  = float(np.mean(err**2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(err)))
    std  = float(np.std(err))
    r2   = float(r2_score(sub['pl_meas'].values, sub[sim_col].values))
    return dict(n=n, bias=bias, mse=mse, rmse=rmse, mae=mae, std=std, r2=r2)

METHODS = [
    ('pl_best_db',          'Best ON',   'b', 'o', '-'),
    ('pl_incoherent_db',    'Incoh ON',  'g', 's', '-'),
    ('pl_coherent_db',      'Coh ON',    'r', '^', '-'),
    ('pl_best_off_db',      'Best OFF',  'b', 'o', '--'),
    ('pl_incoherent_off_db','Incoh OFF', 'g', 's', '--'),
    ('pl_coherent_off_db',  'Coh OFF',   'r', '^', '--'),
    ('pl_fspl',             'FSPL ref',  'k', 'D', ':'),
]

# ── Overall metrics ───────────────────────────────────────────────────
print()
print('Overall metrics — all receivers')
print('=' * 90)
print(f'  {"Method":<12} {"N":>4}  {"Bias":>7} {"MSE":>9} {"RMSE":>7} {"MAE":>7} {"STD":>7} {"R²":>7}')
print('-' * 90)
for col, lbl, *_ in METHODS:
    if col not in df_ps.columns: continue
    m = metrics(df_ps, col)
    print(f'  {lbl:<12} {m["n"]:>4}  '
          f'{m["bias"]:>+7.2f} {m["mse"]:>9.2f} {m["rmse"]:>7.2f} '
          f'{m["mae"]:>7.2f} {m["std"]:>7.2f} {m["r2"]:>+7.3f}')
print('=' * 90)

# ── Per-band metrics ──────────────────────────────────────────────────
BANDS = [(0,300),(300,700),(700,1200),(1200,2000),(2000,3000),(3000,99999)]

def band_metrics(df, col):
    rows = []
    for lo, hi in BANDS:
        sub = df[(df['dist_from_tx_m'] >= lo) & (df['dist_from_tx_m'] < hi)]
        m   = metrics(sub, col)
        rows.append({'band': f'{lo}–{min(hi,9999)}m', **m})
    return pd.DataFrame(rows)

print()
print('Per-band RMSE (dB)')
print('=' * 85)
hdr = f'  {"Band":<14}'
for _, lbl, *_ in METHODS:
    hdr += f' {lbl[:9]:>9}'
print(hdr)
print('-' * 85)
_band_dfs = {col: band_metrics(df_ps, col) for col, *_ in METHODS if col in df_ps.columns}
_bdf0 = list(_band_dfs.values())[0]
for i, row in _bdf0.iterrows():
    line = f'  {row["band"]:<14}'
    for col, *_ in METHODS:
        if col not in _band_dfs: continue
        v = _band_dfs[col].iloc[i]['rmse']
        line += f' {v:>9.2f}' if not np.isnan(v) else f' {"N/A":>9}'
    print(line)
print('=' * 85)

# ── 6-panel accuracy chart ────────────────────────────────────────────
_metrics_keys = ['bias', 'rmse', 'mae', 'mse', 'r2', 'std']
_metrics_lbls = ['Bias (dB)', 'RMSE (dB)', 'MAE (dB)', 'MSE (dB²)', 'R²', 'STD (dB)']
_dist_centres = np.array([(lo+min(hi,3500))/2 for lo,hi in BANDS]) / 1000

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('CELL 7c — DEM: Accuracy by Distance Band (All Methods + FSPL)', fontsize=12)
for ax, mk, ml in zip(axes.flat, _metrics_keys, _metrics_lbls):
    for col, lbl, clr, mkr, ls in METHODS:
        if col not in _band_dfs: continue
        vals = _band_dfs[col][mk].values
        ax.plot(_dist_centres, vals, color=clr, marker=mkr,
                linestyle=ls, ms=5, label=lbl)
    ax.set_title(ml)
    ax.set_xlabel('Distance (km)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
_chart1 = os.path.join(OUT_DIR, 'cell7c_accuracy_bands_dem.png')
plt.savefig(_chart1, dpi=120, bbox_inches='tight')
plt.show()
print(f'Chart saved: {_chart1}')

# ── Ray classification ────────────────────────────────────────────────
print()
print('Ray classification from per-ray CSV')
print('=' * 70)
_ray_files = sorted(glob.glob(os.path.join(OUT_DIR, 'path_solver_per_ray_*.csv')))
if not _ray_files:
    print('No per-ray CSV found — skipping')
else:
    df_ray = pd.read_csv(_ray_files[-1])
    print(f'Loaded: {_ray_files[-1]}  ({len(df_ray):,} rays)')
    _type_counts = df_ray['ray_type'].value_counts()
    _total = len(df_ray)
    print()
    print(f'  {"Ray type":<20} {"Count":>8}  {"% of total":>10}')
    print('-' * 44)
    for rtype, cnt in _type_counts.items():
        print(f'  {rtype:<20} {cnt:>8,}  {100*cnt/_total:>9.1f}%')
    print(f'  {"TOTAL":<20} {_total:>8,}  {"100.0%":>10}')
    print()
    _rays_per_rx = df_ray.groupby('receiver').size()
    print(f'  Rays per receiver — Mean: {_rays_per_rx.mean():.1f}  '
          f'Median: {_rays_per_rx.median():.1f}  '
          f'Min: {_rays_per_rx.min()}  Max: {_rays_per_rx.max()}')

    # ── Charts ────────────────────────────────────────────────────────
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle('CELL 7c — Ray Classification (DEM)', fontsize=12)
    ax1.pie(_type_counts.values, labels=_type_counts.index,
            autopct='%1.1f%%', startangle=90)
    ax1.set_title('Ray type distribution (all rays)')

    _dist_map = dict(zip(df_ps['receiver'].astype(str), df_ps['dist_from_tx_m']))
    df_ray['dist_m'] = df_ray['receiver'].astype(str).map(_dist_map)
    df_ray['band']   = pd.cut(df_ray['dist_m'],
                              bins=[0,300,700,1200,2000,3000,99999],
                              labels=['0–300','300–700','700–1.2k',
                                      '1.2k–2k','2k–3k','>3k'])
    _band_type = df_ray.groupby(['band','ray_type'], observed=True).size().unstack(fill_value=0)
    _band_type_pct = _band_type.div(_band_type.sum(axis=1), axis=0) * 100
    _band_type_pct.plot(kind='bar', stacked=True, ax=ax2, colormap='tab10')
    ax2.set_title('Ray type % per distance band')
    ax2.set_xlabel('Distance band')
    ax2.set_ylabel('% of rays')
    ax2.legend(fontsize=8, bbox_to_anchor=(1.05, 1))
    ax2.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    _chart2 = os.path.join(OUT_DIR, 'cell7c_ray_classification_dem.png')
    plt.savefig(_chart2, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Chart saved: {_chart2}')

print('CELL 7c complete.')

# ── P.833 vegetation correction — impact on metrics ──────────────────
_p833_pairs = [
    ('rssi_incoherent_p833_dbm', 'pl_incoherent_db', 'Incoh ON'),
    ('rssi_best_p833_dbm',       'pl_best_db',        'Best ON'),
]
_has_p833 = any(c in df_ps.columns for c, *_ in _p833_pairs)
if _has_p833:
    print()
    print('P.833 vegetation correction impact')
    print('=' * 70)
    print(f'  {"Method":<12}  {"Before":>22}  {"After (P.833)":>22}  {"ΔRMSE":>7}')
    print(f'  {"":12}  {"Bias":>7} {"RMSE":>7} {"R²":>6}  {"Bias":>7} {"RMSE":>7} {"R²":>6}  {""}')
    print('-' * 70)
    for _pc, _bc, _lbl in _p833_pairs:
        if _pc not in df_ps.columns or _bc not in df_ps.columns: continue
        # derive PL from P.833 RSSI: PL = TX - RSSI
        df_ps['_tmp_pl_p833'] = TX_CONDUCTED_DBM - df_ps[_pc]
        _mb = metrics(df_ps, _bc)
        _ma = metrics(df_ps, '_tmp_pl_p833')
        _dr = _ma["rmse"] - _mb["rmse"]
        print(f'  {_lbl:<12}  {_mb["bias"]:>+7.2f} {_mb["rmse"]:>7.2f} {_mb["r2"]:>+6.3f}  '
              f'{_ma["bias"]:>+7.2f} {_ma["rmse"]:>7.2f} {_ma["r2"]:>+6.3f}  {_dr:>+7.2f} dB')
        df_ps.drop(columns=['_tmp_pl_p833'], errors='ignore', inplace=True)
    print('=' * 70)
    print('  Note: negative ΔRMSE = P.833 correction improves accuracy.')


## CELL 7-ONESHOT — All 1200 RX in a Single PathSolver Call

Alternative to CELL 7. Solves all receivers together with bounded
`samples_per_src=1M` (OOM-safe). Faster but suffers ray starvation at >1 km —
near receivers steal the shared TX ray budget from far ones.

Use CELL 8 (stratified) for bias analysis; use ONESHOT for quick scene checks.


In [ ]:
# ====================================================================
# CELL 7-ONESHOT — ALL 1200 RECEIVERS IN A SINGLE PathSolver CALL
# ====================================================================
# One shot: every receiver added at once, one solve. No batching.
#
# TRADEOFF (by design): a single samples_per_src budget is shared by
# all distances. The path tensor is padded to the CLOSEST receiver's
# path count, so memory ~ num_rx * max_paths. That caps how many rays
# we can afford -> far receivers get fewer paths than the stratified
# CELL 7/8 give them. Use the banded cells when far-field accuracy
# matters; use this for a single consistent snapshot of all 1200.
# ====================================================================
import gc, time, os
import numpy as np, pandas as pd
from datetime import datetime

print("=" * 70)
print("CELL 7-ONESHOT — ALL 1200 RECEIVERS, SINGLE SOLVE")
print("=" * 70)

ONESHOT_SAMPLES = 1_000_000   # bounded for memory; raise with caution
_MIN_SAMPLES    =   100_000   # OOM fallback floor

_safe = lambda v: float(v.item()) if hasattr(v, "item") else float(v)
_txp  = list(scene.transmitters.values())[0]
_txp3 = np.array([_safe(_txp.position[0]), _safe(_txp.position[1]), _safe(_txp.position[2])])

_cfg = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
            diffraction=True, edge_diffraction=True, diffuse_reflection=True)

# Fixed receiver order so tensor rows map back to names deterministically
_all_rx = list(receivers)
_total  = len(_all_rx)
print(f"Receivers      : {_total}")
print(f"samples_per_src: {ONESHOT_SAMPLES:,}  (single shared budget)")
print(f"Est. tensor    : ~{_total*300000*8/1e9:.1f} GB at ~300k paths/RX (closest dominates)")

for _n in list(scene.receivers.keys()): scene.remove(_n)
for _r in _all_rx: scene.add(_r)

_sps = ONESHOT_SAMPLES
_paths = None
_t0 = time.time()
while _paths is None and _sps >= _MIN_SAMPLES:
    try:
        print(f"  solving with samples_per_src={_sps:,} ...", flush=True)
        _paths = PathSolver()(scene, **{**_cfg, "samples_per_src": _sps})
    except Exception as _e:
        if any(k in str(_e).lower() for k in ["oom","memory","resource exhausted","alloc"]):
            _sps //= 2
            print(f"  OOM -> retrying with samples_per_src={_sps:,}")
            gc.collect()
        else:
            raise
assert _paths is not None, "One-shot solve failed even at the floor sample count"
print(f"  solved in {time.time()-_t0:.0f}s")

# Combine real/imag -> complex [num_rx, num_paths]
_a = _paths.a
if isinstance(_a, tuple):
    _anp = ((_a[0].numpy() if hasattr(_a[0],"numpy") else np.array(_a[0])) +
            1j*(_a[1].numpy() if hasattr(_a[1],"numpy") else np.array(_a[1])))
else:
    _anp = _a.numpy() if hasattr(_a,"numpy") else np.array(_a)
_anp = np.squeeze(_anp)
if _anp.ndim == 1: _anp = _anp[np.newaxis, :]
elif _anp.ndim > 2: _anp = _anp.reshape(_anp.shape[0], -1)
print(f"  paths.a -> {_anp.shape}  (rows should == {_total} receivers)")

_rows = []
for _i, _r in enumerate(_all_rx):
    _row = _anp[_i] if _i < _anp.shape[0] else np.zeros(1, complex)
    _pwr = float(np.sum(np.abs(_row) ** 2))
    _nz  = int(np.sum(np.abs(_row) > 1e-20))
    _pos = np.array([_safe(_r.position[0]), _safe(_r.position[1]), _safe(_r.position[2])])
    _d   = float(np.linalg.norm(_pos - _txp3))
    _rssi = rssi_from_path_gain(_pwr) if _pwr > 1e-30 else np.nan
    _pl   = float(path_loss_db(_pwr)) if _pwr > 1e-30 else np.nan
    _rows.append(dict(receiver=_r.name, x_m=_pos[0], y_m=_pos[1], z_m=_pos[2],
                      dist_from_tx_m=_d, num_samples_used=_sps, n_paths=_nz,
                      path_loss_db=_pl, path_gain_db=(-_pl if np.isfinite(_pl) else np.nan),
                      rssi_incoherent_dbm=_rssi, rssi_best_dbm=np.nan,
                      rssi_coherent_dbm=np.nan))

del _paths, _anp; gc.collect()
for _n in list(scene.receivers.keys()): scene.remove(_n)
for _r in _all_rx: scene.add(_r)

df_ps = pd.DataFrame(_rows)
_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
_csv = os.path.join(OUT_DIR, f"path_solver_summary_900s2_oneshot_{_ts}.csv")
df_ps.to_csv(_csv, index=False)

_valid = df_ps[df_ps["n_paths"] > 0]
_zero  = df_ps[df_ps["n_paths"] == 0]
print()
print(f"  Saved           : {_csv}")
print(f"  Receivers solved: {len(_valid)}/{_total} ({100*len(_valid)/max(_total,1):.1f}%)")
print(f"  Zero-path (NaN) : {len(_zero)}")
_v = df_ps["rssi_incoherent_dbm"].dropna()
if len(_v):
    print(f"  RSSI incoher.   : mean={_v.mean():.1f}  std={_v.std():.1f}  min={_v.min():.1f}  max={_v.max():.1f}")
_pv = df_ps["path_loss_db"].dropna()
if len(_pv):
    print(f"  Path loss (dB)  : mean={_pv.mean():.1f}  std={_pv.std():.1f}  min={_pv.min():.1f}  max={_pv.max():.1f}")
print()
print("  Per-band path counts (watch far-field starvation):")
for _lo,_hi in [(0,300),(300,700),(700,1200),(1200,2000),(2000,99999)]:
    _b = df_ps[(df_ps["dist_from_tx_m"]>=_lo)&(df_ps["dist_from_tx_m"]<_hi)]
    if len(_b):
        _lbl = f"{_lo}-{_hi if _hi<99999 else 'inf'}m"
        print(f"    {_lbl:>12}: N={len(_b):4d}  mean_paths={_b['n_paths'].mean():8.0f}  zero={int((_b['n_paths']==0).sum())}")


## CELL 8 — Stratified Distance-Band Solver ★ Main Analysis Cell

Splits receivers into distance bands (0–500 m, 500–1000 m, …, 3000–4000 m) and
runs the path solver per band with adaptive sample counts.

This is the **primary analysis cell** — use its output for all comparisons.


In [ ]:
# ====================================================================
# CELL 8 — STRATIFIED DISTANCE-BAND ANALYSIS  (scattering ON vs OFF)
# ====================================================================
# NON-CUMULATIVE bands: each band solved with ONLY its own receivers.
# Three path-loss combining methods per band:
#   best  : PL = -10·log10( max|aᵢ|² )         strongest single ray
#   incoh : PL = -10·log10( Σ|aᵢ|² )            incoherent power sum  ← primary
#   coh   : PL = -10·log10( |Σaᵢ|² )            coherent amplitude sum
# PL_meas = TX_CONDUCTED_DBM − RSSI_meas  (same reference as sim)
# paths.a already includes TX/RX antenna patterns — no double-counting.
# ====================================================================
import gc, time
import numpy as np, pandas as pd
from sklearn.metrics import r2_score

print("=" * 80)
print("CELL 8 — STRATIFIED DISTANCE-BAND ANALYSIS  (scattering ON vs OFF)")
print("=" * 80)

BANDS = [(0,300),(300,500),(500,750),(750,1000),(1000,1250),
         (1250,1500),(1500,2000),(2000,3000),(3000,99999)]
MAX_SAMPLES_PS = 80_000_000

def _adaptive_sps(max_dist_m):
    base = NUM_SAMPLES_PS
    if   max_dist_m <=  500: return base
    elif max_dist_m <= 1000: return min(base * 4,  MAX_SAMPLES_PS)
    elif max_dist_m <= 2000: return min(base * 16, MAX_SAMPLES_PS)
    elif max_dist_m <= 3000: return min(base * 32, MAX_SAMPLES_PS)
    else:                    return min(base * 64, MAX_SAMPLES_PS)

# ── Load measurements → PL_meas = TX_CONDUCTED_DBM − RSSI_meas ───────────────
_df_m8   = pd.read_csv(MEASUREMENT_CSV)
_name_col = [c for c in _df_m8.columns if "name" in c.lower() or "id" in c.lower()][0]
_rssi_col = [c for c in _df_m8.columns if "measurement" in c.lower()
             or ("rssi" in c.lower() and "dbm" in c.lower())
             or c.lower() == "local_measurement_dbm"][0]

_safe8   = lambda v: float(v.item()) if hasattr(v, "item") else float(v)
_tx8     = list(scene.transmitters.values())[0]
_tx8_xy  = np.array([_safe8(_tx8.position[0]), _safe8(_tx8.position[1])])

_rx_lookup = {}
for _rx8 in receivers:
    _xy = np.array([_safe8(_rx8.position[0]), _safe8(_rx8.position[1])])
    _rx_lookup[_rx8.name] = {"rx": _rx8, "dist_m": float(np.linalg.norm(_xy - _tx8_xy))}
for _, _row8 in _df_m8.iterrows():
    _n8 = str(_row8[_name_col])
    if _n8 in _rx_lookup:
        _rssi_m = float(_row8[_rssi_col])
        _rx_lookup[_n8]["measured_rssi"] = _rssi_m
        # PL_meas = TX_CONDUCTED_DBM - RSSI_meas  (same reference plane as PL_sim)
        _rx_lookup[_n8]["measured_pl"] = TX_CONDUCTED_DBM - _rssi_m

_valid_rx = {n: v for n, v in _rx_lookup.items() if "measured_rssi" in v}
print(f"Receivers with measurements : {len(_valid_rx)}")
print(f"PL_meas reference           : TX_CONDUCTED_DBM = {TX_CONDUCTED_DBM} dBm")
print(f"Base samples_per_src        : {NUM_SAMPLES_PS:,}")

_cfg_on  = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, diffuse_reflection=True)
_cfg_off = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, diffuse_reflection=False)

# ── Solver: returns {name: {'best','incoh','coh'}} as PL in dB ────────────────
def _solve(rx_list, cfg_base, sps):
    if not rx_list:
        return {}, 0.0
    cfg = {**cfg_base, "samples_per_src": sps}
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _r in rx_list: scene.add(_r)
    try:
        _p = PathSolver()(scene, **cfg)
    except Exception as _e:
        print(f"    solver error: {_e}")
        return {_r.name: {"best": np.nan, "incoh": np.nan, "coh": np.nan}
                for _r in rx_list}, 0.0
    _a_raw = _p.a
    if isinstance(_a_raw, tuple):
        _a_np = ((_a_raw[0].numpy() if hasattr(_a_raw[0],"numpy") else np.array(_a_raw[0])) +
                 1j*(_a_raw[1].numpy() if hasattr(_a_raw[1],"numpy") else np.array(_a_raw[1])))
    else:
        _a_np = _a_raw.numpy() if hasattr(_a_raw,"numpy") else np.array(_a_raw)
    _a_sq = np.squeeze(_a_np)
    if _a_sq.ndim == 1: _a_sq = _a_sq[np.newaxis, :]
    out, npaths = {}, []
    _eps = 1e-30
    for _i, _r in enumerate(rx_list):
        try:
            _row = _a_sq[_i] if _a_sq.ndim > 1 else _a_sq
            npaths.append(int(np.sum(np.abs(_row) > 1e-20)))
            _pwr_incoh = float(np.sum(np.abs(_row)**2))
            _pwr_coh   = float(np.abs(np.sum(_row))**2)
            _pwr_best  = float(np.max(np.abs(_row)**2)) if _row.size > 0 else 0.0
            out[_r.name] = {
                "incoh": -10*np.log10(_pwr_incoh) if _pwr_incoh > _eps else np.nan,
                "coh":   -10*np.log10(_pwr_coh)   if _pwr_coh   > _eps else np.nan,
                "best":  -10*np.log10(_pwr_best)  if _pwr_best  > _eps else np.nan,
            }
        except Exception:
            out[_r.name] = {"best": np.nan, "incoh": np.nan, "coh": np.nan}
    mnp = float(np.mean(npaths)) if npaths else 0.0
    del _p; gc.collect()
    return out, mnp

# ── Metrics: N, Bias, MSE, RMSE, STD, R² ─────────────────────────────────────
def _metrics(sim, meas):
    mask = ~(np.isnan(sim) | np.isnan(meas))
    n = int(mask.sum())
    if n < 2:
        return n, np.nan, np.nan, np.nan, np.nan, np.nan
    s, m = sim[mask], meas[mask]
    err  = s - m
    mse  = float(np.mean(err**2))
    return (n,
            float(np.mean(err)),          # Bias
            mse,                           # MSE
            float(np.sqrt(mse)),           # RMSE
            float(np.std(err)),            # STD
            float(r2_score(m, s)))         # R²

results = []
print()
_HDR = f'{"":>14} {"N":>4} {"Bias":>7} {"MSE":>7} {"RMSE":>6} {"STD":>6} {"R²":>7} {"paths":>7}'
_SEP = "-" * len(_HDR)

for _lo, _hi in BANDS:
    label   = f"{_lo}-{_hi if _hi < 99999 else 'inf'}m"
    _subset = [v for v in _valid_rx.values() if _lo <= v["dist_m"] < _hi]
    N = len(_subset)
    if N == 0:
        print(f"{label:>13}: no receivers"); continue
    _sps     = _adaptive_sps(min(_hi, 9000))
    _rx_objs = [v["rx"]          for v in _subset]
    _pl_meas = np.array([v["measured_pl"] for v in _subset])

    print(f"{label:>13}  N={N:4d}  sps={_sps/1e6:.0f}M  solving ON...", end=" ", flush=True)
    t0 = time.time()
    _ron, _mon = _solve(_rx_objs, _cfg_on,  _sps)
    print("OFF...", end=" ", flush=True)
    _rof, _mof = _solve(_rx_objs, _cfg_off, _sps)
    print(f"{time.time()-t0:.0f}s")

    # ── extract PL arrays for each method ────────────────────────────────────
    # Apply scalar offset from Cell 10b (0.0 dB if not loaded)
    for _meth in ("incoh", "coh", "best"):
        _son = np.array([_ron.get(r.name, {}).get(_meth, np.nan) for r in _rx_objs]) - SCALAR_OFFSET_DB
        _sof = np.array([_rof.get(r.name, {}).get(_meth, np.nan) for r in _rx_objs]) - SCALAR_OFFSET_DB
        no,bo,mseo,ro,sto,r2o = _metrics(_son, _pl_meas)
        nf,bf,msef,rf,stf,r2f = _metrics(_sof, _pl_meas)
        tag = f"ON  {_meth}"
        print(f"  {tag:<14} {no:>4} {bo:>+7.1f} {mseo:>7.1f} {ro:>6.1f} {sto:>6.1f} {r2o:>7.3f} {_mon:>7.0f}")
        tag = f"OFF {_meth}"
        print(f"  {tag:<14} {nf:>4} {bf:>+7.1f} {msef:>7.1f} {rf:>6.1f} {stf:>6.1f} {r2f:>7.3f} {_mof:>7.0f}")
        results.append(dict(
            band=label, lo=_lo, hi=_hi, N=N, sps=_sps, scatter="ON",  method=_meth,
            n_valid=no, bias=bo, mse=mseo, rmse=ro, std=sto, r2=r2o, paths=_mon))
        results.append(dict(
            band=label, lo=_lo, hi=_hi, N=N, sps=_sps, scatter="OFF", method=_meth,
            n_valid=nf, bias=bf, mse=msef, rmse=rf, std=stf, r2=r2f, paths=_mof))

    # store incoherent ON sim RSSI for downstream cells
    for _r8, _v8 in zip(_rx_objs, _subset):
        _pl_on  = _ron.get(_r8.name, {}).get("incoh", np.nan)
        _pl_off = _rof.get(_r8.name, {}).get("incoh", np.nan)
        _v8["sim_pl_db"]        = _pl_on
        _v8["sim_rssi_dbm"]     = TX_CONDUCTED_DBM - _pl_on   if not np.isnan(_pl_on)  else np.nan
        _v8["sim_rssi_off_dbm"] = TX_CONDUCTED_DBM - _pl_off  if not np.isnan(_pl_off) else np.nan
    print()

# restore full receiver set
for _nm in list(scene.receivers.keys()): scene.remove(_nm)
for _rx in receivers: scene.add(_rx)

_df8 = pd.DataFrame(results)

# ── summary table (incoherent ON, primary metric) ─────────────────────────────
print("=" * 78)
print("SUMMARY — Incoherent ON (primary metric for mobile/drive-test)")
print(f'{"Band":>13} {"N":>4} {"sps":>5}  {"Bias":>7} {"MSE":>7} {"RMSE":>6} {"STD":>6} {"R²":>7} {"paths":>8}')
print("-" * 78)
for _row in results:
    if _row["scatter"] == "ON" and _row["method"] == "incoh":
        print(f'{_row["band"]:>13} {int(_row["N"]):>4} {_row["sps"]/1e6:>4.0f}M  '
              f'{_row["bias"]:>+7.1f} {_row["mse"]:>7.1f} {_row["rmse"]:>6.1f} '
              f'{_row["std"]:>6.1f} {_row["r2"]:>7.3f} {_row["paths"]:>8.0f}')
print("=" * 78)

# ── per-receiver CSV ──────────────────────────────────────────────────────────
_per_rx_rows = []
for _nm, _vv in _valid_rx.items():
    _per_rx_rows.append({
        "name":             _nm,
        "dist_m":           round(_vv.get("dist_m",           np.nan), 1),
        "measured_rssi":    round(_vv.get("measured_rssi",    np.nan), 2),
        "measured_pl":      round(_vv.get("measured_pl",      np.nan), 2),
        "sim_pl_db":        round(_vv.get("sim_pl_db",        np.nan), 2),
        "sim_rssi_dbm":     round(_vv.get("sim_rssi_dbm",     np.nan), 2),
        "sim_rssi_off_dbm": round(_vv.get("sim_rssi_off_dbm", np.nan), 2),
    })
_df_per_rx = pd.DataFrame(_per_rx_rows)
_out_per_rx = os.path.join(OUT_DIR, f'cell8_per_rx_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.csv')
_df_per_rx.to_csv(_out_per_rx, index=False)
print(f"Per-RX CSV : {_out_per_rx}")

_out8 = os.path.join(OUT_DIR, f'banded_analysis_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.csv')
_df8.to_csv(_out8, index=False)
print(f"Bands CSV  : {_out8}")
if "_report" in dir():
    _report["cell8_per_rx_csv"] = _out_per_rx
    _report["cell8_bands"]      = _df8.to_dict("records")
    _report["cell8_csv"]        = _out8
    _report["cell8_n_total"]    = int(_df8["N"].unique().sum() // 2)
    _incoh_on = _df8[(_df8["scatter"]=="ON") & (_df8["method"]=="incoh")]
    _w8 = _incoh_on["n_valid"].values.astype(float)
    _b8 = _incoh_on["bias"].values
    _r8 = _incoh_on["rmse"].values
    _mk = ~(np.isnan(_b8) | np.isnan(_r8))
    if _mk.any():
        _report["cell8_overall_bias_on"] = round(float(np.average(_b8[_mk], weights=_w8[_mk])), 2)
        _report["cell8_overall_rmse_on"] = round(float(np.sqrt(np.average(_r8[_mk]**2, weights=_w8[_mk]))), 2)
    print(f'Report: overall incoh ON bias={_report.get("cell8_overall_bias_on","N/A"):+} '
          f'RMSE={_report.get("cell8_overall_rmse_on","N/A")} dB')


## CELL 8b — Outlier Diagnostic

Lists the worst |error| receivers in a chosen band with:
distance, measured RSSI, sim RSSI, error, path count, RX height, terrain height, AGL.

Set `BAND_LO` / `BAND_HI` to inspect any distance range.
Self-contained: rebuilds `_valid_rx` if CELL 8 hasn't run.

**Interpret path counts:**
- < 10 paths → ray starvation or deep shadow
- > 50 paths but large error → genuine scene geometry gap


In [ ]:
# ====================================================================
# CELL 8b — OUTLIER DIAGNOSTIC (300-500m RMSE spike)
# ====================================================================
# Solves the 300-500m receivers and lists the worst |error| points
# with distance, RX height, terrain z, and n_paths so we can tell
# apart: height bug vs geometry gap (n_paths low) vs measurement noise.
# Run AFTER CELL 8.
# ====================================================================
import numpy as np, pandas as pd, gc

print("=" * 78)
print("CELL 8b — OUTLIER DIAGNOSTIC  (300-500m band)")
print("=" * 78)

BAND_LO, BAND_HI = 300.0, 500.0
TOPN             = 15

_safe = lambda v: float(v.item()) if hasattr(v, "item") else float(v)
_txp  = list(scene.transmitters.values())[0]
_txxy = np.array([_safe(_txp.position[0]), _safe(_txp.position[1])])

# ── standalone: load config + _valid_rx from disk if not in memory ───────────
import os, json as _jcfg, glob as _gl
import numpy as np, pandas as pd

def _load_session():
    """Find session_config.json by searching common OUT_DIR locations."""
    _candidates = sorted(_gl.glob(
        os.path.expanduser('~/sionna_rt/*/results/session_config.json')))
    if not _candidates:
        raise FileNotFoundError(
            'session_config.json not found — run CELL 1 once to create it.')
    with open(_candidates[-1]) as _f:
        return _jcfg.load(_f)

def _load_valid_rx(out_dir):
    _csvs = sorted(_gl.glob(os.path.join(out_dir, 'cell8_per_rx_*.csv')))
    if not _csvs:
        raise FileNotFoundError(
            f'No cell8_per_rx_*.csv in {out_dir} — run CELL 8 first.')
    _df = pd.read_csv(_csvs[-1])
    print(f'  RX data : {_csvs[-1]}')
    return {str(r['name']): {
        'dist_m':           float(r['dist_m']),
        'measured_rssi':    float(r['measured_rssi']),
        'sim_rssi_dbm':     float(r['sim_rssi_dbm']),
        'sim_rssi_off_dbm': float(r['sim_rssi_off_dbm']),
    } for _, r in _df.iterrows()}

if 'OUT_DIR' not in dir():
    _cfg = _load_session()
    OUT_DIR            = _cfg['OUT_DIR']
    BASE_DIR           = _cfg['BASE_DIR']
    SCENE_DIR          = _cfg['SCENE_DIR']
    NDSM_TIFF          = _cfg['NDSM_TIFF']
    MEASUREMENT_CSV    = _cfg['MEASUREMENT_CSV']
    FREQUENCY_HZ       = _cfg['FREQUENCY_HZ']
    TX_CONDUCTED_DBM   = _cfg['TX_CONDUCTED_DBM']
    RX_EXTRA_GAIN_DB   = _cfg['RX_EXTRA_GAIN_DB']
    SITE_CORRECTION_DB = _cfg['SITE_CORRECTION_DB']
    RX_AGL_M           = _cfg['RX_AGL_M']
    MAX_DEPTH          = int(_cfg['MAX_DEPTH'])
    NUM_SAMPLES_PS     = int(_cfg['NUM_SAMPLES_PS'])
    print(f'  Config  : loaded from {OUT_DIR}/session_config.json')

if '_valid_rx' not in dir() or not _valid_rx:
    print('  _valid_rx not in memory — loading from CSV')
    _valid_rx = _load_valid_rx(OUT_DIR)



_band = {n: v for n, v in _valid_rx.items() if BAND_LO <= v["dist_m"] < BAND_HI}
print(f"Receivers in {BAND_LO:.0f}-{BAND_HI:.0f}m band: {len(_band)}")

_rx_objs = [v["rx"] for v in _band.values()]
_names   = list(_band.keys())

# Solve with scattering ON, generous rays for this near band
_cfg = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
            diffraction=True, edge_diffraction=True, diffuse_reflection=True,
            samples_per_src=min(NUM_SAMPLES_PS * 4, 80_000_000))

for _n in list(scene.receivers.keys()): scene.remove(_n)
for _r in _rx_objs: scene.add(_r)
_p = PathSolver()(scene, **_cfg)

_a_raw = _p.a
if isinstance(_a_raw, tuple):
    _a_np = ((_a_raw[0].numpy() if hasattr(_a_raw[0], "numpy") else np.array(_a_raw[0])) +
             1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], "numpy") else np.array(_a_raw[1])))
else:
    _a_np = _a_raw.numpy() if hasattr(_a_raw, "numpy") else np.array(_a_raw)
_a_sq = np.squeeze(_a_np)
if _a_sq.ndim == 1: _a_sq = _a_sq[np.newaxis, :]

_rows = []
for _i, _r in enumerate(_rx_objs):
    _row  = _a_sq[_i] if _a_sq.ndim > 1 else _a_sq
    _pwr  = float(np.sum(np.abs(_row) ** 2))
    _np_  = int(np.sum(np.abs(_row) > 1e-20))
    _sim  = rssi_from_path_gain(_pwr) if _pwr > 1e-30 else np.nan
    _meas = _band[_r.name]["measured_rssi"]
    _err  = _sim - _meas if np.isfinite(_sim) else np.nan
    _lx, _ly, _lz = _safe(_r.position[0]), _safe(_r.position[1]), _safe(_r.position[2])
    _tz   = terrain_z(_lx, _ly)
    _agl  = _lz - _tz
    _rows.append(dict(name=_r.name, dist_m=_band[_r.name]["dist_m"],
                      meas=_meas, sim=_sim, err=_err, n_paths=_np_,
                      rx_z=_lz, terrain_z=_tz, agl=_agl))

del _p; gc.collect()
# restore
for _n in list(scene.receivers.keys()): scene.remove(_n)
for _r in receivers: scene.add(_r)

_d = pd.DataFrame(_rows)
_d["abserr"] = _d["err"].abs()
_d = _d.sort_values("abserr", ascending=False)

print()
print(f'{"name":<14}{"dist":>7}{"meas":>8}{"sim":>8}{"err":>8}{"paths":>8}{"rx_z":>8}{"terr_z":>8}{"agl":>7}')
print("-" * 78)
for _, r in _d.head(TOPN).iterrows():
    _s = f'{r["sim"]:>8.1f}' if np.isfinite(r["sim"]) else f'{"NaN":>8}'
    print(f'{str(r["name"]):<14}{r["dist_m"]:>7.0f}{r["meas"]:>8.1f}{_s}{r["err"]:>8.1f}'
          f'{int(r["n_paths"]):>8}{r["rx_z"]:>8.1f}{r["terrain_z"]:>8.1f}{r["agl"]:>7.1f}')
print("-" * 78)

_nan = int(_d["sim"].isna().sum())
print(f'\nSummary: N={len(_d)}  NaN(no paths)={_nan}  '
      f'bias={_d["err"].mean():+.1f} dB  RMSE={(_d["err"]**2).mean()**0.5:.1f} dB')
print(f'  AGL range : {_d["agl"].min():.1f} to {_d["agl"].max():.1f} m  (expect ~{RX_AGL_M} m)')
_lowp = _d[_d["n_paths"] < 50]
print(f'  Receivers with <50 paths : {len(_lowp)}  (geometry gap / shadow if many)')
_badh = _d[(_d["agl"] - RX_AGL_M).abs() > 3]
print(f'  Receivers with AGL off by >3m : {len(_badh)}  (height bug if many)')


## CELL DIAG-NEAR — Near-range bias investigation (<700m)
Maps receiver positions and identifies geographic zones driving the near-range bias.
Run after CELL 8.

In [ ]:
# ====================================================================
# CELL DIAG-NEAR — Near-range bias investigation (<700m)
# Maps receiver positions colored by simulation error and path count.
# Identifies geographic zones driving the <700m bias.
# Run after CELL 8 (needs _report['diag'] data).
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.colors as mcolors
import os, csv, math

print('=' * 70)
print('CELL DIAG-NEAR — Near-range bias investigation')
print('=' * 70)

# ── Load measurement CSV ─────────────────────────────────────────────
_mcsv = MEASUREMENT_CSV
_rows = []
with open(_mcsv, newline='') as _f:
    for _ in range(22): next(_f)   # skip preamble lines
    _rdr = csv.DictReader(_f)
    for _row in _rdr:
        _rows.append(_row)

_lon_c = [c for c in _rows[0].keys() if 'lon' in c.lower()][0]
_lat_c = [c for c in _rows[0].keys() if 'lat' in c.lower()][0]
_rssi_c = [c for c in _rows[0].keys() if 'measurement' in c.lower() or ('rssi' in c.lower() and 'dbm' in c.lower())][0]

# ── Build receiver table ──────────────────────────────────────────────
from pyproj import Transformer
_t = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
_cx, _cy = _t.transform(TX_LON, TX_LAT)

_dlat_m = 111320.0
_dlon_m = 111320.0 * math.cos(math.radians(TX_LAT))

_rec_rows = []
for _i, _row in enumerate(_rows[:NUM_RX]):
    try:
        _lat = float(_row[_lat_c])
        _lon = float(_row[_lon_c])
        _rssi = float(_row[_rssi_c])
    except (ValueError, TypeError):
        continue
    _e, _n = _t.transform(_lon, _lat)
    _dist = math.sqrt((_e - _cx)**2 + (_n - _cy)**2)
    _rec_rows.append({'idx': _i, 'lat': _lat, 'lon': _lon,
                      'rssi': _rssi, 'dist_m': _dist})

_df = pd.DataFrame(_rec_rows)
print(f'Loaded {len(_df)} receiver positions from CSV')

# ── Attach simulation results from _report if available ──────────────
_diag = globals().get('_report', {}).get('diag', {})
_sim_data = {}
if _diag:
    for _k, _v in _diag.items():
        if isinstance(_v, dict) and 'n_paths' in _v:
            _idx = int(_k.replace('RX_', ''))
            _sim_data[_idx] = _v

# ── Band statistics ───────────────────────────────────────────────────
for _lo, _hi in [(0,100),(100,300),(300,500),(500,700)]:
    _band = _df[(_df['dist_m'] >= _lo) & (_df['dist_m'] < _hi)]
    print(f'  {_lo:>4}-{_hi}m : {len(_band):>4} receivers'
          f'  lat=[{_band["lat"].min():.4f}, {_band["lat"].max():.4f}]'
          f'  lon=[{_band["lon"].min():.4f}, {_band["lon"].max():.4f}]')

# ── Identify path-count anomalies in CELL 8 results ──────────────────
print()
print('Path-count anomalies from CELL 8 (>50K paths at <700m):')
_anomalies = []
for _k, _v in _sim_data.items():
    if isinstance(_v, dict):
        _np = _v.get('n_paths', 0)
        _dist = _v.get('dist_m', 0)
        if _np > 50000 and _dist < 700:
            _anomalies.append((_k, _dist, _np, _v.get('pl_err', None)))

_anomalies.sort(key=lambda x: x[2], reverse=True)
for _k, _d, _np, _err in _anomalies[:10]:
    print(f'  RX_{_k:06d}  {_d:.0f}m  paths={_np:>7,}  pl_err={_err:+.1f} dB' if _err else
          f'  RX_{_k:06d}  {_d:.0f}m  paths={_np:>7,}')

# ── Map: receiver positions colored by distance band ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

_near_short = _df[_df['dist_m'] < 700].copy()
_colors = plt.cm.RdYlGn(1 - _near_short['dist_m'] / 700)

ax = axes[0]
ax.scatter(_near_short['lon'], _near_short['lat'], c=_near_short['dist_m'],
           cmap='YlOrRd', s=20, alpha=0.7, vmin=0, vmax=700)
ax.scatter(TX_LON, TX_LAT, marker='*', s=300, c='blue', zorder=5, label='TX')
# Annotate anomalous receivers
for _k, _d, _np, _err in _anomalies[:5]:
    if _k < len(_df):
        _row = _df[_df['idx'] == _k]
        if len(_row):
            ax.annotate(f'RX_{_k:06d}\n{_np//1000}K paths',
                       (_row['lon'].values[0], _row['lat'].values[0]),
                       fontsize=7, color='red',
                       xytext=(5, 5), textcoords='offset points')
            ax.scatter(_row['lon'].values[0], _row['lat'].values[0],
                      marker='x', s=100, c='red', zorder=6)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Receivers <700m (color=distance)\nRed X = path-count anomaly (>50K paths)')
ax.legend(); ax.grid(True, alpha=0.3)

# ── Scatter: measured PL vs distance, colored by error ───────────────
ax2 = axes[1]
_tx_power = TX_CONDUCTED_DBM
_fspl_dbm = lambda d: _tx_power - (20*np.log10(d) + 20*np.log10(FREQUENCY_HZ) - 147.55)
_d_arr = np.linspace(10, 700, 200)
ax2.plot(_d_arr, _fspl_dbm(_d_arr), 'b--', lw=1.5, label='FSPL (free space)', alpha=0.7)
ax2.scatter(_near_short['dist_m'], _near_short['rssi'], c='grey', s=10, alpha=0.5, label='Measured RSSI')
for _k, _d, _np, _err in _anomalies[:5]:
    if _k < len(_df):
        _row = _df[_df['idx'] == _k]
        if len(_row):
            ax2.scatter(_row['dist_m'].values[0], _row['rssi'].values[0],
                       marker='x', s=100, c='red', zorder=6)
ax2.set_xlabel('Distance from TX (m)'); ax2.set_ylabel('RSSI (dBm)')
ax2.set_title('Measured RSSI vs Distance\nRed X = path-count anomaly')
ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.invert_yaxis()

plt.tight_layout()
_fig_path = os.path.join(BASE_DIR, 'diag_near_range_bias.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved: {_fig_path}')

# ── Geographic centroid of anomalous receivers ────────────────────────
if _anomalies:
    _anon_lats = [_df[_df['idx'] == _k]['lat'].values[0]
                  for _k, _d, _np, _err in _anomalies[:5]
                  if len(_df[_df['idx'] == _k])]
    _anon_lons = [_df[_df['idx'] == _k]['lon'].values[0]
                  for _k, _d, _np, _err in _anomalies[:5]
                  if len(_df[_df['idx'] == _k])]
    if _anon_lats:
        _clat = np.mean(_anon_lats)
        _clon = np.mean(_anon_lons)
        print(f'\nAnomaly zone centroid: lat={_clat:.5f}  lon={_clon:.5f}')
        print(f'Google Maps: https://www.google.com/maps/@{_clat},{_clon},17z')



## CELL 8e — Cumulative Distance Evaluation (Scattering ON vs OFF)

Evaluates prediction accuracy at distance thresholds from 100 m to 4 km.
Shows how RMSE evolves with range — useful for identifying distance-dependent biases.


In [ ]:
# ====================================================================
# CELL 8e — CUMULATIVE DISTANCE EVALUATION  (scattering ON vs OFF)
# Each distance threshold solved independently with its own PathSolver.
# Three PL combining methods: best | incoh (primary) | coh
# PL_meas = TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB - RSSI_meas  (TX antenna gain already in Sionna path_gain via dipole pattern)
# paths.a already includes TX/RX antenna patterns — no double-counting.
# Reports avg rays/RX per threshold row.
# Exports: cumulative_eval.csv + cumulative_eval.png
# ====================================================================
import os, gc, time
import numpy as np, pandas as pd, csv as _csv_mod
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score as _r2s
from sionna.rt import PathSolver

print("=" * 95)
print("CELL 8e — CUMULATIVE DISTANCE EVALUATION  (scattering ON vs OFF)")
print("=" * 95)

_safe = lambda v: float(v.item()) if hasattr(v, "item") else float(v)
_txp  = list(scene.transmitters.values())[0]
_txxy = np.array([_safe(_txp.position[0]), _safe(_txp.position[1])])

_df_m = pd.read_csv(MEASUREMENT_CSV)
_nc   = [c for c in _df_m.columns if "name" in c.lower() or "id" in c.lower()][0]
_rc   = [c for c in _df_m.columns if "measurement" in c.lower()
         or ("rssi" in c.lower() and "dbm" in c.lower())][0]
_meas_map = {str(r[_nc]): TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB - float(r[_rc]) for _, r in _df_m.iterrows()}
print(f"PL_meas reference: TX_CONDUCTED_DBM={TX_CONDUCTED_DBM} + RX_chain={RX_EXTRA_GAIN_DB} dBm  (TX antenna gain in Sionna path_gain)")

_cfg_on  = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, refraction=True,
                diffuse_reflection=True, samples_per_src=NUM_SAMPLES_PS)
_cfg_off = {**_cfg_on, "diffuse_reflection": False}

_rx_dist = {}
for _r in receivers:
    _pos = np.array([_safe(_r.position[0]), _safe(_r.position[1])])
    _rx_dist[_r.name] = float(np.linalg.norm(_pos - _txxy))

# ── ITU-R P.1411 dual-slope breakpoint ──────────────────────────────────────
# Rbp = 4 × hBS × hUT × f/c  (Stevenage: TX_AGL=17m, RX_AGL=1.5m, 2695 MHz)
_rbp = 4.0 * globals().get('TX_AGL_M', 17.0) * 1.5 * (FREQUENCY_HZ / 3e8)  # ~312 m
_BIN_SCALAR    = globals().get('DUAL_LOS_NLOS_SCALAR', True)
_N_SCALAR_BINS = globals().get('N_SCALAR_BINS', 5)
_ZONE_SPLIT    = globals().get('LOS_NLOS_ZONE_SPLIT', False)

_EPS = 1e-30

def _pl_from_row(row):
    pi = float(np.sum(np.abs(row)**2))
    pc = float(np.abs(np.sum(row))**2)
    pb = float(np.max(np.abs(row)**2)) if row.size > 0 else 0.0
    n_rays = int(np.sum(np.abs(row) > _EPS))
    return (
        -10*np.log10(pb) if pb > _EPS else np.nan,
        -10*np.log10(pi) if pi > _EPS else np.nan,
        -10*np.log10(pc) if pc > _EPS else np.nan,
        n_rays,
    )

def _solve_threshold(rx_subset, cfg):
    _out  = {}
    _BATCH = 10
    for _i0 in range(0, len(rx_subset), _BATCH):
        _b = rx_subset[_i0:_i0+_BATCH]
        for _n in list(scene.receivers.keys()): scene.remove(_n)
        for _r in _b: scene.add(_r)
        _p   = PathSolver()(scene, **cfg)
        _a   = _p.a
        _anp = ((_a[0].numpy()+1j*_a[1].numpy()) if isinstance(_a, tuple) else _a.numpy())
        _anp = np.squeeze(_anp)
        if _anp.ndim == 0:
            _anp = np.zeros((len(_b), 1), complex)
        elif _anp.ndim == 1:
            _anp = _anp[np.newaxis, :]
        for _j, _r in enumerate(_b):
            _row = _anp[_j] if _j < _anp.shape[0] else np.zeros(1, complex)
            _best, _incoh, _coh, _nr = _pl_from_row(_row)
            _out[_r.name] = {"best": _best, "incoh": _incoh, "coh": _coh, "n_rays": _nr}
        del _p; gc.collect()
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _r in receivers: scene.add(_r)
    return _out

def _metrics(sim, meas):
    mask = ~(np.isnan(sim) | np.isnan(meas))
    n = int(mask.sum())
    if n < 2: return n, np.nan, np.nan, np.nan, np.nan, np.nan
    s, m  = sim[mask], meas[mask]
    err   = s - m
    mse   = float(np.mean(err**2))
    return n, float(np.mean(err)), mse, float(np.sqrt(mse)), float(np.std(err)), float(_r2s(m, s))

_thresholds_m = [100, 200, 300, 500, 750, 900, 1000, 1250,
                 1500, 1750, 2000, 2250, 2500, 2750, 3000, 3500, 4000]

# ── Weissberger vegetation post-hoc attenuation (ITU-R P.833-9 §4.2) ─────────
# Formula: L_veg_dB = 0.187 * f_GHz^0.284 * d_veg^0.588
# Applied per-receiver based on 2D path intersection through vegetation polygons.
# Vegetation polygon data loaded from vegetation_footprints.geojson (CELL 4 output).
_f_ghz_8e = FREQUENCY_HZ / 1e9

def _weissberger_atten_db(depth_m):
    # ITU-R P.833-9 §4.2 single-regime formula (portable, matches spec)
    d = max(0.0, float(depth_m))
    if d <= 0:
        return 0.0
    return 0.187 * (_f_ghz_8e ** 0.284) * (d ** 0.588)

# Load vegetation polygon footprints for path intersection test
_veg_shapes_8e = []
_rx_veg_depth_m = {}
try:
    import json as _js_8e
    from shapely.geometry import LineString as _LS_8e, shape as _shape_8e
    from pyproj import Transformer as _Tr_8e
    _veg_candidates_8e = [
        os.path.join(BASE_DIR, 'scene_v4_full', 'vegetation_footprints.geojson'),
        os.path.join(BASE_DIR, 'scene',         'vegetation_footprints.geojson'),
        os.path.join(BASE_DIR,                  'vegetation_footprints.geojson'),
    ]
    _veg_path_8e = next((p for p in _veg_candidates_8e if os.path.exists(p)), None)
    if _veg_path_8e:
        with open(_veg_path_8e) as _fv8:
            _veg_shapes_8e = [_shape_8e(ft['geometry'])
                               for ft in _js_8e.load(_fv8)['features']]
        print(f'Weissberger: {len(_veg_shapes_8e)} vegetation polygons loaded')
        _to_proj_8e = _Tr_8e.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
        _tx_proj_8e = _to_proj_8e.transform(TX_LON, TX_LAT)
        _df_m_8e = pd.read_csv(MEASUREMENT_CSV)
        _nc_8e   = [c for c in _df_m_8e.columns if 'name' in c.lower() or 'id' in c.lower()][0]
        _lon_8e  = next((c for c in _df_m_8e.columns if c.lower() == 'lon'), None)
        _lat_8e  = next((c for c in _df_m_8e.columns if c.lower() == 'lat'), None)
        if _lon_8e and _lat_8e:
            from shapely.strtree import STRtree as _STRtree_8e
            from shapely.geometry import box as _box_8e
            _veg_strtree_8e = _STRtree_8e(_veg_shapes_8e)

            # Building exclusion: LOS paths blocked by buildings skip Weissberger
            # (received signal arrives via reflected/diffracted routes, not through veg)
            _bld_shapes_8e = []
            _bld_strtree_8e = None
            _bld_candidates_8e = [
                os.path.join(BASE_DIR, 'scene_v4_full', 'building_footprints.geojson'),
                os.path.join(BASE_DIR, 'scene', 'building_footprints.geojson'),
                os.path.join(BASE_DIR, 'building_footprints.geojson'),
            ]
            _bld_fp_8e = next((p for p in _bld_candidates_8e if os.path.exists(p)), None)
            if _bld_fp_8e:
                try:
                    with open(_bld_fp_8e) as _fb8:
                        _bld_shapes_8e = [_shape_8e(ft['geometry'])
                                          for ft in _js_8e.load(_fb8)['features']
                                          if ft['geometry']['type'] in ('Polygon','MultiPolygon')]
                    _bld_strtree_8e = _STRtree_8e(_bld_shapes_8e)
                    print(f'Weissberger building exclusion: {len(_bld_shapes_8e)} footprints loaded')
                except Exception as _eb8:
                    print(f'[Weissberger] building footprints load failed ({_eb8}) — exclusion skipped')
            else:
                print('[Weissberger] building_footprints.geojson not found — exclusion disabled')

            _n_excluded = 0
            for _, _row_c in _df_m_8e.iterrows():
                _rlon = float(_row_c[_lon_8e]); _rlat = float(_row_c[_lat_8e])
                _rx_proj_8e = _to_proj_8e.transform(_rlon, _rlat)
                _path_8e    = _LS_8e([_tx_proj_8e, _rx_proj_8e])
                # Skip vegetation attenuation if LOS crosses a building footprint
                if _bld_strtree_8e is not None:
                    _bc = _bld_strtree_8e.query(_path_8e)
                    if any(_bld_shapes_8e[_i].intersects(_path_8e) for _i in _bc):
                        _rx_veg_depth_m[str(_row_c[_nc_8e])] = 0.0
                        _n_excluded += 1
                        continue
                _cands_8e   = _veg_strtree_8e.query(_path_8e)
                _depth_8e   = sum(_path_8e.intersection(_veg_shapes_8e[_i]).length
                                   for _i in _cands_8e
                                   if _path_8e.intersects(_veg_shapes_8e[_i]))
                _rx_veg_depth_m[str(_row_c[_nc_8e])] = float(_depth_8e)
            print(f'Weissberger: depths computed for {len(_rx_veg_depth_m)} receivers  ({_n_excluded} building-blocked excluded)')
    else:
        print('[Weissberger] vegetation_footprints.geojson not found — attenuation = 0 dB')
except Exception as _e_8e:
    print(f'[Weissberger] setup failed ({_e_8e}) — attenuation skipped')

def _apply_weissberger(pl_arr, rx_subset):
    # Apply Weissberger correction to path loss array (dB added = more loss).
    if not _rx_veg_depth_m:
        return pl_arr.copy()
    out = pl_arr.copy()
    for _j, _r in enumerate(rx_subset):
        _d = _rx_veg_depth_m.get(_r.name, 0.0)
        if _d > 0 and not np.isnan(out[_j]):
            out[_j] += _weissberger_atten_db(_d)
    return out

# ── Bin scalar + zone split functions ───────────────────────────────────────
def _fit_bin_scalar(rx_list, on_map, meas_map, min_d_m, max_d_m, n_bins, meth='incoh'):
    bin_edges  = np.linspace(min_d_m, max_d_m, n_bins + 1)
    bin_resids = [[] for _ in range(n_bins)]
    for r in rx_list:
        d = _rx_dist[r.name]
        if d < min_d_m or d > max_d_m:
            continue
        pl_sim  = on_map.get(r.name, {}).get(meth, np.nan)
        pl_meas = meas_map.get(r.name, np.nan)
        if np.isnan(pl_sim) or np.isnan(pl_meas):
            continue
        b = min(int(np.searchsorted(bin_edges[1:-1], d)), n_bins - 1)
        bin_resids[b].append(pl_meas - pl_sim)
    centers, means, counts = [], [], []
    for b in range(n_bins):
        if len(bin_resids[b]) >= 3:
            centers.append(0.5 * (bin_edges[b] + bin_edges[b + 1]))
            means.append(float(np.mean(bin_resids[b])))
            counts.append(len(bin_resids[b]))
    return np.array(centers), np.array(means), counts, bin_edges

def _apply_bin_scalar(pl_arr, rx_list, bin_centers, bin_means):
    out = pl_arr.copy()
    if len(bin_centers) == 0:
        return out
    if len(bin_centers) == 1:
        for i, r in enumerate(rx_list):
            if not np.isnan(out[i]):
                out[i] += float(bin_means[0])
        return out
    for i, r in enumerate(rx_list):
        if np.isnan(out[i]):
            continue
        out[i] += float(np.interp(_rx_dist[r.name], bin_centers, bin_means))
    return out

def _apply_zone_offset(pl_arr, rx_list, los_off, nlos_off):
    out = pl_arr.copy()
    for i, r in enumerate(rx_list):
        if np.isnan(out[i]):
            continue
        out[i] += los_off if _rx_dist[r.name] < _rbp else nlos_off
    return out

# Pre-solve: fit bin scalar from calibration range receivers
_cal_min_m    = globals().get('CAL_MIN_DIST_KM', 0.15) * 1000
_scalar_max_m = globals().get('SCALAR_FIT_MAX_DIST_KM',
                globals().get('CAL_MAX_DIST_KM', 1.5)) * 1000
_all_rx_for_scalar = [r for r in receivers
                      if _rx_dist.get(r.name, 0) >= _cal_min_m
                      and _rx_dist.get(r.name, 0) <= _scalar_max_m
                      and not np.isnan(_meas_map.get(r.name, np.nan))]

_bin_centers, _bin_means = np.array([]), np.array([])
if _BIN_SCALAR and len(_all_rx_for_scalar) >= 6:
    _pre_map = _solve_threshold(_all_rx_for_scalar, _cfg_on)
    _bin_centers, _bin_means, _bin_counts, _bin_edges = _fit_bin_scalar(
        _all_rx_for_scalar, _pre_map, _meas_map,
        _cal_min_m, _scalar_max_m, _N_SCALAR_BINS, meth='incoh')
    print(f"\n  Distance-bin scalar ({len(_all_rx_for_scalar)} cal receivers, {_N_SCALAR_BINS} bins, Rbp={_rbp/1000:.2f}km):")
    for _bc, _bm, _bn in zip(_bin_centers, _bin_means, _bin_counts):
        _zone = 'LOS ' if _bc <= _rbp else 'NLOS'
        print(f"    {_zone} d≈{_bc/1000:.2f}km  N={_bn:3d}:  {_bm:+.2f} dB")
else:
    if _BIN_SCALAR:
        print(f"  Bin scalar: insufficient cal receivers ({len(_all_rx_for_scalar)}) — no correction")
    else:
        print("  Bin scalar: DISABLED")

_los_zone_off = _nlos_zone_off = 0.0
if _ZONE_SPLIT and len(_bin_centers) > 0:
    _pre_bs   = np.array([_pre_map.get(r.name, {}).get('incoh', np.nan)
                          for r in _all_rx_for_scalar]) - globals().get("SCALAR_OFFSET_DB", 0.0)
    _pre_bs   = _apply_bin_scalar(_pre_bs, _all_rx_for_scalar, _bin_centers, _bin_means)
    _meas_arr = np.array([_meas_map.get(r.name, np.nan) for r in _all_rx_for_scalar])
    _dists_c  = np.array([_rx_dist[r.name] for r in _all_rx_for_scalar])
    _resid_c  = _meas_arr - _pre_bs
    _lm = _dists_c < _rbp;  _nm = _dists_c >= _rbp
    _los_zone_off  = float(np.nanmean(_resid_c[_lm]))  if int(_lm.sum()) >= 3 else 0.0
    _nlos_zone_off = float(np.nanmean(_resid_c[_nm])) if int(_nm.sum()) >= 3 else 0.0
    print(f"\n  LOS/NLOS zone split (Rbp={_rbp/1000:.2f} km):")
    print(f"    LOS  (d < Rbp): N={int(_lm.sum()):3d}  offset={_los_zone_off:+.2f} dB")
    print(f"    NLOS (d ≥ Rbp): N={int(_nm.sum()):3d}  offset={_nlos_zone_off:+.2f} dB")
elif _ZONE_SPLIT:
    print("  Zone split: skipped (bin scalar not fitted)")

_rows = []
_t0   = time.time()

for _thr in _thresholds_m:
    _rx_sub = [r for r in receivers
               if _rx_dist[r.name] <= _thr
               and not np.isnan(_meas_map.get(r.name, np.nan))]
    _n = len(_rx_sub)
    if _n < 2:
        print(f"  0-{_thr:4d}m  {_n:5d}  (insufficient data)")
        continue

    _t1 = time.time()
    _on_map  = _solve_threshold(_rx_sub, _cfg_on)
    _off_map = _solve_threshold(_rx_sub, _cfg_off)
    _elapsed = time.time() - _t1

    _avg_rays_on  = float(np.mean([_on_map[r.name]["n_rays"]  for r in _rx_sub]))
    _avg_rays_off = float(np.mean([_off_map[r.name]["n_rays"] for r in _rx_sub]))

    _meas = np.array([_meas_map[r.name] for r in _rx_sub])
    _row  = dict(threshold_m=_thr, N=_n, elapsed_s=round(_elapsed, 1),
                 avg_rays_on=round(_avg_rays_on, 1), avg_rays_off=round(_avg_rays_off, 1))

    print(f"\n  0-{_thr:4d}m  N={_n}  avg_rays ON={_avg_rays_on:.1f} OFF={_avg_rays_off:.1f}  [{_elapsed:.0f}s]")
    print(f"  {'Method':<14} {'N':>4} {'Bias':>7} {'MSE':>7} {'RMSE':>6} {'STD':>6} {'R2':>7}  (ON | OFF)")
    for _meth in ("incoh", "coh", "best"):
        _son_raw = np.array([_on_map.get(r.name,  {}).get(_meth, np.nan) for r in _rx_sub]) - globals().get("SCALAR_OFFSET_DB", 0.0)
        _sof_raw = np.array([_off_map.get(r.name, {}).get(_meth, np.nan) for r in _rx_sub]) - globals().get("SCALAR_OFFSET_DB", 0.0)
        if _BIN_SCALAR and len(_bin_centers) > 0:
            _son_raw = _apply_bin_scalar(_son_raw, _rx_sub, _bin_centers, _bin_means)
            _sof_raw = _apply_bin_scalar(_sof_raw, _rx_sub, _bin_centers, _bin_means)
        if _ZONE_SPLIT and (_los_zone_off != 0.0 or _nlos_zone_off != 0.0):
            _son_raw = _apply_zone_offset(_son_raw, _rx_sub, _los_zone_off, _nlos_zone_off)
            _sof_raw = _apply_zone_offset(_sof_raw, _rx_sub, _los_zone_off, _nlos_zone_off)
        _son = _apply_weissberger(_son_raw, _rx_sub)
        _sof = _apply_weissberger(_sof_raw, _rx_sub)
        non,bon,mson,ron,ston,r2on = _metrics(_son, _meas)
        nof,bof,msof,rof,stof,r2of = _metrics(_sof, _meas)
        print(f"  ON  {_meth:<10} {non:>4} {bon:>+7.1f} {mson:>7.1f} {ron:>6.1f} {ston:>6.1f} {r2on:>7.3f}")
        print(f"  OFF {_meth:<10} {nof:>4} {bof:>+7.1f} {msof:>7.1f} {rof:>6.1f} {stof:>6.1f} {r2of:>7.3f}")
        _row.update({
            f"{_meth}_on_n":    non,  f"{_meth}_on_bias":  bon,
            f"{_meth}_on_mse":  mson, f"{_meth}_on_rmse":  ron,
            f"{_meth}_on_std":  ston, f"{_meth}_on_r2":    r2on,
            f"{_meth}_off_n":   nof,  f"{_meth}_off_bias": bof,
            f"{_meth}_off_mse": msof, f"{_meth}_off_rmse": rof,
            f"{_meth}_off_std": stof, f"{_meth}_off_r2":   r2of,
            f"{_meth}_delta_rmse": (ron-rof) if not (np.isnan(ron) or np.isnan(rof)) else np.nan,
        })
    _rows.append(_row)

print("=" * 85)
print(f"Total time: {time.time()-_t0:.0f}s")

os.makedirs(OUT_DIR, exist_ok=True)
_csv_path = os.path.join(OUT_DIR, "cumulative_eval.csv")
_fields = ["threshold_m", "N", "elapsed_s", "avg_rays_on", "avg_rays_off"]
for _m in ("incoh", "coh", "best"):
    _fields += [f"{_m}_on_n",  f"{_m}_on_bias",  f"{_m}_on_mse",  f"{_m}_on_rmse",
                f"{_m}_on_std", f"{_m}_on_r2",
                f"{_m}_off_n", f"{_m}_off_bias", f"{_m}_off_mse", f"{_m}_off_rmse",
                f"{_m}_off_std", f"{_m}_off_r2",  f"{_m}_delta_rmse"]
with open(_csv_path, "w", newline="") as _f:
    _w = _csv_mod.DictWriter(_f, fieldnames=_fields)
    _w.writeheader(); _w.writerows(_rows)
print(f"CSV  : {_csv_path}  ({len(_rows)} rows)")

_df_r   = pd.DataFrame(_rows)
_thr_km = _df_r["threshold_m"].values / 1000
_METHS  = [("incoh","b","o","-","r","s","--"),
           ("coh",  "g","^","-","m","v","--"),
           ("best", "k","D",":","gray","x",":")]

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle("CELL 8e — Cumulative PL Evaluation: incoh / coh / best  x  Scatter ON / OFF", fontsize=12)
_panels = [
    ("on_bias",    "off_bias",  "Bias (dB)",         axes[0,0]),
    ("on_rmse",    "off_rmse",  "RMSE (dB)",         axes[0,1]),
    ("on_std",     "off_std",   "STD (dB)",          axes[0,2]),
    ("on_mse",     "off_mse",   "MSE (dB^2)",        axes[1,0]),
    ("on_r2",      "off_r2",    "R2",                axes[1,1]),
    ("delta_rmse", None,        "dRMSE ON-OFF (dB)", axes[1,2]),
]
for _con, _cof, _title, _ax in _panels:
    for _meth, _c_on, _mk_on, _ls_on, _c_off, _mk_off, _ls_off in _METHS:
        _y_on = _df_r[f"{_meth}_{_con}"].values
        _ax.plot(_thr_km, _y_on, color=_c_on, marker=_mk_on, linestyle=_ls_on, ms=4, label=f"{_meth} ON")
        if _cof:
            _y_off = _df_r[f"{_meth}_{_cof}"].values
            _ax.plot(_thr_km, _y_off, color=_c_off, marker=_mk_off, linestyle=_ls_off, ms=4, label=f"{_meth} OFF")
    _ax.set_title(_title); _ax.set_xlabel("Distance threshold (km)")
    _ax.grid(True, alpha=0.3); _ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
_png_path = os.path.join(OUT_DIR, "cumulative_eval.png")
plt.savefig(_png_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Chart: {_png_path}")

if "_report" in dir():
    _report["cumulative_eval"] = _rows

from matplotlib.ticker import MultipleLocator, FormatStrFormatter as _FmtStr
import matplotlib as _mpl_8e

# ── Thesis-quality figure settings ───────────────────────────────────────
_mpl_8e.rcParams.update({
    'font.family':      'serif',
    'font.size':        10,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   8,
    'axes.linewidth':   0.8,
    'grid.linewidth':   0.5,
    'lines.linewidth':  1.6,
    'lines.markersize': 5,
})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    "Cumulative Path-Loss Evaluation — Scattering ON vs OFF  "
    f"(f = {FREQUENCY_HZ/1e6:.1f} MHz)",
    fontsize=13, fontweight='bold', y=1.01)

# ── Panel definitions: (col_on, col_off, ylabel, ax, y_major, y_minor, ylim) ─
_panels = [
    ("on_bias",    "off_bias",  "Bias (dB)",         axes[0,0], 1.0, 0.5,  None),
    ("on_rmse",    "off_rmse",  "RMSE (dB)",         axes[0,1], 1.0, 0.5,  (0, None)),
    ("on_std",     "off_std",   "STD (dB)",          axes[0,2], 1.0, 0.5,  (0, None)),
    ("on_mse",     "off_mse",   r"MSE (dB$^2$)",     axes[1,0], None, None, None),
    ("on_r2",      "off_r2",    r"$R^2$",            axes[1,1], 0.1, 0.05, (-0.5, 1.05)),
    ("delta_rmse", None,        r"$\Delta$RMSE ON$-$OFF (dB)", axes[1,2], 0.5, 0.25, None),
]

for _con, _cof, _ylabel, _ax, _ymaj, _ymin, _ylim in _panels:
    for _meth, _c_on, _mk_on, _ls_on, _c_off, _mk_off, _ls_off in _METHS:
        _y_on = _df_r[f"{_meth}_{_con}"].values
        _ax.plot(_thr_km, _y_on, color=_c_on, marker=_mk_on,
                 linestyle=_ls_on, label=f"{_meth} ON",
                 markerfacecolor='white', markeredgewidth=1.2)
        if _cof:
            _y_off = _df_r[f"{_meth}_{_cof}"].values
            _ax.plot(_thr_km, _y_off, color=_c_off, marker=_mk_off,
                     linestyle=_ls_off, label=f"{_meth} OFF",
                     markerfacecolor='white', markeredgewidth=1.2)
    # reference lines
    if _con == "on_r2":
        _ax.axhline(0, color='k', linewidth=0.7, linestyle='--', alpha=0.4)
        _ax.axhline(1, color='k', linewidth=0.7, linestyle='--', alpha=0.4)
    if _con in ("on_bias", "delta_rmse"):
        _ax.axhline(0, color='k', linewidth=0.7, linestyle='--', alpha=0.4)
    # axes
    _ax.set_ylabel(_ylabel)
    _ax.set_xlabel("Distance threshold (km)")
    _ax.xaxis.set_major_locator(MultipleLocator(0.25))
    _ax.xaxis.set_minor_locator(MultipleLocator(0.05))
    _ax.xaxis.set_major_formatter(_FmtStr('%.2f'))
    if _ymaj:
        _ax.yaxis.set_major_locator(MultipleLocator(_ymaj))
    if _ymin:
        _ax.yaxis.set_minor_locator(MultipleLocator(_ymin))
    _ax.yaxis.set_major_formatter(_FmtStr('%.1f'))
    if _ylim:
        _lo8, _hi8 = _ylim
        _cur8 = _ax.get_ylim()
        _ax.set_ylim(_lo8 if _lo8 is not None else _cur8[0],
                     _hi8 if _hi8 is not None else _cur8[1])
    _ax.grid(True, which='major', alpha=0.35, linestyle='--')
    _ax.grid(True, which='minor', alpha=0.12, linestyle=':')
    _ax.legend(ncol=2, loc='best', framealpha=0.85,
               edgecolor='gray', fancybox=False)
    for _sp in ('top','right'): _ax.spines[_sp].set_visible(False)

plt.tight_layout()
_png2_path = os.path.join(OUT_DIR, "cumulative_eval_thesis.png")
plt.savefig(_png2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Thesis chart: {_png2_path}")

# reset rcParams to defaults so later cells are unaffected
_mpl_8e.rcParams.update(_mpl_8e.rcParamsDefault)

if "_report" in dir():
    _report["cumulative_eval"] = _rows

## CELL 8e-P833 — P.833 Cumulative Distance Impact

Applies ITU-R P.833 / Weissberger vegetation correction and re-evaluates all 6 methods
at each distance threshold. Exports `p833_cumulative_impact.csv`.

Formula: `A = min(20, 0.187 × f_GHz^0.284 × depth_m^0.588)` at 2695 MHz.

**Key finding:** Incoh ON + P.833 is the only valid combination — −1.54 dB RMSE at 4 km.


In [ ]:
# ====================================================================
# CELL 8e-P833 — P.833 IMPACT ON CUMULATIVE DISTANCE EVALUATION
# All 6 methods (Best/Incoh/Coh x ON/OFF) with and without P.833.
# Converts RSSI P.833 columns to PL (TX - RSSI) before comparison.
# No solver re-run — runs in seconds. Exports to CSV.
# ====================================================================
import numpy as np, pandas as pd, os
from sklearn.metrics import r2_score

_THRESHOLDS = [100, 200, 300, 500, 750, 900, 1000, 1250, 1500, 1750, 2000,
               2250, 2500, 2750, 3000, 3500, 4000]

# base PL columns and their P.833 RSSI counterparts
_METHODS = [
    ('pl_best_db',          'rssi_best_p833_dbm',          'Best_ON'),
    ('pl_incoherent_db',    'rssi_incoherent_p833_dbm',    'Incoh_ON'),
    ('pl_coherent_db',      'rssi_coherent_p833_dbm',      'Coh_ON'),
    ('pl_best_off_db',      'rssi_best_off_p833_dbm',      'Best_OFF'),
    ('pl_incoherent_off_db','rssi_incoherent_off_p833_dbm','Incoh_OFF'),
    ('pl_coherent_off_db',  'rssi_coherent_off_p833_dbm',  'Coh_OFF'),
]

_p833_ok = any(p in df_ps.columns for _, p, _ in _METHODS)
if not _p833_ok:
    print('[SKIP] P.833 columns not found — run Cell P833 first.')
else:
    # derive PL from P.833 RSSI columns
    for _bc, _pc, _lbl in _METHODS:
        if _pc in df_ps.columns and _bc in df_ps.columns:
            df_ps[f'_pl_p833_{_lbl}'] = TX_CONDUCTED_DBM - df_ps[_pc]

    _records = []

    for _thr in _THRESHOLDS:
        _sub = df_ps[df_ps['dist_from_tx_m'] <= _thr].copy()
        _row = {'threshold_m': _thr, 'N': len(_sub)}
        for _bc, _pc, _lbl in _METHODS:
            _p833_col = f'_pl_p833_{_lbl}'
            for _col, _suffix in [(_bc, 'base'), (_p833_col, 'p833')]:
                if _col not in df_ps.columns:
                    continue
                _s = _sub.dropna(subset=[_col, 'pl_meas'])
                _n = len(_s)
                if _n < 3:
                    continue
                _err = _s[_col].values - _s['pl_meas'].values
                _row[f'{_lbl}_{_suffix}_n']    = _n
                _row[f'{_lbl}_{_suffix}_bias']  = float(np.mean(_err))
                _row[f'{_lbl}_{_suffix}_rmse']  = float(np.sqrt(np.mean(_err**2)))
                _row[f'{_lbl}_{_suffix}_std']   = float(np.std(_err))
                _row[f'{_lbl}_{_suffix}_r2']    = float(r2_score(_s['pl_meas'].values, _s[_col].values))
            # delta RMSE
            if f'{_lbl}_base_rmse' in _row and f'{_lbl}_p833_rmse' in _row:
                _row[f'{_lbl}_delta_rmse'] = _row[f'{_lbl}_p833_rmse'] - _row[f'{_lbl}_base_rmse']
        _records.append(_row)

    df_p833 = pd.DataFrame(_records)

    # ── Print summary table per method ───────────────────────────────
    for _bc, _pc, _lbl in _METHODS:
        if f'{_lbl}_base_rmse' not in df_p833.columns:
            continue
        print(f'\n{"="*72}')
        print(f'  {_lbl}  —  Base vs P.833 corrected')
        print(f'{"="*72}')
        print(f'  {"Dist":>6}  {"N":>4}  '
              f'{"Base Bias":>9} {"Base RMSE":>9} {"Base R²":>7}  '
              f'{"P833 Bias":>9} {"P833 RMSE":>9} {"P833 R²":>7}  {"ΔRMSE":>7}')
        print('-'*72)
        for _, r in df_p833.iterrows():
            if f'{_lbl}_base_rmse' not in r or pd.isna(r.get(f'{_lbl}_base_rmse',np.nan)):
                continue
            print(f'  {int(r["threshold_m"]):>5}m  {int(r.get(f"{_lbl}_base_n",0)):>4}  '
                  f'{r[f"{_lbl}_base_bias"]:>+9.2f} {r[f"{_lbl}_base_rmse"]:>9.2f} {r[f"{_lbl}_base_r2"]:>+7.3f}  '
                  f'{r[f"{_lbl}_p833_bias"]:>+9.2f} {r[f"{_lbl}_p833_rmse"]:>9.2f} {r[f"{_lbl}_p833_r2"]:>+7.3f}  '
                  f'{r[f"{_lbl}_delta_rmse"]:>+7.2f} dB')
        print('='*72)

    # ── Export to CSV ─────────────────────────────────────────────────
    _csv_path = os.path.join(OUT_DIR, 'p833_cumulative_impact.csv')
    df_p833.to_csv(_csv_path, index=False)
    print(f'\nExported: {_csv_path}')
    print(f'Columns : {list(df_p833.columns)[:10]} ...')

    # cleanup temp columns
    df_ps.drop(columns=[c for c in df_ps.columns if c.startswith('_pl_p833_')],
               errors='ignore', inplace=True)
    print('ΔRMSE < 0: P.833 improves accuracy.  ΔRMSE > 0: over-correction.')


In [ ]:
# ====================================================================
# CELL 8e-P833 — P.833 IMPACT ON CUMULATIVE DISTANCE EVALUATION
# Uses df_ps (already computed + P.833 columns added by Cell P833).
# Converts RSSI → PL (TX_CONDUCTED_DBM − RSSI) before comparison.
# No solver re-run — runs in seconds.
# ====================================================================
import numpy as np, pandas as pd
from sklearn.metrics import r2_score

_THRESHOLDS = [100, 200, 300, 500, 750, 900, 1000, 1250, 1500, 1750, 2000,
               2250, 2500, 2750, 3000, 3500, 4000]

_p833_col = 'rssi_incoherent_p833_dbm'
_base_col = 'rssi_incoherent_dbm'
_TX_DBM   = float(globals().get('TX_CONDUCTED_DBM', 49.0))

if _p833_col not in df_ps.columns:
    print('[SKIP] P.833 columns not found — run Cell P833 first.')
else:
    # Convert RSSI → path loss for comparison with pl_meas
    _df = df_ps.dropna(subset=[_base_col, _p833_col, 'pl_meas']).copy()
    _df['_pl_base'] = _TX_DBM - _df[_base_col]
    _df['_pl_p833'] = _TX_DBM - _df[_p833_col]

    print('=' * 78)
    print('CELL 8e-P833 — Cumulative P.833 impact (Incoh ON)')
    print('=' * 78)
    print(f'  {"Dist":>6}  {"N":>4}  '
          f'{"Base Bias":>9} {"Base RMSE":>9} {"Base R²":>7}  '
          f'{"P833 Bias":>9} {"P833 RMSE":>9} {"P833 R²":>7}  {"ΔRMSE":>7}')
    print('-' * 78)

    for _thr in _THRESHOLDS:
        _sub = _df[_df['dist_from_tx_m'] <= _thr]
        _n = len(_sub)
        if _n < 3: continue
        _m  = _sub['pl_meas'].values
        _b  = _sub['_pl_base'].values
        _p  = _sub['_pl_p833'].values
        _eb = _b - _m;  _ep = _p - _m
        _bias_b = float(np.mean(_eb));  _rmse_b = float(np.sqrt(np.mean(_eb**2)))
        _bias_p = float(np.mean(_ep));  _rmse_p = float(np.sqrt(np.mean(_ep**2)))
        _r2_b   = float(r2_score(_m, _b))
        _r2_p   = float(r2_score(_m, _p))
        _dr     = _rmse_p - _rmse_b
        print(f'  {_thr:>5}m  {_n:>4}  '
              f'{_bias_b:>+9.2f} {_rmse_b:>9.2f} {_r2_b:>+7.3f}  '
              f'{_bias_p:>+9.2f} {_rmse_p:>9.2f} {_r2_p:>+7.3f}  {_dr:>+7.2f} dB')
    print('=' * 78)
    print('  ΔRMSE < 0: P.833 improves accuracy.  ΔRMSE > 0: over-correction.')


## CELL P.833 — ITU-R P.833 Vegetation Attenuation Post-Processing

Standalone — **no path solver re-run needed**.

Detects woodland polygons (386 found, 93.4% of scene area), computes per-receiver
vegetation depth from GPS path intersection, applies Weissberger attenuation.

Formula: `A = min(20, 0.187 × f^0.284 × d^0.588)` — aligned with flat notebook.


In [ ]:
# ====================================================================
# CELL P833 — ITU-R P.833 VEGETATION ATTENUATION  [frequency-portable]
# ====================================================================
# Weissberger formula (P.833-9 §4.2):
#   A(dB) = min(20, 0.187 · f_GHz^0.284 · depth_m^0.588)
# Applies to woodland polygons only (landuse=forest / natural=wood).
# Coordinates: GeoJSON is UTM absolute; df_ps local coords + scene centre.
# ====================================================================
import os
import numpy as np
from shapely.geometry import LineString as _LS
import geopandas as _gpd
from pyproj import Transformer as _TrP833

# ── 1. Locate vegetation GeoJSON (try new scene first, fallback old) ─
_veg_candidates = [
    os.path.join(os.path.dirname(SCENE_XML), 'vegetation_footprints.geojson'),  # scene_v4_full/
    os.path.join(SCENE_DIR, 'vegetation_footprints.geojson'),
]
_veg_geojson = next((p for p in _veg_candidates if os.path.exists(p)), None)

if _veg_geojson is None:
    print("[P833] vegetation_footprints.geojson not found in any known location.")
    print("  Run scene builder Cell 4 with INCLUDE_VEGETATION=True first.")
elif 'df_ps' not in dir():
    print("[P833] df_ps not found — run Cell 7 first.")
else:
    # ── 2. Load + reproject vegetation ──────────────────────────────
    _UTM_EPSG = UTM_EPSG  # use the session's PROJECTION_CRS-derived EPSG, not a hardcoded one
    _gdf_veg  = _gpd.read_file(_veg_geojson)
    if _gdf_veg.crs is None or _gdf_veg.crs.to_epsg() != _UTM_EPSG:
        _gdf_veg = _gdf_veg.to_crs(epsg=_UTM_EPSG)
    print(f"[P833] Loaded {len(_gdf_veg)} vegetation polygons  ({_veg_geojson.split('/')[-2]})")
    print(f"       CRS: EPSG:{_UTM_EPSG}  |  Area: {_gdf_veg.geometry.area.sum()/1e6:.2f} km²")

    # ── 3. Scene centre UTM (local → absolute conversion) ───────────
    _to_utm_p = _TrP833.from_crs('EPSG:4326', _UTM_EPSG, always_xy=True)
    _cx, _cy  = _to_utm_p.transform(
        (SCENE_WEST + SCENE_EAST) / 2,
        (SCENE_SOUTH + SCENE_NORTH) / 2)
    print(f"[P833] Scene centre UTM: E={_cx:.1f}  N={_cy:.1f}")

    # ── 4. Weissberger constants at FREQUENCY_HZ ────────────────────
    _f_GHz    = FREQUENCY_HZ / 1e9
    _P833_A   = 0.187 * (_f_GHz ** 0.284)
    _P833_B   = 0.588
    _P833_MAX = 20.0
    print(f"[P833] Weissberger @ {FREQUENCY_HZ/1e6:.1f} MHz: "
          f"A = {_P833_A:.4f}·depth^{_P833_B}  (cap={_P833_MAX} dB)")

    # ── 5. TX position (Sionna tensor → UTM absolute) ───────────────
    def _to_float(v):
        if hasattr(v, 'numpy'): return float(v.numpy().flat[0])
        if hasattr(v, 'item'):  return float(v.item())
        return float(v)

    _tx_pos = list(scene.transmitters.values())[0].position
    _tx_utm = np.array([_to_float(_tx_pos[0]) + _cx,
                        _to_float(_tx_pos[1]) + _cy])

    # ── 6. Auto-detect df_ps column names ───────────────────────────
    _cols    = df_ps.columns.tolist()
    _rx_col  = next((c for c in ['receiver','name','rx_name'] if c in _cols), None)
    _x_col   = next((c for c in ['x_m','rx_x','local_x','x']  if c in _cols), None)
    _y_col   = next((c for c in ['y_m','rx_y','local_y','y']  if c in _cols), None)

    if None in (_rx_col, _x_col, _y_col):
        print(f"[P833] Cannot detect required columns. Available: {_cols}")
    else:
        print(f"[P833] Using columns: name='{_rx_col}'  x='{_x_col}'  y='{_y_col}'")

        # ── 7. Spatial index ─────────────────────────────────────────
        _sindex = _gdf_veg.sindex
        _geoms  = _gdf_veg.geometry.values

        # ── 8. Compute attenuation for every receiver in df_ps ───────
        print(f"[P833] Computing intersections for {len(df_ps)} receivers ...")
        _p833_att = {}
        for _, _row in df_ps.iterrows():
            _rx_utm = np.array([float(_row[_x_col]) + _cx,
                                float(_row[_y_col]) + _cy])
            _line   = _LS([tuple(_tx_utm), tuple(_rx_utm)])
            _cands  = list(_sindex.intersection(_line.bounds))
            _depth  = sum(
                _line.intersection(_geoms[i]).length
                for i in _cands if _line.intersects(_geoms[i])
            )
            _p833_att[str(_row[_rx_col])] = (
                min(_P833_MAX, _P833_A * (_depth ** _P833_B)) if _depth > 0 else 0.0
            )

        # ── 9. Summary ───────────────────────────────────────────────
        _atts    = np.array(list(_p833_att.values()))
        _nonzero = (_atts > 0).sum()
        print(f"\n[P833] Attenuation summary:")
        print(f"  RX with vegetation on path : {_nonzero} / {len(_atts)} "
              f"({100*_nonzero/max(len(_atts),1):.1f}%)")
        print(f"  Attenuation mean (all RX)  : {_atts.mean():.2f} dB")
        if _nonzero:
            print(f"  Attenuation mean (affected): {_atts[_atts>0].mean():.2f} dB")
        print(f"  Attenuation max            : {_atts.max():.2f} dB")

        # ── 10. Apply to ALL RSSI columns in df_ps ───────────────────
        _rssi_cols = [c for c in _cols if c.startswith('rssi_') and c.endswith('_dbm')]
        _added     = []
        for _src_col in _rssi_cols:
            _dst_col = _src_col.replace('_dbm', '_p833_dbm')
            df_ps[_dst_col] = df_ps.apply(
                lambda r, s=_src_col: r[s] - _p833_att.get(str(r[_rx_col]), 0.0), axis=1)
            _added.append(_dst_col)

        print(f"\n[P833] Columns added ({len(_added)}): {_added}")
        print("  Run Cell 7c / Cell 8e to see RMSE with P.833 correction.")


## CELL DIAG-B — Receivers Inside Scene BBox Check

Verifies all 1200 receivers fall within the scene bounding box.
Run after receivers are loaded (after CELL 6).

In [ ]:
# ====================================================================
# DIAG-B — Check all receivers are inside scene bounding box
# ====================================================================
from pyproj import Transformer

_safe_v  = lambda v: float(v.numpy().flat[0]) if hasattr(v,'numpy') else float(v)
_t       = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
_orig_e, _orig_n = _t.transform((SCENE_WEST+SCENE_EAST)/2,
                                (SCENE_SOUTH+SCENE_NORTH)/2)
_sw  = _t.transform(SCENE_WEST,  SCENE_SOUTH)
_ne  = _t.transform(SCENE_EAST,  SCENE_NORTH)
_xmin = _sw[0] - _orig_e;  _xmax = _ne[0] - _orig_e
_ymin = _sw[1] - _orig_n;  _ymax = _ne[1] - _orig_n

print(f'Scene bbox (local): X=[{_xmin:.0f}, {_xmax:.0f}]  Y=[{_ymin:.0f}, {_ymax:.0f}]')
print()

_outside = []
for rx in receivers:
    x = _safe_v(rx.position[0])
    y = _safe_v(rx.position[1])
    if x < _xmin or x > _xmax or y < _ymin or y > _ymax:
        _outside.append((rx.name, x, y))

print(f'Total receivers : {len(receivers)}')
print(f'Inside bbox     : {len(receivers) - len(_outside)}')
print(f'Outside bbox    : {len(_outside)}')
if _outside:
    print('\nFirst 10 outside:')
    for name, x, y in _outside[:10]:
        print(f'  {name}  ({x:.1f}, {y:.1f})')
else:
    print('  All receivers inside bbox.')

# ── CSV export ────────────────────────────────────────────────────────
_diagb_rows = []
for _r in receivers:
    _rx_x = _safe_v(_r.position[0]); _rx_y = _safe_v(_r.position[1])
    _ins = (_xmin <= _rx_x <= _xmax and _ymin <= _rx_y <= _ymax)
    _diagb_rows.append({'name': _r.name, 'x_m': _rx_x, 'y_m': _rx_y, 'inside_bbox': _ins})
import pandas as _pd_b; import os as _os_b
_diagb_df  = _pd_b.DataFrame(_diagb_rows)
_diagb_csv = _os_b.path.join(OUT_DIR, 'diag_b_bbox_check.csv')
_diagb_df.to_csv(_diagb_csv, index=False)
print(f'DIAG-B CSV: {_diagb_csv}')


## CELL DIAG-E — East Corridor Troubleshooting

Three diagnostic cells to identify high-error receivers by direction and distance.
Run after CELL 8e. Requires `df_ps` and `MEASUREMENT_CSV`.

In [ ]:
# ====================================================================
# DIAG 1 — Worst-error receivers in 500-900m band
# ====================================================================
import pandas as pd, numpy as np

_meas_d   = pd.read_csv(MEASUREMENT_CSV)
_merged_d = df_ps.merge(_meas_d[['name','local_measurement_dBm']],
                        left_on='receiver', right_on='name', how='left')

_band = _merged_d[(_merged_d['dist_from_tx_m'] > 500) &
                  (_merged_d['dist_from_tx_m'] < 900)].copy()
_band['error'] = _band['rssi_incoherent_dbm'] - _band['local_measurement_dBm']

print(f'Receivers in 500-900m band: {len(_band)}')
print(_band.nlargest(15,'error')[['receiver','dist_from_tx_m','error',
                                  'rssi_incoherent_dbm','local_measurement_dBm']].to_string())


In [ ]:
# ====================================================================
# DIAG 2 — Direction angle of worst-error receivers from TX
# ====================================================================
_safe_v = lambda v: float(v.numpy().flat[0]) if hasattr(v,'numpy') else float(v)
_tx0  = list(scene.transmitters.values())[0]
_tx_x = _safe_v(_tx0.position[0])
_tx_y = _safe_v(_tx0.position[1])

print(f'TX position: ({_tx_x:.1f}, {_tx_y:.1f}) m')
print()
print(f'{"Receiver":<14} {"Dist(m)":>8} {"Angle":>7} {"Error(dB)":>10} {"Sim":>8} {"Meas":>8}')
print('-' * 62)
for _, row in _band.nlargest(15,'error').iterrows():
    dx    = row['x_m'] - _tx_x
    dy    = row['y_m'] - _tx_y
    angle = np.degrees(np.arctan2(dy, dx))
    print(f'{row["receiver"]:<14} {row["dist_from_tx_m"]:>8.0f} {angle:>7.1f}'
          f' {row["error"]:>10.1f} {row["rssi_incoherent_dbm"]:>8.1f}'
          f' {row["local_measurement_dBm"]:>8.1f}')

# ── CSV export ────────────────────────────────────────────────────────
if '_band' in dir() and len(_band):
    import os as _os_e
    _diage_rows = []
    for _, _row in _band.iterrows():
        _dx = _row['x_m'] - _tx_x; _dy = _row['y_m'] - _tx_y
        _diage_rows.append({
            'receiver':        _row['receiver'],
            'dist_m':          _row['dist_from_tx_m'],
            'azimuth_deg':     float(np.degrees(np.arctan2(_dy, _dx))),
            'error_db':        _row['error'],
            'sim_rssi_dbm':    _row['rssi_incoherent_dbm'],
            'meas_rssi_dbm':   _row['local_measurement_dBm'],
            'x_m':             _row['x_m'],
            'y_m':             _row['y_m'],
        })
    import pandas as _pd_e
    _diage_df  = _pd_e.DataFrame(_diage_rows)
    _diage_csv = _os_e.path.join(OUT_DIR, 'diag_e_worst_errors.csv')
    _diage_df.to_csv(_diage_csv, index=False)
    print(f'DIAG-E CSV: {_diage_csv}')


In [ ]:
# ====================================================================
# DIAG 3 — Map: error magnitude by receiver location (all receivers)
# ====================================================================
import matplotlib.pyplot as plt, numpy as np, pandas as pd, os

# ── Build full receiver table from measurements + any sim results ────
_meas_all = pd.read_csv(MEASUREMENT_CSV)
_nc3 = [c for c in _meas_all.columns if 'name' in c.lower() or c.lower()=='id'][0]
_rc3 = [c for c in _meas_all.columns if 'measurement' in c.lower()
        or ('rssi' in c.lower() and 'dbm' in c.lower())][0]
_meas_all = _meas_all.rename(columns={_nc3:'name', _rc3:'meas_dbm'})

# attach local x,y,dist from receivers list
_safe3 = lambda v: float(v.numpy().flat[0]) if hasattr(v,'numpy') else float(v)
_txp3  = list(scene.transmitters.values())[0]
_tx_x3 = _safe3(_txp3.position[0]); _tx_y3 = _safe3(_txp3.position[1])

_rx_lut = {}
for _r in receivers:
    _rx_lut[_r.name] = {
        'x_m': _safe3(_r.position[0]),
        'y_m': _safe3(_r.position[1]),
    }
    _rx_lut[_r.name]['dist_m'] = float(np.sqrt(
        (_rx_lut[_r.name]['x_m']-_tx_x3)**2 +
        (_rx_lut[_r.name]['y_m']-_tx_y3)**2))

_meas_all['x_m']   = _meas_all['name'].map(lambda n: _rx_lut.get(n,{}).get('x_m',   np.nan))
_meas_all['y_m']   = _meas_all['name'].map(lambda n: _rx_lut.get(n,{}).get('y_m',   np.nan))
_meas_all['dist_m']= _meas_all['name'].map(lambda n: _rx_lut.get(n,{}).get('dist_m',np.nan))

# merge sim results if available
if 'df_ps' in dir() and len(df_ps):
    _sim3 = df_ps[['receiver','rssi_incoherent_dbm']].rename(
                columns={'receiver':'name','rssi_incoherent_dbm':'sim_dbm'})
    _meas_all = _meas_all.merge(_sim3, on='name', how='left')
    _meas_all['error'] = _meas_all['sim_dbm'] - _meas_all['meas_dbm']
else:
    _meas_all['sim_dbm'] = np.nan
    _meas_all['error']   = np.nan

_meas_all = _meas_all.dropna(subset=['x_m','y_m'])
print(f"Total receivers for map : {len(_meas_all)}")
print(f"  With sim error        : {_meas_all['error'].notna().sum()}")
print(f"  Measurement only      : {_meas_all['error'].isna().sum()}")

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Panel 1 — measured RSSI
_v1 = _meas_all.dropna(subset=['meas_dbm'])
sc1 = axes[0].scatter(_v1['x_m'], _v1['y_m'], c=_v1['meas_dbm'],
                      cmap='jet', s=18, alpha=0.85, vmin=-115, vmax=-50)
plt.colorbar(sc1, ax=axes[0], label='Measured RSSI (dBm)')
axes[0].scatter([_tx_x3],[_tx_y3], marker='*', s=500, c='gold',
                edgecolors='black', linewidths=0.8, zorder=5, label='TX')
axes[0].set_title(f'Measured RSSI — all {len(_v1)} receivers')
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Panel 2 — simulation error (if available)
_v2 = _meas_all.dropna(subset=['error'])
if len(_v2):
    sc2 = axes[1].scatter(_v2['x_m'], _v2['y_m'], c=_v2['error'].abs(),
                          cmap='RdYlGn_r', s=18, alpha=0.85, vmin=0, vmax=20)
    plt.colorbar(sc2, ax=axes[1], label='|Sim − Meas| (dB)')
    # label worst 15
    for _, row in _v2.nlargest(15,'error').iterrows():
        axes[1].annotate(f"{row['error']:.0f}dB",
                         xy=(row['x_m'], row['y_m']),
                         fontsize=7, color='red',
                         xytext=(4,4), textcoords='offset points')
    axes[1].set_title(f'|Sim error| — {len(_v2)} receivers with sim results')
else:
    axes[1].text(0.5, 0.5, 'No sim results yet\n(run CELL 7 or CELL 8 first)',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title('Sim error — no data')

axes[1].scatter([_tx_x3],[_tx_y3], marker='*', s=500, c='gold',
                edgecolors='black', linewidths=0.8, zorder=5, label='TX')
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Y (m)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('DIAG 3 — Receiver location map', fontsize=13)
plt.tight_layout()
os.makedirs(OUT_DIR, exist_ok=True)
_png = os.path.join(OUT_DIR, 'diag3_error_map.png')
plt.savefig(_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'PNG: {_png}')

# ── CSV export ────────────────────────────────────────────────────────
_diag3_csv = os.path.join(OUT_DIR, 'diag3_all_receivers.csv')
_meas_all.to_csv(_diag3_csv, index=False)
print(f'CSV: {_diag3_csv}')


## CELL 8f — Sim vs Measured Scatter Plot

Scatter plot of simulated vs measured path loss. Shows per-method fit quality and outlier distribution.

In [ ]:
# ==================================================================
# CELL 8f — SIM vs MEASURED SCATTER + DISTANCE COLOUR
# ==================================================================
import matplotlib.pyplot as plt, numpy as np
import matplotlib.cm as _cm

import pandas as _pd8d
# ── standalone: load config + _valid_rx from disk if not in memory ───────────
import os, json as _jcfg, glob as _gl
import numpy as np, pandas as pd

def _load_session():
    """Find session_config.json by searching common OUT_DIR locations."""
    _candidates = sorted(_gl.glob(
        os.path.expanduser('~/sionna_rt/*/results/session_config.json')))
    if not _candidates:
        raise FileNotFoundError(
            'session_config.json not found — run CELL 1 once to create it.')
    with open(_candidates[-1]) as _f:
        return _jcfg.load(_f)

def _load_valid_rx(out_dir):
    _csvs = sorted(_gl.glob(os.path.join(out_dir, 'cell8_per_rx_*.csv')))
    if not _csvs:
        raise FileNotFoundError(
            f'No cell8_per_rx_*.csv in {out_dir} — run CELL 8 first.')
    _df = pd.read_csv(_csvs[-1])
    print(f'  RX data : {_csvs[-1]}')
    return {str(r['name']): {
        'dist_m':           float(r['dist_m']),
        'measured_rssi':    float(r['measured_rssi']),
        'sim_rssi_dbm':     float(r['sim_rssi_dbm']),
        'sim_rssi_off_dbm': float(r['sim_rssi_off_dbm']),
    } for _, r in _df.iterrows()}

if 'OUT_DIR' not in dir():
    _cfg = _load_session()
    OUT_DIR            = _cfg['OUT_DIR']
    BASE_DIR           = _cfg['BASE_DIR']
    SCENE_DIR          = _cfg['SCENE_DIR']
    NDSM_TIFF          = _cfg['NDSM_TIFF']
    MEASUREMENT_CSV    = _cfg['MEASUREMENT_CSV']
    FREQUENCY_HZ       = _cfg['FREQUENCY_HZ']
    TX_CONDUCTED_DBM   = _cfg['TX_CONDUCTED_DBM']
    RX_EXTRA_GAIN_DB   = _cfg['RX_EXTRA_GAIN_DB']
    SITE_CORRECTION_DB = _cfg['SITE_CORRECTION_DB']
    RX_AGL_M           = _cfg['RX_AGL_M']
    MAX_DEPTH          = int(_cfg['MAX_DEPTH'])
    NUM_SAMPLES_PS     = int(_cfg['NUM_SAMPLES_PS'])
    print(f'  Config  : loaded from {OUT_DIR}/session_config.json')

if '_valid_rx' not in dir() or not _valid_rx:
    print('  _valid_rx not in memory — loading from CSV')
    _valid_rx = _load_valid_rx(OUT_DIR)


if _valid_rx:
    _dists, _meas, _sim, _p833 = [], [], [], []
    for _v in _valid_rx.values():
        if 'sim_rssi_dbm' not in _v or 'measured_rssi' not in _v: continue
        if np.isnan(_v['sim_rssi_dbm']) or np.isnan(_v['measured_rssi']): continue
        _dists.append(_v['dist_m'])
        _meas.append(_v['measured_rssi'])
        _sim.append(_v['sim_rssi_dbm'])
        _p833.append(_v.get('rssi_p833_dbm', _v['sim_rssi_dbm']))
    _dists=np.array(_dists); _meas=np.array(_meas)
    _sim=np.array(_sim);     _p833=np.array(_p833)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, _y, _title in zip(axes,
            [_sim, _p833], ['Sim (raw)', 'Sim + P.833 correction']):
        _sc = ax.scatter(_meas, _y, c=_dists/1000, cmap='plasma',
                         s=12, alpha=0.7, vmin=0, vmax=3)
        _lo = min(_meas.min(), _y.min()) - 2
        _hi = max(_meas.max(), _y.max()) + 2
        ax.plot([_lo,_hi],[_lo,_hi],'k--',lw=1,label='Perfect')
        plt.colorbar(_sc, ax=ax, label='Distance (km)')
        _err = _y - _meas
        _bias = float(np.mean(_err)); _rmse = float(np.sqrt(np.mean(_err**2)))
        ax.set_xlabel('Measured RSSI (dBm)')
        ax.set_ylabel('Simulated RSSI (dBm)')
        ax.set_title(f'{_title}\nbias={_bias:+.1f} dB  RMSE={_rmse:.1f} dB  N={len(_meas)}')
        ax.legend()
    plt.suptitle(f'{SCENARIO_NAME}', fontsize=11)
    plt.tight_layout()
    _fig_path = os.path.join(OUT_DIR, 'sim_vs_meas_scatter.png')
    plt.savefig(_fig_path, dpi=150)
    plt.show()
    print(f'Saved: {_fig_path}')
    
    # ── store in report ─────────────────────────────────────────────
    if "_report" in dir():
        _report["figures"].append(_fig_path)
        print("Figure path saved to _report[figures]")


## CELL 8 — Compare vs Measurements

Bias, RMSE, R² vs Ofcom 2018 drive-test data for all 6 methods. Distance-band breakdown table.

In [ ]:
# ====================================================================
# CELL 8 — SIM vs MEASUREMENTS  [2695 MHz DEM / Sionna 2.0]
# ====================================================================
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt, os, glob

if not MEASUREMENT_CSV or not os.path.exists(MEASUREMENT_CSV):
    print('Set MEASUREMENT_CSV in Cell 1 to compare vs measurements.')
else:
    if 'df_ps' not in dir():
        _files = sorted(glob.glob(os.path.join(OUT_DIR, 'path_solver_summary_900s2_*.csv')))
        assert _files, f'No path solver CSV found in {OUT_DIR}\nRun Cell 7 first.'
        df_ps = pd.read_csv(_files[-1])
        print(f'Loaded: {_files[-1]}')

    df_sim = df_ps.rename(columns={
        'receiver'           : 'name',
        'rssi_incoherent_dbm': 'rssi_sim_dbm',
        'dist_from_tx_m'     : 'dist_m',
    })

    df_meas  = pd.read_csv(MEASUREMENT_CSV)
    df_merge = df_sim.merge(df_meas[['name', 'local_measurement_dBm']], on='name', how='inner')
    df_merge = df_merge.dropna(subset=['rssi_sim_dbm', 'local_measurement_dBm'])
    df_merge['err']     = df_merge['rssi_sim_dbm'] - df_merge['local_measurement_dBm']
    df_merge['dist_km'] = df_merge['dist_m'] / 1000

    bias = df_merge['err'].mean()
    rmse = math.sqrt((df_merge['err']**2).mean())

    print(f'Receivers compared : {len(df_merge)}')
    print(f'Bias (sim-meas)    : {bias:+.2f} dB')
    print(f'RMSE               : {rmse:.2f} dB')
    print()

    print(f'  {"Band":<12} {"N":>4}  {"Bias (dB)":>10}  {"RMSE (dB)":>10}  {"Mean paths":>11}')
    print(f'  {"-"*12} {"-"*4}  {"-"*10}  {"-"*10}  {"-"*11}')
    for lbl, d0, d1 in [("<300m",0,300),("300-700m",300,700),
                        ("700m-1.2km",700,1200),(">1.2km",1200,9999)]:
        s = df_merge[(df_merge['dist_m']>=d0) & (df_merge['dist_m']<d1)]
        if not len(s): continue
        e = s['err']
        p = s['n_paths'].mean() if 'n_paths' in s.columns else float('nan')
        print(f'  {lbl:<12} {len(s):>4}  {e.mean():>+10.1f}  {((e**2).mean()**0.5):>10.1f}  {p:>11.0f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].scatter(df_merge['dist_km'], df_merge['err'], s=6, alpha=0.5, color='steelblue')
    axes[0].axhline(0, color='red', lw=1)
    axes[0].set_xlabel('Distance (km)')
    axes[0].set_ylabel('Error (dB)')
    axes[0].set_title(f'Error vs distance  (bias={bias:+.1f} dB, RMSE={rmse:.1f} dB)')
    axes[0].grid(alpha=0.3)

    axes[1].scatter(df_merge['local_measurement_dBm'], df_merge['rssi_sim_dbm'],
                    s=6, alpha=0.5, color='steelblue')
    _lo = min(df_merge['local_measurement_dBm'].min(), df_merge['rssi_sim_dbm'].min()) - 5
    _hi = max(df_merge['local_measurement_dBm'].max(), df_merge['rssi_sim_dbm'].max()) + 5
    axes[1].plot([_lo, _hi], [_lo, _hi], 'r--', lw=1)
    axes[1].set_xlabel('Measured RSSI (dBm)')
    axes[1].set_ylabel('Simulated RSSI (dBm)')
    axes[1].set_title('Sim vs Measured  (2695 MHz DEM / Sionna 2.0)')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    _p = os.path.join(OUT_DIR, 'rssi_compare_900mhz_s2.png')
    plt.savefig(_p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved → {_p}')

## CELL SAVE — Quick Results Snapshot

Run at any point to save current `_report` state to JSON + print summary.
Does not require CELL 8 or P.833 to have completed.


In [ ]:
# ==================================================================
# CELL SAVE — QUICK SNAPSHOT
# ==================================================================
import json as _js, os
from datetime import datetime as _dt

# Auto-collect whatever is available
if '_df8' in dir():    _report['cell8_bands']  = _df8.to_dict('records')
if '_df833' in dir():  _report['p833_raw']     = _df833.to_dict('records')
if '_valid_rx' in dir():
    _report['n_valid_rx'] = len(_valid_rx)
    _sims = [v.get('sim_rssi_dbm') for v in _valid_rx.values()]
    _meas = [v.get('measured_rssi') for v in _valid_rx.values()]
    import numpy as _np
    _errs = [s-m for s,m in zip(_sims,_meas) if s is not None and not _np.isnan(s) and not _np.isnan(m)]
    if _errs:
        _report['overall_bias'] = round(float(_np.mean(_errs)),2)
        _report['overall_rmse'] = round(float(_np.sqrt(_np.mean(_np.array(_errs)**2))),2)
        print(f'Overall: N={len(_errs)}  bias={_report["overall_bias"]:+.1f} dB  rmse={_report["overall_rmse"]:.1f} dB')

_snap = os.path.join(OUT_DIR, f'snapshot_{_dt.now().strftime("%Y%m%d_%H%M%S")}.json')
os.makedirs(OUT_DIR, exist_ok=True)
with open(_snap,'w') as _f: _js.dump(_report, _f, indent=2, default=str)
print(f'Snapshot saved: {_snap}')


## CELL REPORT — Auto-Collected Results Report

Reads from `_report` (populated by CELL DIAG, CELL 8, CELL P.833).  
Generates `results/report_<timestamp>.md` + `.json` + `band_metrics.csv`.  
Run any time after CELL 8; re-run after CELL P.833 for full results.

In [ ]:
# ====================================================================
# CELL REPORT — AUTO-COLLECTED RESULTS REPORT
# ====================================================================
# Reads from _report (populated by CELL DIAG, CELL 8, CELL P.833)
# Generates:
#   results/report_<timestamp>.md   — full markdown report
#   results/report_<timestamp>.json — raw metrics snapshot
#   results/band_metrics.csv        — CELL 8 band table
# Run any time after CELL 8. Re-run after CELL P.833 for full results.
# ====================================================================
import os, json as _json, numpy as np, pandas as pd
from datetime import datetime

_ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
_now = datetime.now().strftime('%Y-%m-%d %H:%M')
os.makedirs(OUT_DIR, exist_ok=True)

# ── Auto-collect CELL 8 band results ─────────────────────────────────────
if '_df8' in dir() and len(_df8) > 0:
    _report['cell8_bands'] = _df8.to_dict('records')
    print(f'CELL 8: {len(_df8)} bands collected')

# ── Auto-collect P.833 results ────────────────────────────────────────────
if '_df833' in dir() and len(_df833) > 0:
    BANDS_R = [(0,300),(300,500),(500,750),(750,1000),
               (1000,1250),(1250,1500),(1500,2000),(2000,3000),(3000,99999)]
    _p833_summary = []
    for _lo, _hi in BANDS_R:
        _s = _df833[(_df833.dist >= _lo) & (_df833.dist < _hi)]
        _s = _s.dropna(subset=['sim_rssi','p833_rssi','meas'])
        if len(_s) < 2: continue
        _e_raw  = _s.sim_rssi.values  - _s.meas.values
        _e_p833 = _s.p833_rssi.values - _s.meas.values
        _p833_summary.append({
            'band': f'{_lo}-{min(_hi,9999)}m', 'n': len(_s),
            'bias_raw':  round(float(np.mean(_e_raw)),1),
            'rmse_raw':  round(float(np.sqrt(np.mean(_e_raw**2))),1),
            'bias_p833': round(float(np.mean(_e_p833)),1),
            'rmse_p833': round(float(np.sqrt(np.mean(_e_p833**2))),1),
            'mean_veg_depth_m': round(float(_s.veg_depth_m.mean()),1),
            'mean_veg_loss_db': round(float(_s.veg_loss_db.mean()),2),
        })
    _report['p833_bands'] = _p833_summary
    print(f'P.833: {len(_p833_summary)} bands collected')

# ── Auto-collect figures ──────────────────────────────────────────────────
_figs = []
for _fn in ['ndsm_heatmap.png','txrx_map.png','sim_vs_meas_scatter.png']:
    _fp = os.path.join(OUT_DIR, _fn)
    if os.path.exists(_fp): _figs.append(_fn)
_report['figures'] = _figs

# ── Save JSON snapshot ────────────────────────────────────────────────────
_json_path = os.path.join(OUT_DIR, f'report_{_ts}.json')
with open(_json_path, 'w') as _f:
    _json.dump(_report, _f, indent=2, default=str)
print(f'JSON snapshot: {_json_path}')

# ── Save CELL 8 band CSV ──────────────────────────────────────────────────
if _report['cell8_bands']:
    _csv_path = os.path.join(OUT_DIR, f'band_metrics_{_ts}.csv')
    pd.DataFrame(_report['cell8_bands']).to_csv(_csv_path, index=False)
    print(f'Band metrics CSV: {_csv_path}')

# ── Build Markdown report ─────────────────────────────────────────────────
_md_path = os.path.join(OUT_DIR, f'report_{_ts}.md')
with open(_md_path, 'w') as _f:
    _f.write(f'# Sionna RT 2.0 — Results Report\n')
    _f.write(f'**Generated:** {_now}  ·  **Scenario:** {_report["scenario"]}\n\n')
    _f.write('---\n\n')

    # Config table
    _f.write('## Simulation Configuration\n\n')
    _f.write('| Parameter | Value |\n|---|---|\n')
    _f.write(f'| Frequency | {_report["frequency_mhz"]:.2f} MHz |\n')
    _f.write(f'| TX position | lat={_report["tx_lat"]}, lon={_report["tx_lon"]} |\n')
    _f.write(f'| TX height AGL | {_report["tx_agl_m"]} m |\n')
    _f.write(f'| TX power | {_report["tx_conducted_dbm"]} dBm conducted + {_report["tx_antenna_gain_dbi"]} dBi |\n')
    _f.write(f'| RX height AGL | {_report["rx_agl_m"]} m |\n')
    _f.write(f'| RX system gain | {_report["rx_extra_gain_db"]} dB |\n')
    _f.write(f'| Max ray depth | {_report["max_depth"]} |\n')
    _f.write(f'| Base samples/src | {_report["num_samples_ps"]:,} |\n')
    _f.write(f'| Antenna pattern | {_report["antenna_pattern"]} |\n\n')

    # CELL 8 bands
    if _report['cell8_bands']:
        _f.write('## CELL 8 — Stratified Band Results\n\n')
        _f.write('| Band | N | sps | Bias ON | RMSE ON | R² ON | paths ON | Bias OFF | RMSE OFF |\n')
        _f.write('|------|---|-----|---------|---------|-------|----------|----------|----------|\n')
        for _b in _report['cell8_bands']:
            _f.write(f'| {_b["band"]} | {int(_b["N"])} | {_b["sps"]/1e6:.0f}M'
                     f' | {_b["on_bias"]:+.1f} | {_b["on_rmse"]:.1f} | {_b["on_r2"]:.3f}'
                     f' | {_b["on_paths"]:.0f}'
                     f' | {_b["off_bias"]:+.1f} | {_b["off_rmse"]:.1f} |\n')
        _f.write('\n')

    # P.833 bands
    if _report['p833_bands']:
        _f.write('## CELL P.833 — Vegetation Correction\n\n')
        _f.write('| Band | N | Bias raw | RMSE raw | Bias P.833 | RMSE P.833 | Veg depth | Veg loss |\n')
        _f.write('|------|---|----------|----------|------------|------------|-----------|----------|\n')
        for _b in _report['p833_bands']:
            _f.write(f'| {_b["band"]} | {_b["n"]}'
                     f' | {_b["bias_raw"]:+.1f} | {_b["rmse_raw"]:.1f}'
                     f' | {_b["bias_p833"]:+.1f} | {_b["rmse_p833"]:.1f}'
                     f' | {_b["mean_veg_depth_m"]:.1f} m | {_b["mean_veg_loss_db"]:.2f} dB |\n')
        _f.write('\n')

    # DIAG results
    if _report.get('diag'):
        _f.write('## CELL DIAG — Propagation Validation\n\n')
        _f.write('### Step 4: Sample receivers vs measurements\n\n')
        _f.write(f'Overall: bias={_report["diag"].get("bias","n/a")} dB  '
                 f'RMSE={_report["diag"].get("rmse","n/a")} dB\n\n')
        _f.write('### Step 5: Measured RSSI excess loss vs FSPL\n\n')
        _f.write('| Band | N | Mean dist | Excess loss (dB) |\n|------|---|-----------|-----------------|\n')
        for _b in _report['diag'].get('fspl_bands',[]):
            _f.write(f'| {_b["band"]} | {_b["n"]} | {_b["mean_dist"]} | {_b["excess_db"]:+.1f} |\n')
        _f.write('\n')

    # Figures
    if _report['figures']:
        _f.write('## Figures\n\n')
        for _fn in _report['figures']:
            _f.write(f'![]({_fn})\n\n')

    # Key findings
    _f.write('## Key Findings\n\n')
    _f.write('- **Scattering essential**: diffuse reflection provides 100–600x more paths at sub-1 GHz\n')
    _f.write('- **nDSM building heights**: fills OSM clutter gaps, reduces far-field positive bias\n')
    _f.write('- **Vegetation**: flat ground patches (no hard shadow); P.833 Weissberger adds correct excess loss\n')
    _f.write('- **Ray starvation**: stratified band solving prevents near-RX stealing ray budget from far-RX\n')

print(f'Markdown report: {_md_path}')
print()
print('=' * 60)
print('REPORT SUMMARY')
print('=' * 60)
if _report['cell8_bands']:
    print(f'  {"Band":>13}  {"N":>4}  {"Bias ON":>8}  {"RMSE ON":>8}  {"R2 ON":>7}  {"paths":>8}')
    print('  ' + '-'*58)
    for _b in _report['cell8_bands']:
        print(f'  {_b["band"]:>13}  {int(_b["N"]):>4}  '
              f'{_b["on_bias"]:>+8.1f}  {_b["on_rmse"]:>8.1f}  '
              f'{_b["on_r2"]:>7.3f}  {_b["on_paths"]:>8.0f}')
print('=' * 60)
print(f'Saved to: {OUT_DIR}')


## CELL DIAG-CSV — Load Specific New-Scene CSV (scene_v2_infra)

Loads the June 2026 new-scene path solver results directly after a kernel restart.
Use this cell instead of re-running the full path solver when the CSV already exists.


In [ ]:
# ====================================================================
# CELL DIAG-CSV — Restore new-scene df_ps from saved CSV
# ====================================================================
import os, pandas as pd

_NEW_SCENE_CSV = os.path.join(
    OUT_DIR,  # TODO: set to the London path_solver_summary_*.csv filename once generated
    'path_solver_summary_2695mhz_REPLACE_ME.csv')

if not os.path.exists(_NEW_SCENE_CSV):
    print(f'[ERROR] File not found: {_NEW_SCENE_CSV}')
else:
    df_ps = pd.read_csv(_NEW_SCENE_CSV)
    _solved = df_ps['pl_incoherent_db'].notna().sum()
    print(f'Loaded: {_NEW_SCENE_CSV}')
    print(f'  Rows: {len(df_ps)}  |  Solved (pl_incoherent_db): {_solved}')
    print(f'  Columns: {list(df_ps.columns)}')
    print()
    print('Next: run Cell P833 to re-add vegetation attenuation columns.')


## CELL CAL — Derivative-Free Material + Scalar Calibration (Sionna 2.0)

Calibrates `scene.radio_materials` (εr, σ, scattering) and a global scalar offset
directly in this notebook using `PathSolver` — no autodiff required.

**Method:**
1. Build a stratified calibration RX set (0–1.2 km, same filter as the 0.19 notebook)
2. Step 1: optimise a single `scaling_factor_db` (1-D, `scipy.optimize.minimize_scalar`)
3. Step 2: optimise (εr, σ, S) for a small set of dominant materials
   (`scipy.optimize.minimize`, L-BFGS-B, bounded around ITU defaults)
4. Each evaluation: set material props → `PathSolver()` on the calib batch →
   incoherent `PL_sim = -10log10(sum|a|^2)` → RMSE vs `PL_meas`
5. Saves `calibrated_materials_stevenage_2695mhz.json` + `scalar_offset_stevenage_2695mhz.json`
   (same files CELL 4A already loads — re-run CELL 4A after this to apply)


In [ ]:
# ====================================================================
# CELL CAL — Maximum-accuracy calibration  (Sionna 2.0)
# ====================================================================
# Method: Coordinate-descent warm-up → joint Powell refinement
#   Phase 0  Scalar offset          : minimize_scalar, all RX, 500k samples
#   Phase 1  Coord-descent warm-up  : 1 pass (er→sigma→S per material)
#   Phase 2  Joint Powell refinement: all params together, warm start
#   Phase 3  Re-scalar              : final scalar with calibrated materials
# If DrJIT caches material params (sensitivity < 0.5 dB) falls back to
# scalar-only automatically.
# ====================================================================
import numpy as np, json as _jcal, os as _ocal, time as _tcal, gc as _gc
from scipy.optimize import minimize_scalar, minimize, differential_evolution

print("=" * 70)
print("CELL CAL — Maximum-accuracy calibration")
print("=" * 70)

# ── Config ────────────────────────────────────────────────────────────────
CAL_MAX_DIST_KM     = globals().get('CAL_MAX_DIST_KM', 1.0)  # from CELL 1 config
CAL_MIN_DIST_KM     = globals().get('CAL_MIN_DIST_KM', 0.0)  # from CELL 1 config
CAL_SAMPLES_SF      = globals().get('CAL_SAMPLES_PS', 500_000)   # scalar phase samples
_warm_factor = int(globals().get('CAL_SAMPLES_WARM_FACTOR', 1))
CAL_SAMPLES_WARM    = max(300_000, globals().get('CAL_SAMPLES_PS', 500_000) // _warm_factor)  # factor=1 → same as Powell (no landscape mismatch)
CAL_SAMPLES_POWELL  = globals().get('CAL_SAMPLES_PS', 500_000)   # Powell refinement
CAL_SAMPLES_FINAL   = globals().get('CAL_SAMPLES_PS', 500_000)   # final scalar re-calibration
CAL_WARM_CYCLES     = globals().get('CAL_WARM_CYCLES', 0)  # coord-descent cycles before Powell
CAL_WARM_EVALS_1D   = globals().get("CAL_WARM_EVALS_1D", 5)  # evals per 1D search in warm-up
CAL_POWELL_MAXITER  = globals().get('CAL_POWELL_MAXITER', 30)   # Powell max iterations (each iter ≈ N evals)
CAL_POWELL_XTOL     = globals().get('CAL_POWELL_XTOL', 0.01)    # Powell convergence tolerance
CAL_POWELL_FTOL     = globals().get('CAL_POWELL_FTOL', 0.01)
CAL_SCALAR_ONLY     = globals().get('CAL_SCALAR_ONLY', False)  # True = skip all material optimisation
CAL_N_AVG_SOLVE     = globals().get('CAL_N_AVG_SOLVE', 1)    # PathSolver calls to average per _solve_pl
CAL_FIX_SCATTER     = globals().get('CAL_FIX_SCATTER', False)  # True = fix S at init, only tune er+sigma

# Physical EM bounds (ITU-R P.2040-2 ranges + margin)
_ER_MIN,  _ER_MAX  = 1.0,  80.0
_SIG_MIN, _SIG_MAX = 1e-6, 1e4
_S_MIN,   _S_MAX   = 0.0,  globals().get('CAL_S_MAX', 0.95)
# Per-material minimum S — prevents optimizer zeroing out scattering
# for buildings/ground where S=0 kills NLOS path coverage
_S_MIN_PER_MAT = {
    'itu_concrete':          0.20,   # min ensures enough NLOS scatter paths
    'itu_brick':             0.20,
    'itu_wood':              0.40,
    'itu_wet_ground':        0.05,
    'itu_medium_dry_ground': 0.05,
    'itu_very_dry_ground':   0.05,
    'itu_ceiling_board':     0.0,
    'itu_metal':             0.25,
    'itu_glass':             0.10,
}
# Per-material minimum sigma — prevents Powell over-softening buildings
# (without floor, sigma can drop to sig0*0.01, destroying building absorption)
_SIG_MIN_PER_MAT = {
    'itu_brick':             0.030,  # ITU default 0.0479 — floor at 63%
    'itu_concrete':          0.030,  # ITU default 0.0525 — floor at 57%
    'itu_wet_ground':        0.010,  # allow some flexibility for ground
    'itu_very_dry_ground':   0.0003, # dry ground naturally low
}
# Per-material S upper bounds — prevents scatter flooding
# Bounded range: buildings [0.20-0.70], ground [0.05-0.40], metal [0.25-0.75]
# Warm prior S=0.35 falls within all ranges — Powell starts in centre
_S_MAX_PER_MAT = {
    'itu_wet_ground':        0.40,
    'itu_medium_dry_ground': 0.40,
    'itu_very_dry_ground':   0.35,
    'itu_concrete':          0.70,
    'itu_brick':             0.70,
    'itu_metal':             0.75,
    'itu_glass':             0.60,
    'itu_wood':              0.65,
}
_S_MAX_PER_MAT.update(globals().get('S_MAX_OVERRIDE', {}))
# Clip _S_MIN to never exceed _S_MAX after override (avoids ValueError in Powell bounds)
for _mn in list(_S_MIN_PER_MAT.keys()):
    if _mn in _S_MAX_PER_MAT:
        _S_MIN_PER_MAT[_mn] = min(_S_MIN_PER_MAT[_mn], _S_MAX_PER_MAT[_mn])

# ── Helpers ───────────────────────────────────────────────────────────────
def _to_float(v):
    for f in [lambda x: float(np.real(x[0])),
              lambda x: float(np.real(x.numpy())),
              lambda x: float(np.real(complex(x))),
              float]:
        try: return f(v)
        except: pass
    return float(v)

def _set_mat(mat, er=None, sigma=None, s=None):
    import drjit as _dr
    for prop, val in [('relative_permittivity', er),
                      ('conductivity', sigma),
                      ('scattering_coefficient', s)]:
        if val is None: continue
        # Use drjit.Float first — creates a new DrJIT variable, invalidates kernel cache
        for attempt in [lambda p=prop, v=val: setattr(mat, p, _dr.Float(float(v))),
                        lambda p=prop, v=val: setattr(mat, p, float(v))]:
            try: attempt(); break
            except: pass
    try:
        _dr.eval(); _dr.sync_thread()
        # Flush compiled kernel cache so next PathSolver sees new material values
        if hasattr(_dr, 'flush_kernel_cache'): _dr.flush_kernel_cache()
    except: pass

def _flush():
    _gc.collect()
    try:
        import drjit as _dr; _dr.flush_malloc_cache(); _dr.sync_thread()
    except: pass

def _solve_pl(rx_list, n_samples):
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _r in rx_list: scene.add(_r)
    # DrJIT: eval+sync to propagate material param changes without flushing the
    # kernel cache. flush_kernel_cache() was removed because it resets the random
    # seed on every call, causing ~4 dB systematic offset between Phase 0 and
    # Powell evals — killing Powell convergence. DrJIT tracks param dependencies
    # automatically; eval() alone is sufficient to apply material changes.
    try:
        import drjit as _drjit_fl
        _drjit_fl.eval(); _drjit_fl.sync_thread()
    except: pass
    _cfg = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, diffuse_reflection=True,
                samples_per_src=int(n_samples))
    # Average N solves in linear power domain — reduces MC variance by sqrt(N)
    # Correct averaging: compute received power per solve, average across solves,
    # then convert to dB. (Concatenating amplitudes would bias power by ×N.)
    _N_avg = int(globals().get('CAL_N_AVG_SOLVE', 1))
    _pwr_sum   = np.zeros(len(rx_list))
    _pwr_count = np.zeros(len(rx_list), dtype=int)
    def _extract_a(p_obj):
        _ai = getattr(p_obj, 'a', None)
        try:
            _m = (np.array(_ai[0]) + 1j*np.array(_ai[1])) if isinstance(_ai, tuple) else np.array(_ai)
            _m = np.squeeze(_m)
            if _m.ndim == 1: _m = _m[np.newaxis, :]
            elif _m.ndim > 2: _m = _m.reshape(_m.shape[0], -1)
            return _m
        except: pass
        try:
            _ct, _ = p_obj.cir()
            _m = (np.array(_ct[0]) + 1j*np.array(_ct[1])) if isinstance(_ct, tuple) else np.array(_ct)
            _m = np.squeeze(_m)
            if _m.ndim == 1: _m = _m[np.newaxis, :]
            elif _m.ndim > 2: _m = _m.reshape(_m.shape[0], -1)
            return _m
        except: pass
        return np.zeros((len(rx_list), 1), dtype=complex)
    # Fix: set a fixed DrJIT seed before each PathSolver call so every
    # kernel recompile (triggered by scene graph changes) produces the
    # same Monte Carlo sequence. Without this, each call gets a new seed
    # causing ~4 dB systematic offset between Phase 0 and Powell evals.
    _CAL_FIXED_SEED = int(globals().get('CAL_FIXED_SEED', 0))  # 0 = disabled
    for _avg_i in range(_N_avg):
        if _CAL_FIXED_SEED:
            try:
                import drjit as _dr_s; _dr_s.seed(_CAL_FIXED_SEED + _avg_i)
            except: pass
            try:
                _p_i = PathSolver()(scene, seed=_CAL_FIXED_SEED + _avg_i, **_cfg)
            except TypeError:
                _p_i = PathSolver()(scene, **_cfg)
        else:
            _p_i  = PathSolver()(scene, **_cfg)
        _ai_np = _extract_a(_p_i)
        del _p_i; _flush()
        for _j in range(min(len(rx_list), _ai_np.shape[0])):
            _pwr_j = float(np.sum(np.abs(_ai_np[_j])**2))
            if _pwr_j > 1e-30:
                _pwr_sum[_j]   += _pwr_j
                _pwr_count[_j] += 1
    _pl = np.full(len(rx_list), np.nan)
    for _j in range(len(rx_list)):
        if _pwr_count[_j] > 0:
            _pl[_j] = -10*np.log10(_pwr_sum[_j] / _pwr_count[_j])  # mean power → PL
    return _pl

# ── Build calibration RX set — all receivers up to CAL_MAX_DIST_KM ────────
import pandas as _pd_cal
_df_cal  = _pd_cal.read_csv(MEASUREMENT_CSV)
_name_c  = [c for c in _df_cal.columns if "name" in c.lower() or "id" in c.lower()][0]
_rssi_c  = [c for c in _df_cal.columns
            if "measurement" in c.lower()
            or ("rssi" in c.lower() and "dbm" in c.lower())
            or c.lower() == "local_measurement_dbm"][0]
_meas_dict = {str(r[_name_c]): float(r[_rssi_c]) for _, r in _df_cal.iterrows()}
_safe  = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)
_tx_xy = np.array([_safe(list(scene.transmitters.values())[0].position[i]) for i in range(2)])

_cands = []
for _rx in receivers:
    if _rx.name not in _meas_dict: continue
    _dkm = float(np.linalg.norm(
        np.array([_safe(_rx.position[0]), _safe(_rx.position[1])]) - _tx_xy)) / 1000.0
    if CAL_MIN_DIST_KM <= _dkm <= CAL_MAX_DIST_KM:
        _cands.append((_rx, _dkm, _meas_dict[_rx.name]))
_cands.sort(key=lambda t: t[1])
cal_rx = [c[0] for c in _cands]
cal_pl = TX_CONDUCTED_DBM + globals().get('RX_EXTRA_GAIN_DB', 0.0) - np.array([c[2] for c in _cands])
print(f"Calibration RX : {len(cal_rx)}  ({CAL_MIN_DIST_KM}\u2013{CAL_MAX_DIST_KM} km ceiling)")
print(f"PL_meas range  : {cal_pl.min():.1f} – {cal_pl.max():.1f} dB")

def _rmse(pl_sim, sf=0.0):
    _v = np.isfinite(pl_sim)
    if _v.sum() == 0: return 999.0
    return float(np.sqrt(np.mean((pl_sim[_v] - sf - cal_pl[_v])**2)))

# ── Warm prior: fix fresh-start auto-discover ─────────────────────────────
# If most materials have S≈0 (ITU defaults), Phase 0 finds no scatter paths
# → auto-discover stops at 0.90 km (70 RX). Warm prior S=0.35 fixes this.
_WARM_S_PRIOR  = globals().get('CAL_WARM_S_PRIOR', 0.35)
_CAL_N_ITER    = globals().get('CAL_N_ITER', 1)  # default 1 — auto-iteration off unless explicitly requested
_n_mats_scene  = sum(1 for _m in scene.radio_materials if not _m.endswith('_train'))
_n_low_s       = sum(1 for _mn_l, _ml in scene.radio_materials.items()
                     if not _mn_l.endswith('_train') and
                     float(_to_float(getattr(_ml, 'scattering_coefficient', 0.0))) < _WARM_S_PRIOR)
_apply_warm_prior = _n_low_s > _n_mats_scene // 2
if _apply_warm_prior:
    print(f"  Fresh start detected: warm prior S={_WARM_S_PRIOR} will be applied after _mats is built")

# ── Early Option A: pre-assign DrJIT Float variables BEFORE Phase 0 ──────
# Phase 0 and Phase 2 Powell must share the same kernel/seed.
# By assigning DrJIT Float leaf nodes here (before the first PathSolver call),
# Phase 0 registers them in the compiled kernel. Powell's in-place updates
# (_mat_vars[mn]['er'][0] = val) mutate the same leaf — no recompile, same seed.
_mat_vars = {}
_ip_ok = False
try:
    import drjit as _dr_early
    _DrFloat = None
    for _mn_disc in scene.radio_materials:
        if _mn_disc.endswith('_train'): continue
        try:
            _sample_prop = scene.radio_materials[_mn_disc].relative_permittivity
            _DrFloat = type(_sample_prop)
            if callable(_DrFloat): break
        except: pass
    if _DrFloat is None:
        for _try_path in ['drjit.cuda.ad.Float', 'drjit.cuda.Float',
                          'drjit.llvm.ad.Float', 'drjit.llvm.Float']:
            try:
                _mod_p, _cls_p = _try_path.rsplit('.', 1)
                import importlib as _imp_e
                _DrFloat = getattr(_imp_e.import_module(_mod_p), _cls_p)
                _DrFloat(1.0); break
            except: pass
    if _DrFloat is None:
        raise RuntimeError("Could not find a valid DrJIT Float type")
    for _mn_e, _mat_e in scene.radio_materials.items():
        if _mn_e.endswith('_train'): continue
        try:
            _ev  = _DrFloat(float(_to_float(_mat_e.relative_permittivity)))
            _sv  = _DrFloat(float(_to_float(_mat_e.conductivity)))
            _scv = _DrFloat(float(_to_float(getattr(_mat_e, 'scattering_coefficient', 0.2))))
            try: _mat_e.relative_permittivity  = _ev
            except: pass
            try: _mat_e.conductivity            = _sv
            except: pass
            try: _mat_e.scattering_coefficient  = _scv
            except: pass
            _mat_vars[_mn_e] = {'er': _ev, 'sig': _sv, 's': _scv}
        except: pass
    _dr_early.eval(); _dr_early.sync_thread()
    _ip_ok = True
    print(f"Early Option A: {len(_mat_vars)} materials pre-assigned — Phase 0 and Powell share same kernel/seed")
except Exception as _e_early:
    print(f"Early Option A failed ({_e_early}) — seed gap between Phase 0 and Powell may persist")
    _mat_vars = {}; _ip_ok = False

# ── Pre-Phase-0 warm prior: ensure auto-discover reaches CAL_MAX_DIST_KM ──────
# Applying warm prior only after Phase 0 means auto-discover uses ITU S≈0
# materials and terminates early. Apply here so Phase 0 finds paths at full range.
if _apply_warm_prior:
    _cal_fixed_pre = globals().get('CAL_FIXED_MATS', set())
    print(f"  Pre-Phase-0 warm prior S={_WARM_S_PRIOR} (auto-discover)")
    for _mn_wp, _ml_wp in scene.radio_materials.items():
        if _mn_wp.endswith('_train') or _mn_wp in _cal_fixed_pre: continue
        try:
            if float(_to_float(getattr(_ml_wp, 'scattering_coefficient', 0.0))) < _WARM_S_PRIOR:
                if _mn_wp in _mat_vars:
                    _mat_vars[_mn_wp]['s'][0] = _WARM_S_PRIOR
                else:
                    _ml_wp.scattering_coefficient = _WARM_S_PRIOR
        except: pass
    try:
        import drjit as _dr_wp; _dr_wp.eval(); _dr_wp.sync_thread()
    except: pass

# ── TX height pre-scan: find optimal AGL before Phase 0 ─────────────────────
# Tests TX_AGL_SCAN_M heights (list of m AGL from CELL 1), picks lowest RMSE.
_TX_AGL_SCAN = globals().get('TX_AGL_SCAN_M', None)
if _TX_AGL_SCAN:
    _tx_obj_sc    = list(scene.transmitters.values())[0]
    _tx_terr_z_sc = float(_safe(_tx_obj_sc.position[2])) - TX_AGL_M
    print(f"\n{'═'*60}")
    print(f"TX height scan: {_TX_AGL_SCAN} m AGL  ({CAL_SAMPLES_SF//1000}k samples each)")
    _best_agl_sc, _best_rmse_sc = float(TX_AGL_M), 999.0
    for _agl_sc in _TX_AGL_SCAN:
        _tx_obj_sc.position = [float(_safe(_tx_obj_sc.position[0])),
                                float(_safe(_tx_obj_sc.position[1])),
                                _tx_terr_z_sc + float(_agl_sc)]
        try:
            import drjit as _drjsc; _drjsc.eval(); _drjsc.sync_thread()
        except: pass
        _pl_sc  = _solve_pl(cal_rx, CAL_SAMPLES_SF)
        _vs_sc  = np.isfinite(_pl_sc)
        if _vs_sc.sum() == 0:
            print(f"  AGL={_agl_sc:.0f}m  valid=0/{len(cal_rx)}  (no paths)"); continue
        _sf_sc  = float(np.mean(_pl_sc[_vs_sc] - cal_pl[_vs_sc]))
        _rms_sc = float(np.sqrt(np.mean((_pl_sc[_vs_sc] - _sf_sc - cal_pl[_vs_sc])**2)))
        print(f"  AGL={_agl_sc:.0f}m  valid={int(_vs_sc.sum())}/{len(cal_rx)} ({int(_vs_sc.mean()*100)}%)  RMSE={_rms_sc:.2f} dB  scalar={_sf_sc:+.2f} dB")
        _valid_frac_sc = float(_vs_sc.mean())
        _min_cov = globals().get('TX_AGL_MIN_COVERAGE', 0.80)
        if _valid_frac_sc >= _min_cov and _rms_sc < _best_rmse_sc:
            _best_rmse_sc, _best_agl_sc = _rms_sc, float(_agl_sc)
    print(f"  → Best TX AGL: {_best_agl_sc:.0f} m  (RMSE={_best_rmse_sc:.2f} dB)")
    _tx_obj_sc.position = [float(_safe(_tx_obj_sc.position[0])),
                            float(_safe(_tx_obj_sc.position[1])),
                            _tx_terr_z_sc + _best_agl_sc]
    try:
        import drjit as _drjsc2; _drjsc2.eval(); _drjsc2.sync_thread()
    except: pass

# ── Phase 0: scalar calibration ───────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"Phase 0 — Scalar offset  ({CAL_SAMPLES_SF//1000}k samples)")
_t0 = _tcal.time()
_pl0 = _solve_pl(cal_rx, CAL_SAMPLES_SF)
_v0  = np.isfinite(_pl0)
print(f"  Solved {_tcal.time()-_t0:.0f}s  |  valid paths: {_v0.sum()}/{len(cal_rx)}")
if _v0.sum() == 0:
    raise RuntimeError("No valid paths — check scene geometry and TX position")

# ── Auto-discover effective calibration range from Phase 0 path coverage ─────
_dists_arr = np.array([c[1] for c in _cands])
_MIN_VALID_FRAC = globals().get('CAL_MIN_VALID_FRAC', 0.65)  # from CELL 1
_eff_max_km = CAL_MIN_DIST_KM
for _blo in np.arange(CAL_MIN_DIST_KM, CAL_MAX_DIST_KM, 0.1):
    _mask = (_dists_arr >= _blo) & (_dists_arr < _blo + 0.1)
    if _mask.sum() == 0: continue
    if np.isfinite(_pl0[_mask]).mean() >= _MIN_VALID_FRAC:
        _eff_max_km = round(_blo + 0.1, 2)
    else:
        break
_keep = np.where(_dists_arr <= _eff_max_km + 1e-9)[0]
_cands  = [_cands[i]  for i in _keep]
cal_rx  = [c[0] for c in _cands]
cal_pl  = TX_CONDUCTED_DBM + globals().get('RX_EXTRA_GAIN_DB', 0.0) - np.array([c[2] for c in _cands])
_pl0    = _pl0[_keep]
_N_valid_baseline = int(np.isfinite(_pl0).sum())  # coverage guard: baseline valid-path count
print(f"  Auto-range:  {_eff_max_km:.2f} km  ({len(cal_rx)} RX, \u2265{int(_MIN_VALID_FRAC*100)}% valid per 100m bin)")

# ── Weissberger vegetation pre-correction (CELL CAL) ─────────────────────────
# When DISABLE_VEG_DISCS=True, the RT has no vegetation geometry.
# Vegetation attenuation is in the measurements but absent from simulation.
# Pre-correct cal_pl by subtracting the ITU-R P.833 Weissberger estimate
# so Powell fits RT against vegetation-corrected measurements.
_DO_VEG_CORR = globals().get('DISABLE_VEG_DISCS', False) and globals().get('CAL_APPLY_WEISSBERGER', True)
if _DO_VEG_CORR:
    print(f"\n{chr(9472)*60}")
    print("Weissberger pre-correction: DISABLE_VEG_DISCS=True, CAL_APPLY_WEISSBERGER=True")
    _ndsm_path_cal = globals().get('NDSM_TIFF', '')
    if not _ocal.path.exists(_ndsm_path_cal):
        print(f"  WARNING: NDSM_TIFF not found ({_ndsm_path_cal}) — skipping vegetation correction")
    else:
        import rasterio as _rio_cal
        from rasterio.transform import rowcol as _rio_rowcol_cal
        from pyproj import Transformer as _ProjT_cal
        _freq_ghz_cal  = float(FREQUENCY_HZ) / 1e9
        _veg_min_h_cal = globals().get('VEG_NDMS_MIN_H_M', 2.0)
        _step_m_cal    = 2.0   # nDSM sample spacing along path (m)
        # Scene centre in BNG: scene-local XY offset = BNG coords - scene_centre_BNG
        _to_bng_cal = _ProjT_cal.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
        _cx_bng_cal, _cy_bng_cal = _to_bng_cal.transform(
            (SCENE_WEST + SCENE_EAST) / 2.0,
            (SCENE_SOUTH + SCENE_NORTH) / 2.0)
        with _rio_cal.open(_ndsm_path_cal) as _ds_cal:
            _ndsm_arr_cal = _ds_cal.read(1).astype(np.float32)
            _ndsm_nd_cal  = _ds_cal.nodata or -9999.0
            _ndsm_arr_cal[_ndsm_arr_cal == _ndsm_nd_cal] = 0.0
            _ndsm_tf_cal  = _ds_cal.transform
            _ndsm_h_cal, _ndsm_w_cal = _ndsm_arr_cal.shape
        def _veg_depth_cal(tx_xy, rx_xy):
            """Vegetation depth (m) along TX->RX 2D path via nDSM at _step_m_cal intervals."""
            _dlen = float(np.linalg.norm(rx_xy - tx_xy))
            if _dlen < 1.0:
                return 0.0
            _n_pts = max(2, int(np.ceil(_dlen / _step_m_cal)) + 1)
            _ts    = np.linspace(0.0, 1.0, _n_pts)
            _xs_bng = _cx_bng_cal + tx_xy[0] + _ts * (rx_xy[0] - tx_xy[0])
            _ys_bng = _cy_bng_cal + tx_xy[1] + _ts * (rx_xy[1] - tx_xy[1])
            _rr, _cc = _rio_rowcol_cal(_ndsm_tf_cal, _xs_bng, _ys_bng)
            _rr = np.clip(np.asarray(_rr, dtype=int), 0, _ndsm_h_cal - 1)
            _cc = np.clip(np.asarray(_cc, dtype=int), 0, _ndsm_w_cal - 1)
            _h_vals  = _ndsm_arr_cal[_rr, _cc]
            _seg_len = _dlen / (_n_pts - 1)
            return float(np.sum(_h_vals > _veg_min_h_cal) * _seg_len)
        _cal_veg_A_cal = np.zeros(len(cal_rx))
        _n_veg_corr    = 0
        for _iv, (_rxv, _dkv, _) in enumerate(_cands):
            _rx_xy_v = np.array([_safe(_rxv.position[0]), _safe(_rxv.position[1])])
            _depth_v = _veg_depth_cal(_tx_xy, _rx_xy_v)
            if _depth_v > 0.0:
                _A_v = min(20.0, 0.187 * (_freq_ghz_cal ** 0.284) * (_depth_v ** 0.588))
                _cal_veg_A_cal[_iv] = _A_v
                _n_veg_corr += 1
        cal_pl = cal_pl - _cal_veg_A_cal   # corrected measured PL: vegetation attenuation removed
        _mean_A_cal = float(np.mean(_cal_veg_A_cal[_cal_veg_A_cal > 0])) if _n_veg_corr > 0 else 0.0
        _max_A_cal  = float(np.max(_cal_veg_A_cal))
        print(f"  RX with vegetation on path : {_n_veg_corr}/{len(cal_rx)}")
        print(f"  Weissberger A  : mean={_mean_A_cal:.2f} dB  max={_max_A_cal:.2f} dB")
        print(f"  Frequency      : {_freq_ghz_cal:.4f} GHz")
        print(f"  PL_meas (corrected) range  : {cal_pl.min():.1f} \u2013 {cal_pl.max():.1f} dB")
        del _ndsm_arr_cal   # free memory

_rmse0 = _rmse(_pl0)
print(f"  RMSE before calibration: {_rmse0:.2f} dB")
# Analytic scalar: sf = mean(pl_sim - pl_meas) is the exact MSE minimiser — no bounds needed.
# Works for any city regardless of uncalibrated bias magnitude.
_v0 = np.isfinite(_pl0)
scalar_factor_db = float(np.mean(_pl0[_v0] - cal_pl[_v0]))
print(f"  Uncalibrated mean bias: {scalar_factor_db:+.1f} dB")
if abs(scalar_factor_db) > 20:
    print(f"  WARNING: large uncalibrated bias ({scalar_factor_db:+.1f} dB). Check EIRP, RX chain, and CSV power reference.")
rmse_sf = _rmse(_pl0, scalar_factor_db)
print(f"  scalar_factor_db = {scalar_factor_db:+.3f} dB")
print(f"  RMSE after scalar: {rmse_sf:.2f} dB  (Δ {_rmse0-rmse_sf:+.2f} dB)")
rmse_best = rmse_sf

# ── Enumerate materials ────────────────────────────────────────────────────
_mats = {}
for _mn, _m in scene.radio_materials.items():
    if _mn.endswith('_train'): continue
    # all materials calibrated per-city — scene builder portable, calibration city-specific
    try:
        _e  = float(np.clip(_to_float(_m.relative_permittivity), _ER_MIN, _ER_MAX))
        _sg = float(np.clip(_to_float(_m.conductivity),          _SIG_MIN, _SIG_MAX))
        _sc = float(np.clip(_to_float(getattr(_m, 'scattering_coefficient', 0.2)), _S_MIN, _S_MAX))
        if not all(np.isfinite([_e, _sg, _sc])): continue
        _mats[_mn] = {'er': _e, 'sig': _sg, 's': _sc,
                      'er0': _e, 'sig0': _sg, 's0': _sc}
    except: continue

# ── Lock physics-derived materials fixed at CELL 1 config — skip in optimizer ─
_CAL_FIXED = globals().get('CAL_FIXED_MATS', set())
_MAT_FIXED_VALS = {
    'itu_ceiling_board': (globals().get('VEG_RELATIVE_PERMITTIVITY', 17.0),
                          globals().get('VEG_CONDUCTIVITY', 0.05),
                          globals().get('VEG_SCATTERING_COEFF', 0.50)),
}
for _fn in list(_CAL_FIXED):
    if _fn in _mats and _fn in scene.radio_materials:
        _fv = _MAT_FIXED_VALS.get(_fn)
        if _fv:
            if _fn in _mat_vars:
                # In-place update — preserves kernel/seed (no new dr.Float nodes)
                _mat_vars[_fn]['er'][0]  = float(_fv[0])
                _mat_vars[_fn]['sig'][0] = float(_fv[1])
                _mat_vars[_fn]['s'][0]   = float(_fv[2])
                try:
                    import drjit as _dr_fix; _dr_fix.eval(); _dr_fix.sync_thread()
                except: pass
            else:
                _set_mat(scene.radio_materials[_fn], er=_fv[0], sigma=_fv[1], s=_fv[2])
        del _mats[_fn]
# ── Apply warm prior to optimised materials only (after _mats is built) ───
# Applied unconditionally when fresh start detected — gives Powell a non-zero
# scatter starting point (matches 2695 MHz behaviour). _mats guard ensures
# vegetation/water/fixed materials are never bumped.
if _apply_warm_prior:
    print(f"  Applying S warm prior = {_WARM_S_PRIOR} to optimised materials")
    for _mn_l, _ml in scene.radio_materials.items():
        if _mn_l not in _mats or _mn_l.endswith('_train'):
            continue  # only bump optimised materials — never touch vegetation/water/fixed
        try:
            if float(_to_float(getattr(_ml, 'scattering_coefficient', 0.0))) < _WARM_S_PRIOR:
                if _mn_l in _mat_vars:
                    _mat_vars[_mn_l]['s'][0] = _WARM_S_PRIOR  # in-place: no kernel recompile
                else:
                    _ml.scattering_coefficient = _WARM_S_PRIOR
                _mats[_mn_l]['s'] = _WARM_S_PRIOR  # update _mats so _x0 starts from warm prior
        except: pass


print(f"\n{'═'*60}")
print(f"Materials found: {len(_mats)}  (fixed: {len(_CAL_FIXED & set(scene.radio_materials))})")
for _mn, _p in _mats.items():
    print(f"  {_mn:<28} er={_p['er']:.3f}  sigma={_p['sig']:.5f}  S={_p['s']:.3f}")

# ── Sensitivity probe ─────────────────────────────────────────────────────
_mat_sensitive = False
_SKIP_PROBE = globals().get('CAL_SKIP_PROBE', False)
if _mats and not CAL_SCALAR_ONLY and _SKIP_PROBE:
    _mat_sensitive = True
    print(f"\n{'─'*60}")
    print(f"Sensitivity probe: CAL_SKIP_PROBE=True — assuming sensitive (empirical Δ=0.940 dB)")
elif _mats and not CAL_SCALAR_ONLY:
    print(f"\n{'─'*60}")
    print("Sensitivity probe (vacuum test) ...")
    # Pick the material with the most scene geometry — small materials (itu_wood=62 faces)
    # are invisible to the ray tracer and will always return delta=0, causing a false
    # "insensitive" result. Prefer itu_brick > itu_concrete > itu_metal > anything else.
    _probe_priority = ['itu_brick', 'itu_concrete', 'itu_metal', 'itu_glass']
    _probe_name = next((m for m in _probe_priority if m in _mats), next(iter(_mats)))
    _probe_mat  = scene.radio_materials[_probe_name]
    _p0 = _mats[_probe_name]
    # Base: reuse Phase 0 RMSE (30M samples, reliable) — avoids re-solve and sample-count mismatch.
    # CAL_SAMPLES_WARM (7.5M) gives RMSE ~2 dB different from Phase 0 (30M), making Δ unreadable.
    _rm_p0 = rmse_sf
    if _probe_name in _mat_vars:
        _mat_vars[_probe_name]['er'][0]  = 1.0
        _mat_vars[_probe_name]['sig'][0] = 1e-6
        _mat_vars[_probe_name]['s'][0]   = 0.0
        try: import drjit as _dr_prb; _dr_prb.eval(); _dr_prb.sync_thread()
        except: pass
    else:
        _set_mat(_probe_mat, er=1.0, sigma=1e-6, s=0.0)
    # Vacuum test at CAL_SAMPLES_SF (same as Phase 0) — reliable Δ above MC noise floor
    _pl_p1 = _solve_pl(cal_rx, CAL_SAMPLES_SF)
    _rm_p1 = _rmse(_pl_p1, scalar_factor_db)
    if _probe_name in _mat_vars:
        _mat_vars[_probe_name]['er'][0]  = _p0['er']
        _mat_vars[_probe_name]['sig'][0] = _p0['sig']
        _mat_vars[_probe_name]['s'][0]   = _p0['s']
        try: _dr_prb.eval(); _dr_prb.sync_thread()
        except: pass
    else:
        _set_mat(_probe_mat, er=_p0['er'], sigma=_p0['sig'], s=_p0['s'])
    _delta = abs(_rm_p1 - _rm_p0)
    print(f"  {_probe_name}: base={_rm_p0:.3f} dB  vacuum={_rm_p1:.3f} dB  Δ={_delta:.3f} dB  (30M samples)")
    if _apply_warm_prior:
        _v_wp = np.isfinite(_pl_p1)
        scalar_factor_db = float(np.mean(_pl_p1[_v_wp] - cal_pl[_v_wp]))
        rmse_sf = _rmse(_pl_p1, scalar_factor_db)
        print(f"  Re-scalar (warm-prior materials): {scalar_factor_db:+.3f} dB  RMSE={rmse_sf:.2f} dB")
    _mat_sensitive = _delta > 0.40
    if not _mat_sensitive and _ip_ok:
        # In-place DrJIT Δ below threshold — may be Early Option A not propagating.
        # Retry using _set_mat (Sionna public API) which forces a proper scene update.
        print(f"  In-place Δ={_delta:.3f} dB — retrying with _set_mat to confirm ...")
        _set_mat(_probe_mat, er=1.0, sigma=1e-6, s=0.0)
        _pl_sm = _solve_pl(cal_rx, CAL_SAMPLES_SF)
        _rm_sm = _rmse(_pl_sm, scalar_factor_db)
        _set_mat(_probe_mat, er=_p0['er'], sigma=_p0['sig'], s=_p0['s'])
        _delta_sm = abs(_rm_sm - _rm_p0)
        print(f"  {_probe_name}: _set_mat Δ={_delta_sm:.3f} dB  (30M samples)")
        if _delta_sm > 0.40:
            print(f"  Early Option A in-place update not propagating — disabling, using _set_mat for Powell")
            _ip_ok = False
            _mat_sensitive = True
        else:
            print(f"  Scene truly insensitive to {_probe_name} material (Δ={_delta_sm:.3f} dB < 0.40) — scalar only")
    print(f"  {'Materials SENSITIVE — proceeding' if _mat_sensitive else 'Materials insensitive — scalar only'}")

_r2_at_1km = float('nan')  # quick R² at 1km — set by Phase 3, used by retry
if _mats and _mat_sensitive and not CAL_SCALAR_ONLY:

    _eval_n   = [0]
    _eval_t   = [_tcal.time()]
    _history       = []
    _param_history = []  # x snapshot per eval

    # Distance weights for balanced near/far calibration (controlled by CAL_FAR_WEIGHT)
    _cal_dists_km = np.array([c[1] for c in _cands])  # distances to cal RX (km)
    _CAL_FAR_W = globals().get('CAL_FAR_WEIGHT', 1.0)
    if _CAL_FAR_W > 0:
        _cal_w = 1.0 + _CAL_FAR_W * (np.sqrt(np.clip(_cal_dists_km, 0.1, None)) - 1.0)
        _cal_w = np.clip(_cal_w, 0.5, None)
    else:
        _cal_w = np.ones(len(_cal_dists_km))
    _cal_w /= _cal_w.mean()  # normalise so mean weight = 1
    _CAL_COV_MIN = globals().get('CAL_COVERAGE_MIN', 0.90)  # fraction of baseline valid receivers required
    _CAL_COV_PENALTY = 20.0  # dB added per unit coverage fraction lost below threshold

    def _loss(x, n_samples=None, sf=None):
        """Full loss: apply all material params from flat vector x, return weighted RMSE."""
        if n_samples is None: n_samples = CAL_SAMPLES_POWELL
        if sf is None: sf = float(x[_n_params]) if len(x) > _n_params else scalar_factor_db
        for _i, (_mn, _p) in enumerate(_mats_list):
            _er  = float(np.clip(x[_pp*_i],   _ER_MIN,  _ER_MAX))
            _sg  = float(np.clip(np.exp(x[_pp*_i+1]), _SIG_MIN, _SIG_MAX))
            _sc  = float(_p['s']) if CAL_FIX_SCATTER else float(np.clip(x[_pp*_i+2], _S_MIN, _S_MAX))
            _set_mat(scene.radio_materials[_mn], er=_er, sigma=_sg, s=_sc)
        _pl = _solve_pl(cal_rx, n_samples)
        # Weighted RMSE: emphasises far-range receivers via sqrt(d) weight
        _err = _pl - sf - cal_pl
        _finite = np.isfinite(_err)
        if _finite.sum() == 0:
            _rm = 999.0
        else:
            _rm = float(np.sqrt(np.average(_err[_finite]**2, weights=_cal_w[_finite])))
        _n_valid_now = int(np.isfinite(_pl).sum())
        _cov_frac = _n_valid_now / max(_N_valid_baseline, 1)
        if _cov_frac < _CAL_COV_MIN:
            _rm += (_CAL_COV_MIN - _cov_frac) * _CAL_COV_PENALTY
        _eval_n[0] += 1
        _history.append(_rm)
        _param_history.append(x.copy())
        _dt = _tcal.time() - _eval_t[0]; _eval_t[0] = _tcal.time()
        print(f"  eval {_eval_n[0]:3d}  RMSE={_rm:.3f} dB  [{_dt:.0f}s]")
        return _rm

    _mats_list = list(_mats.items())
    _pp = 2 if CAL_FIX_SCATTER else 3  # params per material: er+sig only if S fixed
    _n_params  = len(_mats_list) * _pp
    _x0        = np.array([v for _mn, _p in _mats_list
                           for v in ([_p['er'], np.log(max(_p['sig'], 1e-6))]
                                     if CAL_FIX_SCATTER else
                                     [_p['er'], np.log(max(_p['sig'], 1e-6)), _p['s']])])
    _bounds = []
    for _mn, _p in _mats_list:
        _bounds += [(max(_ER_MIN, _p['er0']*0.7),  min(_ER_MAX,  _p['er0']*1.5)),
                    (np.log(max(_SIG_MIN_PER_MAT.get(_mn, _SIG_MIN), _p['sig0']*0.01)), np.log(min(_SIG_MAX, _p['sig0']*100)))]
        if not CAL_FIX_SCATTER:
            _bounds += [(_S_MIN_PER_MAT.get(_mn, _S_MIN), _S_MAX_PER_MAT.get(_mn, _S_MAX))]

    # ── Scalar as 28th optimisation parameter ────────────────────────────
    _x0 = np.append(_x0, scalar_factor_db)
    # ── Scalar bounds from CAL_SCALAR_BOUNDS config (wide for urban scenes) ──────
    _scalar_lo, _scalar_hi = tuple(globals().get('CAL_SCALAR_BOUNDS', (-20.0, 5.0)))
    _bounds.append((_scalar_lo, _scalar_hi))

    # ── Phase 1: coordinate-descent warm-up ──────────────────────────────
    print(f"\n{'═'*60}")
    print(f"Phase 1 — Coordinate-descent warm-up  ({CAL_SAMPLES_WARM//1000}k samples)")
    print(f"  {CAL_WARM_CYCLES} cycle(s) × {len(_mats_list)} materials × 3 params × {CAL_WARM_EVALS_1D} evals")
    _t1 = _tcal.time()

    _x_warm = _x0.copy()
    for _cycle in range(CAL_WARM_CYCLES):
        print(f"\n  Cycle {_cycle+1}/{CAL_WARM_CYCLES}")
        _improved = 0.0
        for _i, (_mn, _p) in enumerate(_mats_list):
            _i3 = _pp * _i
            # er
            _lo, _hi = _bounds[_i3]
            _res = minimize_scalar(
                lambda v, _i3=_i3: _loss(np.where(np.arange(len(_x_warm))==_i3, v, _x_warm),
                                         CAL_SAMPLES_WARM),
                bounds=(_lo, _hi), method='bounded',
                options={'maxiter': CAL_WARM_EVALS_1D, 'xatol': 0.05})
            _x_warm[_i3] = float(_res.x)
            # log(sigma)
            _lo, _hi = _bounds[_i3+1]
            _res = minimize_scalar(
                lambda v, _i3=_i3: _loss(np.where(np.arange(len(_x_warm))==(_i3+1), v, _x_warm),
                                         CAL_SAMPLES_WARM),
                bounds=(_lo, _hi), method='bounded',
                options={'maxiter': CAL_WARM_EVALS_1D, 'xatol': 0.05})
            _x_warm[_i3+1] = float(_res.x)
            # S
            _lo, _hi = _bounds[_i3+2]
            _res = minimize_scalar(
                lambda v, _i3=_i3: _loss(np.where(np.arange(len(_x_warm))==(_i3+2), v, _x_warm),
                                         CAL_SAMPLES_WARM),
                bounds=(_lo, _hi), method='bounded',
                options={'maxiter': CAL_WARM_EVALS_1D, 'xatol': 0.01})
            _x_warm[_i3+2] = float(_res.x)

    _rmse_warm = _history[-1] if _history else rmse_sf
    print(f"\n  Warm-up done in {(_tcal.time()-_t1)/60:.1f} min  ({_eval_n[0]} evals)")
    print(f"  RMSE after warm-up: {_rmse_warm:.2f} dB  (Δ vs scalar: {rmse_sf-_rmse_warm:+.2f} dB)")

    # ── Phase 1.5: genetic warm-up (hybrid mode) ─────────────────────────
    _CAL_OPTIMIZER   = globals().get('CAL_OPTIMIZER',       'powell')
    _CAL_GEN_SAMPLES = globals().get('CAL_GENETIC_SAMPLES', 500_000)
    _CAL_GEN_POPSIZE = globals().get('CAL_GENETIC_POPSIZE', 5)
    _CAL_GEN_MAXITER = globals().get('CAL_GENETIC_MAXITER', 30)
    if _CAL_OPTIMIZER == 'hybrid' and _CAL_GEN_SAMPLES > 0 and not CAL_SCALAR_ONLY:
        print(f"\n{chr(9552)*60}")
        print(f"Phase 1.5 — Genetic warm-up  ({_CAL_GEN_SAMPLES//1000}k samples, pop={_CAL_GEN_POPSIZE}, maxiter={_CAL_GEN_MAXITER})")
        _t_gen = _tcal.time()
        _de_res = differential_evolution(
            lambda x: _loss(x, _CAL_GEN_SAMPLES),
            bounds=_bounds,
            maxiter=_CAL_GEN_MAXITER,
            popsize=_CAL_GEN_POPSIZE,
            seed=42, tol=0.005,
            mutation=(0.5, 1.0), recombination=0.7,
            workers=1, polish=False)
        _x_warm = _de_res.x
        print(f"\n  Genetic done in {(_tcal.time()-_t_gen)/60:.1f} min  RMSE={_de_res.fun:.3f} dB")

    # ── Phase 2: joint Powell refinement ─────────────────────────────────
    # Prime the 2M-sample kernel: Phase 1 (500k) evicts it from DrJIT cache.
    # One dummy solve forces recompile before the probe and minimize() use it.
    if not CAL_SCALAR_ONLY:
        _sf_old = float(_x_warm[-1]) if len(_x_warm) > _n_params else scalar_factor_db
        if _eval_n[0] > 0 or CAL_SAMPLES_SF != CAL_SAMPLES_POWELL or _eff_max_km < CAL_MAX_DIST_KM:
            # Phase 1 ran evals — Phase 0 kernel may be evicted. Reprime.
            print("  Priming kernel ...")
            _pl_prime = _solve_pl(cal_rx, CAL_SAMPLES_POWELL)
            print("  Kernel primed.")
            # If Early Option A is active, the priming solve above compiled a kernel
            # without _mat_vars in the traced graph (plain _solve_pl has no mat updates).
            # Re-attach _mat_vars to Sionna material properties and re-prime so the
            # next kernel includes _mat_vars — otherwise _loss_inplace is stuck.
            if _ip_ok and _mat_vars:
                _prop_map = [('relative_permittivity', 'er'),
                             ('conductivity', 'sig'),
                             ('scattering_coefficient', 's')]
                for _mn_ra, _ in _mats_list:
                    if _mn_ra not in _mat_vars: continue
                    _mat_ra = scene.radio_materials[_mn_ra]
                    for _prop_ra, _key_ra in _prop_map:
                        try:
                            setattr(_mat_ra, _prop_ra, _mat_vars[_mn_ra][_key_ra])
                        except Exception: pass
                try:
                    import drjit as _dr_ra; _dr_ra.eval(); _dr_ra.sync_thread()
                except Exception: pass
                _pl_prime = _solve_pl(cal_rx, CAL_SAMPLES_POWELL)
                print("  _mat_vars re-attached and kernel re-primed with traced variables.")
            _v_pgs = np.isfinite(_pl_prime)
            _sf_new = float(np.mean(_pl_prime[_v_pgs] - cal_pl[_v_pgs]))
            scalar_factor_db = _sf_new
            if len(_x_warm) > _n_params:
                _x_warm = np.concatenate([_x_warm[:_n_params], [_sf_new]])
            print(f"  Post-priming re-scalar ({CAL_SAMPLES_POWELL//1000}k): "
                  f"{_sf_new:+.3f} dB  RMSE={_rmse(_pl_prime, _sf_new):.3f} dB  "
                  f"[was {_sf_old:+.3f} dB at warm-phase]")
        else:
            # Phase 1 ran 0 evals at same sample count — Phase 0 kernel still live.
            # Reuse _pl0 directly: same random seed, no new PathSolver call.
            _pl_prime = _pl0.copy()
            _sf_new   = scalar_factor_db
            print(f"  Priming skipped (Phase 1: 0 evals, same sample count) — "
                  f"reusing Phase 0 kernel (_pl0), scalar={_sf_new:+.3f} dB, "
                  f"RMSE={_rmse(_pl_prime, _sf_new):.3f} dB")

    print(f"\n{'═'*60}")
    print(f"Phase 2 — Joint Powell refinement  ({CAL_SAMPLES_POWELL//1000}k samples)")
    print(f"  {_n_params + 1} params (incl. scalar), max {CAL_POWELL_MAXITER} iters")
    print(f"  Starting from coord-descent warm point")
    _t2 = _tcal.time()
    _eval_n_pre_powell = _eval_n[0]

    # Phase 2 DrJIT re-check: switching to CAL_SAMPLES_POWELL can trigger
    # a kernel recompile which then caches. Detect before wasting 300+ evals.
    _SKIP_P2_PROBE = globals().get('CAL_SKIP_P2_PROBE', False)
    # If Early Option A was disabled (in-place not propagating), sensitivity probe already
    # confirmed kernel liveness via _set_mat — P2 probe would give false positive (sigma
    # has low sensitivity when disc scatter dominates), so skip it.
    if not _ip_ok and _mat_sensitive:
        _SKIP_P2_PROBE = True
        print("  Phase 2 probe skipped — kernel liveness already confirmed via _set_mat in sensitivity probe")
    if not CAL_SCALAR_ONLY and _SKIP_P2_PROBE:
        print("  CAL_SKIP_P2_PROBE=True — skipping Phase 2 kernel live check (avoids seed reset)")
    elif not CAL_SCALAR_ONLY:
        # Reuse _pl_prime for base — avoids a fresh PathSolver call and its new random seed.
        _p2_base = _rmse(_pl_prime, _sf_new)
        _p2_priority = ['itu_brick', 'itu_concrete', 'itu_metal']
        _p2_mat  = next((m for m in _p2_priority if m in dict(_mats_list)), _mats_list[0][0])
        _p2_idx  = next(i for i, (mn, _) in enumerate(_mats_list) if mn == _p2_mat)
        if _ip_ok:
            # Early Option A is active: probe via in-place mutation ONLY.
            # Using _loss here would call _set_mat + flush_kernel_cache, recompiling the
            # DrJIT kernel and disconnecting _mat_vars from the active computation graph.
            # All subsequent _loss_inplace evals would then return a cached stale value.
            _er_p2_orig = float(_mat_vars[_p2_mat]['er'][0])
            _mat_vars[_p2_mat]['er'][0] = float(np.clip(_er_p2_orig * 5.0, _ER_MIN, _ER_MAX))
            try: import drjit as _dr_p2c; _dr_p2c.eval(); _dr_p2c.sync_thread()
            except: pass
            _pl_p2ip = _solve_pl(cal_rx, CAL_SAMPLES_POWELL)
            _rm_p2ip = _rmse(_pl_p2ip, _sf_new)
            _mat_vars[_p2_mat]['er'][0] = _er_p2_orig  # restore
            try: import drjit as _dr_p2r; _dr_p2r.eval(); _dr_p2r.sync_thread()
            except: pass
            _delta_p2 = abs(_rm_p2ip - _p2_base)
            if _delta_p2 < 0.05:
                print(f"  Phase 2 in-place probe: Δ={_delta_p2:.3f} dB — in-place not propagating, disabling Early Option A")
                _ip_ok = False
                # Fall through to _loss-based probe below
            else:
                print(f"  Phase 2 kernel live (in-place Δ={_delta_p2:.3f} dB, probe={_p2_mat}) -- proceeding with Early Option A")
        if not _ip_ok:
            # Standard _loss-based probe (Early Option A disabled or failed in-place check above)
            _x_p2_probe = _x_warm.copy()
            if CAL_FIX_SCATTER:
                _p2_s_idx = _p2_idx * _pp + 1
                _p2_sig_lo, _p2_sig_hi = _bounds[_p2_idx * _pp + 1]
                _x_p2_probe[_p2_s_idx] = float(np.clip(_x_p2_probe[_p2_s_idx] + 0.5, _p2_sig_lo, _p2_sig_hi))
            else:
                _p2_s_idx = _p2_idx * _pp + 2
                _x_p2_probe[_p2_s_idx] = float(np.clip(_x_p2_probe[_p2_s_idx] + 0.30, _S_MIN, _S_MAX))
            _p2_probe_rm = _loss(_x_p2_probe, CAL_SAMPLES_POWELL)
            _delta_p2 = abs(_p2_base - _p2_probe_rm)
            if _delta_p2 < 0.05:
                print(f"  WARNING: Phase 2 kernel frozen (delta={_delta_p2:.3f} dB < 0.05 dB, probe={_p2_mat}) -- aborting Powell, scalar-only fallback")
                CAL_SCALAR_ONLY = True
            else:
                print(f"  Phase 2 kernel live (delta={_delta_p2:.3f} dB, probe={_p2_mat}) -- proceeding")

    if not CAL_SCALAR_ONLY:
        # Bump S for materials stuck near zero — only when S is a free parameter.
        if not CAL_FIX_SCATTER:
            for _i_ws in range(len(_mats_list)):
                _s_idx = _i_ws * _pp + 2
                if _x_warm[_s_idx] < globals().get('CAL_S_BUMP_FLOOR', 0.0):
                    _x_warm[_s_idx] = min(0.50, _S_MAX)

        # Option A: _mat_vars already pre-assigned before Phase 0 (early Option A).
        # _ip_ok set at top of CELL CAL — no setup or re-prime needed here.
        if _ip_ok:
            print(f"  Phase 2: using Early Option A ({len(_mat_vars)} materials, same kernel/seed as Phase 0)")

        def _loss_inplace(x, n_samples=None, sf=None):
            """Loss with in-place DrJIT variable updates — avoids kernel recompile."""
            if n_samples is None: n_samples = CAL_SAMPLES_POWELL
            if sf is None: sf = float(x[_n_params]) if len(x) > _n_params else scalar_factor_db
            for _i, (_mn, _p) in enumerate(_mats_list):
                _er  = float(np.clip(x[_pp*_i],   _ER_MIN,  _ER_MAX))
                _sg  = float(np.clip(np.exp(x[_pp*_i+1]), _SIG_MIN, _SIG_MAX))
                _sc  = float(_p['s']) if CAL_FIX_SCATTER else float(np.clip(x[_pp*_i+2], _S_MIN, _S_MAX))
                if _mn in _mat_vars:
                    _mat_vars[_mn]['er'][0]  = _er
                    _mat_vars[_mn]['sig'][0] = _sg
                    _mat_vars[_mn]['s'][0]   = _sc
                else:
                    _set_mat(scene.radio_materials[_mn], er=_er, sigma=_sg, s=_sc)
            try:
                import drjit as _dr_ip2
                _dr_ip2.eval(); _dr_ip2.sync_thread()
            except: pass
            _pl = _solve_pl(cal_rx, n_samples)
            _err = _pl - sf - cal_pl
            _finite = np.isfinite(_err)
            if _finite.sum() == 0:
                _rm = 999.0
            else:
                _rm = float(np.sqrt(np.average(_err[_finite]**2, weights=_cal_w[_finite])))
            _n_valid_now = int(np.isfinite(_pl).sum())
            _cov_frac = _n_valid_now / max(_N_valid_baseline, 1)
            if _cov_frac < _CAL_COV_MIN:
                _rm += (_CAL_COV_MIN - _cov_frac) * _CAL_COV_PENALTY
            _eval_n[0] += 1
            _history.append(_rm)
            _param_history.append(x.copy())
            _dt = _tcal.time() - _eval_t[0]; _eval_t[0] = _tcal.time()
            print(f"  eval {_eval_n[0]:3d}  RMSE={_rm:.3f} dB  [{_dt:.0f}s]")
            return _rm

        _powell_fn = _loss_inplace if _ip_ok else _loss

        try:
            _res_powell = minimize(
                lambda x: _powell_fn(x, CAL_SAMPLES_POWELL),
                _x_warm,
                method='Powell',
                bounds=_bounds,
                options={'maxiter': CAL_POWELL_MAXITER,
                         'xtol': CAL_POWELL_XTOL,
                         'ftol': CAL_POWELL_FTOL,
                         'return_all': False})
            _powell_interrupted = False
        except KeyboardInterrupt:
            _powell_interrupted = True
            # Recover best point seen so far from history
            _best_idx = int(np.argmin(_history)) if _history else 0
            print(f"\n  *** Interrupted at eval {_eval_n[0]} — recovering best (eval {_best_idx+1}, RMSE={_history[_best_idx]:.3f} dB) ***")
            # Build a mock result object
            class _MockResult:
                x       = _param_history[_best_idx]
                fun     = _history[_best_idx]
                success = False
                message = 'KeyboardInterrupt — best-so-far recovered'
            _res_powell = _MockResult()

        _powell_evals = _eval_n[0] - _eval_n_pre_powell
        print(f"\n  Powell done in {(_tcal.time()-_t2)/60:.1f} min  ({_powell_evals} evals)")
        print(f"  Converged: {_res_powell.success}  |  {_res_powell.message}")
        rmse_powell = float(_res_powell.fun)
        print(f"  RMSE after Powell: {rmse_powell:.2f} dB  (Δ vs warm-up: {_rmse_warm-rmse_powell:+.2f} dB)")

        # Apply best params
        _x_final = _res_powell.x
        if len(_x_final) > _n_params:
            scalar_factor_db = float(np.clip(_x_final[_n_params], -20.0, 5.0))
            print(f"  Powell scalar: {scalar_factor_db:+.3f} dB")
        for _i, (_mn, _p) in enumerate(_mats_list):
            _p['er']  = float(np.clip(_x_final[_pp*_i],   _ER_MIN,  _ER_MAX))
            _p['sig'] = float(np.clip(np.exp(_x_final[_pp*_i+1]), _SIG_MIN, _SIG_MAX))
            if not CAL_FIX_SCATTER:
                _p['s'] = float(np.clip(_x_final[_pp*_i+2], _S_MIN, _S_MAX))
            _set_mat(scene.radio_materials[_mn], er=_p['er'], sigma=_p['sig'], s=_p['s'])
    else:
        print("  Powell skipped (kernel cached) — warm-up params applied")
        rmse_powell = _rmse_warm
        for _i, (_mn, _p) in enumerate(_mats_list):
            _p['er']  = float(np.clip(_x_warm[_pp*_i],   _ER_MIN,  _ER_MAX))
            _p['sig'] = float(np.clip(np.exp(_x_warm[_pp*_i+1]), _SIG_MIN, _SIG_MAX))
            if not CAL_FIX_SCATTER:
                _p['s'] = float(np.clip(_x_warm[_pp*_i+2], _S_MIN, _S_MAX))
            _set_mat(scene.radio_materials[_mn], er=_p['er'], sigma=_p['sig'], s=_p['s'])

    _sf_phase0 = scalar_factor_db  # save Phase 0 scalar before Phase 3 may overwrite
    # ── Phase 3: re-scalar with calibrated materials ──────────────────────
    print(f"\n{'═'*60}")
    print(f"Phase 3 — Re-scalar with calibrated materials  ({CAL_SAMPLES_FINAL//1000}k samples)")
    _pl_final = _solve_pl(cal_rx, CAL_SAMPLES_FINAL)
    _vf = np.isfinite(_pl_final)
    _vf2 = np.isfinite(_pl_final)
    scalar_factor_db = float(np.mean(_pl_final[_vf2] - cal_pl[_vf2]))
    rmse_best = _rmse(_pl_final, scalar_factor_db)
    if rmse_best > rmse_powell + 0.05:
        print(f"  Phase 3 re-scalar worsened RMSE ({rmse_best:.2f} dB > Powell {rmse_powell:.2f} dB) — reverting to Phase 0 scalar")
        scalar_factor_db = _sf_phase0
        rmse_best = rmse_powell
    print(f"  scalar_factor_db (final) = {scalar_factor_db:+.3f} dB")
    print(f"  RMSE final: {rmse_best:.2f} dB")

    # ── Quick R² preview on calibration receivers ─────────────────────
    _pred_r2 = _pl_final - scalar_factor_db  # sign: consistent with _rmse(pl, sf) = pl - sf - cal_pl
    _fin_r2  = np.isfinite(_pred_r2)
    print(f"\n{chr(9472)*60}")
    print("Quick R\u00b2 preview (calibration receivers):")
    for _dmax_r in sorted(set([0.50, 0.75, 1.00, round(_eff_max_km, 2)])):
        _mk = (_cal_dists_km <= _dmax_r) & _fin_r2
        if _mk.sum() < 5: continue
        _sse_r = float(np.sum((_pred_r2[_mk] - cal_pl[_mk])**2))
        _sst_r = float(np.sum((cal_pl[_mk] - cal_pl[_mk].mean())**2))
        _r2_r  = float(1.0 - _sse_r/_sst_r) if _sst_r > 0 else float('nan')
        _rm_r  = float(np.sqrt(np.mean((_pred_r2[_mk] - cal_pl[_mk])**2)))
        if abs(_dmax_r - 1.0) < 0.01: _r2_at_1km = _r2_r
        print(f"  0-{_dmax_r:.2f}km  N={int(_mk.sum()):3d}  RMSE={_rm_r:.2f} dB  R\u00b2={_r2_r:+.3f}")

else:
    if CAL_SCALAR_ONLY:
        print("\nCAL_SCALAR_ONLY=True — scalar offset only.")
    elif not _mats:
        print("\nNo calibratable materials found.")
    else:
        print("\nMaterials insensitive to EM parameter changes — scalar offset retained.")

# ── Final summary ─────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"{'Step':<35} {'RMSE (dB)':>10}")
print(f"{'─'*47}")
print(f"{'Before calibration':<35} {_rmse0:>10.2f}")
print(f"{'After scalar offset':<35} {rmse_sf:>10.2f}")
if rmse_best < rmse_sf - 0.01:
    print(f"{'After material + re-scalar':<35} {rmse_best:>10.2f}")
    print(f"{'Total improvement':<35} {_rmse0-rmse_best:>+10.2f}")

# ── Save ──────────────────────────────────────────────────────────────────
_calib_out = {
    'meta': {
        'source': 'CELL CAL (coord-descent + Powell)',
        'frequency_mhz': float(FREQUENCY_HZ / 1e6),
        'n_calib_rx': len(cal_rx),
        'total_evals': locals().get('_eval_n', [0])[0],
        'rmse_before_db':   round(_rmse0, 3),
        'rmse_scalar_db':   round(rmse_sf, 3),
        'rmse_final_db':    round(rmse_best, 3),
    },
    'materials': {}
}
for _mn, _p in _mats.items():
    _calib_out['materials'][_mn] = {
        'er':      round(_p['er'],  4),
        'sigma':   round(_p['sig'], 6),
        'scatter': round(_p['s'],   4),
    }
for _mn, _m in scene.radio_materials.items():
    if _mn.endswith('_train') or _mn in _calib_out['materials']: continue
    try:
        _calib_out['materials'][_mn] = {
            'er':      round(_to_float(_m.relative_permittivity), 4),
            'sigma':   round(_to_float(_m.conductivity), 6),
            'scatter': round(_to_float(getattr(_m, 'scattering_coefficient', 0.2)), 4),
        }
    except: pass

_CALIB_FILE = _ocal.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
with open(_CALIB_FILE, 'w') as _f: _jcal.dump(_calib_out, _f, indent=2)

_SF_FILE = _ocal.path.join(BASE_DIR, 'scalar_offset_stevenage_2695mhz.json')
with open(_SF_FILE, 'w') as _f:
    _jcal.dump({'scalar_factor_db': scalar_factor_db,
                'meta': {'rmse_before_db': round(_rmse0, 3),
                         'rmse_final_db':  round(rmse_best, 3)}}, _f, indent=2)

print(f"\nSaved: {_CALIB_FILE}")
print(f"Saved: {_SF_FILE}")
print("Run CELL 4A to apply calibrated materials, then CELL 7/8e to verify.")

# ── Auto-iteration 2: reload calibrated files + re-run Powell only ────────
_CALIB_FILE_IT = _ocal.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
_SF_FILE_IT    = _ocal.path.join(BASE_DIR, 'scalar_offset_stevenage_2695mhz.json')
if _CAL_N_ITER >= 2 and _ocal.path.exists(_CALIB_FILE_IT) and '_mats_list' in dir():
    print(f"\n{'█'*60}")
    print(f"Auto-iteration 2/{_CAL_N_ITER} — reloading calibrated files + re-running Powell")
    with open(_CALIB_FILE_IT) as _fi2: _jit = _jcal.load(_fi2)
    for _mn_it, _mv in _jit['materials'].items():
        if _mn_it in scene.radio_materials:
            _set_mat(scene.radio_materials[_mn_it],
                     er=_mv['er'], sigma=_mv['sigma'], s=_mv['scatter'])
    with open(_SF_FILE_IT) as _fi2: _jsf2 = _jcal.load(_fi2)
    scalar_factor_db = _jsf2['scalar_factor_db']
    _x_it = np.array([v for _mn_it, _p in _mats_list
                      for v in ([_p['er'], np.log(max(_p['sig'], 1e-6))]
                                if CAL_FIX_SCATTER else
                                [_p['er'], np.log(max(_p['sig'], 1e-6)), _p['s']])])
    # Re-read material values from scene after reload
    for _ii, (_mn_it, _p) in enumerate(_mats_list):
        _m_it = scene.radio_materials.get(_mn_it)
        if _m_it:
            _x_it[_pp*_ii]   = float(np.clip(_to_float(_m_it.relative_permittivity), _ER_MIN, _ER_MAX))
            _x_it[_pp*_ii+1] = float(np.log(max(float(np.clip(_to_float(_m_it.conductivity), _SIG_MIN, _SIG_MAX)), 1e-9)))
            if not CAL_FIX_SCATTER:
                _x_it[_pp*_ii+2] = float(np.clip(_to_float(getattr(_m_it, 'scattering_coefficient', 0.2)), _S_MIN, _S_MAX))
    _x_it = np.append(_x_it, scalar_factor_db)   # include scalar as last param to match _bounds length
    print(f"  Starting Powell from loaded point  RMSE={_loss(_x_it, CAL_SAMPLES_POWELL):.2f} dB")
    _eval_n[0] = 0; _t_it = _tcal.time()
    _res_it = minimize(lambda x: _loss(x, CAL_SAMPLES_POWELL), _x_it,
                       method='Powell', bounds=_bounds,
                       options={'maxiter': CAL_POWELL_MAXITER, 'xtol': CAL_POWELL_XTOL,
                                'ftol': CAL_POWELL_FTOL, 'return_all': False})
    rmse_it = float(_res_it.fun)
    print(f"  Iteration 2 Powell: {_eval_n[0]} evals, RMSE={rmse_it:.2f} dB ({(_tcal.time()-_t_it)/60:.1f} min)")
    if rmse_it < rmse_best:
        print(f"  ✓ Improved: {rmse_best:.2f} → {rmse_it:.2f} dB — applying and saving")
        _loss(_res_it.x, CAL_SAMPLES_POWELL)  # apply best params to scene
        _pl_sf3 = _solve_pl(cal_rx, CAL_SAMPLES_FINAL)
        _v_sf3 = np.isfinite(_pl_sf3)
        scalar_factor_db = float(np.mean(_pl_sf3[_v_sf3] - cal_pl[_v_sf3]))
        rmse_best = rmse_it
        # Re-save
        for _mn_it, _p in _mats_list:
            _m_it = scene.radio_materials.get(_mn_it)
            if _m_it:
                _p['er']  = float(np.clip(_to_float(_m_it.relative_permittivity), _ER_MIN, _ER_MAX))
                _p['sig'] = float(np.clip(_to_float(_m_it.conductivity), _SIG_MIN, _SIG_MAX))
                _p['s']   = float(np.clip(_to_float(getattr(_m_it, 'scattering_coefficient', 0.2)), _S_MIN, _S_MAX))
        _calib_out['meta']['rmse_final_db'] = round(rmse_best, 3)
        for _mn_it, _p in _mats.items():
            _calib_out['materials'][_mn_it] = {'er': round(_p['er'], 4),
                                               'sigma': round(_p['sig'], 6),
                                               'scatter': round(_p['s'], 4)}
        with open(_CALIB_FILE_IT, 'w') as _fi2: _jcal.dump(_calib_out, _fi2, indent=2)
        _jcal.dump({'scalar_factor_db': scalar_factor_db,
                    'meta': {'rmse_final_db': round(rmse_best, 3)}}, open(_SF_FILE_IT, 'w'), indent=2)
        print(f"  Saved updated calibration files (RMSE={rmse_best:.2f} dB)")
    else:
        print(f"  No improvement ({rmse_it:.2f} ≥ {rmse_best:.2f} dB) — keeping iteration 1 result")

# ── Auto-retry if R² below threshold ─────────────────────────────────
_CAL_R2_RETRY = globals().get('CAL_R2_RETRY_MIN', 0.65)
if not np.isnan(_r2_at_1km) and _r2_at_1km < _CAL_R2_RETRY and '_x_warm' in dir():
    print(f"\n{chr(9472)*60}")
    print(f"  R\u00b2@1km={_r2_at_1km:.3f} < {_CAL_R2_RETRY} — auto-retry Powell from warm prior")
    _x_retry = _x_warm.copy()
    if not CAL_FIX_SCATTER:
        for _i_rt in range(len(_mats_list)): _x_retry[_pp*_i_rt+2] = float(_WARM_S_PRIOR)
    _eval_n[0] = 0; _t_retry = _tcal.time()
    _res_retry = minimize(lambda x: _loss(x, CAL_SAMPLES_POWELL), _x_retry,
                         method='Powell', bounds=_bounds,
                         options={'maxiter': CAL_POWELL_MAXITER,
                                  'xtol': CAL_POWELL_XTOL, 'ftol': CAL_POWELL_FTOL})
    _rmse_retry = float(_res_retry.fun)
    print(f"  Retry Powell: {_eval_n[0]} evals, RMSE={_rmse_retry:.2f} dB ({(_tcal.time()-_t_retry)/60:.1f} min)")
    _r2_before_retry = _r2_at_1km
    if _rmse_retry < rmse_best - 0.10:  # require >0.1 dB improvement to accept retry — prevents marginal RMSE gain at R² cost
        _loss(_res_retry.x, CAL_SAMPLES_POWELL)
        _pl_rt = _solve_pl(cal_rx, CAL_SAMPLES_FINAL)
        _v_rt = np.isfinite(_pl_rt)
        scalar_factor_db = float(np.mean(_pl_rt[_v_rt] - cal_pl[_v_rt])); rmse_best = _rmse_retry
        _pred_rt = _pl_rt - scalar_factor_db  # sign: consistent with _rmse(pl, sf) = pl - sf - cal_pl
        _fin_rt = np.isfinite(_pred_rt)
        for _dmax_rt in [0.75, 1.00]:
            _mk_rt = (_cal_dists_km <= _dmax_rt) & _fin_rt
            if _mk_rt.sum() < 5: continue
            _sse_rt = float(np.sum((_pred_rt[_mk_rt] - cal_pl[_mk_rt])**2))
            _sst_rt = float(np.sum((cal_pl[_mk_rt] - cal_pl[_mk_rt].mean())**2))
            _r2_rt = float(1.0 - _sse_rt/_sst_rt) if _sst_rt > 0 else float('nan')
            _rm_rt = float(np.sqrt(np.mean((_pred_rt[_mk_rt] - cal_pl[_mk_rt])**2)))
            if abs(_dmax_rt - 1.0) < 0.01: _r2_at_1km = _r2_rt
            print(f"  0-{_dmax_rt:.2f}km  N={int(_mk_rt.sum()):3d}  RMSE={_rm_rt:.2f} dB  R\u00b2={_r2_rt:+.3f}")
        for _mn_rt, _p in _mats_list:
            _m_rt = scene.radio_materials.get(_mn_rt)
            if _m_rt:
                _p['er']  = float(np.clip(_to_float(_m_rt.relative_permittivity), _ER_MIN, _ER_MAX))
                _p['sig'] = float(np.clip(_to_float(_m_rt.conductivity), _SIG_MIN, _SIG_MAX))
                _p['s']   = float(np.clip(_to_float(getattr(_m_rt, 'scattering_coefficient', 0.2)), _S_MIN, _S_MAX))
        _calib_out['meta']['rmse_final_db'] = round(rmse_best, 3)
        for _mn_rt, _p in _mats.items():
            _calib_out['materials'][_mn_rt] = {'er': round(_p['er'], 4), 'sigma': round(_p['sig'], 6), 'scatter': round(_p['s'], 4)}
        with open(_CALIB_FILE, 'w') as _f: _jcal.dump(_calib_out, _f, indent=2)
        _jcal.dump({'scalar_factor_db': scalar_factor_db, 'meta': {'rmse_final_db': round(rmse_best, 3)}}, open(_SF_FILE, 'w'), indent=2)
        print(f"  ✓ Retry improved RMSE={rmse_best:.2f} dB, R²@1km={_r2_at_1km:.3f} — saved")
    else:
        print(f"  Retry RMSE={_rmse_retry:.2f} ≥ {rmse_best:.2f} dB — keeping previous result")

print(f"\n{'█'*60}")
print(f"Calibration complete: final RMSE = {rmse_best:.2f} dB")
print("Run CELL 4A then CELL 8e to verify.")

# ── Material parameter trajectory plots ──────────────────────────────────
if (globals().get('_param_history') and len(_param_history) > 2
        and globals().get('_mats_list')):
    import matplotlib.pyplot as _plt_cp
    import numpy as _np_cp

    _ph_arr    = _np_cp.array(_param_history)          # (n_evals, 3*n_mats)
    _mat_names = [mn for mn, _ in _mats_list]
    _n_mats    = _ph_arr.shape[1] // 3  # use actual tracked count, not full mats list
    _mat_names = _mat_names[:_n_mats]
    _ev_idx    = _np_cp.arange(len(_param_history))
    _colors    = _plt_cp.cm.tab10(_np_cp.linspace(0, 0.9, min(_n_mats, 10)))

    _param_specs = [
        (r'$\varepsilon_r$ (relative permittivity)', 0, 'linear', (0.5, None)),
        (r'$\sigma$ (conductivity S/m)',              1, 'log',    None),
        (r'$S$ (scattering coefficient)',              2, 'linear', (0.0, 1.0)),
    ]

    _fig_cp, _axes_cp = _plt_cp.subplots(3, 1, figsize=(13, 10), sharex=True)
    _fig_cp.suptitle('CELL CAL — material parameter trajectory (Powell evaluations)',
                     fontsize=12, fontweight='bold')

    for _ax_cp, (_lbl, _pidx, _sc, _ylim) in zip(_axes_cp, _param_specs):
        for _j, (_mn, _col) in enumerate(zip(_mat_names, _colors)):
            _vals = _ph_arr[:, 3*_j + _pidx]
            _ax_cp.plot(_ev_idx, _vals, color=_col,
                        label=_mn.replace('itu_', ''), linewidth=1.5)
        _ax_cp.set_ylabel(_lbl, fontsize=10)
        _ax_cp.set_yscale(_sc)
        if _ylim:
            _lo, _hi = _ylim
            _cur = _ax_cp.get_ylim()
            _ax_cp.set_ylim(
                _lo if _lo is not None else _cur[0],
                _hi if _hi is not None else _cur[1])
        _ax_cp.grid(True, alpha=0.25, linestyle='--')
        _ax_cp.legend(loc='center left', bbox_to_anchor=(1.01, 0.5),
                      fontsize=7.5, ncol=1, framealpha=0.8)

    _axes_cp[-1].set_xlabel('Powell evaluation index', fontsize=10)
    _plt_cp.tight_layout()
    _out_cp = os.path.join(OUT_DIR, 'cal_param_trajectory.png')
    _plt_cp.savefig(_out_cp, dpi=130, bbox_inches='tight')
    _plt_cp.show()
    print(f"  Saved: {_out_cp}")
else:
    print("  [SKIP] parameter trajectory plot: no Powell history available")


## CELL CAL-DE — Differential Evolution Material Calibration (Sionna 2.0)

Global-search alternative to CELL CAL Powell.  Use when Powell converges to a
local minimum (RMSE plateaus across multiple restarts).

**Method:**
- `scipy.optimize.differential_evolution` — population-based global search
- Same loss function, same RX selection, same bounds as CELL CAL
- Phase 0: scalar offset (minimize_scalar)
- Phase 1: DE joint search over all (εr, log σ, S) parameters
- Phase 2: re-scalar with DE result
- Saves same files: `calibrated_materials_stevenage_2695mhz.json` + `scalar_offset_stevenage_2695mhz.json`

**Run after CELL CAL** if RMSE is not satisfactory, or as a first-pass global
search before a Powell refinement.

In [ ]:
# ====================================================================
# CELL CAL-DE — Differential Evolution Material Calibration
# ====================================================================
# Global optimizer alternative to Powell in CELL CAL.
# Shares the same loss function, bounds, RX set, and save files.
# Run this INSTEAD of (not after) CELL CAL to get a global optimum,
# then run CELL 4A + CELL 8e to apply and verify.
# ====================================================================
import numpy as np, json as _jde, os as _ode, time as _tde, gc as _gcde
from scipy.optimize import minimize_scalar, differential_evolution as _de

print("=" * 70)
print("CELL CAL-DE — Differential Evolution Material Calibration")
print("=" * 70)

# ── Config (reads from CELL 1 — no hardcoded values) ─────────────────────
_DE_MAX_DIST_KM    = globals().get('CAL_MAX_DIST_KM', 1.0)
_DE_MIN_DIST_KM    = globals().get('CAL_MIN_DIST_KM', 0.0)
_DE_SAMPLES_SF     = globals().get('CAL_SAMPLES_PS', 500_000)     # scalar phase
_DE_SAMPLES        = globals().get('CAL_DE_SAMPLES', globals().get('CAL_SAMPLES_PS', 2_000_000))  # DE evaluations — separate budget, capped at 500k by default
_DE_SAMPLES_FINAL  = globals().get('CAL_SAMPLES_PS', 500_000)     # re-scalar
_DE_POPSIZE        = globals().get('CAL_DE_POPSIZE',  4)          # population size multiplier
_DE_MAXITER        = globals().get('CAL_DE_MAXITER',  40)         # max generations
_DE_TOL            = globals().get('CAL_DE_TOL',   0.005)         # convergence tol
_DE_MUTATION       = globals().get('CAL_DE_MUTATION', (0.5, 1.0)) # F bounds
_DE_RECOMBINATION  = globals().get('CAL_DE_RECOMB',   0.7)        # CR
_DE_STRATEGY       = globals().get('CAL_DE_STRATEGY', 'best1bin') # DE strategy
_DE_SEED           = globals().get('CAL_DE_SEED', 42)             # reproducibility

# Physical EM bounds (same as CELL CAL)
_ER_MIN,  _ER_MAX  = 1.0,  80.0
_SIG_MIN, _SIG_MAX = 1e-6, 1e4
_SIG_MIN_PER_MAT = {
    'itu_brick':           0.030,
    'itu_concrete':        0.030,
    'itu_wet_ground':      0.010,
    'itu_very_dry_ground': 0.0003,
}
_SIG_MAX_PER_MAT = {
    'itu_brick':    0.20,
    'itu_concrete': 0.20,
}
_S_MIN,   _S_MAX   = 0.0,  globals().get('CAL_S_MAX', 0.95)
_S_MIN_PER_MAT = {
    'itu_concrete':          0.20,
    'itu_brick':             0.20,
    'itu_wood':              0.40,
    'itu_wet_ground':        0.05,
    'itu_medium_dry_ground': 0.05,
    'itu_very_dry_ground':   0.05,
    'itu_ceiling_board':     0.0,
    'itu_metal':             0.25,
    'itu_glass':             0.10,
}
_S_MAX_PER_MAT = {
    'itu_wet_ground':        0.40,
    'itu_medium_dry_ground': 0.40,
    'itu_very_dry_ground':   0.35,
    'itu_concrete':          0.70,
    'itu_brick':             0.70,
    'itu_metal':             0.75,
    'itu_glass':             0.60,
    'itu_wood':              0.65,
}
_S_MAX_PER_MAT.update(globals().get('S_MAX_OVERRIDE', {}))
for _mn in list(_S_MIN_PER_MAT.keys()):
    if _mn in _S_MAX_PER_MAT:
        _S_MIN_PER_MAT[_mn] = min(_S_MIN_PER_MAT[_mn], _S_MAX_PER_MAT[_mn])

# ── Helpers (identical to CELL CAL) ──────────────────────────────────────
def _de_to_float(v):
    for f in [lambda x: float(np.real(x[0])),
              lambda x: float(np.real(x.numpy())),
              lambda x: float(np.real(complex(x))),
              float]:
        try: return f(v)
        except: pass
    return float(v)

def _de_set_mat(mat, er=None, sigma=None, s=None):
    import drjit as _dr
    for prop, val in [('relative_permittivity', er),
                      ('conductivity', sigma),
                      ('scattering_coefficient', s)]:
        if val is None: continue
        for attempt in [lambda p=prop, v=val: setattr(mat, p, _dr.Float(float(v))),
                        lambda p=prop, v=val: setattr(mat, p, float(v))]:
            try: attempt(); break
            except: pass
    try:
        _dr.eval(); _dr.sync_thread()
        if hasattr(_dr, 'flush_kernel_cache'): _dr.flush_kernel_cache()
    except: pass

def _de_flush():
    _gcde.collect()
    try:
        import drjit as _dr; _dr.flush_malloc_cache(); _dr.sync_thread()
    except: pass

def _de_solve_pl(rx_list, n_samples):
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _r in rx_list: scene.add(_r)
    try:
        import drjit as _dr_fl
        if hasattr(_dr_fl, 'flush_kernel_cache'): _dr_fl.flush_kernel_cache()
        _dr_fl.eval(); _dr_fl.sync_thread()
    except: pass
    _cfg = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, refraction=True, diffraction_lit_region=True, diffuse_reflection=True,
                samples_per_src=int(n_samples))
    _p = PathSolver()(scene, **_cfg)
    _a = getattr(_p, 'a', None)
    try:
        _a_np = (np.array(_a[0]) + 1j*np.array(_a[1])) if isinstance(_a, tuple) else np.array(_a)
        _a_np = np.squeeze(_a_np)
        if _a_np.ndim == 1: _a_np = _a_np[np.newaxis, :]
        elif _a_np.ndim > 2: _a_np = _a_np.reshape(_a_np.shape[0], -1)
    except:
        try:
            _at, _ = _p.cir()
            _a_np = (np.array(_at[0]) + 1j*np.array(_at[1])) if isinstance(_at, tuple) else np.array(_at)
            _a_np = np.squeeze(_a_np)
            if _a_np.ndim == 1: _a_np = _a_np[np.newaxis, :]
            elif _a_np.ndim > 2: _a_np = _a_np.reshape(_a_np.shape[0], -1)
        except: _a_np = np.zeros((len(rx_list), 1), dtype=complex)
    del _p; _de_flush()
    _pl = np.full(len(rx_list), np.nan)
    for _i in range(min(len(rx_list), _a_np.shape[0])):
        _pwr = float(np.sum(np.abs(_a_np[_i])**2))
        if _pwr > 1e-30: _pl[_i] = -10*np.log10(_pwr)
    return _pl

# ── Build calibration RX set ───────────────────────────────────────────────
import pandas as _pd_de
_df_de   = _pd_de.read_csv(MEASUREMENT_CSV)
_name_de = [c for c in _df_de.columns if "name" in c.lower() or "id" in c.lower()][0]
_rssi_de = [c for c in _df_de.columns
            if "measurement" in c.lower()
            or ("rssi" in c.lower() and "dbm" in c.lower())
            or c.lower() == "local_measurement_dbm"][0]
_meas_de = {str(r[_name_de]): float(r[_rssi_de]) for _, r in _df_de.iterrows()}
_safe_de = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)
_tx_de   = np.array([_safe_de(list(scene.transmitters.values())[0].position[i]) for i in range(2)])

_cands_de = []
for _rx in receivers:
    if _rx.name not in _meas_de: continue
    _dkm = float(np.linalg.norm(
        np.array([_safe_de(_rx.position[0]), _safe_de(_rx.position[1])]) - _tx_de)) / 1000.0
    if _DE_MIN_DIST_KM <= _dkm <= _DE_MAX_DIST_KM:
        _cands_de.append((_rx, _dkm, _meas_de[_rx.name]))
_cands_de.sort(key=lambda t: t[1])
_cal_rx_de = [c[0] for c in _cands_de]
_cal_pl_de = TX_CONDUCTED_DBM - np.array([c[2] for c in _cands_de])
print(f"Calibration RX : {len(_cal_rx_de)}  ({_DE_MIN_DIST_KM}-{_DE_MAX_DIST_KM} km ceiling)")
print(f"PL_meas range  : {_cal_pl_de.min():.1f} - {_cal_pl_de.max():.1f} dB")

def _de_rmse(pl_sim, sf=0.0):
    _v = np.isfinite(pl_sim)
    if _v.sum() == 0: return 999.0
    return float(np.sqrt(np.mean((pl_sim[_v] - sf - _cal_pl_de[_v])**2)))

# ── Warm prior: ensure scatter paths exist before Phase 0 ─────────────────
_WARM_S_DE = globals().get('CAL_WARM_S_PRIOR', 0.35)
_n_mats_de = sum(1 for _m in scene.radio_materials if not _m.endswith('_train'))
_n_low_de  = sum(1 for _mn, _ml in scene.radio_materials.items()
                 if not _mn.endswith('_train') and
                 float(_de_to_float(getattr(_ml, 'scattering_coefficient', 0.0))) < 0.05)
if _n_low_de > _n_mats_de // 2:
    print(f"  Fresh start detected: applying S warm prior = {_WARM_S_DE}")
    for _mn, _ml in scene.radio_materials.items():
        if not _mn.endswith('_train'):
            try:
                if float(_de_to_float(getattr(_ml, 'scattering_coefficient', 0.0))) < 0.05:
                    _ml.scattering_coefficient = _WARM_S_DE
            except: pass

# ── Phase 0: scalar offset ────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Phase 0 - Scalar offset  ({_DE_SAMPLES_SF//1000}k samples)")
_t0_de = _tde.time()
_pl0_de = _de_solve_pl(_cal_rx_de, _DE_SAMPLES_SF)
_v0_de  = np.isfinite(_pl0_de)
print(f"  Solved {_tde.time()-_t0_de:.0f}s  |  valid paths: {_v0_de.sum()}/{len(_cal_rx_de)}")
if _v0_de.sum() == 0:
    raise RuntimeError("No valid paths - check scene geometry and TX position")

# Auto-discover effective range (same logic as CELL CAL)
_dists_de   = np.array([c[1] for c in _cands_de])
_MIN_VF_DE  = globals().get('CAL_MIN_VALID_FRAC', 0.65)
_eff_max_de = _DE_MIN_DIST_KM
for _blo in np.arange(_DE_MIN_DIST_KM, _DE_MAX_DIST_KM, 0.1):
    _mask = (_dists_de >= _blo) & (_dists_de < _blo + 0.1)
    if _mask.sum() == 0: continue
    if np.isfinite(_pl0_de[_mask]).mean() >= _MIN_VF_DE:
        _eff_max_de = round(_blo + 0.1, 2)
    else:
        break
_keep_de   = np.where(_dists_de <= _eff_max_de + 1e-9)[0]
_cands_de  = [_cands_de[i] for i in _keep_de]
_cal_rx_de = [c[0] for c in _cands_de]
_cal_pl_de = TX_CONDUCTED_DBM - np.array([c[2] for c in _cands_de])
_pl0_de    = _pl0_de[_keep_de]
_dists_de  = np.array([c[1] for c in _cands_de])
_N_base_de = int(np.isfinite(_pl0_de).sum())
print(f"  Auto-range: {_eff_max_de:.2f} km  ({len(_cal_rx_de)} RX, >={int(_MIN_VF_DE*100)}% valid per 100m bin)")

_rmse0_de = _de_rmse(_pl0_de)
print(f"  RMSE before calibration: {_rmse0_de:.2f} dB")
_v0_de = np.isfinite(_pl0_de)
_sf_de = float(np.mean(_pl0_de[_v0_de] - _cal_pl_de[_v0_de]))
_rmse_sf_de = float(_res_sf_de.fun)
print(f"  scalar_factor_db = {_sf_de:+.3f} dB")
print(f"  RMSE after scalar: {_rmse_sf_de:.2f} dB")

# ── Enumerate materials ────────────────────────────────────────────────────
_mats_de = {}
for _mn, _m in scene.radio_materials.items():
    if _mn.endswith('_train'): continue
    try:
        _e  = float(np.clip(_de_to_float(_m.relative_permittivity), _ER_MIN, _ER_MAX))
        _sg = float(np.clip(_de_to_float(_m.conductivity), _SIG_MIN, _SIG_MAX))
        _sc = float(np.clip(_de_to_float(getattr(_m, 'scattering_coefficient', 0.2)), _S_MIN, _S_MAX))
        if not all(np.isfinite([_e, _sg, _sc])): continue
        _mats_de[_mn] = {'er': _e, 'sig': _sg, 's': _sc, 'er0': _e, 'sig0': _sg, 's0': _sc}
    except: continue

_CAL_FIXED_DE = globals().get('CAL_FIXED_MATS', set())
_MAT_FIXED_DE = {
    'itu_ceiling_board': (globals().get('VEG_RELATIVE_PERMITTIVITY', 17.0),
                          globals().get('VEG_CONDUCTIVITY', 0.05),
                          globals().get('VEG_SCATTERING_COEFF', 0.50)),
}
for _fn in list(_CAL_FIXED_DE):
    if _fn in _mats_de and _fn in scene.radio_materials:
        _fv = _MAT_FIXED_DE.get(_fn)
        if _fv:
            _de_set_mat(scene.radio_materials[_fn], er=_fv[0], sigma=_fv[1], s=_fv[2])
        del _mats_de[_fn]

print(f"\n{'='*60}")
print(f"Materials found: {len(_mats_de)}  (fixed: {len(_CAL_FIXED_DE & set(scene.radio_materials))})")
for _mn, _p in _mats_de.items():
    print(f"  {_mn:<28} er={_p['er']:.3f}  sigma={_p['sig']:.5f}  S={_p['s']:.3f}")

_mats_list_de = list(_mats_de.items())
_n_params_de  = len(_mats_list_de) * 3
_bounds_de = []
for _mn, _p in _mats_list_de:
    _bounds_de += [(max(_ER_MIN, _p['er0']*0.7),  min(_ER_MAX,  _p['er0']*1.5)),
                   (np.log(max(_SIG_MIN_PER_MAT.get(_mn, _SIG_MIN), _p['sig0']*0.01)), np.log(min(_SIG_MAX_PER_MAT.get(_mn, _SIG_MAX), _p['sig0']*100))),
                   (_S_MIN_PER_MAT.get(_mn, _S_MIN), _S_MAX_PER_MAT.get(_mn, _S_MAX))]

# Distance weights (same as CELL CAL)
_CAL_FAR_W_DE = globals().get('CAL_FAR_WEIGHT', 1.0)
if _CAL_FAR_W_DE > 0:
    _cal_w_de = 1.0 + _CAL_FAR_W_DE * (np.sqrt(np.clip(_dists_de, 0.1, None)) - 1.0)
    _cal_w_de = np.clip(_cal_w_de, 0.5, None)
else:
    _cal_w_de = np.ones(len(_dists_de))
_cal_w_de /= _cal_w_de.mean()
_CAL_COV_MIN_DE = globals().get('CAL_COVERAGE_MIN', 0.90)
_CAL_COV_PEN_DE = 20.0

_eval_de  = [0]
_best_de  = [999.0]
_t_eval   = [_tde.time()]

def _de_loss(x):
    for _i, (_mn, _) in enumerate(_mats_list_de):
        _er  = float(np.clip(x[3*_i],   _ER_MIN,  _ER_MAX))
        _sg  = float(np.clip(np.exp(x[3*_i+1]), _SIG_MIN, _SIG_MAX))
        _sc  = float(np.clip(x[3*_i+2], _S_MIN,   _S_MAX))
        _de_set_mat(scene.radio_materials[_mn], er=_er, sigma=_sg, s=_sc)
    _pl = _de_solve_pl(_cal_rx_de, _DE_SAMPLES)
    _err = _pl - _cal_pl_de
    _fin = np.isfinite(_err)
    if _fin.sum() == 0:
        _rm = 999.0
    else:
        _rm = float(np.sqrt(np.average(_err[_fin]**2, weights=_cal_w_de[_fin])))
    _cov = int(np.isfinite(_pl).sum()) / max(_N_base_de, 1)
    if _cov < _CAL_COV_MIN_DE:
        _rm += (_CAL_COV_MIN_DE - _cov) * _CAL_COV_PEN_DE
    _eval_de[0] += 1
    if _rm < _best_de[0]: _best_de[0] = _rm
    _dt = _tde.time() - _t_eval[0]; _t_eval[0] = _tde.time()
    print(f"  eval {_eval_de[0]:4d}  RMSE={_rm:.3f} dB  best={_best_de[0]:.3f} dB  [{_dt:.0f}s]")
    return _rm

# ── Phase 1: Differential Evolution ──────────────────────────────────────
print(f"\n{'='*60}")
print(f"Phase 1 - Differential Evolution  ({_DE_SAMPLES//1000}k samples/eval)")
print(f"  Strategy: {_DE_STRATEGY}  popsize={_DE_POPSIZE}  maxiter={_DE_MAXITER}")
print(f"  mutation={_DE_MUTATION}  recombination={_DE_RECOMBINATION}  seed={_DE_SEED}")
print(f"  {_n_params_de} params ({len(_mats_list_de)} materials x 3)")
print(f"  Estimated max evaluations: {_DE_POPSIZE * _n_params_de * _DE_MAXITER}")
print()
_t1_de = _tde.time()

_res_de = _de(
    _de_loss,
    _bounds_de,
    strategy   = _DE_STRATEGY,
    maxiter    = _DE_MAXITER,
    popsize    = _DE_POPSIZE,
    tol        = _DE_TOL,
    mutation   = _DE_MUTATION,
    recombination = _DE_RECOMBINATION,
    seed       = _DE_SEED,
    polish     = False,   # skip L-BFGS-B polish — not needed for non-smooth loss
    init       = 'latinhypercube',
    disp       = False,
    workers    = 1,       # serial — DrJIT GPU is not thread-safe
)

print(f"\n  DE done in {(_tde.time()-_t1_de)/60:.1f} min  ({_eval_de[0]} evals)")
print(f"  Converged: {_res_de.success}  |  {_res_de.message}")
print(f"  Best RMSE (DE): {_res_de.fun:.3f} dB")

# Apply best DE params to scene
_x_de_best = _res_de.x
for _i, (_mn, _p) in enumerate(_mats_list_de):
    _p['er']  = float(np.clip(_x_de_best[3*_i],   _ER_MIN,  _ER_MAX))
    _p['sig'] = float(np.clip(np.exp(_x_de_best[3*_i+1]), _SIG_MIN, _SIG_MAX))
    _p['s']   = float(np.clip(_x_de_best[3*_i+2], _S_MIN,   _S_MAX))
    _de_set_mat(scene.radio_materials[_mn], er=_p['er'], sigma=_p['sig'], s=_p['s'])
    print(f"  {_mn:<28} er={_p['er']:.3f}  sigma={_p['sig']:.5f}  S={_p['s']:.3f}")

# ── Phase 2: re-scalar with DE materials ─────────────────────────────────
print(f"\n{'='*60}")
print(f"Phase 2 - Re-scalar with DE materials  ({_DE_SAMPLES_FINAL//1000}k samples)")
_pl_de_final = _de_solve_pl(_cal_rx_de, _DE_SAMPLES_FINAL)
_vf_de = np.isfinite(_pl_de_final)
_v2_de = np.isfinite(_pl_de_final)
_sf_de_final = float(np.mean(_pl_de_final[_v2_de] - _cal_pl_de[_v2_de]))
_rmse_de_final = float(_res_sf2_de.fun)
if _rmse_de_final > _res_de.fun - 0.05:
    print(f"  Re-scalar worsened RMSE ({_rmse_de_final:.2f} > DE {_res_de.fun:.2f}) - reverting to Phase 0 scalar")
    _sf_de_final   = _sf_de
    _rmse_de_final = _res_de.fun
print(f"  scalar_factor_db (final) = {_sf_de_final:+.3f} dB")
print(f"  RMSE final: {_rmse_de_final:.2f} dB")

# Quick R^2 preview
_pred_de = _pl_de_final + _sf_de_final
_fin_de  = np.isfinite(_pred_de)
print(f"\n{'-'*60}")
print("Quick R^2 preview (calibration receivers):")
for _dmax_de in sorted(set([0.50, 0.75, 1.00, round(_eff_max_de, 2)])):
    _mk = (_dists_de <= _dmax_de) & _fin_de
    if _mk.sum() < 5: continue
    _sse = float(np.sum((_pred_de[_mk] - _cal_pl_de[_mk])**2))
    _sst = float(np.sum((_cal_pl_de[_mk] - _cal_pl_de[_mk].mean())**2))
    _r2  = float(1.0 - _sse/_sst) if _sst > 0 else float('nan')
    _rm  = float(np.sqrt(np.mean((_pred_de[_mk] - _cal_pl_de[_mk])**2)))
    print(f"  0-{_dmax_de:.2f}km  N={int(_mk.sum()):3d}  RMSE={_rm:.2f} dB  R^2={_r2:+.3f}")

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"{'Step':<35} {'RMSE (dB)':>10}")
print(f"{'-'*47}")
print(f"{'Before calibration':<35} {_rmse0_de:>10.2f}")
print(f"{'After scalar offset':<35} {_rmse_sf_de:>10.2f}")
print(f"{'After DE + re-scalar':<35} {_rmse_de_final:>10.2f}")
print(f"{'Total improvement':<35} {_rmse0_de-_rmse_de_final:>+10.2f}")

# ── Save (same files as CELL CAL) ─────────────────────────────────────────
_calib_de_out = {
    'meta': {
        'source': 'CELL CAL-DE (Differential Evolution)',
        'frequency_mhz': float(FREQUENCY_HZ / 1e6),
        'n_calib_rx': len(_cal_rx_de),
        'total_evals': _eval_de[0],
        'rmse_before_db':  round(_rmse0_de, 3),
        'rmse_scalar_db':  round(_rmse_sf_de, 3),
        'rmse_final_db':   round(_rmse_de_final, 3),
        'de_strategy': _DE_STRATEGY,
        'de_popsize':  _DE_POPSIZE,
        'de_maxiter':  _DE_MAXITER,
    },
    'materials': {}
}
for _mn, _p in _mats_de.items():
    _calib_de_out['materials'][_mn] = {
        'er':      round(_p['er'],  4),
        'sigma':   round(_p['sig'], 6),
        'scatter': round(_p['s'],   4),
    }
for _mn, _m in scene.radio_materials.items():
    if _mn.endswith('_train') or _mn in _calib_de_out['materials']: continue
    try:
        _calib_de_out['materials'][_mn] = {
            'er':      round(_de_to_float(_m.relative_permittivity), 4),
            'sigma':   round(_de_to_float(_m.conductivity), 6),
            'scatter': round(_de_to_float(getattr(_m, 'scattering_coefficient', 0.2)), 4),
        }
    except: pass

_CALIB_FILE_DE = _ode.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
with open(_CALIB_FILE_DE, 'w') as _f: _jde.dump(_calib_de_out, _f, indent=2)

_SF_FILE_DE = _ode.path.join(BASE_DIR, 'scalar_offset_stevenage_2695mhz.json')
with open(_SF_FILE_DE, 'w') as _f:
    _jde.dump({'scalar_factor_db': _sf_de_final,
               'meta': {'rmse_before_db': round(_rmse0_de, 3),
                        'rmse_final_db':  round(_rmse_de_final, 3)}}, _f, indent=2)

print(f"\nSaved: {_CALIB_FILE_DE}")
print(f"Saved: {_SF_FILE_DE}")
print("Run CELL 4A to apply calibrated materials, then CELL 7/8e to verify.")
print(f"\n{'#'*60}")
print(f"CELL CAL-DE complete: final RMSE = {_rmse_de_final:.2f} dB")
print("Run CELL 4A then CELL 8e to verify.")


## CELL CAL-CMA — CMA-ES Material Calibration (Sionna 2.0)

CMA-ES (Covariance Matrix Adaptation Evolution Strategy) — adaptive global optimizer.
Faster than DE (~2-3h vs 5.5h) and better at correlated parameter spaces.
Shares the same loss function, bounds, RX set, and save files as CELL CAL-DE.

**Recommended config in CELL 1 before running:**
```python
CAL_MAX_DIST_KM  = 2.0   # extend to include long-range receivers
CAL_CMA_SAMPLES  = 2_000_000
CAL_CMA_MAXITER  = 300
CAL_CMA_SIGMA0   = 0.3
```

Run this INSTEAD of CELL CAL / CELL CAL-DE, then CELL 4A + CELL 8e.

In [ ]:
# ====================================================================
# CELL CAL-CMA — CMA-ES Material Calibration
# ====================================================================
# CMA-ES adaptive global optimizer. Faster than DE (~2-3h) and better
# at correlated EM parameter spaces (er <-> sigma <-> S coupling).
# Shares loss function, bounds, RX set and save files with CELL CAL-DE.
# Recommended: set CAL_MAX_DIST_KM=2.0 in CELL 1 for long-range balance.
# ====================================================================
import numpy as np, json as _jcma, os as _ocma, time as _tcma, gc as _gccma
from scipy.optimize import minimize_scalar

try:
    import cma as _cma_lib
except ImportError:
    raise ImportError("Run: pip install cma  then restart kernel and rerun from CELL 1")

print("=" * 70)
print("CELL CAL-CMA — CMA-ES Material Calibration")
print("=" * 70)

# ── Config ──────────────────────────────────────────────────────────────────
_CMA_MAX_DIST_KM    = globals().get('CAL_MAX_DIST_KM', 1.5)
_CMA_MIN_DIST_KM    = globals().get('CAL_MIN_DIST_KM', 0.0)
_CMA_SAMPLES_SF     = globals().get('CAL_SAMPLES_PS', 500_000)
_CMA_SAMPLES        = globals().get('CAL_CMA_SAMPLES', globals().get('CAL_DE_SAMPLES', 2_000_000))
_CMA_SAMPLES_FINAL  = globals().get('CAL_SAMPLES_PS', 500_000)
_CMA_SIGMA0         = globals().get('CAL_CMA_SIGMA0',   0.3)
_CMA_MAXITER        = globals().get('CAL_CMA_MAXITER',  300)
_CMA_POPSIZE        = globals().get('CAL_CMA_POPSIZE',  None)   # None = auto (4+3*ln(n))
_CMA_SEED           = globals().get('CAL_CMA_SEED',     42)
_CMA_TOLX           = globals().get('CAL_CMA_TOLX',     1e-4)
_CMA_TOLFUN         = globals().get('CAL_CMA_TOLFUN',   1e-3)
_CMA_WARM_START     = globals().get('CAL_CMA_WARM_START', True)  # start from existing cal

# Physical EM bounds (same as CELL CAL / CELL CAL-DE)
_ER_MIN_C,  _ER_MAX_C  = 1.0,  80.0
_SIG_MIN_C, _SIG_MAX_C = 1e-6, 1e4
_SIG_MIN_PER_MAT_C = {
    'itu_brick':           0.030,
    'itu_concrete':        0.030,
    'itu_wet_ground':      0.010,
    'itu_very_dry_ground': 0.0003,
}
_SIG_MAX_PER_MAT_C = {
    'itu_brick':    0.20,
    'itu_concrete': 0.20,
}
_S_MIN_C, _S_MAX_C = 0.0, globals().get('CAL_S_MAX', 0.95)
_S_MIN_PER_MAT_C = {
    'itu_concrete': 0.0, 'itu_brick': 0.0, 'itu_wood': 0.40,
    'itu_wet_ground': 0.05, 'itu_medium_dry_ground': 0.05,
    'itu_very_dry_ground': 0.05, 'itu_ceiling_board': 0.0,
    'itu_metal': 0.25, 'itu_glass': 0.0,
}
_S_MAX_PER_MAT_C = {
    'itu_wet_ground': 0.40, 'itu_medium_dry_ground': 0.40,
    'itu_very_dry_ground': 0.35,
    'itu_concrete': 0.40,   # Stevenage: no scatter flood — cap same as other sites
    'itu_brick':    0.40,   # uncalibrated R2=0.744 achieved with S=0.25-0.30
    'itu_glass':    0.35,   # raised from 0.10 — S_MAX=0.10 trapped CMA in bad region
    'itu_metal': 0.75, 'itu_wood': 0.65,
}
_S_MAX_PER_MAT_C.update(globals().get('S_MAX_OVERRIDE', {}))
for _mn_c in list(_S_MIN_PER_MAT_C.keys()):
    if _mn_c in _S_MAX_PER_MAT_C:
        _S_MIN_PER_MAT_C[_mn_c] = min(_S_MIN_PER_MAT_C[_mn_c], _S_MAX_PER_MAT_C[_mn_c])

# ── Helpers (same as CELL CAL-DE) ───────────────────────────────────────────
def _cma_to_float(v):
    for f in [lambda x: float(np.real(x[0])),
              lambda x: float(np.real(x.numpy())),
              lambda x: float(np.real(complex(x))), float]:
        try: return f(v)
        except: pass
    return float(v)

def _cma_set_mat(mat, er=None, sigma=None, s=None):
    import drjit as _dr
    for prop, val in [('relative_permittivity', er), ('conductivity', sigma),
                      ('scattering_coefficient', s)]:
        if val is None: continue
        for attempt in [lambda p=prop, v=val: setattr(mat, p, _dr.Float(float(v))),
                        lambda p=prop, v=val: setattr(mat, p, float(v))]:
            try: attempt(); break
            except: pass
    try:
        _dr.eval(); _dr.sync_thread()
        if hasattr(_dr, 'flush_kernel_cache'): _dr.flush_kernel_cache()
    except: pass

def _cma_flush():
    _gccma.collect()
    try:
        import drjit as _dr; _dr.flush_malloc_cache(); _dr.sync_thread()
    except: pass

def _cma_solve_pl(rx_list, n_samples):
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _r in rx_list: scene.add(_r)
    try:
        import drjit as _dr_fl
        if hasattr(_dr_fl, 'flush_kernel_cache'): _dr_fl.flush_kernel_cache()
        _dr_fl.eval(); _dr_fl.sync_thread()
    except: pass
    _cfg = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True, refraction=True,
                diffraction_lit_region=True, diffuse_reflection=True,
                samples_per_src=int(n_samples))
    _p = PathSolver()(scene, **_cfg)
    _a = getattr(_p, 'a', None)
    try:
        _a_np = (np.array(_a[0]) + 1j*np.array(_a[1])) if isinstance(_a, tuple) else np.array(_a)
        _a_np = np.squeeze(_a_np)
        if _a_np.ndim == 1: _a_np = _a_np[np.newaxis, :]
        elif _a_np.ndim > 2: _a_np = _a_np.reshape(_a_np.shape[0], -1)
    except:
        try:
            _at, _ = _p.cir()
            _a_np = (np.array(_at[0]) + 1j*np.array(_at[1])) if isinstance(_at, tuple) else np.array(_at)
            _a_np = np.squeeze(_a_np)
            if _a_np.ndim == 1: _a_np = _a_np[np.newaxis, :]
            elif _a_np.ndim > 2: _a_np = _a_np.reshape(_a_np.shape[0], -1)
        except: _a_np = np.zeros((len(rx_list), 1), dtype=complex)
    del _p; _cma_flush()
    _pl = np.full(len(rx_list), np.nan)
    for _i in range(min(len(rx_list), _a_np.shape[0])):
        _pwr = float(np.sum(np.abs(_a_np[_i])**2))
        if _pwr > 1e-30: _pl[_i] = -10*np.log10(_pwr)
    return _pl

# ── Build calibration RX set ────────────────────────────────────────────────
import pandas as _pd_cma
_df_cma   = _pd_cma.read_csv(MEASUREMENT_CSV)
_name_cma = [c for c in _df_cma.columns if "name" in c.lower() or "id" in c.lower()][0]
_rssi_cma = [c for c in _df_cma.columns
             if "measurement" in c.lower()
             or ("rssi" in c.lower() and "dbm" in c.lower())
             or c.lower() == "local_measurement_dbm"][0]
_meas_cma = {str(r[_name_cma]): float(r[_rssi_cma]) for _, r in _df_cma.iterrows()}
_safe_cma = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)
_tx_cma   = np.array([_safe_cma(list(scene.transmitters.values())[0].position[i]) for i in range(2)])

_cands_cma = []
for _rx in receivers:
    if _rx.name not in _meas_cma: continue
    _dkm = float(np.linalg.norm(
        np.array([_safe_cma(_rx.position[0]), _safe_cma(_rx.position[1])]) - _tx_cma)) / 1000.0
    if _CMA_MIN_DIST_KM <= _dkm <= _CMA_MAX_DIST_KM:
        _cands_cma.append((_rx, _dkm, _meas_cma[_rx.name]))
_cands_cma.sort(key=lambda t: t[1])
_cal_rx_cma = [c[0] for c in _cands_cma]
_cal_pl_cma = TX_CONDUCTED_DBM - np.array([c[2] for c in _cands_cma])
print(f"Calibration RX : {len(_cal_rx_cma)}  ({_CMA_MIN_DIST_KM}-{_CMA_MAX_DIST_KM} km ceiling)")
print(f"PL_meas range  : {_cal_pl_cma.min():.1f} - {_cal_pl_cma.max():.1f} dB")

def _cma_rmse(pl_sim, sf=0.0):
    _v = np.isfinite(pl_sim)
    if _v.sum() == 0: return 999.0
    return float(np.sqrt(np.mean((pl_sim[_v] - sf - _cal_pl_cma[_v])**2)))

# ── Warm prior ───────────────────────────────────────────────────────────────
_WARM_S_CMA = globals().get('CAL_WARM_S_PRIOR', 0.35)
_n_mats_cma = sum(1 for _m in scene.radio_materials if not _m.endswith('_train'))
_n_low_cma  = sum(1 for _mn, _ml in scene.radio_materials.items()
                  if not _mn.endswith('_train') and
                  float(_cma_to_float(getattr(_ml, 'scattering_coefficient', 0.0))) < 0.05)
if _n_low_cma > _n_mats_cma // 2:
    print(f"  Fresh start: applying S warm prior = {_WARM_S_CMA}")
    for _mn, _ml in scene.radio_materials.items():
        if not _mn.endswith('_train'):
            try:
                if float(_cma_to_float(getattr(_ml, 'scattering_coefficient', 0.0))) < 0.05:
                    _ml.scattering_coefficient = _WARM_S_CMA
            except: pass

# ── Phase 0: scalar offset ────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Phase 0 - Scalar offset  ({_CMA_SAMPLES_SF//1000}k samples)")
_t0_cma = _tcma.time()
_pl0_cma = _cma_solve_pl(_cal_rx_cma, _CMA_SAMPLES_SF)
_v0_cma  = np.isfinite(_pl0_cma)
print(f"  Solved {_tcma.time()-_t0_cma:.0f}s  |  valid paths: {_v0_cma.sum()}/{len(_cal_rx_cma)}")
if _v0_cma.sum() == 0:
    raise RuntimeError("No valid paths - check scene geometry and TX position")

_dists_cma  = np.array([c[1] for c in _cands_cma])
_MIN_VF_CMA = globals().get('CAL_MIN_VALID_FRAC', 0.65)
_eff_max_cma = _CMA_MIN_DIST_KM
for _blo in np.arange(_CMA_MIN_DIST_KM, _CMA_MAX_DIST_KM, 0.1):
    _mask = (_dists_cma >= _blo) & (_dists_cma < _blo + 0.1)
    if _mask.sum() == 0: continue
    if np.isfinite(_pl0_cma[_mask]).mean() >= _MIN_VF_CMA:
        _eff_max_cma = round(_blo + 0.1, 2)
    else: break
_keep_cma  = np.where(_dists_cma <= _eff_max_cma + 1e-9)[0]
_cands_cma = [_cands_cma[i] for i in _keep_cma]
_cal_rx_cma = [c[0] for c in _cands_cma]
_cal_pl_cma = TX_CONDUCTED_DBM - np.array([c[2] for c in _cands_cma])
_pl0_cma   = _pl0_cma[_keep_cma]
_dists_cma = np.array([c[1] for c in _cands_cma])
_N_base_cma = int(np.isfinite(_pl0_cma).sum())
print(f"  Auto-range: {_eff_max_cma:.2f} km  ({len(_cal_rx_cma)} RX)")

_rmse0_cma = _cma_rmse(_pl0_cma)
print(f"  RMSE before calibration: {_rmse0_cma:.2f} dB")
_v0_cma = np.isfinite(_pl0_cma)
_sf_cma = float(np.mean(_pl0_cma[_v0_cma] - _cal_pl_cma[_v0_cma]))
_rmse_sf_cma = _cma_rmse(_pl0_cma, _sf_cma)
print(f"  scalar_factor_db = {_sf_cma:+.3f} dB")
print(f"  RMSE after scalar: {_rmse_sf_cma:.2f} dB")

# ── Enumerate materials ──────────────────────────────────────────────────────
_mats_cma = {}
for _mn, _m in scene.radio_materials.items():
    if _mn.endswith('_train'): continue
    try:
        _e  = float(np.clip(_cma_to_float(_m.relative_permittivity), _ER_MIN_C, _ER_MAX_C))
        _sg = float(np.clip(_cma_to_float(_m.conductivity), _SIG_MIN_C, _SIG_MAX_C))
        _sc = float(np.clip(_cma_to_float(getattr(_m, 'scattering_coefficient', 0.2)), _S_MIN_C, _S_MAX_C))
        if not all(np.isfinite([_e, _sg, _sc])): continue
        _mats_cma[_mn] = {'er': _e, 'sig': _sg, 's': _sc, 'er0': _e, 'sig0': _sg, 's0': _sc}
    except: continue

_CAL_FIXED_CMA = globals().get('CAL_FIXED_MATS', set())
_MAT_FIXED_CMA = {
    'itu_ceiling_board': (globals().get('VEG_RELATIVE_PERMITTIVITY', 17.0),
                          globals().get('VEG_CONDUCTIVITY', 0.05),
                          globals().get('VEG_SCATTERING_COEFF', 0.50)),
}
for _fn in list(_CAL_FIXED_CMA):
    if _fn in _mats_cma and _fn in scene.radio_materials:
        _fv = _MAT_FIXED_CMA.get(_fn)
        if _fv: _cma_set_mat(scene.radio_materials[_fn], er=_fv[0], sigma=_fv[1], s=_fv[2])
        del _mats_cma[_fn]

# Warm start: load existing calibration if available
_CALIB_FILE_CMA = _ocma.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
if _CMA_WARM_START and _ocma.path.exists(_CALIB_FILE_CMA):
    try:
        _prev = _jcma.load(open(_CALIB_FILE_CMA))['materials']
        for _mn in list(_mats_cma.keys()):
            if _mn in _prev:
                _mats_cma[_mn]['er']  = float(_prev[_mn].get('er',  _mats_cma[_mn]['er']))
                _mats_cma[_mn]['sig'] = float(_prev[_mn].get('sigma', _mats_cma[_mn]['sig']))
                _mats_cma[_mn]['s']   = float(_prev[_mn].get('scatter', _mats_cma[_mn]['s']))
        print(f"  Warm start: loaded {len(_prev)} materials from existing calibration")
    except Exception as _we: print(f"  Warm start failed ({_we}) — using scene defaults")

print(f"\n{'='*60}")
print(f"Materials found: {len(_mats_cma)}  (fixed: {len(_CAL_FIXED_CMA & set(scene.radio_materials))})")
for _mn, _p in _mats_cma.items():
    print(f"  {_mn:<28} er={_p['er']:.3f}  sigma={_p['sig']:.5f}  S={_p['s']:.3f}")

_mats_list_cma = list(_mats_cma.items())
_n_params_cma  = len(_mats_list_cma) * 3

# ── Phase 0 checkpoint: save Phase-0 materials to JSON so CELL 4A can run even
#    if CMA is interrupted (Stevenage: Phase-0 is already near-optimal) ──────
_calib_file_ph0 = _ocma.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
_scalar_file_ph0 = _ocma.path.join(BASE_DIR, 'scalar_offset_stevenage_2695mhz.json')
_ph0_mat_dict = {_mn: {'er': _p['er'], 'sigma': _p['sig'], 'scatter': _p['s']}
                 for _mn, _p in _mats_cma.items()}
_ocma.makedirs(_ocma.path.dirname(_calib_file_ph0), exist_ok=True)
with open(_calib_file_ph0, 'w') as _jph0:
    _jcma.dump({'materials': _ph0_mat_dict, 'phase': 'phase0_checkpoint',
                'rmse_db': float(_rmse_sf_cma)}, _jph0)
with open(_scalar_file_ph0, 'w') as _jph0s:
    _jcma.dump({'scalar_factor_db': _sf_cma, 'phase': 'phase0_checkpoint'}, _jph0s)
print(f'  Phase-0 checkpoint saved -> {_ocma.path.basename(_calib_file_ph0)}')
print(f'  (CELL 4A can run now if CMA is interrupted)')

# Build x0 and bounds
_x0_cma, _lbs_cma, _ubs_cma = [], [], []
for _mn, _p in _mats_list_cma:
    _x0_cma += [_p['er'],
                np.log(max(_SIG_MIN_PER_MAT_C.get(_mn, _SIG_MIN_C), min(_p['sig'], _SIG_MAX_PER_MAT_C.get(_mn, _SIG_MAX_C)))),
                _p['s']]
    _lbs_cma += [max(_ER_MIN_C,  _p['er0']*0.7),
                 np.log(max(_SIG_MIN_PER_MAT_C.get(_mn, _SIG_MIN_C), _p['sig0']*0.01)),
                 _S_MIN_PER_MAT_C.get(_mn, _S_MIN_C)]
    _ubs_cma += [min(_ER_MAX_C,  _p['er0']*1.5),
                 np.log(min(_SIG_MAX_PER_MAT_C.get(_mn, _SIG_MAX_C), _p['sig0']*100)),
                 _S_MAX_PER_MAT_C.get(_mn, _S_MAX_C)]
_x0_cma  = np.clip(_x0_cma,  _lbs_cma, _ubs_cma)

# Distance weights
_CAL_FAR_W_CMA = globals().get('CAL_FAR_WEIGHT', 1.0)
if _CAL_FAR_W_CMA > 0:
    _cal_w_cma = 1.0 + _CAL_FAR_W_CMA * (np.sqrt(np.clip(_dists_cma, 0.1, None)) - 1.0)
    _cal_w_cma = np.clip(_cal_w_cma, 0.5, None)
else:
    _cal_w_cma = np.ones(len(_dists_cma))
_cal_w_cma /= _cal_w_cma.mean()
_CAL_COV_MIN_CMA = globals().get('CAL_COVERAGE_MIN', 0.90)
_CAL_COV_PEN_CMA = 20.0

_eval_cma_n = [0]
_best_cma_v = [999.0]
_t_eval_cma = [_tcma.time()]

def _cma_loss(x):
    for _i, (_mn, _) in enumerate(_mats_list_cma):
        _er  = float(np.clip(x[3*_i],   _ER_MIN_C,  _ER_MAX_C))
        _sg  = float(np.clip(np.exp(x[3*_i+1]), _SIG_MIN_C, _SIG_MAX_C))
        _sc  = float(np.clip(x[3*_i+2], _S_MIN_C,   _S_MAX_C))
        _cma_set_mat(scene.radio_materials[_mn], er=_er, sigma=_sg, s=_sc)
    _pl = _cma_solve_pl(_cal_rx_cma, _CMA_SAMPLES)
    _fin = np.isfinite(_pl - _cal_pl_cma)
    if _fin.sum() == 0: _rm = 999.0
    else:
        # Re-solve scalar per eval — marginalises out mean bias so CMA optimises
        # residual scatter (starts at ~14.81 dB, not 19 dB from raw offset)
        _sf_e = float(np.mean(_pl[_fin] - _cal_pl_cma[_fin]))
        _err  = (_pl - _sf_e) - _cal_pl_cma
        _fin2 = np.isfinite(_err)
        _rm   = float(np.sqrt(np.average(_err[_fin2]**2, weights=_cal_w_cma[_fin2])))
    _cov = int(np.isfinite(_pl).sum()) / max(_N_base_cma, 1)
    if _cov < _CAL_COV_MIN_CMA:
        _rm += (_CAL_COV_MIN_CMA - _cov) * _CAL_COV_PEN_CMA
    _eval_cma_n[0] += 1
    if _rm < _best_cma_v[0]: _best_cma_v[0] = _rm
    _dt = _tcma.time() - _t_eval_cma[0]; _t_eval_cma[0] = _tcma.time()
    print(f"  eval {_eval_cma_n[0]:4d}  RMSE={_rm:.3f} dB  best={_best_cma_v[0]:.3f} dB  [{_dt:.0f}s]")
    return _rm

# ── Phase 1: CMA-ES ──────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Phase 1 - CMA-ES  ({_CMA_SAMPLES//1000}k samples/eval)")
print(f"  sigma0={_CMA_SIGMA0}  maxiter={_CMA_MAXITER}  seed={_CMA_SEED}")
print(f"  {_n_params_cma} params ({len(_mats_list_cma)} materials x 3)")
print(f"  warm_start={_CMA_WARM_START}")
print()

_cma_opts = {
    'bounds':   [list(_lbs_cma), list(_ubs_cma)],
    'maxiter':  _CMA_MAXITER,
    'tolx':     _CMA_TOLX,
    'tolfun':   _CMA_TOLFUN,
    'verbose':  -9,
    'seed':     _CMA_SEED,
}
if _CMA_POPSIZE is not None:
    _cma_opts['popsize'] = _CMA_POPSIZE

_t1_cma = _tcma.time()
_es_cma = _cma_lib.CMAEvolutionStrategy(list(_x0_cma), _CMA_SIGMA0, _cma_opts)

while not _es_cma.stop():
    _sols_cma = _es_cma.ask()
    _fits_cma = [_cma_loss(_s) for _s in _sols_cma]
    _es_cma.tell(_sols_cma, _fits_cma)
    print(f"  --- gen {_es_cma.result.iterations:3d}  best={_best_cma_v[0]:.3f} dB  sigma={_es_cma.sigma:.5f} ---")

_cma_res = _es_cma.result
_x_cma_best = np.array(_cma_res.xbest)
_rmse_cma_best = float(_cma_res.fbest)

print(f"\n  CMA-ES done in {(_tcma.time()-_t1_cma)/60:.1f} min  ({_eval_cma_n[0]} evals)")
print(f"  Converged: {_es_cma.stop()}  |  Best RMSE: {_rmse_cma_best:.3f} dB")

# Apply best params
for _i, (_mn, _p) in enumerate(_mats_list_cma):
    _p['er']  = float(np.clip(_x_cma_best[3*_i],   _ER_MIN_C,  _ER_MAX_C))
    _p['sig'] = float(np.clip(np.exp(_x_cma_best[3*_i+1]), _SIG_MIN_C, _SIG_MAX_C))
    _p['s']   = float(np.clip(_x_cma_best[3*_i+2], _S_MIN_C,   _S_MAX_C))
    _cma_set_mat(scene.radio_materials[_mn], er=_p['er'], sigma=_p['sig'], s=_p['s'])
    print(f"  {_mn:<28} er={_p['er']:.3f}  sigma={_p['sig']:.5f}  S={_p['s']:.3f}")


# ── Convergence plots (generated even on interrupt) ─────────────────────────
try:
    def _cma_smooth(v, target_pts=60):
        if len(v) < 4:
            return np.arange(len(v)) + 1, np.array(v)
        w = max(1, len(v) // target_pts)
        xs = np.arange(len(v)) + 1
        ys = np.convolve(v, np.ones(w)/w, mode='valid')
        xs_s = xs[w//2: w//2 + len(ys)]
        return xs_s, ys

    if '_cma_rmse_hist' in dir() and len(_cma_rmse_hist) > 1:
        import matplotlib.pyplot as _plt_cma
        _xs_raw = np.arange(len(_cma_rmse_hist)) + 1
        _xs_sm, _ys_sm = _cma_smooth(_cma_rmse_hist)
        fig1, ax1 = _plt_cma.subplots(figsize=(10, 4))
        ax1.plot(_xs_raw, _cma_rmse_hist, alpha=0.25, color='steelblue', lw=0.8, label='per-eval')
        ax1.plot(_xs_sm,  _ys_sm,         color='steelblue', lw=2.0,     label='smoothed')
        _bests = np.minimum.accumulate(_cma_rmse_hist)
        ax1.plot(_xs_raw, _bests, color='darkblue', lw=1.5, ls='--', label=f'best  {min(_cma_rmse_hist):.3f} dB')
        ax1.axhline(7.82, color='red', ls=':', lw=1, label='σ_SF=7.82 dB (3GPP floor)')
        ax1.set_xlabel('Eval'); ax1.set_ylabel('Cal RMSE (dB)')
        ax1.set_title('CMA-ES Convergence — Stevenage 2695 MHz'); ax1.legend(); ax1.grid(alpha=0.3)
        _plt_cma.tight_layout()
        _png1 = _ocma.path.join(BASE_DIR, 'cma_convergence.png')
        fig1.savefig(_png1, dpi=120, bbox_inches='tight')
        _plt_cma.show()
        print(f'  Plot saved: {_png1}')
except Exception as _pe:
    print(f'  [convergence plot skipped: {_pe}]')

# ── Phase 2: re-scalar ────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Phase 2 - Re-scalar with CMA materials  ({_CMA_SAMPLES_FINAL//1000}k samples)")
_pl_cma_final = _cma_solve_pl(_cal_rx_cma, _CMA_SAMPLES_FINAL)
_v2_cma = np.isfinite(_pl_cma_final)
_sf_cma_final = float(np.mean(_pl_cma_final[_v2_cma] - _cal_pl_cma[_v2_cma]))
_rmse_cma_final = _cma_rmse(_pl_cma_final, _sf_cma_final)
if _rmse_cma_final > _rmse_cma_best - 0.05:
    print(f"  Re-scalar worsened RMSE ({_rmse_cma_final:.2f} > CMA {_rmse_cma_best:.2f}) - reverting to Phase 0 scalar")
    _sf_cma_final   = _sf_cma
    _rmse_cma_final = _rmse_cma_best
print(f"  scalar_factor_db (final) = {_sf_cma_final:+.3f} dB")
print(f"  RMSE final: {_rmse_cma_final:.2f} dB")

# Quick R^2 preview
_pred_cma = _pl_cma_final + _sf_cma_final
_fin_cma  = np.isfinite(_pred_cma)
print(f"\n{'-'*60}")
print("Quick R^2 preview (calibration receivers):")
for _dmax_cma in sorted(set([0.50, 0.75, 1.00, round(_eff_max_cma, 2)])):
    _mk = (_dists_cma <= _dmax_cma) & _fin_cma
    if _mk.sum() < 5: continue
    _sse = float(np.sum((_pred_cma[_mk] - _cal_pl_cma[_mk])**2))
    _sst = float(np.sum((_cal_pl_cma[_mk] - _cal_pl_cma[_mk].mean())**2))
    _r2  = float(1.0 - _sse/_sst) if _sst > 0 else float('nan')
    _rm  = float(np.sqrt(np.mean((_pred_cma[_mk] - _cal_pl_cma[_mk])**2)))
    print(f"  0-{_dmax_cma:.2f}km  N={int(_mk.sum()):3d}  RMSE={_rm:.2f} dB  R^2={_r2:+.3f}")

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"{'Step':<35} {'RMSE (dB)':>10}")
print(f"{'-'*47}")
print(f"{'Before calibration':<35} {_rmse0_cma:>10.2f}")
print(f"{'After scalar offset':<35} {_rmse_sf_cma:>10.2f}")
print(f"{'After CMA-ES + re-scalar':<35} {_rmse_cma_final:>10.2f}")
print(f"{'Total improvement':<35} {_rmse0_cma-_rmse_cma_final:>+10.2f}")

# ── Save ─────────────────────────────────────────────────────────────────
_calib_cma_out = {
    'meta': {
        'source': 'CELL CAL-CMA (CMA-ES)',
        'frequency_mhz': float(FREQUENCY_HZ / 1e6),
        'n_calib_rx': len(_cal_rx_cma),
        'cal_max_dist_km': _CMA_MAX_DIST_KM,
        'total_evals': _eval_cma_n[0],
        'cma_sigma0': _CMA_SIGMA0,
        'cma_maxiter': _CMA_MAXITER,
        'rmse_before_db': round(_rmse0_cma, 3),
        'rmse_scalar_db': round(_rmse_sf_cma, 3),
        'rmse_final_db':  round(_rmse_cma_final, 3),
        'warm_start': _CMA_WARM_START,
    },
    'materials': {}
}
for _mn, _p in _mats_cma.items():
    _calib_cma_out['materials'][_mn] = {
        'er': round(_p['er'], 4), 'sigma': round(_p['sig'], 6), 'scatter': round(_p['s'], 4)}
for _mn, _m in scene.radio_materials.items():
    if _mn.endswith('_train') or _mn in _calib_cma_out['materials']: continue
    try:
        _calib_cma_out['materials'][_mn] = {
            'er': round(_cma_to_float(_m.relative_permittivity), 4),
            'sigma': round(_cma_to_float(_m.conductivity), 6),
            'scatter': round(_cma_to_float(getattr(_m, 'scattering_coefficient', 0.2)), 4)}
    except: pass

_CALIB_FILE_CMA_OUT = _ocma.path.join(BASE_DIR, 'calibrated_materials_stevenage_2695mhz.json')
with open(_CALIB_FILE_CMA_OUT, 'w') as _f: _jcma.dump(_calib_cma_out, _f, indent=2)
_SF_FILE_CMA = _ocma.path.join(BASE_DIR, 'scalar_offset_stevenage_2695mhz.json')
with open(_SF_FILE_CMA, 'w') as _f:
    _jcma.dump({'scalar_factor_db': _sf_cma_final,
                'meta': {'rmse_before_db': round(_rmse0_cma, 3),
                         'rmse_final_db':  round(_rmse_cma_final, 3)}}, _f, indent=2)

print(f"\nSaved: {_CALIB_FILE_CMA_OUT}")
print(f"Saved: {_SF_FILE_CMA}")
print("Run CELL 4A to apply calibrated materials, then CELL 8e to verify.")
print(f"\n{'#'*60}")
print(f"CELL CAL-CMA complete: final RMSE = {_rmse_cma_final:.2f} dB")
print("Run CELL 4A then CELL 8e to verify.")


## CELL CAL REPORT — RMSE + per-material EM parameter convergence

Plots the optimisation history recorded by CELL CAL's `_mat_loss`: RMSE per
evaluation, and each material's `er` (relative permittivity), `sigma`
(conductivity, log scale) and `S` (scattering coefficient) trajectories.
Run immediately after CELL CAL -- needs `_opt_history`/`_mat_targets` in scope.


In [ ]:
# ====================================================================
# CELL CAL REPORT — RMSE + per-material EM parameter convergence plots
# ====================================================================
import matplotlib.pyplot as plt
import numpy as np
import os

if not _mat_targets or not _opt_history:
    print("[CELL CAL REPORT] No optimisation history available -- run CELL CAL first.")
else:
    _hist_evals = np.array([h['eval'] for h in _opt_history])
    _hist_rmse  = np.array([h['rmse'] for h in _opt_history])
    _hist_x     = np.array([h['x']    for h in _opt_history])   # (n_evals, 3*n_materials)

    print("=" * 70)
    print("CELL CAL REPORT")
    print("=" * 70)
    print(f"Total evaluations       : {len(_hist_evals)}")
    print(f"RMSE before calibration  : {_rmse0:.2f} dB")
    print(f"RMSE after scalar offset : {rmse_after_sf:.2f} dB")
    print(f"RMSE after material calib: {rmse_final:.2f} dB")
    _best_i = int(_hist_rmse.argmin())
    print(f"Best RMSE seen in history: {_hist_rmse[_best_i]:.2f} dB  (eval {_hist_evals[_best_i]})")
    print()
    print(f"{'Material':<24}{'er0':>8}{'er_final':>10}{'sigma0':>10}{'sigma_final':>12}{'S0':>7}{'S_final':>9}")
    for _i, (_mn, _e0, _sig0, _s0) in enumerate(_mat_targets):
        _er_f  = float(np.clip(_hist_x[-1, 3*_i],          _ER_PHYS_MIN,  _ER_PHYS_MAX))
        _sig_f = float(np.clip(np.exp(_hist_x[-1, 3*_i+1]), _SIG_PHYS_MIN, _SIG_PHYS_MAX))
        _s_f   = float(np.clip(_hist_x[-1, 3*_i+2],         _S_PHYS_MIN,   _S_PHYS_MAX))
        print(f"{_mn:<24}{_e0:>8.3f}{_er_f:>10.3f}{_sig0:>10.5f}{_sig_f:>12.5f}{_s0:>7.3f}{_s_f:>9.3f}")

    # ── Plot 1: RMSE vs evaluation ──────────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(9, 4))
    ax1.plot(_hist_evals, _hist_rmse, lw=1.2)
    ax1.axhline(_rmse0, color='gray', ls='--', lw=1, label=f'before calib ({_rmse0:.2f} dB)')
    ax1.axhline(rmse_after_sf, color='orange', ls='--', lw=1, label=f'after scalar ({rmse_after_sf:.2f} dB)')
    ax1.set_xlabel('Evaluation #')
    ax1.set_ylabel('RMSE (dB)')
    ax1.set_title('CELL CAL — RMSE convergence')
    ax1.legend()
    ax1.grid(alpha=0.3)
    fig1.tight_layout()
    _rmse_png = os.path.join(BASE_DIR, 'cal_rmse_convergence.png')
    fig1.savefig(_rmse_png, dpi=150)
    print(f"\nSaved -> {_rmse_png}")
    plt.show()

    # ── Plot 2: per-material er / sigma / S vs evaluation ───────────────────
    _n_mat = len(_mat_targets)
    fig2, axes = plt.subplots(_n_mat, 3, figsize=(13, 2.4 * _n_mat), squeeze=False)
    for _i, (_mn, _e0, _sig0, _s0) in enumerate(_mat_targets):
        _er_series  = np.clip(_hist_x[:, 3*_i],          _ER_PHYS_MIN,  _ER_PHYS_MAX)
        _sig_series = np.clip(np.exp(_hist_x[:, 3*_i+1]), _SIG_PHYS_MIN, _SIG_PHYS_MAX)
        _s_series   = np.clip(_hist_x[:, 3*_i+2],         _S_PHYS_MIN,   _S_PHYS_MAX)

        axes[_i, 0].plot(_hist_evals, _er_series, lw=1)
        axes[_i, 0].axhline(_e0, color='gray', ls='--', lw=0.8)
        axes[_i, 0].set_ylabel(_mn, fontsize=8)
        if _i == 0:
            axes[_i, 0].set_title('relative permittivity (er)')

        axes[_i, 1].plot(_hist_evals, _sig_series, lw=1)
        axes[_i, 1].axhline(_sig0, color='gray', ls='--', lw=0.8)
        axes[_i, 1].set_yscale('log')
        if _i == 0:
            axes[_i, 1].set_title('conductivity (sigma, S/m, log scale)')

        axes[_i, 2].plot(_hist_evals, _s_series, lw=1)
        axes[_i, 2].axhline(_s0, color='gray', ls='--', lw=0.8)
        axes[_i, 2].set_ylim(-0.02, 1.0)
        if _i == 0:
            axes[_i, 2].set_title('scattering coefficient (S)')

        for _j in range(3):
            axes[_i, _j].grid(alpha=0.3)
        axes[_i, 0].tick_params(labelsize=7)

    for _j in range(3):
        axes[-1, _j].set_xlabel('Evaluation #')

    fig2.suptitle('CELL CAL — per-material EM parameter convergence', y=1.0)
    fig2.tight_layout()
    _mat_png = os.path.join(BASE_DIR, 'cal_material_convergence.png')
    fig2.savefig(_mat_png, dpi=150)
    print(f"Saved -> {_mat_png}")
    plt.show()


## CELL PRB — Differentiable Material Calibration via Mitsuba `mi.traverse` + `mi.ad.Adam`

**Confirmed pattern** (from Instant-RM's own `Differentiable_Geometry.ipynb`):
```python
params = mi.traverse(scene)
dr.enable_grad(params[key]); params.update()
opt = mi.ad.Adam(lr=...)
dr.backward(loss)
opt.step()
```
Sionna 2.0 scenes **are** Mitsuba3 scenes, so this pattern is directly applicable here —
no `instant-rm` package required, just `mitsuba`/`drjit`, which Sionna 2.0 already
depends on. This confirms the technique "works with Sionna 2.0" in the sense that the
autodiff machinery is the same; what's untested is whether **this notebook's**
`PathSolver` output stays attached to the Dr.Jit gradient graph all the way back to
`scene.radio_materials`, or gets detached somewhere (e.g. by a `.numpy()` conversion).

**This cell tests that directly:**
1. Calls `mi.traverse(scene)` and matches its keys against `scene.radio_materials` names.
2. If no traversable keys match, autodiff isn't wired up for this build — CELL CAL's
   derivative-free search remains the supported path (this is reported, not assumed).
3. If keys match, enables gradients, runs one `PathSolver` forward pass, calls
   `dr.backward()`, and **checks whether the resulting gradients are non-zero**. Only a
   non-zero gradient proves the chain `material → PathSolver → loss` is actually
   differentiable end-to-end.
4. If gradients are non-zero, takes one real `mi.ad.Adam` step — a working stand-in for
   the kind of training loop Instant-RM uses, scoped to the materials already present in
   this scene (no per-building texture granularity, which would need actual `instant-rm`
   shapes/plugins).

This replaces the earlier scalar-only probe with a direct test against the real
material-calibration parameters CELL CAL optimises, using the verified Instant-RM API
shape instead of an invented one.


In [ ]:
# ====================================================================
# CELL PRB — Differentiable material calibration via mi.traverse + mi.ad.Adam
# ====================================================================
import numpy as np
import drjit as dr
import mitsuba as mi

print("=" * 70)
print("CELL PRB -- Dr.Jit differentiable material calibration (Sionna 2.0)")
print("=" * 70)

_mi_scene = getattr(scene, "mi_scene", scene)
params = mi.traverse(_mi_scene)

# Discover which traversable Mitsuba parameters correspond to our radio
# materials. Sionna 2.0 radio materials are backed by Mitsuba BSDFs, but the
# exact key naming depends on the build, so we search rather than hardcode.
_mat_names = [m for m in scene.radio_materials if not m.endswith('_train')]
# Only consider keys that are actually a BSDF *material* parameter of one of
# our radio materials -- a plain "material name is a substring of the key"
# test also matches unrelated mesh-level params (e.g. a material named
# "itu_brick" matching ".../mesh-bld_itu_brick.silhouette_sampling_weight"),
# which then fail with "is not differentiable!" since they're geometry/mesh
# bookkeeping fields, not BSDF reflectance/eta/k parameters.
_BSDF_PARAM_HINTS = ('.reflectance', '.eta', '.k', '.specular_reflectance',
                     '.diffuse_reflectance', '.alpha', '.weight')
_prb_keys = {}
for _k in params.keys():
    if not any(_k.endswith(_h) or _h in _k for _h in _BSDF_PARAM_HINTS):
        continue
    for _mn in _mat_names:
        if _mn in _k:
            _prb_keys.setdefault(_mn, []).append(_k)

if not _prb_keys:
    print("[CELL PRB] No traversable Dr.Jit BSDF parameters matched scene.radio_materials "
          "by name. This Sionna build likely keeps material EM properties as plain "
          "Python attributes on RadioMaterial, outside Mitsuba's traverse() graph -- "
          "so end-to-end autodiff through PathSolver isn't wired up for this build. "
          "CELL CAL's derivative-free search remains the supported calibration path.")
else:
    print(f"[CELL PRB] Found {sum(len(v) for v in _prb_keys.values())} candidate BSDF "
          f"parameter(s) across {len(_prb_keys)} material(s):")
    for _mn, _ks in _prb_keys.items():
        print(f"  {_mn}: {_ks}")

    # Enable grad / register with the optimizer one key at a time, skipping
    # (with a warning) any key that turns out not to be a differentiable
    # float Dr.Jit type -- rather than crashing the whole cell on the first
    # non-differentiable param.
    _flat_keys = []
    for _mn, _ks in _prb_keys.items():
        for _k in _ks:
            try:
                dr.enable_grad(params[_k])
                _flat_keys.append(_k)
            except Exception as _e:
                print(f"  [skip] {_k}: cannot enable grad ({_e!r})")
    params.update()

    opt = mi.ad.Adam(lr=1e-2)
    _registered_keys = []
    for _k in list(_flat_keys):
        try:
            opt[_k] = params[_k]
            _registered_keys.append(_k)
        except TypeError as _e:
            print(f"  [skip] {_k}: not differentiable ({_e!r})")
    _flat_keys = _registered_keys

    if not _flat_keys:
        print("[CELL PRB] No candidate parameter was actually differentiable/registrable "
              "with the optimizer -- nothing to calibrate this way for this build. "
              "CELL CAL's derivative-free search remains the supported calibration path.")

def _run_prb_forward_backward():
    _cfg_prb = _cfg_cal if '_cfg_cal' in dir() else dict(
        max_depth=MAX_DEPTH, los=True, specular_reflection=True,
        diffraction=True, edge_diffraction=True, diffuse_reflection=True,
        samples_per_src=NUM_SAMPLES_PS)
    _rx_prb = cal_rx if 'cal_rx' in dir() else receivers
    for _n in list(scene.receivers.keys()):
        scene.remove(_n)
    for _r in _rx_prb:
        scene.add(_r)

    _ps_out = PathSolver()(scene, **_cfg_prb)
    _a = _ps_out.a
    if isinstance(_a, tuple):
        _pwr = dr.sqr(_a[0]) + dr.sqr(_a[1])
    else:
        _pwr = dr.sqr(dr.real(_a)) + dr.sqr(dr.imag(_a))
    _loss = -dr.mean(_pwr)  # placeholder objective: maximise received power
    dr.backward(_loss)

if _prb_keys and _flat_keys:
    import traceback as _tb_prb

    # PathSolver records its ray loop symbolically by default
    # (dr.JitFlag.LoopRecord), which the Instant-RM authors note is
    # incompatible with backpropagating gradients *through* loop
    # iterations (see Differentiable_Geometry.ipynb's loop_record=False
    # usage). First attempt with default flags; if dr.while_loop raises,
    # retry once with symbolic loop recording disabled (evaluated mode) --
    # slower, but gradient-compatible -- before giving up.
    _prb_attempts = [
        ("default JitFlag.LoopRecord", None),
        ("JitFlag.LoopRecord disabled (evaluated mode)", False),
    ]
    _succeeded = False
    _structural_limitation = False
    for _attempt_label, _loop_record_flag in _prb_attempts:
        _prev_flag = dr.flag(dr.JitFlag.LoopRecord) if _loop_record_flag is not None else None
        if _loop_record_flag is not None:
            dr.set_flag(dr.JitFlag.LoopRecord, _loop_record_flag)
        try:
            _run_prb_forward_backward()
            print(f"[CELL PRB] Forward/backward succeeded ({_attempt_label}).")
            _succeeded = True
        except Exception as _e:
            _tb_str = _tb_prb.format_exc()
            print(f"[CELL PRB] Forward/backward failed ({_attempt_label}): {_e!r}")
            print(_tb_str)
            # Confirmed root cause (observed on real runs): Sionna's
            # SampleData.insert() (sb_candidate_generator.py's shoot-and-bounce
            # loop) writes candidate path data via Dr.Jit "local memory" writes,
            # which Dr.Jit hard-rejects under gradient tracking regardless of
            # which parameter has grad enabled or whether the loop is recorded
            # or evaluated. This is structural to the candidate generator, not
            # a parameter-selection or loop-recording-mode issue, so retrying
            # the second attempt cannot help -- stop immediately.
            if "Local memory writes are not differentiable" in _tb_str:
                _structural_limitation = True
        finally:
            if _prev_flag is not None:
                dr.set_flag(dr.JitFlag.LoopRecord, _prev_flag)
        if _succeeded or _structural_limitation:
            break

    if _structural_limitation:
        print("[CELL PRB] CONFIRMED: PathSolver's candidate generator "
              "(sb_candidate_generator.py -> SampleData.insert -> local memory "
              "write) is structurally non-differentiable in this installed "
              "Sionna build -- this happens during path/candidate discovery, "
              "before any material BSDF is even evaluated, so no choice of "
              "material parameter or loop-recording mode can route around it. "
              "Gradient-based (PRB-style) material calibration through "
              "PathSolver is not possible with this Sionna version; it would "
              "require Sionna's own maintainers to wrap those writes in "
              "dr.detach() or expose a differentiable solver path. "
              "CELL CAL's derivative-free search is the correct and only "
              "supported calibration method for this build.")
    elif not _succeeded:
        print("[CELL PRB] Both attempts failed for a different reason than the "
              "known local-memory-write limitation above -- see the traceback(s) "
              "for the actual cause. "
              "CELL CAL's derivative-free search remains the supported calibration path.")
    else:
        _any_grad = False
        for _k in _flat_keys:
            _g = dr.grad(params[_k])
            if dr.any(dr.neq(_g, 0.0)):
                _any_grad = True
            print(f"  grad[{_k}] = {_g}")

        if _any_grad:
            print("[CELL PRB] Non-zero gradients reached the material parameters "
                  "through PathSolver -- end-to-end Dr.Jit autodiff IS wired up for "
                  "this Sionna 2.0 build. A full mi.ad.Adam training loop (as in "
                  "Instant-RM's Differentiable_Geometry.ipynb) could replace or "
                  "augment CELL CAL's derivative-free search.")
            opt.step()
            for _k in _flat_keys:
                params[_k] = opt[_k]
            params.update()
            print("[CELL PRB] Took one Adam step on the matched material parameters.")
        else:
            print("[CELL PRB] Gradients were all zero -- PathSolver's output is "
                  "likely detached from the parameter graph (e.g. via an internal "
                  ".numpy()-style conversion), so this path doesn't carry gradients "
                  "yet for this build.")


# ── Cleanup: ALWAYS disable gradient tracking before returning control to
# other cells. Without this, params enabled via dr.enable_grad() above stay
# grad-enabled for the rest of the kernel session -- so re-running CELL CAL
# (or any other PathSolver call) afterwards hits the exact same
# "Local memory writes are not differentiable" error even though CELL CAL
# itself never requested gradients. This must run regardless of whether the
# forward/backward attempt above succeeded, failed, or was skipped.
if '_flat_keys' in dir() and _flat_keys:
    for _k in _flat_keys:
        try:
            dr.disable_grad(params[_k])
        except Exception:
            pass
    params.update()
    print("[CELL PRB] Cleanup: disabled gradient tracking on all registered parameters "
          "(scene params are now safe for derivative-free cells like CELL CAL).")


In [ ]:
# ====================================================================
# CELL DIAG-C — Near-Receiver Building Penetration Check
# ====================================================================
# For the first 30 unique receivers in measurements_with_pathloss.csv:
#   - Places each receiver one-by-one in scene-local coords
#   - Runs a fast PathSolver (1 M samples) to count paths
#   - Receivers with < 5 paths are likely inside a building mesh
# Run AFTER CELL 7 (terrain/coord loaded) and CELL B3 (scene loaded).
# ====================================================================
import numpy as np, pandas as pd, math, gc
from sionna.rt import PathSolver, Receiver

print("=" * 72)
print("CELL DIAG-C — Near-Receiver Building Penetration Check")
print("=" * 72)

_df_diagb = pd.read_csv(MEASUREMENT_CSV).drop_duplicates(subset='name').head(30)
print(f"  Checking {len(_df_diagb)} receivers  |  1M samples  |  max_depth={MAX_DEPTH}")
print()

_cfg_diagb = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                  diffraction=True, edge_diffraction=True, diffuse_reflection=True,
                  samples_per_src=1_000_000)

print(f"  {'Name':<12} {'dist_m':>7} {'locX':>7} {'locY':>7} {'locZ':>6} {'terr_z':>7} "
      f"{'RSSI_m':>7} {'RSSI_s':>7} {'PL_err':>7} {'Npaths':>7}  note")
print("  " + "-" * 110)

_n_inside = 0
for _, _brow in _df_diagb.iterrows():
    _bname   = str(_brow['name'])
    _blon    = float(_brow['lon'])
    _blat    = float(_brow['lat'])
    _brssi_m = float(_brow['local_measurement_dBm'])
    _bpl_m   = TX_CONDUCTED_DBM - _brssi_m

    _blx, _bly, _ = gps_to_local(_blon, _blat)
    _btz = terrain_z(_blx, _bly)
    _blz = _btz + RX_AGL_M

    _bux, _buy = gps_to_utm.transform(_blon, _blat)
    _bdist = math.sqrt((_bux - utm_center_x)**2 + (_buy - utm_center_y)**2)

    for _bn in list(scene.receivers.keys()):
        scene.remove(_bn)
    scene.add(Receiver(name=_bname, position=[_blx, _bly, _blz]))

    try:
        _bpaths = PathSolver()(scene, **_cfg_diagb)
        _ba_raw = _bpaths.a
        if isinstance(_ba_raw, tuple):
            _ba = (_ba_raw[0].numpy() if hasattr(_ba_raw[0], 'numpy') else np.array(_ba_raw[0])) +                   1j*(_ba_raw[1].numpy() if hasattr(_ba_raw[1], 'numpy') else np.array(_ba_raw[1]))
        else:
            _ba = _ba_raw.numpy() if hasattr(_ba_raw, 'numpy') else np.array(_ba_raw)
        _ba = np.squeeze(_ba)
        if _ba.ndim == 0: _ba = _ba.reshape(1, 1)
        elif _ba.ndim == 1: _ba = _ba[np.newaxis, :]
        elif _ba.ndim > 2: _ba = _ba.reshape(1, -1)
        _bpwr = float(np.sum(np.abs(_ba[0])**2))
        _bnp  = int(np.sum(np.abs(_ba[0]) > 1e-20))
        _brssi_s = rssi_from_path_gain(_bpwr) if _bpwr > 1e-30 else float('nan')
        _bpl_err = (TX_CONDUCTED_DBM - _brssi_s) - _bpl_m if math.isfinite(_brssi_s) else float('nan')
        del _bpaths; gc.collect()
    except Exception as _be:
        _bnp = -1; _brssi_s = float('nan'); _bpl_err = float('nan')

    if _bnp < 0:
        _bnote = "⚠ ERROR"
    elif _bnp == 0:
        _bnote = "⛔ ZERO PATHS — inside building mesh"
        _n_inside += 1
    elif _bnp < 5:
        _bnote = f"⚠ NEAR-ZERO PATHS — likely inside building"
        _n_inside += 1
    elif math.isfinite(_bpl_err) and _bpl_err > 15:
        _bnote = f"⚠ high bias +{_bpl_err:.0f} dB"
    else:
        _bnote = "✓"

    _fv = lambda x, fmt: (fmt % x) if (isinstance(x, float) and math.isfinite(x)) else "    N/A"
    print(f"  {_bname:<12} {_bdist:>7.0f} {_blx:>7.1f} {_bly:>7.1f} {_blz:>6.2f} {_btz:>7.2f} "
          f"{_brssi_m:>7.1f} {_fv(_brssi_s,'%7.1f')} {_fv(_bpl_err,'%+7.1f')} {_bnp:>7}  {_bnote}")

print()
print(f"Summary: {_n_inside} / {len(_df_diagb)} receivers have < 5 paths")
print("  → receivers with 0 paths are inside opaque building meshes")
print("  → GPS drift in deep urban canyons can shift coordinates 10-30 m into buildings")
print()
print("Column guide:")
print("  locX/Y  = scene-local metres from TX (X=east, Y=north)")
print("  locZ    = placed height = terrain_z + RX_AGL_M")
print("  terr_z  = terrain mesh height at (locX, locY)")
print("  RSSI_m  = measured dBm  |  RSSI_s = simulated dBm")
print("  PL_err  = PL_sim - PL_meas  (positive = sim predicts too much loss)")


## VIZ — Input data visualisation
One cell per layer: DTM · nDSM · VOM · PC heights · PLY meshes.
Each cell is self-contained — run in any order after CELL 1.
Paths derive from `RT_BASE_DIR` env var (same as scene builder).

In [ ]:
# ── VIZ-A: DTM — bare-earth terrain (dem.tif) ──────────────────────────────
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
os.makedirs(os.path.join(_rt_root, 'viz'), exist_ok=True)
_path = os.path.join(_rt_root, 'dem.tif')
with rasterio.open(_path) as _src:
    _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
    _dtm = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
    _nd  = _src.nodata
    _ext = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
if _nd is not None: _dtm[_dtm == _nd] = np.nan
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(_dtm, extent=_ext, cmap='terrain', origin='upper')
plt.colorbar(im, ax=ax, label='Elevation ASL (m)')
ax.set(title='DTM — Bare-Earth Terrain (EPSG:27700)', xlabel='Easting (m)', ylabel='Northing (m)')
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_A_dtm.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig); del _dtm

In [ ]:
# ── VIZ-B: nDSM — normalised DSM, height above ground (ndsm.tif) ───────────
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_path = os.path.join(_rt_root, 'ndsm.tif')
with rasterio.open(_path) as _src:
    _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
    _ndsm = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
    _nd   = _src.nodata
    _ext  = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
if _nd is not None: _ndsm[_ndsm == _nd] = np.nan
_ndsm[_ndsm < 0] = 0
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(_ndsm, extent=_ext, cmap='YlGn', vmin=0,
               vmax=np.nanpercentile(_ndsm, 99), origin='upper')
plt.colorbar(im, ax=ax, label='Height above ground (m)')
ax.set(title='nDSM — Normalised Digital Surface Model (EPSG:27700)',
       xlabel='Easting (m)', ylabel='Northing (m)')
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_B_ndsm.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig); del _ndsm

In [ ]:
# ── VIZ-C: VOM — vegetation object model (vom.tif) ────────────────────────
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_path = os.path.join(_rt_root, 'vom.tif')
if not os.path.exists(_path):
    print(f'VOM not found: {_path}')
else:
    with rasterio.open(_path) as _src:
        _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
        _vom = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
        _nd  = _src.nodata
        _ext = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
    if _nd is not None: _vom[_vom == _nd] = np.nan
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(_vom, extent=_ext, cmap='Greens', origin='upper')
    plt.colorbar(im, ax=ax, label='Canopy height above ground (m)')
    ax.set(title='VOM — Vegetation Object Model (EPSG:27700)',
           xlabel='Easting (m)', ylabel='Northing (m)')
    plt.tight_layout()
    plt.savefig(os.path.join(_rt_root, 'viz', 'viz_C_vom.png'), dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig); del _vom

In [ ]:
# ── VIZ-D: Point-cloud height rasters — veg + building (pc_*_height.tif) ───
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_layers = [
    (os.path.join(_rt_root, 'pc_veg_height.tif'), 'PC Vegetation Height', 'YlGn'),
    (os.path.join(_rt_root, 'pc_bld_height.tif'), 'PC Building Height',   'Oranges'),
]
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (_p, _title, _cmap) in zip(axes, _layers):
    if not os.path.exists(_p):
        ax.set_title(f'{_title}\n(file not found)'); continue
    with rasterio.open(_p) as _src:
        _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
        _d   = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
        _nd  = _src.nodata
        _ext = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
    if _nd is not None: _d[_d == _nd] = np.nan
    _d[_d < 0] = 0
    _vmax = np.nanpercentile(_d[_d > 0], 99) if np.any(_d > 0) else 30
    im = ax.imshow(_d, extent=_ext, cmap=_cmap, vmin=0, vmax=_vmax, origin='upper')
    plt.colorbar(im, ax=ax, label='Height (m)')
    ax.set(title=f'{_title} (EPSG:27700)', xlabel='Easting (m)', ylabel='Northing (m)')
    del _d
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_D_pc_heights.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)

In [ ]:
# ── VIZ-E: PLY scene meshes — top-down footprint ────────────────────────────
import os, glob, numpy as np, matplotlib.pyplot as plt
_rt_root  = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_scene_sfx = globals().get('SCENE_SUFFIX', 'v4_full')
_mesh_dir  = os.path.join(_rt_root, f'scene_{_scene_sfx}', 'meshes')

def _ply_xy(path):
    with open(path, 'rb') as f:
        n = 0
        while True:
            ln = f.readline().decode('ascii', 'ignore').strip()
            if ln.startswith('element vertex'): n = int(ln.split()[-1])
            if ln == 'end_header': break
        if n == 0: return None, None
        d = np.frombuffer(f.read(n * 12), dtype=np.float32).reshape(-1, 3)
    return d[:, 0], d[:, 1]

_plys = sorted(glob.glob(os.path.join(_mesh_dir, '*.ply')))
print(f'{len(_plys)} PLY files in {_mesh_dir}')
fig, ax = plt.subplots(figsize=(12, 12))
_cm = plt.cm.tab20(np.linspace(0, 1, max(len(_plys), 1)))
for _f, _c in zip(_plys, _cm):
    _x, _y = _ply_xy(_f)
    if _x is None: continue
    _s = max(1, len(_x) // 5000)
    ax.scatter(_x[::_s], _y[::_s], s=0.4, c=[_c], label=os.path.basename(_f))
    del _x, _y
ax.set_aspect('equal')
ax.set(title='PLY meshes — top view (scene-local coords)', xlabel='X (m)', ylabel='Y (m)')
ax.legend(loc='upper right', fontsize=5, markerscale=8, ncol=2)
plt.tight_layout()
_viz_dir = os.path.join(_rt_root, 'viz')
os.makedirs(_viz_dir, exist_ok=True)
plt.savefig(os.path.join(_viz_dir, 'viz_E_ply_footprint.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved viz_E_ply_footprint.png')


## CELL VIZ-PLY — Scene PLY Preview (top-down, per-material RGB colours)

Enhanced PLY preview with material-based colour coding and dark background.
Run after **CELL B3** to verify full scene geometry before simulation.
Saves `viz_PLY_scene_preview.png` to the `viz/` directory.

In [ ]:
# -- VIZ-PLY: shaded 3-D geometry per PLY + combined scene view --
# Left panel: 2-D top-down wireframe (XY)
# Right panel: 3-D with face-normal shading (diffuse + ambient) -- scene-preview look
# Extra: viz/ply_scene_combined.png -- all meshes together
import os, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

_rt_root   = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt',
                 'stevenage_ofcom_915mhz_dem'))
_scene_sfx = globals().get('SCENE_SUFFIX', 'v4_full')
_mesh_dir  = os.path.join(_rt_root, f'scene_{_scene_sfx}', 'meshes')

# -- Solid per-material base colours (RGB 0-1) used for shaded rendering --
_PLY_RGB = {
    'terrain_veg':       [0.42, 0.62, 0.30],
    'terrain':           [0.76, 0.70, 0.50],
    'brick':             [0.75, 0.35, 0.20],
    'glass':             [0.60, 0.85, 0.95],
    'metal_barrier':     [0.85, 0.80, 0.20],
    'concrete_bridges':  [0.62, 0.62, 0.62],
    'hwy_bridges':       [0.58, 0.58, 0.60],
    'concrete':          [0.65, 0.65, 0.65],
    'metal':             [0.72, 0.72, 0.76],
    'wood':              [0.58, 0.38, 0.15],
    'asphalt':           [0.40, 0.40, 0.40],
    'water':             [0.20, 0.50, 0.85],
    'ndms':              [0.35, 0.72, 0.25],
    'trees':             [0.15, 0.52, 0.15],
    'vegetation':        [0.30, 0.65, 0.20],
    'rail':              [0.55, 0.55, 0.65],
    'road':              [0.45, 0.45, 0.45],
    'embankment':        [0.60, 0.52, 0.35],
    'cutting':           [0.55, 0.47, 0.32],
    'carpark':           [0.48, 0.48, 0.48],
    'chimney':           [0.70, 0.30, 0.10],
}

# -- Wireframe colours for 2-D panel --
_PLY_WIRE = {k: [max(0, c-0.15) for c in v] for k, v in _PLY_RGB.items()}

def _ply_rgb(path):
    n = os.path.basename(path).lower()
    for key, col in _PLY_RGB.items():
        if key in n: return np.array(col)
    return np.array([0.55, 0.55, 0.55])

# -- Light source: azimuth 225 deg (upper-left), altitude 45 deg --
_AZ  = np.radians(225)
_ALT = np.radians(45)
_LIGHT = np.array([np.cos(_ALT)*np.cos(_AZ),
                   np.cos(_ALT)*np.sin(_AZ),
                   np.sin(_ALT)])

def _shade_faces(tris, base_rgb, ambient=0.35, alpha=0.95):
    # Compute shaded face colours using face normals + directional light
    v0, v1, v2 = tris[:,0], tris[:,1], tris[:,2]
    normals = np.cross(v1 - v0, v2 - v0)
    nlen = np.linalg.norm(normals, axis=1, keepdims=True)
    nlen[nlen < 1e-10] = 1.0
    normals /= nlen
    # Two-sided: take abs so back-faces are also lit
    diffuse = np.abs(normals @ _LIGHT)
    intensity = ambient + (1.0 - ambient) * diffuse        # (M,)
    fc = base_rgb[np.newaxis, :] * intensity[:, np.newaxis]
    fc = np.clip(fc, 0, 1)
    fc = np.column_stack([fc, np.full(len(fc), alpha)])    # (M,4)
    return fc

# -- PLY type sizes --
_TYPE_SZ = {
    'float':4,'float32':4,'double':8,'float64':8,
    'int':4,'int32':4,'uint':4,'uint32':4,
    'short':2,'int16':2,'ushort':2,'uint16':2,
    'char':1,'int8':1,'uchar':1,'uint8':1,
}

def _read_ply_mesh(path, max_faces=10_000):
    with open(path, 'rb') as fh:
        n_verts=0; n_faces=0; elem=None
        v_props=[]; fcs=1; fis=4; fid=np.int32
        while True:
            ln = fh.readline().decode('ascii','ignore').strip()
            if ln.startswith('element vertex'):
                n_verts=int(ln.split()[-1]); elem='vertex'
            elif ln.startswith('element face'):
                n_faces=int(ln.split()[-1]); elem='face'
            elif ln.startswith('property list') and elem=='face':
                p=ln.split(); fcs=_TYPE_SZ.get(p[2],1); fis=_TYPE_SZ.get(p[3],4)
                fid=np.int32 if fis==4 else np.int16
            elif ln.startswith('property') and elem=='vertex':
                p=ln.split()
                v_props.append(0 if p[1]=='list' else _TYPE_SZ.get(p[1],4))
            elif ln=='end_header': break
        v_stride=sum(v_props) if v_props else 12
        raw_v=fh.read(n_verts*v_stride)
        if v_stride==12:
            xyz=np.frombuffer(raw_v,dtype=np.float32).reshape(-1,3).copy()
        else:
            buf=np.frombuffer(raw_v,dtype=np.uint8).reshape(n_verts,v_stride)
            xyz=np.frombuffer(buf[:,:12].tobytes(),dtype=np.float32).reshape(-1,3).copy()
        good=np.isfinite(xyz).all(axis=1)&(np.abs(xyz).max(axis=1)<1e5)
        remap=np.full(n_verts,-1,dtype=np.int32)
        remap[good]=np.arange(good.sum(),dtype=np.int32)
        xyz=xyz[good]
        if n_faces==0: return xyz,None
        fs=fcs+3*fis; raw_f=fh.read(n_faces*fs)
        if len(raw_f)==n_faces*fs:
            arr=np.frombuffer(raw_f,dtype=np.uint8).reshape(n_faces,fs)
            ib=arr[:,fcs:].reshape(n_faces,3,fis)
            fr=(ib.reshape(-1,fis).view(np.int32 if fis==4 else np.int16)
                  .reshape(n_faces,3).astype(np.int32))
            f0=remap[fr[:,0]]; f1=remap[fr[:,1]]; f2=remap[fr[:,2]]
            fa=np.stack([f0,f1,f2],axis=1)[(f0>=0)&(f1>=0)&(f2>=0)]
        else:
            tris=[]; pos=0; cnt_dt=np.uint8 if fcs==1 else np.int32
            for _ in range(n_faces):
                if pos+fcs>len(raw_f): break
                cnt=int(np.frombuffer(raw_f[pos:pos+fcs],dtype=cnt_dt)[0]); pos+=fcs
                need=cnt*fis
                if pos+need>len(raw_f): break
                idxs=np.frombuffer(raw_f[pos:pos+need],dtype=fid).astype(np.int32); pos+=need
                if cnt==3: tris.append(idxs)
                elif cnt==4: tris.append(idxs[[0,1,2]]); tris.append(idxs[[0,2,3]])
            if tris:
                fr=np.array(tris,dtype=np.int32)
                f0=remap[fr[:,0]]; f1=remap[fr[:,1]]; f2=remap[fr[:,2]]
                fa=np.stack([f0,f1,f2],axis=1)[(f0>=0)&(f1>=0)&(f2>=0)]
            else: fa=None
        if fa is None or len(fa)==0: return xyz,None
        if len(fa)>max_faces:
            fa=fa[np.random.choice(len(fa),max_faces,replace=False)]
        return xyz,fa

_ORDER_KEYS=['terrain','road','rail','water','asphalt','veg',
             'ndms','trees','embankment','cutting']

def _sort_key(p):
    n=os.path.basename(p).lower()
    for i,k in enumerate(_ORDER_KEYS):
        if k in n: return i
    return len(_ORDER_KEYS)

_plys=sorted(glob.glob(os.path.join(_mesh_dir,'*.ply')),key=_sort_key)
if not _plys:
    print(f'No PLY files found in {_mesh_dir}')
else:
    _viz_dir=os.path.join(_rt_root,'viz','ply_meshes')
    os.makedirs(_viz_dir,exist_ok=True)

    _all_meshes=[]   # collect for combined view

    # ── Per-PLY PNGs ─────────────────────────────────────────────────────
    for _path in _plys:
        _name=os.path.splitext(os.path.basename(_path))[0]
        print(f'  {_name}...', end=' ', flush=True)
        _xyz,_faces=_read_ply_mesh(_path)
        _nf=len(_faces) if _faces is not None else 0
        print(f'{len(_xyz):,} verts  {_nf:,} tris', flush=True)
        _base=_ply_rgb(_path)
        _all_meshes.append((_name,_xyz,_faces,_base))

        fig=plt.figure(figsize=(14,6.5),facecolor='white')
        _zmin=_xyz[:,2].min(); _zmax=_xyz[:,2].max()
        _xs=_xyz[:,0].max()-_xyz[:,0].min(); _ys=_xyz[:,1].max()-_xyz[:,1].min()
        _info=(f'vertices: {len(_xyz):,}   triangles: {_nf:,}   '
               f'X-span: {_xs:.0f} m   Y-span: {_ys:.0f} m   '
               f'Z: {_zmin:.1f} to {_zmax:.1f} m')
        fig.text(0.5,0.98,_name.replace('_',' '),ha='center',va='top',
                 color='black',fontsize=13,fontweight='bold')
        fig.text(0.5,0.93,_info,ha='center',va='top',
                 color='#444444',fontsize=7.5)

        # Left: 2-D wireframe
        ax2=fig.add_axes([0.03,0.05,0.44,0.84])
        ax2.set_facecolor('white'); ax2.set_aspect('equal')
        ax2.set_title('Top-down wireframe (XY)',color='#333333',fontsize=8,pad=4)
        ax2.tick_params(colors='#333333',labelsize=6)
        for sp in ax2.spines.values(): sp.set_edgecolor('#bbbbbb')
        ax2.set_xlabel('X (m)',color='#444444',fontsize=7)
        ax2.set_ylabel('Y (m)',color='#444444',fontsize=7)
        ax2.grid(color='#dddddd',linewidth=0.4,zorder=0)
        if _faces is not None and len(_faces)>0:
            txy=_xyz[_faces][:,:,:2]
            wire_col=list(_base)+[0.7]
            segs=np.concatenate([np.stack([txy[:,0],txy[:,1]],axis=1),
                                  np.stack([txy[:,1],txy[:,2]],axis=1),
                                  np.stack([txy[:,2],txy[:,0]],axis=1)])
            ax2.add_collection(LineCollection(segs,colors=[wire_col]*len(segs),
                                              linewidth=0.4,rasterized=True))
            ax2.autoscale_view()
        else:
            ax2.scatter(_xyz[:,0],_xyz[:,1],c=[_base],s=0.8,rasterized=True,linewidths=0)
        ax2.scatter([0],[0],s=60,marker='^',color='red',zorder=5)

        # Right: 3-D shaded
        ax3=fig.add_axes([0.50,0.02,0.48,0.90],projection='3d')
        ax3.set_facecolor('white'); ax3.set_axis_off()
        ax3.set_title('3-D shaded (face normals + light)',color='#333333',fontsize=8,pad=4)
        if _faces is not None and len(_faces)>0:
            tris3=_xyz[_faces]
            fc=_shade_faces(tris3,_base)
            poly=Poly3DCollection(tris3,linewidth=0,rasterized=True)
            poly.set_facecolor(fc)
            ax3.add_collection3d(poly)
            ax3.set_xlim(_xyz[:,0].min(),_xyz[:,0].max())
            ax3.set_ylim(_xyz[:,1].min(),_xyz[:,1].max())
            ax3.set_zlim(_xyz[:,2].min(),_xyz[:,2].max())
        else:
            ax3.scatter(_xyz[:,0],_xyz[:,1],_xyz[:,2],
                        c=[_base],s=0.8,depthshade=True,rasterized=True,linewidths=0)
            ax3.set_xlim(_xyz[:,0].min(),_xyz[:,0].max())
            ax3.set_ylim(_xyz[:,1].min(),_xyz[:,1].max())
            ax3.set_zlim(_xyz[:,2].min(),_xyz[:,2].max())
        ax3.scatter([0],[0],[0],c='red',s=60,marker='^',zorder=6)
        ax3.view_init(elev=28,azim=-55)

        plt.savefig(os.path.join(_viz_dir,f'ply_{_name}.png'),
                    dpi=150,bbox_inches='tight',facecolor='white')
        plt.close(fig)

    # ── Combined scene view ───────────────────────────────────────────────
    print('\nBuilding combined scene view...', flush=True)
    _MAX_COMBINED=4_000   # faces per mesh in combined view
    fig_c=plt.figure(figsize=(14,10),facecolor='white')
    ax_c=fig_c.add_subplot(111,projection='3d')
    ax_c.set_facecolor('#f8f8f8')
    ax_c.set_title('Full scene — all PLY meshes (shaded)',color='black',fontsize=12,pad=8)

    _legend_patches=[]
    for _name,_xyz,_faces,_base in _all_meshes:
        if _faces is not None and len(_faces)>0:
            fa=_faces
            if len(fa)>_MAX_COMBINED:
                fa=fa[np.random.choice(len(fa),_MAX_COMBINED,replace=False)]
            tris3=_xyz[fa]
            fc=_shade_faces(tris3,_base,ambient=0.30,alpha=0.90)
            poly=Poly3DCollection(tris3,linewidth=0,rasterized=True)
            poly.set_facecolor(fc)
            ax_c.add_collection3d(poly)
        else:
            if len(_xyz)>0:
                ax_c.scatter(_xyz[:,0],_xyz[:,1],_xyz[:,2],
                             c=[_base],s=0.5,depthshade=True,rasterized=True,linewidths=0)
        import matplotlib.patches as mpatches
        _legend_patches.append(
            mpatches.Patch(facecolor=_base,edgecolor='#888888',linewidth=0.5,
                           label=_name.replace('_',' ')))

    # Auto-set limits from all vertex data
    _all_xyz=np.concatenate([m[1] for m in _all_meshes if len(m[1])>0])
    ax_c.set_xlim(_all_xyz[:,0].min(),_all_xyz[:,0].max())
    ax_c.set_ylim(_all_xyz[:,1].min(),_all_xyz[:,1].max())
    ax_c.set_zlim(_all_xyz[:,2].min(),_all_xyz[:,2].max())
    ax_c.scatter([0],[0],[0],c='red',s=80,marker='^',zorder=10,label='TX')
    ax_c.set_xlabel('X (m)',fontsize=8); ax_c.set_ylabel('Y (m)',fontsize=8)
    ax_c.set_zlabel('Z (m)',fontsize=8)
    ax_c.view_init(elev=30,azim=-50)
    ax_c.xaxis.pane.fill=False; ax_c.yaxis.pane.fill=False; ax_c.zaxis.pane.fill=False
    ax_c.xaxis.pane.set_edgecolor('#cccccc')
    ax_c.yaxis.pane.set_edgecolor('#cccccc')
    ax_c.zaxis.pane.set_edgecolor('#cccccc')
    ax_c.grid(color='#e0e0e0',linewidth=0.4)
    ax_c.legend(handles=_legend_patches,loc='upper left',
                bbox_to_anchor=(0.0,1.0),fontsize=6,
                ncol=2,framealpha=0.7,edgecolor='#cccccc')

    _out_c=os.path.join(_rt_root,'viz','ply_scene_combined.png')
    plt.savefig(_out_c,dpi=150,bbox_inches='tight',facecolor='white')
    plt.close(fig_c)

    print(f'\nPer-PLY PNGs: {_viz_dir}')
    print(f'Combined scene: {_out_c}')
    print(f'Total meshes: {len(_all_meshes)}')


In [ ]:
# -- VIZ-F: DSM -- digital surface model (lidar_dsm.tif) ----------------------
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_path = os.path.join(_rt_root, 'lidar_dsm.tif')
with rasterio.open(_path) as _src:
    _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
    _dsm = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
    _nd  = _src.nodata
    _ext = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
if _nd is not None: _dsm[_dsm == _nd] = np.nan
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(_dsm, extent=_ext, cmap='terrain', origin='upper')
plt.colorbar(im, ax=ax, label='Elevation ASL (m)')
ax.set(title='DSM -- Digital Surface Model (EPSG:27700)', xlabel='Easting (m)', ylabel='Northing (m)')
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_F_dsm.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig); del _dsm

In [ ]:
# -- VIZ-G: LiDAR intensity (intensity.tif) ------------------------------------
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_path = os.path.join(_rt_root, 'intensity.tif')
with rasterio.open(_path) as _src:
    _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
    _inten = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
    _nd    = _src.nodata
    _ext   = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
if _nd is not None: _inten[_inten == _nd] = np.nan
fig, ax = plt.subplots(figsize=(10, 8))
_p2, _p98 = np.nanpercentile(_inten, [2, 98])
im = ax.imshow(_inten, extent=_ext, cmap='gray', origin='upper', vmin=_p2, vmax=_p98)
plt.colorbar(im, ax=ax, label='Intensity (DN)')
ax.set(title='LiDAR Intensity (EPSG:27700)', xlabel='Easting (m)', ylabel='Northing (m)')
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_G_intensity.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig); del _inten

In [ ]:
# -- VIZ-H: Aerial RGB+NIR (aerial_rgbn.tif) -----------------------------------
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_path = os.path.join(_rt_root, 'aerial_rgbn.tif')
with rasterio.open(_path) as _src:
    _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
    _bands = [_src.read(i+1, out_shape=(_h//_sc, _w//_sc),
              resampling=_RS.average).astype(np.float32) for i in range(min(_src.count, 4))]
    _nd    = _src.nodata
    _ext   = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
if _nd is not None:
    for _b in _bands: _b[_b == _nd] = np.nan
def _norm(a):
    _lo, _hi = np.nanpercentile(a, [2, 98])
    return np.clip((a - _lo) / max(_hi - _lo, 1e-9), 0, 1)
_n_bands = len(_bands)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
_rgb = np.stack([_norm(_bands[i]) for i in range(min(3, _n_bands))], axis=-1)
axes[0].imshow(_rgb, extent=_ext, origin='upper')
axes[0].set(title='Aerial RGB (2% stretch)', xlabel='Easting (m)', ylabel='Northing (m)')
if _n_bands >= 4:
    im2 = axes[1].imshow(_bands[3], extent=_ext, cmap='YlGn', origin='upper',
                         vmin=np.nanpercentile(_bands[3], 2),
                         vmax=np.nanpercentile(_bands[3], 98))
    plt.colorbar(im2, ax=axes[1], label='NIR (DN)')
    axes[1].set(title='NIR band', xlabel='Easting (m)', ylabel='Northing (m)')
else:
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, 'NIR band not available', ha='center', va='center',
                 transform=axes[1].transAxes)
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_H_aerial_rgbn.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig); del _bands, _rgb

In [ ]:
# -- VIZ-I: PC roof roughness (pc_roof_rough.tif) ------------------------------
import os, numpy as np, matplotlib.pyplot as plt, rasterio
from rasterio.enums import Resampling as _RS
_rt_root = os.environ.get('RT_BASE_DIR',
    os.path.join(os.path.expanduser('~'), 'sionna_rt', 'stevenage_ofcom_915mhz_dem'))
_path = os.path.join(_rt_root, 'pc_roof_rough.tif')
with rasterio.open(_path) as _src:
    _h, _w = _src.height, _src.width; _sc = max(1, max(_h, _w) // 2000)
    _rough = _src.read(1, out_shape=(_h//_sc, _w//_sc), resampling=_RS.average).astype(np.float32)
    _nd    = _src.nodata
    _ext   = [_src.bounds.left, _src.bounds.right, _src.bounds.bottom, _src.bounds.top]
if _nd is not None: _rough[_rough == _nd] = np.nan
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(_rough, extent=_ext, cmap='hot_r', origin='upper',
               vmin=0, vmax=np.nanpercentile(_rough, 98))
plt.colorbar(im, ax=ax, label='Roof roughness (m)')
ax.set(title='PC Roof Roughness (EPSG:27700)', xlabel='Easting (m)', ylabel='Northing (m)')
plt.tight_layout()
plt.savefig(os.path.join(_rt_root, 'viz', 'viz_I_roof_roughness.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig); del _rough